In [2]:
import json
from pathlib import Path
import pandas as pd

# Project root (assuming notebook runs from notebooks/)
BASE_DIR = Path.cwd().parent

folder = BASE_DIR / "data" / "raw" / "train_positions"

# Find the latest JSON file
latest_file = max(
    folder.glob("*.json"),
    key=lambda p: p.stat().st_mtime
)

with open(latest_file, "r", encoding="utf-8") as f:
    data = json.load(f)

print("File:", latest_file.name)
print("Number of train entities:", len(data["entity"]))

File: train_positions_20261028_123125_709169.json
Number of train entities: 180


In [3]:
df_trains = pd.json_normalize(data["entity"])

print("Shape:", df_trains.shape)
print("\nColumns:")
print(df_trains.columns.tolist())

display(df_trains.head())

Shape: (180, 12)

Columns:
['id', 'vehicle.trip.trip_id', 'vehicle.trip.schedule_relationship', 'vehicle.position.latitude', 'vehicle.position.longitude', 'vehicle.timestamp', 'vehicle.congestion_level', 'vehicle.stop_id', 'vehicle.vehicle.id', 'vehicle.trip.route_id', 'vehicle.vehicle.label', 'vehicle.occupancy_status']


,id,vehicle.trip.trip_id,vehicle.trip.schedule_relationship,vehicle.position.latitude,vehicle.position.longitude,vehicle.timestamp,vehicle.congestion_level,vehicle.stop_id,vehicle.vehicle.id,vehicle.trip.route_id,vehicle.vehicle.label,vehicle.occupancy_status
0,1,NonTimetabled.M971,UNSCHEDULED,-33.457146,151.31435,1793190628,UNKNOWN_CONGESTION_LEVEL,CentralCoast.N47.3 Loc,N47.3 Loc,NaN,NaN,NaN
1,2,NonTimetabled.927-,UNSCHEDULED,-33.883743,151.20650,1793189966,UNKNOWN_CONGESTION_LEVEL,Sydney.Sydney 12 Loc,Sydney 12 Loc,NaN,NaN,NaN
2,3,NonTimetabled.GRID,UNSCHEDULED,-33.837920,151.02164,1793190204,UNKNOWN_CONGESTION_LEVEL,Lidcombe.CE72L.72R/CE7R.7L.10L Loc,CE72L.72R/CE7R.7L.10L Loc,NaN,NaN,NaN
3,4,NonTimetabled.5114,UNSCHEDULED,-33.497032,151.30660,1793190654,UNKNOWN_CONGESTION_LEVEL,CentralCoast.N44.0 Loc,N44.0 Loc,NaN,NaN,NaN
4,5,NonTimetabled.*385,UNSCHEDULED,-34.430668,150.88756,1793188252,UNKNOWN_CONGESTION_LEVEL,SouthCoast.WOLL-446-425,WOLL-446-425,NaN,NaN,NaN


In [4]:
df_train_clean = df_trains[[
    "vehicle.trip.trip_id",
    "vehicle.trip.route_id",
    "vehicle.position.latitude",
    "vehicle.position.longitude",
    "vehicle.timestamp",
    "vehicle.stop_id",
    "vehicle.vehicle.label"
]].copy()

df_train_clean = df_train_clean.rename(columns={
    "vehicle.trip.trip_id": "trip_id",
    "vehicle.trip.route_id": "route_id",
    "vehicle.position.latitude": "latitude",
    "vehicle.position.longitude": "longitude",
    "vehicle.timestamp": "timestamp",
    "vehicle.stop_id": "stop_id",
    "vehicle.vehicle.label": "train_label"
})

# Convert Unix timestamps to readable UTC dates
df_train_clean["timestamp"] = pd.to_datetime(
    df_train_clean["timestamp"],
    unit="s",
    utc=True,
    errors="coerce"
)

display(df_train_clean.head())

,trip_id,route_id,latitude,longitude,timestamp,stop_id,train_label
0,NonTimetabled.M971,NaN,-33.457146,151.31435,NaT,CentralCoast.N47.3 Loc,NaN
1,NonTimetabled.927-,NaN,-33.883743,151.20650,NaT,Sydney.Sydney 12 Loc,NaN
2,NonTimetabled.GRID,NaN,-33.837920,151.02164,NaT,Lidcombe.CE72L.72R/CE7R.7L.10L Loc,NaN
3,NonTimetabled.5114,NaN,-33.497032,151.30660,NaT,CentralCoast.N44.0 Loc,NaN
4,NonTimetabled.*385,NaN,-34.430668,150.88756,NaT,SouthCoast.WOLL-446-425,NaN


In [5]:
df_train_clean["timestamp"] = pd.to_datetime(
    pd.to_numeric(df_train_clean["timestamp"], errors="coerce"),
    unit="s",
    utc=True,
    errors="coerce"
)

# Convert UTC to Sydney time
df_train_clean["timestamp"] = (
    df_train_clean["timestamp"]
    .dt.tz_convert("Australia/Sydney")
)

display(df_train_clean.head())

,trip_id,route_id,latitude,longitude,timestamp,stop_id,train_label
0,NonTimetabled.M971,NaN,-33.457146,151.31435,NaT,CentralCoast.N47.3 Loc,NaN
1,NonTimetabled.927-,NaN,-33.883743,151.20650,NaT,Sydney.Sydney 12 Loc,NaN
2,NonTimetabled.GRID,NaN,-33.837920,151.02164,NaT,Lidcombe.CE72L.72R/CE7R.7L.10L Loc,NaN
3,NonTimetabled.5114,NaN,-33.497032,151.30660,NaT,CentralCoast.N44.0 Loc,NaN
4,NonTimetabled.*385,NaN,-34.430668,150.88756,NaT,SouthCoast.WOLL-446-425,NaN


In [6]:
import json
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent
folder = BASE_DIR / "data" / "raw" / "train_positions"

latest_file = max(folder.glob("*.json"), key=lambda p: p.stat().st_mtime)

with open(latest_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# Recreate DataFrame from original data
df_trains = pd.json_normalize(data["entity"])

# Check the original timestamp values
print(df_trains["vehicle.timestamp"].head())

0    1793190628
1    1793189966
2    1793190204
3    1793190654
4    1793188252
Name: vehicle.timestamp, dtype: str


In [7]:
# Start fresh from the original DataFrame
df_train_clean["timestamp"] = df_trains["vehicle.timestamp"]

# Convert strings to numbers
df_train_clean["timestamp"] = pd.to_numeric(
    df_train_clean["timestamp"],
    errors="coerce"
)

# Convert Unix seconds to Sydney local time
df_train_clean["timestamp"] = pd.to_datetime(
    df_train_clean["timestamp"],
    unit="s",
    utc=True
).dt.tz_convert("Australia/Sydney")

display(df_train_clean.head())

,trip_id,route_id,latitude,longitude,timestamp,stop_id,train_label
0,NonTimetabled.M971,NaN,-33.457146,151.31435,2026-10-28 23:30:28+11:00,CentralCoast.N47.3 Loc,NaN
1,NonTimetabled.927-,NaN,-33.883743,151.20650,2026-10-28 23:19:26+11:00,Sydney.Sydney 12 Loc,NaN
2,NonTimetabled.GRID,NaN,-33.837920,151.02164,2026-10-28 23:23:24+11:00,Lidcombe.CE72L.72R/CE7R.7L.10L Loc,NaN
3,NonTimetabled.5114,NaN,-33.497032,151.30660,2026-10-28 23:30:54+11:00,CentralCoast.N44.0 Loc,NaN
4,NonTimetabled.*385,NaN,-34.430668,150.88756,2026-10-28 22:50:52+11:00,SouthCoast.WOLL-446-425,NaN


In [8]:
# Calculate how old each train position is
now = pd.Timestamp.now(tz="Australia/Sydney")

df_train_clean["minutes_old"] = (
    (now - df_train_clean["timestamp"])
    .dt.total_seconds() / 60
)

# Show the freshest positions first
display(
    df_train_clean[
        ["train_label", "route_id", "timestamp", "minutes_old"]
    ].sort_values("minutes_old").head(10)
)

,train_label,route_id,timestamp,minutes_old
57,23:23 Lidcombe Station to Olympic Park Station,OLY_2b,2026-10-28 23:31:08+11:00,860.370148
71,23:28 Central Station to Berowra Station,NSN_1a,2026-10-28 23:31:07+11:00,860.386815
111,NaN,RTTA_DEF,2026-10-28 23:31:06+11:00,860.403482
70,23:22 Richmond Station to Leppington Station,CMB_1a,2026-10-28 23:31:06+11:00,860.403482
9,NaN,NaN,2026-10-28 23:31:06+11:00,860.403482
149,21:09 Central Station to Mount Victoria Station,BMT_1,2026-10-28 23:31:06+11:00,860.403482
121,23:23 Central Station to Redfern Station,IWL_1b,2026-10-28 23:31:06+11:00,860.403482
69,22:54 Waterfall Station to Central Station,ESI_2b,2026-10-28 23:31:04+11:00,860.436815
88,22:33 Central Station to Penrith Station,WST_1a,2026-10-28 23:31:04+11:00,860.436815
100,23:26 Central Station to Kiama Station,SCO_1a,2026-10-28 23:31:04+11:00,860.436815


In [9]:
# Get the collection time from the filename
snapshot_time = pd.to_datetime(
    latest_file.stem.split("_")[-3] + "_" +
    latest_file.stem.split("_")[-2] + "_" +
    latest_file.stem.split("_")[-1],
    format="%Y%m%d_%H%M%S_%f",
    utc=True
).tz_convert("Australia/Sydney")

# Compare each train's timestamp with collection time
df_train_clean["minutes_old_at_collection"] = (
    (snapshot_time - df_train_clean["timestamp"])
    .dt.total_seconds() / 60
)

print("Snapshot collected:", snapshot_time)

display(
    df_train_clean[
        ["train_label", "route_id", "timestamp",
         "minutes_old_at_collection"]
    ].sort_values("minutes_old_at_collection").head(10)
)

Snapshot collected: 2026-10-28 23:31:25.709169+11:00


,train_label,route_id,timestamp,minutes_old_at_collection
57,23:23 Lidcombe Station to Olympic Park Station,OLY_2b,2026-10-28 23:31:08+11:00,0.295153
71,23:28 Central Station to Berowra Station,NSN_1a,2026-10-28 23:31:07+11:00,0.311819
111,NaN,RTTA_DEF,2026-10-28 23:31:06+11:00,0.328486
70,23:22 Richmond Station to Leppington Station,CMB_1a,2026-10-28 23:31:06+11:00,0.328486
9,NaN,NaN,2026-10-28 23:31:06+11:00,0.328486
149,21:09 Central Station to Mount Victoria Station,BMT_1,2026-10-28 23:31:06+11:00,0.328486
121,23:23 Central Station to Redfern Station,IWL_1b,2026-10-28 23:31:06+11:00,0.328486
69,22:54 Waterfall Station to Central Station,ESI_2b,2026-10-28 23:31:04+11:00,0.361819
88,22:33 Central Station to Penrith Station,WST_1a,2026-10-28 23:31:04+11:00,0.361819
100,23:26 Central Station to Kiama Station,SCO_1a,2026-10-28 23:31:04+11:00,0.361819


In [10]:
# Convert position age into seconds
df_train_clean["seconds_old"] = (
    df_train_clean["minutes_old_at_collection"] * 60
)

print(df_train_clean["seconds_old"].describe())

print("\nPositions within 30 seconds:",
      df_train_clean["seconds_old"].between(0, 30).sum())

print("Positions within 60 seconds:",
      df_train_clean["seconds_old"].between(0, 60).sum())

print("Positions older than 5 minutes:",
      (df_train_clean["seconds_old"] > 300).sum())

count       180.000000
mean       7623.514725
std       36398.053687
min          17.709169
25%          35.459169
50%          92.709169
75%         997.209169
max      379138.709169
Name: seconds_old, dtype: float64

Positions within 30 seconds: 34
Positions within 60 seconds: 75
Positions older than 5 minutes: 65


In [11]:
import folium
import pandas as pd

train_map = folium.Map(
    location=[-33.8688, 151.2093],
    zoom_start=10,
    tiles="OpenStreetMap"
)

for _, train in df_train_clean.iterrows():

    if pd.isna(train["latitude"]) or pd.isna(train["longitude"]):
        continue

    age = train["seconds_old"]

    if pd.isna(age) or age < 0:
        continue
    elif age <= 60:
        colour = "green"
    elif age <= 300:
        colour = "orange"
    else:
        colour = "gray"

    label = train["train_label"]

    if pd.isna(label):
        label = "Unknown train"

    popup = f"""
    <b>Train:</b> {label}<br>
    <b>Route:</b> {train['route_id']}<br>
    <b>Last position:</b> {train['timestamp']}<br>
    <b>Age at collection:</b> {age:.0f} seconds
    """

    folium.CircleMarker(
        location=[train["latitude"], train["longitude"]],
        radius=5,
        color=colour,
        fill=True,
        fill_color=colour,
        fill_opacity=0.8,
        popup=folium.Popup(popup, max_width=350)
    ).add_to(train_map)

train_map.save("sydney_train_map.html")

In [12]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent

gtfs_dir = BASE_DIR / "data" / "raw" / "gtfs_static" / "gtfs_static"

routes = pd.read_csv(gtfs_dir / "routes.txt", dtype=str)
trips = pd.read_csv(gtfs_dir / "trips.txt", dtype=str)

print("Routes:", routes.shape)
print("Trips:", trips.shape)

display(routes.head())

Routes: (10237, 9)
Trips: (211419, 11)


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_color,route_text_color,exact_times
0,1-SC0-1-sj2-1,701,SC01,Kiama to Bomaderry,Illawarra Buses Network,700,00B5EF,FFFFFF,1
1,1-SH1-0-sj2-2,701,SH100,Moss Vale to Goulburn,Illawarra Buses Network,700,00B5EF,FFFFFF,1
2,11-400-1-sj2-1,2501,4001,Penrith Station to McCarthy College,School buses,712,00B5EF,FFFFFF,1
3,11-400-2-sj2-1,2501,4002,Swallow Dr opp Skylark Cres to Caroline Chisho...,School buses,712,00B5EF,FFFFFF,1
4,11-400-3-sj2-1,2501,4003,Caroline Chisholm College to York PS via Regen...,School buses,712,00B5EF,FFFFFF,1


In [13]:
train_routes = routes[routes["agency_id"] == "x0001"].copy()

print("Number of Sydney Trains routes:", len(train_routes))

display(
    train_routes[
        ["route_short_name", "route_long_name", "route_type"]
    ]
)

Number of Sydney Trains routes: 10


,route_short_name,route_long_name,route_type
2320,T1,T1 North Shore and Western Line,2
2321,T1,T1 North Shore and Western Line,2
2322,T2,T2 Leppington and Inner West Line,2
2323,T3,T3 Liverpool and Inner West Line,2
2324,T4,T4 Eastern Suburbs and Illawarra Line,2
2325,T5,T5 Cumberland Line,2
2326,T6,T6 Lidcombe and Bankstown Line,2
2327,T7,T7 Olympic Park Line,2
2328,T8,T8 Airport and South Line,2
2329,T9,T9 Northern Line,2


In [14]:
train_trips = trips.merge(
    train_routes[["route_id", "route_short_name"]],
    on="route_id",
    how="inner"
)

shape_ids = set(train_trips["shape_id"].dropna())

print("Total Sydney Trains trips:", len(train_trips))
print("Unique railway shapes:", len(shape_ids))

display(
    train_trips[
        ["route_short_name", "trip_id", "shape_id"]
    ].head(10)
)

Total Sydney Trains trips: 48298
Unique railway shapes: 1298


,route_short_name,trip_id,shape_id
0,T1,101A.2010.103.128.T.8.91040144,2-T1-N-sj2-1.13.R
1,T1,101A.2010.103.2.T.8.91040144,2-T1-N-sj2-1.13.R
2,T1,101D.2010.103.128.T.8.91040148,2-T1-N-sj2-1.23.H
3,T1,101D.2010.103.2.T.8.91040148,2-T1-N-sj2-1.23.H
4,T1,101E.2010.103.128.T.8.91040150,2-T1-N-sj2-1.9.R
5,T1,101E.2010.103.2.T.8.91040150,2-T1-N-sj2-1.9.R
6,T1,101H.2010.103.128.T.8.91040156,2-T1-N-sj2-1.23.H
7,T1,101H.2010.103.2.T.8.91040156,2-T1-N-sj2-1.23.H
8,T1,101J.2010.103.128.T.8.91040158,2-T1-N-sj2-1.9.R
9,T1,101J.2010.103.2.T.8.91040158,2-T1-N-sj2-1.9.R


In [15]:
train_shape_chunks = []

for chunk in pd.read_csv(
    gtfs_dir / "shapes.txt",
    dtype={"shape_id": str},
    chunksize=100_000
):
    selected = chunk[
        chunk["shape_id"].isin(shape_ids)
    ]

    if not selected.empty:
        train_shape_chunks.append(selected)

train_shapes = pd.concat(
    train_shape_chunks,
    ignore_index=True
)

train_shapes["shape_pt_sequence"] = pd.to_numeric(
    train_shapes["shape_pt_sequence"]
)

train_shapes = train_shapes.sort_values(
    ["shape_id", "shape_pt_sequence"]
)

print("Total railway coordinates:", len(train_shapes))
print("Shapes extracted:", train_shapes["shape_id"].nunique())

display(train_shapes.head())

Total railway coordinates: 852072
Shapes extracted: 1298


,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,2-T1-N-sj2-1.1.R,-33.623511,151.153121,1,0.00
1,2-T1-N-sj2-1.1.R,-33.624566,151.151528,2,188.53
2,2-T1-N-sj2-1.1.R,-33.624789,151.151271,3,222.85
3,2-T1-N-sj2-1.1.R,-33.625196,151.150681,4,293.81
4,2-T1-N-sj2-1.1.R,-33.625464,151.150281,5,341.41


In [16]:
output_dir = BASE_DIR / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

train_shapes.to_parquet(
    output_dir / "sydney_train_shapes.parquet",
    index=False
)

print("Railway shapes saved!")

Railway shapes saved!


In [17]:
import pandas as pd

stops = pd.read_csv(
    gtfs_dir / "stops.txt",
    dtype=str
)

stop_times = pd.read_csv(
    gtfs_dir / "stop_times.txt",
    dtype=str
)

print("Stops:", stops.shape)
print("Stop times:", stop_times.shape)

display(stops.head())
display(stop_times.head())

Stops: (171285, 10)
Stop times: (5705967, 11)


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code
0,2533211,2533211,"Kiama Station, Platform 1",-34.67255547,150.85455900,NaN,253330,1,Level -1,1
1,2533212,2533212,"Kiama Station, Platform 2",-34.67260704,150.85463774,NaN,253330,1,Level -1,2
2,253330,NaN,Kiama Station,-34.67266689,150.85445510,1,NaN,0,NaN,NaN
3,253330_CCP1,NaN,CCP Railway Pde N,-34.67188486,150.85496415,NaN,253330,NaN,Level 0,NaN
4,253330_CCP2,NaN,CCP Railway Pde S,-34.67326859,150.85366501,NaN,253330,NaN,Level 0,NaN


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,stop_note
0,1.AA61.1-SC0-1-sj2-1.1.R,22:50:00,22:50:00,254111,1,NaN,0,0,0.00,1,NaN
1,1.AA61.1-SC0-1-sj2-1.1.R,23:04:00,23:04:00,253572,2,NaN,0,0,16208.96,1,NaN
2,1.AA61.1-SC0-1-sj2-1.1.R,23:20:00,23:20:00,253476,3,NaN,0,0,29527.41,1,NaN
3,1.AA61.1-SC0-1-sj2-1.1.R,23:30:00,23:30:00,253332,4,NaN,0,0,40708.07,1,NaN
4,1.AB22.1-SC0-1-sj2-1.1.R,03:25:00,03:25:00,254111,1,NaN,0,0,0.00,1,NaN


In [18]:
station_names = ["Central Station", "Parramatta Station"]

selected_stations = stops[
    (stops["stop_name"].isin(station_names))
    & (stops["location_type"] == "1")
]

display(
    selected_stations[
        ["stop_id", "stop_name", "location_type"]
    ]
)

,stop_id,stop_name,location_type
800,215020,Parramatta Station,1
22172,200060,Central Station,1


In [19]:
central_id = "200060"
parramatta_id = "215020"

platforms = stops[
    stops["parent_station"].isin([
        central_id,
        parramatta_id
    ])
].copy()

display(
    platforms[
        [
            "stop_id",
            "stop_name",
            "parent_station",
            "platform_code"
        ]
    ]
)

,stop_id,stop_name,parent_station,platform_code
789,2150106,"Parramatta Station, Stand B1",215020,NaN
790,2150107,"Parramatta Station, Stand B2",215020,NaN
791,2150109,"Parramatta Station, Stand B4",215020,NaN
792,2150111,"Parramatta Station, Stand A2",215020,NaN
793,2150114,"Parramatta Station, Stand A1",215020,NaN
...,...,...,...,...
22236,200060_MZ,TicketG Intercity,200060,NaN
22269,2010107,Central Chalmers Street Light Rail,200060,NaN
22270,201016,"Central Station, Chalmers St, Stand G",200060,NaN
22271,201039,"Central Station, Foveaux St, Stand F",200060,NaN


In [20]:
# Get all Sydney Trains trip IDs
train_trip_ids = set(train_trips["trip_id"])

# Find timetable records belonging to Sydney Trains
train_stop_times = stop_times[
    stop_times["trip_id"].isin(train_trip_ids)
].copy()

# Find stops used by these trips
train_stop_ids = set(train_stop_times["stop_id"])

# Filter Central and Parramatta child stops
train_platforms = platforms[
    platforms["stop_id"].isin(train_stop_ids)
].copy()

display(
    train_platforms[
        [
            "stop_id",
            "stop_name",
            "parent_station",
            "platform_code"
        ]
    ]
)

print("Train platforms found:", len(train_platforms))

,stop_id,stop_name,parent_station,platform_code
833,2150411,"Parramatta Station, Platform 1",215020,1
834,2150412,"Parramatta Station, Platform 2",215020,2
835,2150413,"Parramatta Station, Platform 3",215020,3
836,2150414,"Parramatta Station, Platform 4",215020,4
22117,2000321,"Central Station, Platform 1",200060,1
22118,2000322,"Central Station, Platform 2",200060,2
22119,2000323,"Central Station, Platform 3",200060,3
22120,2000324,"Central Station, Platform 4",200060,4
22121,2000325,"Central Station, Platform 5",200060,5
22122,2000326,"Central Station, Platform 6",200060,6


Train platforms found: 28


In [21]:
# Get Central and Parramatta train platform IDs
central_stops = set(
    train_platforms.loc[
        train_platforms["parent_station"] == "200060",
        "stop_id"
    ]
)

parramatta_stops = set(
    train_platforms.loc[
        train_platforms["parent_station"] == "215020",
        "stop_id"
    ]
)

# Find departures from Central
central_departures = train_stop_times[
    train_stop_times["stop_id"].isin(central_stops)
][
    ["trip_id", "departure_time", "stop_sequence", "stop_id"]
].copy()

central_departures = central_departures.rename(columns={
    "departure_time": "central_departure",
    "stop_sequence": "central_sequence",
    "stop_id": "central_stop_id"
})

# Find arrivals at Parramatta
parramatta_arrivals = train_stop_times[
    train_stop_times["stop_id"].isin(parramatta_stops)
][
    ["trip_id", "arrival_time", "stop_sequence", "stop_id"]
].copy()

parramatta_arrivals = parramatta_arrivals.rename(columns={
    "arrival_time": "parramatta_arrival",
    "stop_sequence": "parramatta_sequence",
    "stop_id": "parramatta_stop_id"
})

# Join both stations on the same train trip
journeys = central_departures.merge(
    parramatta_arrivals,
    on="trip_id",
    how="inner"
)

# Make sure the train stops at Central first
journeys["central_sequence"] = pd.to_numeric(
    journeys["central_sequence"]
)

journeys["parramatta_sequence"] = pd.to_numeric(
    journeys["parramatta_sequence"]
)

journeys = journeys[
    journeys["central_sequence"] < journeys["parramatta_sequence"]
].copy()

print("Matching scheduled journeys:", len(journeys))

display(journeys.head())

Matching scheduled journeys: 4067


,trip_id,central_departure,central_sequence,central_stop_id,parramatta_arrival,parramatta_sequence,parramatta_stop_id
0,101B.2010.103.128.T.8.91040143,04:56:01,1,2000338,05:27:00,16,2150414
1,101B.2010.103.2.T.8.91040143,04:56:01,1,2000338,05:27:00,16,2150414
4,101F.2010.103.128.T.8.91040152,08:41:01,1,2000338,09:13:00,16,2150414
5,101F.2010.103.2.T.8.91040152,08:41:01,1,2000338,09:13:00,16,2150414
8,101K.2010.103.128.T.8.91040160,12:26:01,1,2000338,12:58:00,16,2150414


In [22]:
calendar = pd.read_csv(
    gtfs_dir / "calendar.txt",
    dtype=str
)

calendar_dates = pd.read_csv(
    gtfs_dir / "calendar_dates.txt",
    dtype=str
)

print("Calendar:", calendar.shape)
print("Calendar exceptions:", calendar_dates.shape)

display(calendar.head())
display(calendar_dates.head())

Calendar: (2336, 10)
Calendar exceptions: (37650, 3)


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,AA61+1,1,1,1,1,1,0,0,20261027,20270125
1,AA61#1+1,1,1,1,1,1,0,0,20261027,20270125
2,AB22+1,0,0,0,0,0,1,1,20261027,20270125
3,AB22#1+1,0,0,0,0,0,1,1,20261027,20270125
4,TA+2,1,1,1,1,1,1,1,20261027,20270125


,service_id,date,exception_type
0,AA61+1,20261225,2
1,AA61+1,20261228,2
2,AA61+1,20270101,2
3,AA61#1+1,20261225,2
4,AA61#1+1,20261228,2


In [23]:
from datetime import datetime

travel_date = "2026-10-28"

date_obj = datetime.strptime(travel_date, "%Y-%m-%d")
date_str = date_obj.strftime("%Y%m%d")
weekday = date_obj.strftime("%A").lower()

# Normal calendar services
regular_services = calendar[
    (calendar["start_date"] <= date_str) &
    (calendar["end_date"] >= date_str) &
    (calendar[weekday] == "1")
]

active_service_ids = set(regular_services["service_id"])

# Exceptions for the selected date
exceptions = calendar_dates[
    calendar_dates["date"] == date_str
]

# Remove cancelled calendar services
removed = set(
    exceptions.loc[
        exceptions["exception_type"] == "2",
        "service_id"
    ]
)

# Add special services
added = set(
    exceptions.loc[
        exceptions["exception_type"] == "1",
        "service_id"
    ]
)

active_service_ids = (active_service_ids - removed) | added

print("Active service patterns:", len(active_service_ids))

Active service patterns: 662


In [24]:
# Add calendar service IDs and train line information
journeys_with_services = journeys.merge(
    train_trips[
        ["trip_id", "service_id", "route_short_name"]
    ],
    on="trip_id",
    how="inner"
)

# Keep only services operating on the travel date
valid_journeys = journeys_with_services[
    journeys_with_services["service_id"].isin(
        active_service_ids
    )
].copy()

print("Valid journeys on", travel_date, ":", len(valid_journeys))

Valid journeys on 2026-10-28 : 259


In [25]:
requested_time = "17:30:00"

upcoming = valid_journeys[
    valid_journeys["central_departure"] >= requested_time
].copy()

upcoming = upcoming.sort_values("central_departure")

# Convert departure times into seconds for interval calculations
def time_to_seconds(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

upcoming["departure_seconds"] = (
    upcoming["central_departure"].apply(time_to_seconds)
)

# Calculate gap to the following train
upcoming["next_train_gap_min"] = (
    upcoming["departure_seconds"].shift(-1)
    - upcoming["departure_seconds"]
) / 60

display(
    upcoming[
        [
            "route_short_name",
            "central_departure",
            "parramatta_arrival",
            "next_train_gap_min"
        ]
    ].head(5)
)

,route_short_name,central_departure,parramatta_arrival,next_train_gap_min
3198,T2,17:32:01,18:36:30,2.000000
1258,T1,17:34:01,17:59:00,0.983333
296,T2,17:35:00,18:21:30,2.016667
214,T1,17:37:01,18:02:00,6.000000
1558,T1,17:43:01,18:08:00,3.000000


In [26]:
# Calculate arrival time in seconds
upcoming["arrival_seconds"] = (
    upcoming["parramatta_arrival"].apply(time_to_seconds)
)

# Journey duration in minutes
upcoming["journey_duration_min"] = (
    upcoming["arrival_seconds"]
    - upcoming["departure_seconds"]
) / 60

# Waiting time from requested departure
requested_seconds = time_to_seconds(requested_time)

upcoming["waiting_time_min"] = (
    upcoming["departure_seconds"]
    - requested_seconds
) / 60

# Display the results
display(
    upcoming[
        [
            "route_short_name",
            "central_departure",
            "parramatta_arrival",
            "journey_duration_min",
            "waiting_time_min",
            "next_train_gap_min"
        ]
    ].head(10)
)

,route_short_name,central_departure,parramatta_arrival,journey_duration_min,waiting_time_min,next_train_gap_min
3198,T2,17:32:01,18:36:30,64.483333,2.016667,2.000000
1258,T1,17:34:01,17:59:00,24.983333,4.016667,0.983333
296,T2,17:35:00,18:21:30,46.500000,5.000000,2.016667
214,T1,17:37:01,18:02:00,24.983333,7.016667,6.000000
1558,T1,17:43:01,18:08:00,24.983333,13.016667,3.000000
1686,T1,17:46:01,18:11:00,24.983333,16.016667,1.000000
3967,T2,17:47:01,18:51:30,64.483333,17.016667,2.000000
1327,T1,17:49:01,18:14:00,24.983333,19.016667,0.983333
3199,T2,17:50:00,18:36:30,46.500000,20.000000,2.016667
42,T1,17:52:01,18:17:00,24.983333,22.016667,6.000000


In [27]:
def find_journeys(limit=5):
    results = upcoming[
        [
            "route_short_name",
            "central_departure",
            "parramatta_arrival",
            "journey_duration_min",
            "waiting_time_min",
            "next_train_gap_min"
        ]
    ].head(limit).copy()

    return results

In [28]:
import pandas as pd
from datetime import datetime


# Convert GTFS time to seconds
# Supports times such as 17:30:00 and 25:10:00
def time_to_seconds(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s


def find_journeys(origin, destination, travel_date, departure_time, limit=5):

    # 1. Find parent station IDs
    origin_ids = set(
        stops.loc[
            (stops["stop_name"] == origin) &
            (stops["location_type"] == "1"),
            "stop_id"
        ]
    )

    destination_ids = set(
        stops.loc[
            (stops["stop_name"] == destination) &
            (stops["location_type"] == "1"),
            "stop_id"
        ]
    )

    if not origin_ids or not destination_ids:
        raise ValueError("Origin or destination station not found")

    if origin_ids == destination_ids:
        raise ValueError("Origin and destination must be different")

    # 2. Find platforms used by Sydney Trains
    origin_platforms = set(
        stops.loc[
            stops["parent_station"].isin(origin_ids) &
            stops["stop_id"].isin(train_stop_ids),
            "stop_id"
        ]
    )

    destination_platforms = set(
        stops.loc[
            stops["parent_station"].isin(destination_ids) &
            stops["stop_id"].isin(train_stop_ids),
            "stop_id"
        ]
    )

    # 3. Find departure records from origin
    departures = train_stop_times[
        train_stop_times["stop_id"].isin(origin_platforms)
    ][
        ["trip_id", "departure_time", "stop_sequence", "stop_id"]
    ].rename(
        columns={
            "departure_time": "departure",
            "stop_sequence": "origin_sequence",
            "stop_id": "origin_platform_id"
        }
    )

    # 4. Find arrival records at destination
    arrivals = train_stop_times[
        train_stop_times["stop_id"].isin(destination_platforms)
    ][
        ["trip_id", "arrival_time", "stop_sequence", "stop_id"]
    ].rename(
        columns={
            "arrival_time": "arrival",
            "stop_sequence": "destination_sequence",
            "stop_id": "destination_platform_id"
        }
    )

    # 5. Match trips that visit both stations
    results = departures.merge(
        arrivals,
        on="trip_id",
        how="inner"
    )

    # Ensure the train visits origin before destination
    results["origin_sequence"] = pd.to_numeric(
        results["origin_sequence"]
    )

    results["destination_sequence"] = pd.to_numeric(
        results["destination_sequence"]
    )

    results = results[
        results["origin_sequence"] <
        results["destination_sequence"]
    ].copy()

    # 6. Determine active services for the travel date
    date_obj = datetime.strptime(travel_date, "%Y-%m-%d")
    date_str = date_obj.strftime("%Y%m%d")
    weekday = date_obj.strftime("%A").lower()

    regular = calendar[
        (calendar["start_date"] <= date_str) &
        (calendar["end_date"] >= date_str) &
        (calendar[weekday] == "1")
    ]

    active = set(regular["service_id"])

    # Apply calendar exceptions
    exceptions = calendar_dates[
        calendar_dates["date"] == date_str
    ]

    removed = set(
        exceptions.loc[
            exceptions["exception_type"] == "2",
            "service_id"
        ]
    )

    added = set(
        exceptions.loc[
            exceptions["exception_type"] == "1",
            "service_id"
        ]
    )

    active = (active - removed) | added

    # 7. Connect trips to their service calendars
    results = results.merge(
        train_trips[
            ["trip_id", "service_id", "route_short_name"]
        ],
        on="trip_id",
        how="inner"
    )

    results = results[
        results["service_id"].isin(active)
    ].copy()

    # 8. Convert times into seconds
    if len(departure_time) == 5:
        departure_time += ":00"

    requested_seconds = time_to_seconds(departure_time)

    results["departure_seconds"] = (
        results["departure"].apply(time_to_seconds)
    )

    results["arrival_seconds"] = (
        results["arrival"].apply(time_to_seconds)
    )

    # 9. Filter trains departing at or after requested time
    results = results[
        results["departure_seconds"] >= requested_seconds
    ].copy()

    # 10. Calculate journey duration
    results["journey_duration_min"] = (
        results["arrival_seconds"] -
        results["departure_seconds"]
    ) / 60

    # Remove invalid durations
    results = results[
        results["journey_duration_min"] >= 0
    ].copy()

    # 11. Handle trains visiting the origin more than once
    # Keep the shortest valid journey for each trip
    results = results.sort_values(
        ["trip_id", "journey_duration_min", "departure_seconds"]
    )

    results = results.drop_duplicates(
        subset=["trip_id"],
        keep="first"
    )

    # 12. Sort chronologically
    results = results.sort_values(
        ["departure_seconds", "arrival_seconds"]
    ).copy()

    # 13. Calculate waiting time
    results["waiting_time_min"] = (
        results["departure_seconds"] -
        requested_seconds
    ) / 60

    # 14. Calculate gap until the next departure
    results["next_train_gap_min"] = (
        results["departure_seconds"].shift(-1) -
        results["departure_seconds"]
    ) / 60

    # 15. Return next scheduled journeys
    return results[
        [
            "trip_id",
            "service_id",
            "route_short_name",
            "origin_platform_id",
            "destination_platform_id",
            "departure",
            "arrival",
            "journey_duration_min",
            "waiting_time_min",
            "next_train_gap_min"
        ]
    ].head(limit).reset_index(drop=True)

In [29]:
find_journeys(
    origin="Central Station",
    destination="Parramatta Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=5
)

,trip_id,service_id,route_short_name,origin_platform_id,destination_platform_id,departure,arrival,journey_duration_min,waiting_time_min,next_train_gap_min
0,152P.1324.165.16.A.8.91999393,TA+bk000+7,T1,2000338,2150412,17:34:01,17:59:00,24.983333,4.016667,0.983333
1,11-R.1324.165.16.B.8.92001395,TA+bk000+7,T2,2000339,2150414,17:35:00,18:21:30,46.500000,5.000000,2.016667
2,113E.1324.165.48.T.8.92000803,TA+bk000+7,T1,2000338,2150412,17:37:01,18:02:00,24.983333,7.016667,6.000000
3,161D.1324.165.48.A.8.92220122,TA+bk000+7,T1,2000338,2150412,17:43:01,18:08:00,24.983333,13.016667,3.000000
4,169H.1324.165.124.A.8.92004232,TA+hv000+7,T1,2000338,2150412,17:46:01,18:11:00,24.983333,16.016667,3.000000


In [30]:
find_journeys(
    origin="Parramatta Station",
    destination="Central Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=5
)

,trip_id,service_id,route_short_name,origin_platform_id,destination_platform_id,departure,arrival,journey_duration_min,waiting_time_min,next_train_gap_min
0,202F.1324.165.60.H.8.92002815,TA+hv000+7,T1,2150411,2000336,17:32:00,17:58:00,26.000000,2.000000,11.000000
1,163K.1324.165.16.A.8.92003187,TA+bk000+7,T1,2150411,2000336,17:43:00,18:09:00,26.000000,13.000000,0.516667
2,9--R.1324.165.60.B.8.92000511,TA+hv000+7,T2,2150414,2000337,17:43:31,18:28:30,44.983333,13.516667,3.483333
3,148J.1324.165.16.A.8.92002655,TA+bk000+7,T1,2150411,2000336,17:47:00,18:13:00,26.000000,17.000000,11.000000
4,193K.1324.165.16.B.8.91999630,TA+bk000+7,T1,2150411,2000336,17:58:00,18:24:00,26.000000,28.000000,0.516667


In [31]:
# Check whether multiple trips have identical departure/arrival times
duplicate_trains = valid_journeys[
    valid_journeys.duplicated(
        subset=[
            "central_departure",
            "parramatta_arrival",
            "route_short_name"
        ],
        keep=False
    )
]

print("Potential duplicates:", len(duplicate_trains))

display(
    duplicate_trains[
        [
            "trip_id",
            "service_id",
            "route_short_name",
            "central_departure",
            "parramatta_arrival"
        ]
    ].head(10)
)

Potential duplicates: 0


,trip_id,service_id,route_short_name,central_departure,parramatta_arrival


In [32]:
find_journeys(
    origin="Central Station",
    destination="Strathfield Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=5
)

,trip_id,service_id,route_short_name,origin_platform_id,destination_platform_id,departure,arrival,journey_duration_min,waiting_time_min,next_train_gap_min
0,40-E.1324.165.124.A.8.92002717,TA+hv000+7,T2,2000339,2135238,17:32:00,17:56:00,24.000000,2.000000,2.016667
1,152P.1324.165.16.A.8.91999393,TA+bk000+7,T1,2000338,2135236,17:34:01,17:46:30,12.483333,4.016667,0.983333
2,11-R.1324.165.16.B.8.92001395,TA+bk000+7,T2,2000339,2135238,17:35:00,18:00:00,25.000000,5.000000,1.016667
3,192S.1324.165.16.B.8.92002023,TA+bk000+7,T9,2000332,2135233,17:36:01,17:49:00,12.983333,6.016667,1.000000
4,113E.1324.165.48.T.8.92000803,TA+bk000+7,T1,2000338,2135236,17:37:01,17:49:30,12.483333,7.016667,2.983333


In [33]:
results = find_journeys(
    origin="Central Station",
    destination="Strathfield Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=100
)

display(
    results[
        results["trip_id"] == "78-P.1324.165.16.A.8.92002733"
    ]
)

,trip_id,service_id,route_short_name,origin_platform_id,destination_platform_id,departure,arrival,journey_duration_min,waiting_time_min,next_train_gap_min
12,78-P.1324.165.16.A.8.92002733,TA+bk000+7,T2,2000339,2135238,17:50:00,18:15:00,25.0,20.0,1.016667


In [34]:
find_journeys(
    origin="Town Hall Station",
    destination="Central Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=5
)

,trip_id,service_id,route_short_name,origin_platform_id,destination_platform_id,departure,arrival,journey_duration_min,waiting_time_min,next_train_gap_min
0,152N.1324.165.16.A.8.92003560,TA+bk000+7,T1,2000392,2000338,17:30:00,17:33:00,3.0,0.0,1.0
1,11-R.1324.165.16.B.8.92001395,TA+bk000+7,T2,2000391,2000339,17:31:00,17:34:00,3.0,1.0,1.7
2,43-V.1324.165.120.A.8.91999817,TA+hv000+7,T8,2000396,2000343,17:32:42,17:46:00,13.3,2.7,0.3
3,601F.1324.165.48.T.8.92004281,TA+bk000+7,T4,2000394,2000345,17:33:00,17:35:00,2.0,3.0,0.0
4,113D.1324.165.48.T.8.92000801,TA+bk000+7,T1,2000392,2000338,17:33:00,17:36:00,3.0,3.0,2.7


In [35]:
t8_trip = "43-V.1324.165.120.A.8.91999817"

t8_stops = train_stop_times[
    train_stop_times["trip_id"] == t8_trip
].merge(
    stops[["stop_id", "stop_name"]],
    on="stop_id",
    how="left"
)

t8_stops["stop_sequence"] = pd.to_numeric(
    t8_stops["stop_sequence"]
)

display(
    t8_stops.sort_values("stop_sequence")[
        [
            "stop_sequence",
            "stop_name",
            "arrival_time",
            "departure_time"
        ]
    ]
)

,stop_sequence,stop_name,arrival_time,departure_time
0,1,"Central Station, Platform 17",17:29:01,17:29:01
1,2,"Town Hall Station, Platform 6",17:31:42,17:32:42
2,3,"Wynyard Station, Platform 6",17:34:30,17:35:30
3,4,"Circular Quay Station, Platform 2",17:38:00,17:38:30
4,5,"St James Station, Platform 2",17:41:18,17:41:48
5,6,"Museum Station, Platform 2",17:43:12,17:43:42
6,7,"Central Station, Platform 23",17:46:00,17:47:00
7,8,"Green Square Station, Platform 2",17:50:18,17:50:48
8,9,"Mascot Station, Platform 2",17:54:00,17:54:30
9,10,"Domestic Airport Station, Platform 2",17:57:00,17:57:30


In [36]:
journeys = find_journeys(
    origin="Town Hall Station",
    destination="Central Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=10
)

recommended = journeys.loc[
    journeys["arrival"].apply(time_to_seconds).idxmin()
]

print("Recommended journey:")
print("Line:", recommended["route_short_name"])
print("Departure:", recommended["departure"])
print("Arrival:", recommended["arrival"])
print("Journey duration:", round(recommended["journey_duration_min"], 1), "minutes")
print("Waiting time:", round(recommended["waiting_time_min"], 1), "minutes")

Recommended journey:
Line: T1
Departure: 17:30:00
Arrival: 17:33:00
Journey duration: 3.0 minutes
Waiting time: 0.0 minutes


In [37]:
routes = pd.read_csv(
    gtfs_dir / "routes.txt",
    dtype=str
)

print("Total routes:", len(routes))

display(
    routes.groupby(
        ["agency_id", "route_type"],
        dropna=False
    ).agg(
        route_count=("route_id", "count"),
        route_names=("route_short_name", lambda x: ", ".join(
            sorted(x.dropna().unique())
        ))
    ).reset_index()
)

Total routes: 10237


,agency_id,route_type,route_count,route_names
0,2447,700,11,"160, 161, 162, 163, 164, 165, 166, 167, 168, 1..."
1,2447,712,95,"S850, S851, S852, S853, S854, S855, S856, S857..."
2,2448,700,25,"136, 137, 138, 140, 141, 145, 177, 178, 179, 1..."
3,2448,712,419,"0011, 0012, 0032, 0037, 0050, 0051, 0052, 0061..."
4,2449,700,6,"130, 131, 132, 133, 134, 135"
...,...,...,...,...
784,SLR,900,3,"L2, L3, LX"
785,SMNW,401,1,M1
786,X000,2,4,"BMT, HUN, SCO, SHL"
787,X0000,2,3,"BMT, CCN, SCO"


In [38]:
train_routes = routes[routes["agency_id"] == "x0001"]

In [39]:
selected_routes = routes[
    routes["agency_id"].isin([
        "x0001",  # Sydney Trains
        "SMNW",   # Sydney Metro
        "SLR"     # Sydney Light Rail
    ])
].copy()

mode_mapping = {
    "x0001": "Train",
    "SMNW": "Metro",
    "SLR": "Light Rail"
}

selected_routes["transport_mode"] = (
    selected_routes["agency_id"].map(mode_mapping)
)

display(
    selected_routes[
        ["route_short_name", "transport_mode", "route_type"]
    ].sort_values(["transport_mode", "route_short_name"])
)

,route_short_name,transport_mode,route_type
10154,L2,Light Rail,900
10155,L3,Light Rail,900
10157,LX,Light Rail,900
3272,M1,Metro,401
2320,T1,Train,2
2321,T1,Train,2
2322,T2,Train,2
2323,T3,Train,2
2324,T4,Train,2
2325,T5,Train,2


In [40]:
all_transport_trips = trips.merge(
    selected_routes[
        ["route_id", "route_short_name", "transport_mode"]
    ],
    on="route_id",
    how="inner"
)

print("Total multimodal trips:", len(all_transport_trips))

display(
    all_transport_trips.groupby("transport_mode").agg(
        trip_count=("trip_id", "count"),
        route_count=("route_id", "nunique")
    )
)

Total multimodal trips: 58455


,trip_count,route_count
transport_mode,,
Light Rail,7790,3
Metro,2367,1
Train,48298,10


In [41]:
# Get trip IDs for Train, Metro and Light Rail
all_trip_ids = set(all_transport_trips["trip_id"])

# Keep stop times belonging to those trips
all_transport_stop_times = stop_times[
    stop_times["trip_id"].isin(all_trip_ids)
].copy()

# Add route and transport mode information
all_transport_stop_times = all_transport_stop_times.merge(
    all_transport_trips[
        ["trip_id", "route_short_name", "transport_mode"]
    ],
    on="trip_id",
    how="left"
)

print("Total stop-time records:", len(all_transport_stop_times))

display(
    all_transport_stop_times.groupby("transport_mode").agg(
        stop_time_records=("stop_id", "count"),
        unique_stops=("stop_id", "nunique"),
        unique_trips=("trip_id", "nunique")
    )
)

Total stop-time records: 938621


,stop_time_records,unique_stops,unique_trips
transport_mode,,,
Light Rail,110048,40,7790
Metro,68749,62,2367
Train,759824,466,48298


In [42]:
# Get all stops used by our three transport modes
used_stop_ids = set(all_transport_stop_times["stop_id"])

# Select relevant stops
transport_stops = stops[
    stops["stop_id"].isin(used_stop_ids)
].copy()

# Create a station identifier
transport_stops["station_id"] = (
    transport_stops["parent_station"]
    .fillna(transport_stops["stop_id"])
)

# Attach transport mode information
stop_modes = (
    all_transport_stop_times[
        ["stop_id", "transport_mode"]
    ]
    .drop_duplicates()
)

transport_stops = transport_stops.merge(
    stop_modes,
    on="stop_id",
    how="left"
)

print("Total transport stops:", len(transport_stops))

display(
    transport_stops[
        [
            "stop_id",
            "stop_name",
            "station_id",
            "transport_mode"
        ]
    ].head(20)
)

Total transport stops: 568


,stop_id,stop_name,station_id,transport_mode
0,2113341,"Macquarie Park Station, Platform 1",211340,Metro
1,2113342,"Macquarie Park Station, Platform 2",211340,Metro
2,2113351,"Macquarie University Station, Platform 1",211310,Metro
3,2113352,"Macquarie University Station, Platform 2",211310,Metro
4,2122231,"Eastwood Station, Platform 1",212210,Train
5,2122232,"Eastwood Station, Platform 2",212210,Train
6,2122233,"Eastwood Station, Platform 3",212210,Train
7,2122234,"Eastwood Station, Platform 4",212210,Train
8,2147421,"Seven Hills Station, Platform 1",214710,Train
9,2147422,"Seven Hills Station, Platform 2",214710,Train


In [43]:
# Count the transport modes available at each station
station_modes = (
    transport_stops
    .groupby("station_id")
    .agg(
        transport_modes=(
            "transport_mode",
            lambda x: sorted(x.dropna().unique())
        ),
        platform_count=("stop_id", "nunique")
    )
    .reset_index()
)

# Count how many different modes each station supports
station_modes["mode_count"] = (
    station_modes["transport_modes"].apply(len)
)

# Keep stations with more than one transport mode
multimodal_stations = station_modes[
    station_modes["mode_count"] > 1
].copy()

print("Multimodal stations:", len(multimodal_stations))

display(multimodal_stations.head(20))

Multimodal stations: 9


,station_id,transport_modes,platform_count,mode_count
4,200020,"[Light Rail, Train]",5,2
5,200030,"[Metro, Train]",4,2
9,200060,"[Light Rail, Metro, Train]",29,3
11,200070,"[Light Rail, Train]",8,2
12,200080,"[Light Rail, Train]",6,2
37,204420,"[Metro, Train]",6,2
50,206710,"[Metro, Train]",4,2
76,212110,"[Metro, Train]",5,2
139,220010,"[Metro, Train]",4,2


In [44]:
# Get parent station names
parent_stations = stops[
    stops["location_type"] == "1"
][["stop_id", "stop_name"]].drop_duplicates()

# Add station names to our multimodal stations
multimodal_stations_named = multimodal_stations.merge(
    parent_stations,
    left_on="station_id",
    right_on="stop_id",
    how="left"
)

display(
    multimodal_stations_named[
        [
            "station_id",
            "stop_name",
            "transport_modes",
            "platform_count"
        ]
    ].sort_values("stop_name")
)

,station_id,stop_name,transport_modes,platform_count
8,220010,Bankstown Station,"[Metro, Train]",4
2,200060,Central Station,"[Light Rail, Metro, Train]",29
6,206710,Chatswood Station,"[Metro, Train]",4
0,200020,Circular Quay,"[Light Rail, Train]",5
7,212110,Epping Station,"[Metro, Train]",5
1,200030,Martin Place Station,"[Metro, Train]",4
5,204420,Sydenham Station,"[Metro, Train]",6
3,200070,Town Hall Station,"[Light Rail, Train]",8
4,200080,Wynyard Station,"[Light Rail, Train]",6


In [45]:
import numpy as np

# Get parent station information
station_locations = stops[
    stops["stop_id"].isin(
        transport_stops["station_id"]
    )
][
    ["stop_id", "stop_name", "stop_lat", "stop_lon"]
].copy()

station_locations = station_locations.rename(
    columns={"stop_id": "station_id"}
)

station_locations["stop_lat"] = pd.to_numeric(
    station_locations["stop_lat"], errors="coerce"
)

station_locations["stop_lon"] = pd.to_numeric(
    station_locations["stop_lon"], errors="coerce"
)

station_locations = station_locations.dropna(
    subset=["stop_lat", "stop_lon"]
)

# Attach available transport modes
station_locations = station_locations.merge(
    station_modes[
        ["station_id", "transport_modes"]
    ],
    on="station_id",
    how="left"
)

print("Stations with coordinates:", len(station_locations))

display(station_locations.head(10))

Stations with coordinates: 200


,station_id,stop_name,stop_lat,stop_lon,transport_modes
0,211310,Macquarie University Station,-33.777282,151.118267,[Metro]
1,211340,Macquarie Park Station,-33.785080,151.128374,[Metro]
2,212210,Eastwood Station,-33.790301,151.082297,[Train]
3,214710,Seven Hills Station,-33.774351,150.936123,[Train]
4,214810,Blacktown Station,-33.768544,150.907080,[Train]
5,215020,Parramatta Station,-33.817490,151.005325,[Train]
6,2153477,Norwest Station,-33.734617,150.963522,[Metro]
7,2153478,Bella Vista Station,-33.730566,150.944020,[Metro]
8,2154391,Castle Hill Station,-33.731626,151.007288,[Metro]
9,2154392,Hills Showground Station,-33.727912,150.987025,[Metro]


In [46]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [47]:
from sklearn.neighbors import BallTree

print("scikit-learn installed successfully!")

scikit-learn installed successfully!


In [48]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# Convert station coordinates to radians
coords = np.radians(
    station_locations[["stop_lat", "stop_lon"]].to_numpy()
)

# Build spatial search tree
tree = BallTree(coords, metric="haversine")

# Search within 500 metres
earth_radius_m = 6371000
radius_m = 500

nearby_indices, nearby_distances = tree.query_radius(
    coords,
    r=radius_m / earth_radius_m,
    return_distance=True,
    sort_results=True
)

connections = []

for i, (indices, distances) in enumerate(
    zip(nearby_indices, nearby_distances)
):
    for j, distance in zip(indices, distances):

        # Avoid matching a station with itself
        # and duplicate station pairs
        if j <= i:
            continue

        station_a = station_locations.iloc[i]
        station_b = station_locations.iloc[j]

        # Keep pairs that offer different transport modes
        modes_a = set(station_a["transport_modes"])
        modes_b = set(station_b["transport_modes"])

        if not (modes_a - modes_b or modes_b - modes_a):
            continue

        connections.append({
            "from_station_id": station_a["station_id"],
            "from_station": station_a["stop_name"],
            "from_modes": ", ".join(sorted(modes_a)),
            "to_station_id": station_b["station_id"],
            "to_station": station_b["stop_name"],
            "to_modes": ", ".join(sorted(modes_b)),
            "distance_m": round(distance * earth_radius_m),
        })

walking_connections = pd.DataFrame(connections)

print("Potential walking connections:", len(walking_connections))

display(
    walking_connections.sort_values("distance_m").head(20)
)

Potential walking connections: 14


,from_station_id,from_station,from_modes,to_station_id,to_station,to_modes,distance_m
0,200070,Town Hall Station,"Light Rail, Train",200066,Gadigal Station,Metro,173
1,200070,Town Hall Station,"Light Rail, Train",2000171,QVB,Light Rail,207
6,200080,Wynyard Station,"Light Rail, Train",2000177,Bridge Street Light Rail,Light Rail,236
3,2000171,QVB,Light Rail,200066,Gadigal Station,Metro,245
12,200040,Museum Station,Train,200066,Gadigal Station,Metro,299
5,200060,Central Station,"Light Rail, Metro, Train",2000168,Haymarket Light Rail,Light Rail,305
10,200030,Martin Place Station,"Metro, Train",200050,St James Station,Train,351
2,200070,Town Hall Station,"Light Rail, Train",200040,Museum Station,Train,359
8,206010,North Sydney Station,Train,206044,Victoria Cross Station,Metro,381
13,200050,St James Station,Train,200066,Gadigal Station,Metro,399


In [49]:
import numpy as np

walking_connections["estimated_walk_distance_m"] = (
    walking_connections["distance_m"] * 1.3
)

walking_connections["transfer_time_min"] = np.ceil(
    walking_connections["estimated_walk_distance_m"] / 75 + 2
).astype(int)

display(
    walking_connections[
        [
            "from_station",
            "to_station",
            "distance_m",
            "transfer_time_min"
        ]
    ].sort_values("distance_m")
)

,from_station,to_station,distance_m,transfer_time_min
0,Town Hall Station,Gadigal Station,173,5
1,Town Hall Station,QVB,207,6
6,Wynyard Station,Bridge Street Light Rail,236,7
3,QVB,Gadigal Station,245,7
12,Museum Station,Gadigal Station,299,8
5,Central Station,Haymarket Light Rail,305,8
10,Martin Place Station,St James Station,351,9
2,Town Hall Station,Museum Station,359,9
8,North Sydney Station,Victoria Cross Station,381,9
13,St James Station,Gadigal Station,399,9


In [50]:
print("transfers.txt:", (gtfs_dir / "transfers.txt").exists())
print("pathways.txt:", (gtfs_dir / "pathways.txt").exists())

transfers.txt: False
pathways.txt: True


In [51]:
pathways = pd.read_csv(
    gtfs_dir / "pathways.txt",
    dtype=str
)

print("Total pathways:", len(pathways))
print("\nColumns:")
print(pathways.columns.tolist())

display(pathways.head(10))

Total pathways: 6924

Columns:
['pathway_id', 'from_stop_id', 'to_stop_id', 'pathway_mode', 'is_bidirectional', 'traversal_time']


,pathway_id,from_stop_id,to_stop_id,pathway_mode,is_bidirectional,traversal_time
0,257861_257810_ENT2_walkway_1,257861,257810_ENT2,1,1,30
1,257862_257810_ENT1_walkway_1,257862,257810_ENT1,1,1,30
2,257811_257810_ENT1_walkway_1,257811,257810_ENT1,1,1,30
3,257812_257810_ENT1_walkway_1,257812,257810_ENT1,1,1,84
4,257821_257810_ENT2_walkway_1,257821,257810_ENT2,1,1,108
5,257823_257810_ENT2_walkway_1,257823,257810_ENT2,1,1,108
6,257810_ENT1_257810_LVX1_walkway_1,257810_ENT1,257810_LVX1,1,1,42
7,257810_LVX1_257810_ENT2_walkway_1,257810_LVX1,257810_ENT2,1,1,30
8,257821_257823_walkway_1,257821,257823,1,1,120
9,257991_257920_DP2_walkway_1,257991,257920_DP2,1,1,42


In [52]:
# All platforms used by Train, Metro and Light Rail
platform_ids = set(all_transport_stop_times["stop_id"])

# All stop IDs referenced by pathways
pathway_stop_ids = set(pathways["from_stop_id"]) | set(
    pathways["to_stop_id"]
)

# Platforms represented in the pathway network
matched_platforms = platform_ids & pathway_stop_ids

print("Total transport platform IDs:", len(platform_ids))
print("Platforms found in pathways:", len(matched_platforms))
print(
    "Platform coverage:",
    round(len(matched_platforms) / len(platform_ids) * 100, 1),
    "%"
)

# Check pathway records involving timetable platforms
matched_pathways = pathways[
    pathways["from_stop_id"].isin(platform_ids)
    | pathways["to_stop_id"].isin(platform_ids)
]

print("Pathway records involving platforms:", len(matched_pathways))

display(matched_pathways.head(10))

Total transport platform IDs: 568
Platforms found in pathways: 566
Platform coverage: 99.6 %
Pathway records involving platforms: 1606


,pathway_id,from_stop_id,to_stop_id,pathway_mode,is_bidirectional,traversal_time
79,2770591_277010_DP5_elevator_1,2770591,277010_DP5,5,1,60
80,2770592_277010_DP5_elevator_1,2770592,277010_DP5,5,1,60
81,2770591_277010_CNC_stairs_1,2770591,277010_CNC,2,1,30
82,2770592_277010_CNC_stairs_1,2770592,277010_CNC,2,1,30
83,2770591_277010_DP7_walkway_1,2770591,277010_DP7,1,1,30
84,2770592_277010_DP7_walkway_1,2770592,277010_DP7,1,1,30
85,2770593_277010_DP6_elevator_1,2770593,277010_DP6,5,1,60
86,2770594_277010_DP6_elevator_1,2770594,277010_DP6,5,1,60
87,2770593_277010_CNC_stairs_1,2770593,277010_CNC,2,1,30
88,2770594_277010_CNC_stairs_1,2770594,277010_CNC,2,1,30


In [53]:
%pip install networkx

Note: you may need to restart the kernel to use updated packages.


In [54]:
import networkx as nx
import pandas as pd

G = nx.DiGraph()

for row in pathways.itertuples(index=False):

    # Skip pathways without valid traversal times
    if pd.isna(row.traversal_time):
        continue

    traversal_time = float(row.traversal_time)

    if traversal_time < 0:
        continue

    # Add the permitted direction
    G.add_edge(
        row.from_stop_id,
        row.to_stop_id,
        weight=traversal_time
    )

    # Add reverse direction only when allowed
    if row.is_bidirectional == "1":
        G.add_edge(
            row.to_stop_id,
            row.from_stop_id,
            weight=traversal_time
        )

print("Pathway nodes:", G.number_of_nodes())
print("Pathway edges:", G.number_of_edges())

Pathway nodes: 4988
Pathway edges: 12162


In [55]:
central_platforms = transport_stops[
    transport_stops["station_id"] == "200060"
][
    ["stop_id", "stop_name", "transport_mode"]
].drop_duplicates()

display(central_platforms)

,stop_id,stop_name,transport_mode
206,2000321,"Central Station, Platform 1",Train
207,2000322,"Central Station, Platform 2",Train
208,2000323,"Central Station, Platform 3",Train
209,2000324,"Central Station, Platform 4",Train
210,2000325,"Central Station, Platform 5",Train
211,2000326,"Central Station, Platform 6",Train
212,2000327,"Central Station, Platform 7",Train
213,2000328,"Central Station, Platform 8",Train
214,2000329,"Central Station, Platform 9",Train
215,2000330,"Central Station, Platform 10",Train


In [56]:
transfer_results = []

platform_records = central_platforms.to_dict("records")

for origin in platform_records:
    for destination in platform_records:

        if origin["stop_id"] == destination["stop_id"]:
            continue

        try:
            seconds = nx.shortest_path_length(
                G,
                source=origin["stop_id"],
                target=destination["stop_id"],
                weight="weight"
            )

            transfer_results.append({
                "from_platform": origin["stop_name"],
                "from_mode": origin["transport_mode"],
                "to_platform": destination["stop_name"],
                "to_mode": destination["transport_mode"],
                "transfer_seconds": seconds,
                "transfer_minutes": round(seconds / 60, 1)
            })

        except (nx.NetworkXNoPath, nx.NodeNotFound):
            continue

central_transfers = pd.DataFrame(transfer_results)

print("Reachable Central platform pairs:", len(central_transfers))

display(central_transfers.head(20))

Reachable Central platform pairs: 812


,from_platform,from_mode,to_platform,to_mode,transfer_seconds,transfer_minutes
0,"Central Station, Platform 1",Train,"Central Station, Platform 2",Train,60.0,1.0
1,"Central Station, Platform 1",Train,"Central Station, Platform 3",Train,60.0,1.0
2,"Central Station, Platform 1",Train,"Central Station, Platform 4",Train,168.0,2.8
3,"Central Station, Platform 1",Train,"Central Station, Platform 5",Train,168.0,2.8
4,"Central Station, Platform 1",Train,"Central Station, Platform 6",Train,150.0,2.5
5,"Central Station, Platform 1",Train,"Central Station, Platform 7",Train,150.0,2.5
6,"Central Station, Platform 1",Train,"Central Station, Platform 8",Train,132.0,2.2
7,"Central Station, Platform 1",Train,"Central Station, Platform 9",Train,132.0,2.2
8,"Central Station, Platform 1",Train,"Central Station, Platform 10",Train,156.0,2.6
9,"Central Station, Platform 1",Train,"Central Station, Platform 11",Train,156.0,2.6


In [57]:
# Keep transfers between different transport modes
cross_mode_transfers = central_transfers[
    central_transfers["from_mode"] !=
    central_transfers["to_mode"]
].copy()

print(
    "Cross-mode platform connections:",
    len(cross_mode_transfers)
)

# Summarise transfer times by mode combination
cross_mode_summary = (
    cross_mode_transfers
    .groupby(["from_mode", "to_mode"])
    .agg(
        connections=("transfer_minutes", "count"),
        min_minutes=("transfer_minutes", "min"),
        median_minutes=("transfer_minutes", "median"),
        max_minutes=("transfer_minutes", "max")
    )
    .reset_index()
)

display(cross_mode_summary)

Cross-mode platform connections: 252


,from_mode,to_mode,connections,min_minutes,median_minutes,max_minutes
0,Light Rail,Metro,6,4.5,4.5,4.5
1,Light Rail,Train,72,2.8,4.4,5.4
2,Metro,Light Rail,6,4.5,4.5,4.5
3,Metro,Train,48,1.5,2.4,4.1
4,Train,Light Rail,72,2.8,4.4,5.4
5,Train,Metro,48,1.5,2.4,4.1


In [58]:
import networkx as nx
import pandas as pd
from itertools import permutations

transfer_records = []

# Group platforms by parent station
for station_id, group in transport_stops.groupby("station_id"):

    platforms = (
        group[
            ["stop_id", "stop_name", "transport_mode"]
        ]
        .drop_duplicates(subset=["stop_id"])
        .to_dict("records")
    )

    # Test every directed platform pair within the station
    for origin, destination in permutations(platforms, 2):

        try:
            seconds = nx.shortest_path_length(
                G,
                source=origin["stop_id"],
                target=destination["stop_id"],
                weight="weight"
            )

            transfer_records.append({
                "station_id": station_id,
                "from_stop_id": origin["stop_id"],
                "to_stop_id": destination["stop_id"],
                "from_mode": origin["transport_mode"],
                "to_mode": destination["transport_mode"],
                "pathway_seconds": seconds,
                "pathway_minutes": round(seconds / 60, 2)
            })

        except (nx.NetworkXNoPath, nx.NodeNotFound):
            continue

platform_transfers = pd.DataFrame(transfer_records)

print("Total platform transfer connections:", len(platform_transfers))

display(
    platform_transfers.groupby(
        ["from_mode", "to_mode"]
    ).agg(
        connections=("pathway_seconds", "count"),
        median_minutes=("pathway_minutes", "median")
    ).reset_index()
)

Total platform transfer connections: 2108


,from_mode,to_mode,connections,median_minutes
0,Light Rail,Light Rail,44,0.60
1,Light Rail,Metro,6,4.50
2,Light Rail,Train,98,4.40
3,Metro,Light Rail,6,4.50
4,Metro,Metro,62,1.40
5,Metro,Train,74,2.45
6,Train,Light Rail,98,4.40
7,Train,Metro,74,2.45
8,Train,Train,1646,2.30


In [59]:
# Define additional transfer buffers in seconds
transfer_buffers = {
    ("Train", "Train"): 180,
    ("Metro", "Metro"): 180,
    ("Light Rail", "Light Rail"): 120,
    ("Train", "Metro"): 240,
    ("Metro", "Train"): 240,
    ("Train", "Light Rail"): 300,
    ("Light Rail", "Train"): 300,
    ("Metro", "Light Rail"): 300,
    ("Light Rail", "Metro"): 300
}

# Assign buffer based on transport modes
platform_transfers["transfer_buffer_seconds"] = (
    platform_transfers.apply(
        lambda row: transfer_buffers[
            (row["from_mode"], row["to_mode"])
        ],
        axis=1
    )
)

# Calculate total minimum connection time
platform_transfers["minimum_transfer_seconds"] = (
    platform_transfers["pathway_seconds"]
    + platform_transfers["transfer_buffer_seconds"]
)

platform_transfers["minimum_transfer_minutes"] = (
    platform_transfers["minimum_transfer_seconds"] / 60
).round(2)

# Check for duplicate platform pairs
duplicates = platform_transfers.duplicated(
    subset=["from_stop_id", "to_stop_id"]
).sum()

print("Duplicate platform pairs:", duplicates)

display(
    platform_transfers.groupby(
        ["from_mode", "to_mode"]
    )["minimum_transfer_minutes"]
    .agg(["min", "median", "max"])
    .round(1)
)

Duplicate platform pairs: 0


min  median   max
from_mode  to_mode                      
Light Rail Light Rail  2.4     2.6   4.5
           Metro       9.5     9.5   9.5
           Train       7.2     9.4  10.4
Metro      Light Rail  9.5     9.5   9.5
           Metro       3.6     4.4   7.5
           Train       5.0     6.4   8.9
Train      Light Rail  7.2     9.4  10.4
           Metro       5.0     6.4   8.9
           Train       3.4     5.3   9.3

In [60]:
# Build a clean multimodal timetable
timetable = all_transport_stop_times[
    [
        "trip_id",
        "stop_id",
        "arrival_time",
        "departure_time",
        "stop_sequence",
        "route_short_name",
        "transport_mode"
    ]
].copy()

# Add service IDs to identify which dates trips operate
timetable = timetable.merge(
    all_transport_trips[
        ["trip_id", "service_id"]
    ],
    on="trip_id",
    how="left",
    validate="many_to_one"
)

# Convert stop sequences to integers
timetable["stop_sequence"] = pd.to_numeric(
    timetable["stop_sequence"]
)

# Convert GTFS times to seconds
timetable["arrival_seconds"] = (
    timetable["arrival_time"].apply(time_to_seconds)
)

timetable["departure_seconds"] = (
    timetable["departure_time"].apply(time_to_seconds)
)

# Add parent station IDs
timetable = timetable.merge(
    transport_stops[
        ["stop_id", "station_id"]
    ].drop_duplicates("stop_id"),
    on="stop_id",
    how="left",
    validate="many_to_one"
)

print("Timetable records:", len(timetable))
print("Missing station IDs:", timetable["station_id"].isna().sum())

display(timetable.head(10))

Timetable records: 938621
Missing station IDs: 0


,trip_id,stop_id,arrival_time,departure_time,stop_sequence,route_short_name,transport_mode,service_id,arrival_seconds,departure_seconds,station_id
0,101A.1324.165.16.T.8.92005031,207262,05:02:00,05:02:00,1,T9,Train,TA+bk000+7,18120,18120,207210
1,101A.1324.165.16.T.8.92005031,207191,05:03:42,05:04:12,2,T9,Train,TA+bk000+7,18222,18252,207110
2,101A.1324.165.16.T.8.92005031,2070141,05:05:48,05:06:18,3,T9,Train,TA+bk000+7,18348,18378,207010
3,101A.1324.165.16.T.8.92005031,2069131,05:08:06,05:08:36,4,T9,Train,TA+bk000+7,18486,18516,206910
4,101A.1324.165.16.T.8.92005031,2067141,05:11:00,05:12:00,5,T9,Train,TA+bk000+7,18660,18720,206710
5,101A.1324.165.16.T.8.92005031,206451,05:14:00,05:14:30,6,T9,Train,TA+bk000+7,18840,18870,206410
6,101A.1324.165.16.T.8.92005031,2065152,05:16:36,05:17:06,7,T9,Train,TA+bk000+7,18996,19026,206520
7,101A.1324.165.16.T.8.92005031,2065161,05:19:00,05:19:30,8,T9,Train,TA+bk000+7,19140,19170,206510
8,101A.1324.165.16.T.8.92005031,2060111,05:21:12,05:21:42,9,T9,Train,TA+bk000+7,19272,19302,206020
9,101A.1324.165.16.T.8.92005031,2060101,05:24:00,05:25:00,10,T9,Train,TA+bk000+7,19440,19500,206010


In [61]:
from datetime import datetime

def get_active_services(travel_date):
    """
    Return the GTFS service IDs operating on a given date.

    travel_date format: YYYY-MM-DD
    """

    date_obj = datetime.strptime(travel_date, "%Y-%m-%d")

    date_str = date_obj.strftime("%Y%m%d")
    weekday = date_obj.strftime("%A").lower()

    # Regular services operating on this weekday
    regular_services = calendar[
        (calendar["start_date"] <= date_str)
        & (calendar["end_date"] >= date_str)
        & (calendar[weekday] == "1")
    ]

    active_services = set(regular_services["service_id"])

    # Special service additions and cancellations
    exceptions = calendar_dates[
        calendar_dates["date"] == date_str
    ]

    removed_services = set(
        exceptions.loc[
            exceptions["exception_type"] == "2",
            "service_id"
        ]
    )

    added_services = set(
        exceptions.loc[
            exceptions["exception_type"] == "1",
            "service_id"
        ]
    )

    return (active_services - removed_services) | added_services

In [62]:
travel_date = "2026-10-28"

active_services = get_active_services(travel_date)

daily_timetable = timetable[
    timetable["service_id"].isin(active_services)
].copy()

print("Active service IDs:", len(active_services))
print("Daily timetable records:", len(daily_timetable))

display(
    daily_timetable.groupby("transport_mode").agg(
        trips=("trip_id", "nunique"),
        stop_times=("stop_id", "count")
    )
)

Active service IDs: 662
Daily timetable records: 59541


,trips,stop_times
transport_mode,,
Light Rail,563,7997
Metro,434,9114
Train,2708,42430


In [63]:
def find_departures(station_name, departure_time, limit=20):
    # Find the station ID
    matching_stations = stops[
        (stops["stop_name"] == station_name)
        & (stops["location_type"] == "1")
    ]

    if matching_stations.empty:
        raise ValueError(f"Station not found: {station_name}")

    station_ids = set(matching_stations["stop_id"])

    # Find departures from any platform at this station
    departures = daily_timetable[
        daily_timetable["station_id"].isin(station_ids)
    ].copy()

    # Convert requested departure time to seconds
    requested_seconds = time_to_seconds(
        departure_time if len(departure_time) == 8
        else departure_time + ":00"
    )

    # Only keep departures after the requested time
    departures = departures[
        departures["departure_seconds"] >= requested_seconds
    ]

    # Sort by departure time
    departures = departures.sort_values("departure_seconds")

    return departures[
        [
            "trip_id",
            "route_short_name",
            "transport_mode",
            "stop_id",
            "departure_time",
            "departure_seconds",
            "stop_sequence"
        ]
    ].head(limit).reset_index(drop=True)

In [64]:
departures = find_departures(
    station_name="Central Station",
    departure_time="17:30",
    limit=20
)

display(departures)

,trip_id,route_short_name,transport_mode,stop_id,departure_time,departure_seconds,stop_sequence
0,78-N.1324.165.16.A.8.92004860,T8,Train,2000341,17:30:00,63000,15
1,136H.1324.165.16.A.8.92002355,T1,Train,2000336,17:30:01,63001,1
2,0160-006-102-015:1000,M1,Metro,2000466,17:30:02,63002,3
3,612M.1324.165.16.T.8.92000044,T4,Train,2000344,17:31:00,63060,14
4,47197-135:1001,LX,Light Rail,2010107,17:31:10,63070,8
5,47197-10561:1001,L2,Light Rail,2000447,17:31:35,63095,8
6,40-E.1324.165.124.A.8.92002717,T2,Train,2000339,17:32:00,63120,7
7,625J.1324.165.16.T.8.92002566,T4,Train,2000345,17:32:00,63120,6
8,5--R.1324.165.16.B.8.92002199,T8,Train,2000343,17:32:00,63120,7
9,78-P.1324.165.16.A.8.92002733,T2,Train,2000341,17:32:01,63121,1


In [65]:
# Select the first departure
selected_departure = departures.iloc[0]

trip_id = selected_departure["trip_id"]
origin_sequence = selected_departure["stop_sequence"]

# Find stops AFTER the selected departure
downstream_stops = daily_timetable[
    (daily_timetable["trip_id"] == trip_id)
    & (daily_timetable["stop_sequence"] > origin_sequence)
].copy()

# Add station names
station_names = stops[
    (stops["location_type"] == "1")
][["stop_id", "stop_name"]].drop_duplicates()

downstream_stops = downstream_stops.merge(
    station_names,
    left_on="station_id",
    right_on="stop_id",
    how="left",
    suffixes=("", "_station")
)

display(
    downstream_stops[
        [
            "route_short_name",
            "stop_name",
            "arrival_time",
            "stop_sequence",
            "transport_mode"
        ]
    ].sort_values("stop_sequence")
)

,route_short_name,stop_name,arrival_time,stop_sequence,transport_mode


In [66]:
selected_departure = departures.iloc[0]

trip_stops = daily_timetable[
    daily_timetable["trip_id"] == selected_departure["trip_id"]
].sort_values("stop_sequence")

print("Selected line:", selected_departure["route_short_name"])
print("Central stop sequence:", selected_departure["stop_sequence"])
print("Total stops on trip:", len(trip_stops))
print("Last stop sequence:", trip_stops["stop_sequence"].max())

display(
    trip_stops[
        ["stop_id", "arrival_time", "departure_time", "stop_sequence"]
    ].tail(10)
)

Selected line: T8
Central stop sequence: 15
Total stops on trip: 15
Last stop sequence: 15


,stop_id,arrival_time,departure_time,stop_sequence
606011,2208141,17:04:00,17:04:30,6
606012,2207281,17:06:30,17:07:00,7
606013,2207271,17:08:48,17:09:18,8
606014,2205101,17:11:24,17:11:54,9
606015,2205111,17:14:06,17:14:36,10
606016,2020101,17:16:30,17:17:00,11
606017,202091,17:19:06,17:19:36,12
606018,2020111,17:21:54,17:22:24,13
606019,2015111,17:25:30,17:26:00,14
606020,2000341,17:30:00,17:30:00,15


In [67]:
# Find the last stop sequence for every trip
last_stop_sequences = (
    daily_timetable
    .groupby("trip_id")["stop_sequence"]
    .max()
    .rename("last_stop_sequence")
)

# Add this information to our departures
departures_checked = departures.merge(
    last_stop_sequences,
    on="trip_id",
    how="left"
)

# Keep services that continue beyond Central
valid_departures = departures_checked[
    departures_checked["stop_sequence"]
    < departures_checked["last_stop_sequence"]
].copy()

print("Original departures:", len(departures))
print("Departures continuing beyond Central:", len(valid_departures))

display(
    valid_departures[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "stop_sequence",
            "last_stop_sequence"
        ]
    ].head(15)
)

Original departures: 20
Departures continuing beyond Central: 16


,route_short_name,transport_mode,departure_time,stop_sequence,last_stop_sequence
1,T1,Train,17:30:01,1,20
2,M1,Metro,17:30:02,3,21
3,T4,Train,17:31:00,14,19
5,L2,Light Rail,17:31:35,8,14
6,T2,Train,17:32:00,7,36
7,T4,Train,17:32:00,6,19
8,T8,Train,17:32:00,7,21
9,T2,Train,17:32:01,1,26
10,T8,Train,17:32:01,1,22
11,M1,Metro,17:32:49,19,21


In [68]:
# Select the first valid departure (T1 at 17:30:01)
selected_departure = valid_departures.iloc[0]

# Get all stops after boarding
downstream_stops = daily_timetable[
    (daily_timetable["trip_id"] == selected_departure["trip_id"])
    & (
        daily_timetable["stop_sequence"]
        > selected_departure["stop_sequence"]
    )
].copy()

# Add readable station names
station_lookup = (
    stops[stops["location_type"] == "1"]
    [["stop_id", "stop_name"]]
    .drop_duplicates("stop_id")
    .rename(columns={"stop_id": "station_id"})
)

downstream_stops = downstream_stops.merge(
    station_lookup,
    on="station_id",
    how="left",
    validate="many_to_one"
)

# Check which arrival platforms have valid transfer connections
transfer_platform_ids = set(
    platform_transfers["from_stop_id"]
)

downstream_stops["has_transfer"] = (
    downstream_stops["stop_id"].isin(transfer_platform_ids)
)

print("Line:", selected_departure["route_short_name"])
print("Departure:", selected_departure["departure_time"])
print("Downstream stops:", len(downstream_stops))
print(
    "Stops with transfer pathways:",
    downstream_stops["has_transfer"].sum()
)

display(
    downstream_stops[
        [
            "stop_name",
            "arrival_time",
            "stop_id",
            "stop_sequence",
            "has_transfer"
        ]
    ].sort_values("stop_sequence")
)

Line: T1
Departure: 17:30:01
Downstream stops: 19
Stops with transfer pathways: 19


,stop_name,arrival_time,stop_id,stop_sequence,has_transfer
0,Town Hall Station,17:32:36,2000393,2,True
1,Wynyard Station,17:35:30,2000404,3,True
2,Milsons Point Station,17:39:30,206142,4,True
3,North Sydney Station,17:41:30,2060104,5,True
4,Waverton Station,17:44:12,2060112,6,True
5,Wollstonecraft Station,17:46:18,2065162,7,True
6,St Leonards Station,17:49:30,2065153,8,True
7,Artarmon Station,17:52:12,206452,9,True
8,Chatswood Station,17:55:00,2067144,10,True
9,Roseville Station,17:57:48,2069132,11,True


In [69]:
# Select Chatswood as our test interchange
arrival = downstream_stops[
    downstream_stops["stop_name"] == "Chatswood Station"
].iloc[0]

arrival_platform = arrival["stop_id"]
arrival_seconds = arrival["arrival_seconds"]

print("Arriving at:", arrival["stop_name"])
print("Arrival time:", arrival["arrival_time"])
print("Arrival platform:", arrival_platform)

# Get valid pathways from the arrival platform
possible_transfers = platform_transfers[
    platform_transfers["from_stop_id"] == arrival_platform
].copy()

# Add the arrival time to each possible transfer
possible_transfers["earliest_boarding_seconds"] = (
    arrival_seconds
    + possible_transfers["minimum_transfer_seconds"]
)

# Find services departing from reachable platforms
connecting_services = daily_timetable.merge(
    possible_transfers[
        [
            "to_stop_id",
            "earliest_boarding_seconds",
            "minimum_transfer_minutes"
        ]
    ],
    left_on="stop_id",
    right_on="to_stop_id",
    how="inner"
)

# Keep departures within 30 minutes of arriving at Chatswood
connecting_services = connecting_services[
    (connecting_services["departure_seconds"]
     >= connecting_services["earliest_boarding_seconds"])
    & (connecting_services["departure_seconds"]
       <= arrival_seconds + 30 * 60)
    & (connecting_services["trip_id"]
       != selected_departure["trip_id"])
].copy()

# Exclude services terminating at the interchange
last_sequences = (
    daily_timetable.groupby("trip_id")["stop_sequence"].max()
)

connecting_services["last_stop_sequence"] = (
    connecting_services["trip_id"].map(last_sequences)
)

connecting_services = connecting_services[
    connecting_services["stop_sequence"]
    < connecting_services["last_stop_sequence"]
]

# Show possible connecting services
display(
    connecting_services[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "stop_id",
            "minimum_transfer_minutes"
        ]
    ]
    .sort_values("departure_time")
    .head(20)
)

Arriving at: Chatswood Station
Arrival time: 17:55:00
Arrival platform: 2067144


,route_short_name,transport_mode,departure_time,stop_id,minimum_transfer_minutes
235,M1,Metro,18:01:04,2067143,5.0
321,M1,Metro,18:01:55,2067142,5.4
13,T1,Train,18:03:00,2067141,4.4
255,M1,Metro,18:05:04,2067143,5.0
500,M1,Metro,18:05:55,2067142,5.4
89,T1,Train,18:06:00,2067141,4.4
437,M1,Metro,18:09:04,2067143,5.0
517,M1,Metro,18:09:55,2067142,5.4
65,T9,Train,18:12:00,2067141,4.4
455,M1,Metro,18:13:04,2067143,5.0


In [70]:
# Select the earliest feasible Metro connection
metro_connection = (
    connecting_services[
        connecting_services["transport_mode"] == "Metro"
    ]
    .sort_values("departure_seconds")
    .iloc[0]
)

# Find all downstream stops on that Metro trip
metro_destinations = daily_timetable[
    (daily_timetable["trip_id"] == metro_connection["trip_id"])
    & (
        daily_timetable["stop_sequence"]
        > metro_connection["stop_sequence"]
    )
].copy()

# Add readable station names
metro_destinations = metro_destinations.merge(
    station_lookup,
    on="station_id",
    how="left",
    validate="many_to_one"
)

print("First leg: T1")
print("Arrive at Chatswood:", arrival["arrival_time"])
print()
print("Second leg:", metro_connection["route_short_name"])
print("Depart Chatswood:", metro_connection["departure_time"])
print("Transfer platform:", metro_connection["stop_id"])
print()

display(
    metro_destinations[
        [
            "stop_name",
            "arrival_time",
            "stop_sequence"
        ]
    ].sort_values("stop_sequence")
)

First leg: T1
Arrive at Chatswood: 17:55:00

Second leg: M1
Depart Chatswood: 18:01:04
Transfer platform: 2067143



,stop_name,arrival_time,stop_sequence
0,North Ryde Station,18:05:56,10
1,Macquarie Park Station,18:08:06,11
2,Macquarie University Station,18:10:07,12
3,Epping Station,18:13:51,13
4,Cherrybrook Station,18:19:38,14
5,Castle Hill Station,18:22:22,15
6,Hills Showground Station,18:24:55,16
7,Norwest Station,18:27:30,17
8,Bella Vista Station,18:30:10,18
9,Kellyville Station,18:32:45,19


In [71]:
destination_name = "Macquarie University Station"

destination_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == destination_name,
        "station_id"
    ]
)

journey_options = []

for _, connection in connecting_services.iterrows():

    # Find downstream stops on this connecting trip
    destinations = daily_timetable[
        (daily_timetable["trip_id"] == connection["trip_id"])
        & (
            daily_timetable["stop_sequence"]
            > connection["stop_sequence"]
        )
        & (
            daily_timetable["station_id"].isin(destination_ids)
        )
    ]

    for _, destination in destinations.iterrows():

        journey_options.append({
            "first_line": selected_departure["route_short_name"],
            "first_departure": selected_departure["departure_time"],
            "transfer_station": "Chatswood Station",
            "transfer_arrival": arrival["arrival_time"],
            "second_line": connection["route_short_name"],
            "second_departure": connection["departure_time"],
            "destination_arrival": destination["arrival_time"],
            "transfer_minutes": round(
                (
                    connection["departure_seconds"]
                    - arrival_seconds
                ) / 60,
                2
            ),
            "total_journey_minutes": round(
                (
                    destination["arrival_seconds"]
                    - selected_departure["departure_seconds"]
                ) / 60,
                2
            )
        })

journey_options = pd.DataFrame(journey_options)

if journey_options.empty:
    print("No connecting journeys found.")
else:
    journey_options = journey_options.sort_values(
        "total_journey_minutes"
    )

    display(journey_options)

,first_line,first_departure,transfer_station,transfer_arrival,second_line,second_departure,destination_arrival,transfer_minutes,total_journey_minutes
0,T1,17:30:01,Chatswood Station,17:55:00,M1,18:01:04,18:10:07,6.07,40.1
1,T1,17:30:01,Chatswood Station,17:55:00,M1,18:05:04,18:14:07,10.07,44.1
3,T1,17:30:01,Chatswood Station,17:55:00,M1,18:09:04,18:18:07,14.07,48.1
4,T1,17:30:01,Chatswood Station,17:55:00,M1,18:13:04,18:22:07,18.07,52.1
5,T1,17:30:01,Chatswood Station,17:55:00,M1,18:17:04,18:26:07,22.07,56.1
2,T1,17:30:01,Chatswood Station,17:55:00,M1,18:21:04,18:30:07,26.07,60.1


In [72]:
def get_origin_departures(
    origin_station,
    departure_time,
    window_minutes=30
):
    # Find the parent station ID
    origin_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == origin_station,
            "station_id"
        ]
    )

    if not origin_ids:
        raise ValueError(f"Station not found: {origin_station}")

    requested_seconds = time_to_seconds(
        departure_time
        if len(departure_time) == 8
        else departure_time + ":00"
    )

    # Find services within the departure window
    origin_departures = daily_timetable[
        daily_timetable["station_id"].isin(origin_ids)
        & daily_timetable["departure_seconds"].between(
            requested_seconds,
            requested_seconds + window_minutes * 60
        )
    ].copy()

    # Identify the final stop sequence of each trip
    last_sequences = (
        daily_timetable.groupby("trip_id")["stop_sequence"].max()
    )

    origin_departures["last_stop_sequence"] = (
        origin_departures["trip_id"].map(last_sequences)
    )

    # Exclude terminating services
    origin_departures = origin_departures[
        origin_departures["stop_sequence"]
        < origin_departures["last_stop_sequence"]
    ]

    return origin_departures.sort_values(
        "departure_seconds"
    ).reset_index(drop=True)

In [73]:
origin_departures = get_origin_departures(
    origin_station="Central Station",
    departure_time="17:30",
    window_minutes=30
)

print("Valid departures:", len(origin_departures))

display(
    origin_departures.groupby("transport_mode").agg(
        services=("trip_id", "nunique"),
        departures=("trip_id", "count")
    )
)

display(
    origin_departures[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "stop_sequence"
        ]
    ].head(15)
)

Valid departures: 91


,services,departures
transport_mode,,
Light Rail,16,16
Metro,15,15
Train,55,60


,route_short_name,transport_mode,departure_time,stop_sequence
0,T1,Train,17:30:01,1
1,M1,Metro,17:30:02,3
2,T4,Train,17:31:00,14
3,L2,Light Rail,17:31:35,8
4,T4,Train,17:32:00,6
5,T2,Train,17:32:00,7
6,T8,Train,17:32:00,7
7,T8,Train,17:32:01,1
8,T2,Train,17:32:01,1
9,M1,Metro,17:32:49,19


In [74]:
# Give every origin boarding event a unique ID
first_legs = origin_departures.copy()
first_legs["boarding_id"] = range(len(first_legs))

# Keep the information needed for each boarding
boarding_events = first_legs[
    [
        "boarding_id",
        "trip_id",
        "stop_id",
        "stop_sequence",
        "departure_seconds",
        "departure_time",
        "route_short_name",
        "transport_mode"
    ]
].rename(columns={
    "stop_id": "origin_platform",
    "stop_sequence": "origin_sequence",
    "departure_seconds": "origin_departure_seconds",
    "departure_time": "origin_departure_time",
    "route_short_name": "first_line",
    "transport_mode": "first_mode"
})

# Match each boarding to stops on the same trip
first_leg_stops = boarding_events.merge(
    daily_timetable[
        [
            "trip_id",
            "stop_id",
            "station_id",
            "stop_sequence",
            "arrival_seconds",
            "arrival_time"
        ]
    ],
    on="trip_id",
    how="inner"
)

# Only keep stops AFTER the boarding event
first_leg_stops = first_leg_stops[
    first_leg_stops["stop_sequence"]
    > first_leg_stops["origin_sequence"]
].copy()

# Attach readable station names
first_leg_stops = first_leg_stops.merge(
    station_lookup,
    on="station_id",
    how="left",
    validate="many_to_one"
)

# Identify platforms with available pathway connections
transfer_platform_ids = set(
    platform_transfers["from_stop_id"]
)

first_leg_stops["has_transfer"] = (
    first_leg_stops["stop_id"].isin(transfer_platform_ids)
)

print("Origin boarding events:", len(boarding_events))
print("Total downstream stop events:", len(first_leg_stops))
print(
    "Downstream events with transfer pathways:",
    first_leg_stops["has_transfer"].sum()
)
print(
    "Unique potential interchange stations:",
    first_leg_stops.loc[
        first_leg_stops["has_transfer"],
        "station_id"
    ].nunique()
)

display(
    first_leg_stops[
        [
            "first_line",
            "first_mode",
            "origin_departure_time",
            "stop_name",
            "arrival_time",
            "has_transfer"
        ]
    ].head(20)
)

Origin boarding events: 91
Total downstream stop events: 1252
Downstream events with transfer pathways: 1240
Unique potential interchange stations: 182


,first_line,first_mode,origin_departure_time,stop_name,arrival_time,has_transfer
0,T1,Train,17:30:01,Town Hall Station,17:32:36,True
1,T1,Train,17:30:01,Wynyard Station,17:35:30,True
2,T1,Train,17:30:01,Milsons Point Station,17:39:30,True
3,T1,Train,17:30:01,North Sydney Station,17:41:30,True
4,T1,Train,17:30:01,Waverton Station,17:44:12,True
5,T1,Train,17:30:01,Wollstonecraft Station,17:46:18,True
6,T1,Train,17:30:01,St Leonards Station,17:49:30,True
7,T1,Train,17:30:01,Artarmon Station,17:52:12,True
8,T1,Train,17:30:01,Chatswood Station,17:55:00,True
9,T1,Train,17:30:01,Roseville Station,17:57:48,True


In [75]:
# 1. Create same-platform transfers

same_platform_transfers = (
    transport_stops[
        ["stop_id", "station_id", "transport_mode"]
    ]
    .drop_duplicates("stop_id")
    .rename(columns={
        "stop_id": "from_stop_id",
        "transport_mode": "from_mode"
    })
    .copy()
)

same_platform_transfers["to_stop_id"] = (
    same_platform_transfers["from_stop_id"]
)

same_platform_transfers["to_mode"] = (
    same_platform_transfers["from_mode"]
)

same_platform_transfers["pathway_seconds"] = 0
same_platform_transfers["pathway_minutes"] = 0.0

# Allow 2 minutes for same-platform boarding
same_platform_transfers["transfer_buffer_seconds"] = 120
same_platform_transfers["minimum_transfer_seconds"] = 120
same_platform_transfers["minimum_transfer_minutes"] = 2.0

# 2. Combine pathway and same-platform transfers

all_platform_transfers = pd.concat(
    [
        platform_transfers,
        same_platform_transfers.reindex(
            columns=platform_transfers.columns
        )
    ],
    ignore_index=True
)

# 3. Match downstream arrivals with transfer connections

transfer_candidates = first_leg_stops.merge(
    all_platform_transfers[
        [
            "from_stop_id",
            "to_stop_id",
            "minimum_transfer_seconds",
            "minimum_transfer_minutes"
        ]
    ],
    left_on="stop_id",
    right_on="from_stop_id",
    how="inner"
)

# 4. Calculate earliest feasible boarding time

transfer_candidates["earliest_boarding_seconds"] = (
    transfer_candidates["arrival_seconds"]
    + transfer_candidates["minimum_transfer_seconds"]
)

print("Total transfer candidates:", len(transfer_candidates))

print(
    "Unique first-leg boarding events:",
    transfer_candidates["boarding_id"].nunique()
)

print(
    "Same-platform connections:",
    (
        all_platform_transfers["from_stop_id"]
        == all_platform_transfers["to_stop_id"]
    ).sum()
)

display(
    transfer_candidates[
        [
            "first_line",
            "origin_departure_time",
            "stop_name",
            "arrival_time",
            "to_stop_id",
            "minimum_transfer_minutes"
        ]
    ].head(20)
)

Total transfer candidates: 4667
Unique first-leg boarding events: 91
Same-platform connections: 568


,first_line,origin_departure_time,stop_name,arrival_time,to_stop_id,minimum_transfer_minutes
0,T1,17:30:01,Town Hall Station,17:32:36,2000391,4.6
1,T1,17:30:01,Town Hall Station,17:32:36,2000392,4.6
2,T1,17:30:01,Town Hall Station,17:32:36,2000394,4.5
3,T1,17:30:01,Town Hall Station,17:32:36,2000395,4.5
4,T1,17:30:01,Town Hall Station,17:32:36,2000396,4.5
5,T1,17:30:01,Town Hall Station,17:32:36,2000458,7.5
6,T1,17:30:01,Town Hall Station,17:32:36,2000459,7.5
7,T1,17:30:01,Town Hall Station,17:32:36,2000393,2.0
8,T1,17:30:01,Wynyard Station,17:35:30,2000403,5.0
9,T1,17:30:01,Wynyard Station,17:35:30,2000405,5.0


In [76]:
# 1. Prepare second-leg boarding events
second_leg_departures = daily_timetable[
    [
        "trip_id",
        "stop_id",
        "stop_sequence",
        "departure_seconds",
        "departure_time",
        "route_short_name",
        "transport_mode"
    ]
].rename(columns={
    "trip_id": "second_trip_id",
    "stop_id": "second_platform",
    "stop_sequence": "second_boarding_sequence",
    "departure_seconds": "second_departure_seconds",
    "departure_time": "second_departure_time",
    "route_short_name": "second_line",
    "transport_mode": "second_mode"
})

# 2. Join reachable platforms with actual departures
feasible_connections = transfer_candidates.merge(
    second_leg_departures,
    left_on="to_stop_id",
    right_on="second_platform",
    how="inner"
)

# 3. Apply timetable feasibility rules
feasible_connections = feasible_connections[
    (
        feasible_connections["second_departure_seconds"]
        >= feasible_connections["earliest_boarding_seconds"]
    )
    & (
        feasible_connections["second_departure_seconds"]
        <= feasible_connections["arrival_seconds"] + 30 * 60
    )
    & (
        feasible_connections["trip_id"]
        != feasible_connections["second_trip_id"]
    )
].copy()

# 4. Exclude terminating second-leg services
last_sequences = (
    daily_timetable.groupby("trip_id")["stop_sequence"].max()
)

feasible_connections["second_last_sequence"] = (
    feasible_connections["second_trip_id"].map(last_sequences)
)

feasible_connections = feasible_connections[
    feasible_connections["second_boarding_sequence"]
    < feasible_connections["second_last_sequence"]
].copy()

# 5. Calculate connection waiting time
feasible_connections["connection_wait_minutes"] = (
    (
        feasible_connections["second_departure_seconds"]
        - feasible_connections["arrival_seconds"]
    ) / 60
).round(2)

print("Feasible second-leg boarding events:", len(feasible_connections))

print(
    "First-leg boarding events with connections:",
    feasible_connections["boarding_id"].nunique()
)

display(
    feasible_connections[
        [
            "first_line",
            "origin_departure_time",
            "stop_name",
            "arrival_time",
            "second_line",
            "second_mode",
            "second_departure_time",
            "minimum_transfer_minutes",
            "connection_wait_minutes"
        ]
    ]
    .sort_values(
        ["origin_departure_time", "arrival_time", "second_departure_time"]
    )
    .head(20)
)

Feasible second-leg boarding events: 15011
First-leg boarding events with connections: 91


,first_line,origin_departure_time,stop_name,arrival_time,second_line,second_mode,second_departure_time,minimum_transfer_minutes,connection_wait_minutes
1779,T1,17:30:01,Town Hall Station,17:32:36,T9,Train,17:37:36,2.0,5.00
393,T1,17:30:01,Town Hall Station,17:32:36,T1,Train,17:39:00,4.6,6.40
551,T1,17:30:01,Town Hall Station,17:32:36,T4,Train,17:39:00,4.5,6.40
141,T1,17:30:01,Town Hall Station,17:32:36,T3,Train,17:40:00,4.6,7.40
739,T1,17:30:01,Town Hall Station,17:32:36,T4,Train,17:40:00,4.5,7.40
1202,T1,17:30:01,Town Hall Station,17:32:36,L2,Light Rail,17:40:40,7.5,8.07
954,T1,17:30:01,Town Hall Station,17:32:36,T8,Train,17:40:42,4.5,8.10
1473,T1,17:30:01,Town Hall Station,17:32:36,L3,Light Rail,17:41:10,7.5,8.57
401,T1,17:30:01,Town Hall Station,17:32:36,T1,Train,17:42:00,4.6,9.40
98,T1,17:30:01,Town Hall Station,17:32:36,T2,Train,17:43:00,4.6,10.40


In [77]:
destination_name = "Macquarie University Station"

# 1. Find destination station IDs
destination_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == destination_name,
        "station_id"
    ]
)

if not destination_ids:
    raise ValueError(f"Destination not found: {destination_name}")

# 2. Find all timetable arrivals at the destination
destination_arrivals = daily_timetable[
    daily_timetable["station_id"].isin(destination_ids)
][
    [
        "trip_id",
        "stop_sequence",
        "arrival_seconds",
        "arrival_time"
    ]
].rename(columns={
    "trip_id": "second_trip_id",
    "stop_sequence": "destination_sequence",
    "arrival_seconds": "destination_arrival_seconds",
    "arrival_time": "destination_arrival_time"
})

# 3. Match second-leg trips to destination arrivals
complete_journeys = feasible_connections.merge(
    destination_arrivals,
    on="second_trip_id",
    how="inner"
)

# 4. Ensure destination is AFTER second-leg boarding
complete_journeys = complete_journeys[
    complete_journeys["destination_sequence"]
    > complete_journeys["second_boarding_sequence"]
].copy()

# 5. Calculate complete journey duration
complete_journeys["total_journey_minutes"] = (
    (
        complete_journeys["destination_arrival_seconds"]
        - complete_journeys["origin_departure_seconds"]
    ) / 60
).round(2)

# 6. Remove exact duplicate journey combinations
complete_journeys = complete_journeys.drop_duplicates(
    subset=[
        "boarding_id",
        "stop_sequence",
        "second_trip_id",
        "second_boarding_sequence",
        "destination_sequence"
    ]
)

# 7. Sort by earliest destination arrival
complete_journeys = complete_journeys.sort_values(
    [
        "destination_arrival_seconds",
        "origin_departure_seconds"
    ]
)

print("Complete one-transfer journeys:", len(complete_journeys))

display(
    complete_journeys[
        [
            "first_line",
            "origin_departure_time",
            "stop_name",
            "arrival_time",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "minimum_transfer_minutes",
            "total_journey_minutes"
        ]
    ].head(15)
)

Complete one-transfer journeys: 1129


,first_line,origin_departure_time,stop_name,arrival_time,second_line,second_departure_time,destination_arrival_time,minimum_transfer_minutes,total_journey_minutes
20,M1,17:30:02,Gadigal Station,17:31:31,M1,17:36:01,17:58:07,2.0,28.08
34,M1,17:30:02,Martin Place Station,17:33:19,M1,17:37:59,17:58:07,2.0,28.08
47,M1,17:30:02,Barangaroo Station,17:35:41,M1,17:40:11,17:58:07,2.0,28.08
61,M1,17:30:02,Victoria Cross Station,17:38:39,M1,17:43:09,17:58:07,2.0,28.08
74,M1,17:30:02,Crows Nest Station,17:40:46,M1,17:45:16,17:58:07,2.0,28.08
88,M1,17:30:02,Chatswood Station,17:44:19,M1,17:49:04,17:58:07,2.0,28.08
101,M1,17:30:02,North Ryde Station,17:49:56,M1,17:54:26,17:58:07,2.0,28.08
115,M1,17:30:02,Macquarie Park Station,17:52:06,M1,17:56:36,17:58:07,2.0,28.08
21,M1,17:30:02,Gadigal Station,17:31:31,M1,17:40:01,18:02:07,2.0,32.08
35,M1,17:30:02,Martin Place Station,17:33:19,M1,17:41:59,18:02:07,2.0,32.08


In [78]:
# Find destination arrivals on the original first-leg trips
direct_destinations = daily_timetable[
    daily_timetable["station_id"].isin(destination_ids)
][
    [
        "trip_id",
        "stop_sequence",
        "arrival_seconds",
        "arrival_time"
    ]
].rename(columns={
    "stop_sequence": "destination_sequence",
    "arrival_seconds": "destination_arrival_seconds",
    "arrival_time": "destination_arrival_time"
})

# Match origin boarding events to destination arrivals
direct_journeys = boarding_events.merge(
    direct_destinations,
    on="trip_id",
    how="inner"
)

# Destination must occur AFTER boarding
direct_journeys = direct_journeys[
    direct_journeys["destination_sequence"]
    > direct_journeys["origin_sequence"]
].copy()

# Calculate journey duration
direct_journeys["total_journey_minutes"] = (
    (
        direct_journeys["destination_arrival_seconds"]
        - direct_journeys["origin_departure_seconds"]
    ) / 60
).round(2)

# Sort by earliest arrival
direct_journeys = direct_journeys.sort_values(
    [
        "destination_arrival_seconds",
        "origin_departure_seconds"
    ]
)

print("Direct journeys found:", len(direct_journeys))

display(
    direct_journeys[
        [
            "first_line",
            "first_mode",
            "origin_departure_time",
            "destination_arrival_time",
            "total_journey_minutes"
        ]
    ].head(15)
)

Direct journeys found: 8


,first_line,first_mode,origin_departure_time,destination_arrival_time,total_journey_minutes
0,M1,Metro,17:30:02,17:54:07,24.08
2,M1,Metro,17:34:02,17:58:07,24.08
4,M1,Metro,17:38:02,18:02:07,24.08
6,M1,Metro,17:42:02,18:06:07,24.08
8,M1,Metro,17:46:02,18:10:07,24.08
10,M1,Metro,17:50:02,18:14:07,24.08
12,M1,Metro,17:54:02,18:18:07,24.08
14,M1,Metro,17:58:02,18:22:07,24.08


In [79]:
# 1. Prepare direct journeys
direct_results = direct_journeys[
    [
        "first_line",
        "origin_departure_time",
        "origin_departure_seconds",
        "destination_arrival_time",
        "destination_arrival_seconds",
        "total_journey_minutes"
    ]
].copy()

direct_results["journey_type"] = "Direct"
direct_results["transfers"] = 0
direct_results["second_line"] = None
direct_results["transfer_station"] = None

# 2. Prepare one-transfer journeys
transfer_results = complete_journeys[
    [
        "first_line",
        "origin_departure_time",
        "origin_departure_seconds",
        "stop_name",
        "second_line",
        "destination_arrival_time",
        "destination_arrival_seconds",
        "total_journey_minutes"
    ]
].copy()

transfer_results = transfer_results.rename(
    columns={"stop_name": "transfer_station"}
)

transfer_results["journey_type"] = "One transfer"
transfer_results["transfers"] = 1

# 3. Combine results
all_journeys = pd.concat(
    [direct_results, transfer_results],
    ignore_index=True
)

# 4. Calculate total time from the requested departure
requested_seconds = time_to_seconds("17:30:00")

all_journeys["elapsed_from_request_min"] = (
    (
        all_journeys["destination_arrival_seconds"]
        - requested_seconds
    ) / 60
).round(2)

# 5. Rank journeys
all_journeys = all_journeys.sort_values(
    [
        "destination_arrival_seconds",
        "transfers",
        "origin_departure_seconds"
    ],
    ascending=[True, True, True]
).reset_index(drop=True)

# 6. Show the best journey
print("Total journey candidates:", len(all_journeys))

display(
    all_journeys[
        [
            "journey_type",
            "first_line",
            "second_line",
            "origin_departure_time",
            "transfer_station",
            "destination_arrival_time",
            "total_journey_minutes",
            "elapsed_from_request_min"
        ]
    ].head(15)
)

Total journey candidates: 1137


,journey_type,first_line,second_line,origin_departure_time,transfer_station,destination_arrival_time,total_journey_minutes,elapsed_from_request_min
0,Direct,M1,None,17:30:02,None,17:54:07,24.08,24.12
1,Direct,M1,None,17:34:02,None,17:58:07,24.08,28.12
2,One transfer,M1,M1,17:30:02,Gadigal Station,17:58:07,28.08,28.12
3,One transfer,M1,M1,17:30:02,Martin Place Station,17:58:07,28.08,28.12
4,One transfer,M1,M1,17:30:02,Barangaroo Station,17:58:07,28.08,28.12
5,One transfer,M1,M1,17:30:02,Victoria Cross Station,17:58:07,28.08,28.12
6,One transfer,M1,M1,17:30:02,Crows Nest Station,17:58:07,28.08,28.12
7,One transfer,M1,M1,17:30:02,Chatswood Station,17:58:07,28.08,28.12
8,One transfer,M1,M1,17:30:02,North Ryde Station,17:58:07,28.08,28.12
9,One transfer,M1,M1,17:30:02,Macquarie Park Station,17:58:07,28.08,28.12


In [80]:
# Separate direct and transfer journeys
direct_options = all_journeys[
    all_journeys["transfers"] == 0
].copy()

transfer_options = all_journeys[
    all_journeys["transfers"] == 1
].copy()

# Compare each transfer journey against direct alternatives
comparisons = transfer_options[
    [
        "origin_departure_seconds",
        "destination_arrival_seconds"
    ]
].merge(
    direct_options[
        [
            "origin_departure_seconds",
            "destination_arrival_seconds"
        ]
    ],
    how="cross",
    suffixes=("_transfer", "_direct")
)

# A direct journey dominates a transfer journey if:
# 1. It departs no earlier
# 2. It arrives no later
comparisons["dominated"] = (
    (
        comparisons["origin_departure_seconds_direct"]
        >= comparisons["origin_departure_seconds_transfer"]
    )
    &
    (
        comparisons["destination_arrival_seconds_direct"]
        <= comparisons["destination_arrival_seconds_transfer"]
    )
)

# Identify dominated transfer journeys
dominated_pairs = comparisons[
    comparisons["dominated"]
][
    [
        "origin_departure_seconds_transfer",
        "destination_arrival_seconds_transfer"
    ]
].drop_duplicates()

# Mark transfer journeys that have a better direct alternative
transfer_options = transfer_options.merge(
    dominated_pairs.assign(dominated=True),
    left_on=[
        "origin_departure_seconds",
        "destination_arrival_seconds"
    ],
    right_on=[
        "origin_departure_seconds_transfer",
        "destination_arrival_seconds_transfer"
    ],
    how="left"
)

transfer_options = transfer_options[
    transfer_options["dominated"].isna()
].drop(
    columns=[
        "dominated",
        "origin_departure_seconds_transfer",
        "destination_arrival_seconds_transfer"
    ]
)

# Combine direct journeys with remaining transfers
recommended_journeys = pd.concat(
    [direct_options, transfer_options],
    ignore_index=True
).sort_values(
    [
        "destination_arrival_seconds",
        "transfers",
        "origin_departure_seconds"
    ],
    ascending=[True, True, False]
).reset_index(drop=True)

print("Original candidates:", len(all_journeys))
print("After removing dominated transfers:", len(recommended_journeys))

display(
    recommended_journeys[
        [
            "journey_type",
            "first_line",
            "second_line",
            "origin_departure_time",
            "transfer_station",
            "destination_arrival_time",
            "elapsed_from_request_min"
        ]
    ].head(15)
)

Original candidates: 1137
After removing dominated transfers: 18


,journey_type,first_line,second_line,origin_departure_time,transfer_station,destination_arrival_time,elapsed_from_request_min
0,Direct,M1,None,17:30:02,None,17:54:07,24.12
1,Direct,M1,None,17:34:02,None,17:58:07,28.12
2,Direct,M1,None,17:38:02,None,18:02:07,32.12
3,Direct,M1,None,17:42:02,None,18:06:07,36.12
4,Direct,M1,None,17:46:02,None,18:10:07,40.12
5,Direct,M1,None,17:50:02,None,18:14:07,44.12
6,Direct,M1,None,17:54:02,None,18:18:07,48.12
7,Direct,M1,None,17:58:02,None,18:22:07,52.12
8,One transfer,T2,M1,17:59:01,Central Station,18:50:07,80.12
9,One transfer,T8,M1,17:59:01,Central Station,18:50:07,80.12


In [81]:
# Identify origin station IDs
origin_station_name = "Central Station"

origin_station_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == origin_station_name,
        "station_id"
    ]
)

# Remove one-transfer journeys returning to the origin
valid_transfer_journeys = complete_journeys[
    ~complete_journeys["station_id"].isin(origin_station_ids)
].copy()

print(
    "Original one-transfer journeys:",
    len(complete_journeys)
)

print(
    "After removing transfers at origin:",
    len(valid_transfer_journeys)
)

print(
    "Removed:",
    len(complete_journeys) - len(valid_transfer_journeys)
)

Original one-transfer journeys: 1129
After removing transfers at origin: 1049
Removed: 80


In [82]:
# Identify journeys where both legs use the same line
valid_transfer_journeys = valid_transfer_journeys.copy()

valid_transfer_journeys["same_line_transfer"] = (
    valid_transfer_journeys["first_line"]
    == valid_transfer_journeys["second_line"]
)

print("Remaining transfer journeys:", len(valid_transfer_journeys))

print(
    "Same-line transfers:",
    valid_transfer_journeys["same_line_transfer"].sum()
)

print(
    "Different-line transfers:",
    (~valid_transfer_journeys["same_line_transfer"]).sum()
)

display(
    valid_transfer_journeys.groupby(
        ["first_line", "second_line"]
    ).size()
    .reset_index(name="journeys")
    .sort_values("journeys", ascending=False)
    .head(15)
)

Remaining transfer journeys: 1049
Same-line transfers: 912
Different-line transfers: 137


,first_line,second_line,journeys
0,M1,M1,912
2,T4,M1,56
4,T9,M1,36
1,T1,M1,24
3,T8,M1,21


In [83]:
# Get the first trip's arrival at the destination
first_trip_destinations = daily_timetable[
    daily_timetable["station_id"].isin(destination_ids)
][
    ["trip_id", "stop_sequence", "arrival_seconds"]
].rename(columns={
    "stop_sequence": "first_trip_destination_sequence",
    "arrival_seconds": "first_trip_destination_arrival"
})

# Remove any previous merge columns if the cell was partially run
base_journeys = valid_transfer_journeys.drop(
    columns=[
        "first_trip_destination_sequence",
        "first_trip_destination_arrival"
    ],
    errors="ignore"
)

# Merge with explicitly named columns
transfer_check = base_journeys.merge(
    first_trip_destinations,
    on="trip_id",
    how="left"
)

# Check if first service reaches destination AFTER original boarding
first_trip_reaches_destination = (
    transfer_check["first_trip_destination_sequence"]
    > transfer_check["origin_sequence"]
)

# Check if staying on the first service is faster
staying_onboard_is_better = (
    first_trip_reaches_destination
    &
    (
        transfer_check["first_trip_destination_arrival"]
        <= transfer_check["destination_arrival_seconds"]
    )
)

# Remove unnecessary transfers
useful_transfer_journeys = transfer_check[
    ~staying_onboard_is_better
].copy()

print("Before filtering:", len(transfer_check))
print("Unnecessary transfers:", staying_onboard_is_better.sum())
print("Remaining transfers:", len(useful_transfer_journeys))

display(
    useful_transfer_journeys[
        [
            "first_line",
            "origin_departure_time",
            "stop_name",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "total_journey_minutes"
        ]
    ].sort_values(
        "total_journey_minutes"
    ).head(15)
)

Before filtering: 1049
Unnecessary transfers: 814
Remaining transfers: 235


,first_line,origin_departure_time,stop_name,second_line,second_departure_time,destination_arrival_time,total_journey_minutes
71,T4,17:37:00,Martin Place Station,M1,17:49:59,18:10:07,33.12
449,T4,17:57:00,Martin Place Station,M1,18:09:59,18:30:07,33.12
122,T4,17:41:00,Martin Place Station,M1,17:53:59,18:14:07,33.12
33,M1,17:32:49,Waterloo Station,M1,17:39:42,18:06:07,33.30
266,M1,17:48:49,Waterloo Station,M1,17:55:42,18:22:07,33.30
70,M1,17:36:49,Waterloo Station,M1,17:43:42,18:10:07,33.30
187,M1,17:44:49,Waterloo Station,M1,17:51:42,18:18:07,33.30
353,M1,17:52:49,Waterloo Station,M1,17:59:42,18:26:07,33.30
448,M1,17:56:49,Waterloo Station,M1,18:03:42,18:30:07,33.30
121,M1,17:40:49,Waterloo Station,M1,17:47:42,18:14:07,33.30


In [84]:
# Prepare the remaining transfer journeys
clean_transfers = useful_transfer_journeys[
    [
        "first_line",
        "second_line",
        "origin_departure_time",
        "origin_departure_seconds",
        "stop_name",
        "destination_arrival_time",
        "destination_arrival_seconds",
        "total_journey_minutes"
    ]
].copy()

clean_transfers = clean_transfers.rename(
    columns={"stop_name": "transfer_station"}
)

clean_transfers["journey_type"] = "One transfer"
clean_transfers["transfers"] = 1

# Prepare direct journeys
clean_direct = direct_journeys[
    [
        "first_line",
        "origin_departure_time",
        "origin_departure_seconds",
        "destination_arrival_time",
        "destination_arrival_seconds",
        "total_journey_minutes"
    ]
].copy()

clean_direct["second_line"] = None
clean_direct["transfer_station"] = None
clean_direct["journey_type"] = "Direct"
clean_direct["transfers"] = 0

# Combine
candidates = pd.concat(
    [clean_direct, clean_transfers],
    ignore_index=True
).reset_index(drop=True)

# Pareto dominance filtering
keep = []

for i, journey in candidates.iterrows():

    alternatives = candidates.drop(index=i)

    dominated = (
        (
            alternatives["origin_departure_seconds"]
            >= journey["origin_departure_seconds"]
        )
        &
        (
            alternatives["destination_arrival_seconds"]
            <= journey["destination_arrival_seconds"]
        )
        &
        (
            alternatives["transfers"]
            <= journey["transfers"]
        )
        &
        (
            (alternatives["origin_departure_seconds"]
             > journey["origin_departure_seconds"])
            |
            (alternatives["destination_arrival_seconds"]
             < journey["destination_arrival_seconds"])
            |
            (alternatives["transfers"] < journey["transfers"])
        )
    ).any()

    if not dominated:
        keep.append(i)

best_journeys = candidates.loc[keep].sort_values(
    ["destination_arrival_seconds", "transfers"]
).reset_index(drop=True)

print("Candidates before filtering:", len(candidates))
print("Non-dominated journeys:", len(best_journeys))

display(
    best_journeys[
        [
            "journey_type",
            "first_line",
            "second_line",
            "origin_departure_time",
            "transfer_station",
            "destination_arrival_time",
            "total_journey_minutes"
        ]
    ].head(15)
)

Candidates before filtering: 243
Non-dominated journeys: 8


,journey_type,first_line,second_line,origin_departure_time,transfer_station,destination_arrival_time,total_journey_minutes
0,Direct,M1,None,17:30:02,None,17:54:07,24.08
1,Direct,M1,None,17:34:02,None,17:58:07,24.08
2,Direct,M1,None,17:38:02,None,18:02:07,24.08
3,Direct,M1,None,17:42:02,None,18:06:07,24.08
4,Direct,M1,None,17:46:02,None,18:10:07,24.08
5,Direct,M1,None,17:50:02,None,18:14:07,24.08
6,Direct,M1,None,17:54:02,None,18:18:07,24.08
7,Direct,M1,None,17:58:02,None,18:22:07,24.08


In [85]:
# Test only journeys starting on a Train
train_first_journeys = useful_transfer_journeys[
    useful_transfer_journeys["first_mode"] == "Train"
].copy()

# Rank by earliest arrival
train_first_journeys = train_first_journeys.sort_values(
    [
        "destination_arrival_seconds",
        "origin_departure_seconds"
    ]
)

print(
    "Train → Metro journey candidates:",
    len(train_first_journeys)
)

display(
    train_first_journeys[
        [
            "first_line",
            "origin_departure_time",
            "stop_name",
            "arrival_time",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "minimum_transfer_minutes",
            "total_journey_minutes"
        ]
    ].head(10)
)

Train → Metro journey candidates: 137


,first_line,origin_departure_time,stop_name,arrival_time,second_line,second_departure_time,destination_arrival_time,minimum_transfer_minutes,total_journey_minutes
32,T4,17:31:00,Martin Place Station,17:35:30,M1,17:45:59,18:06:07,6.6,35.12
51,T1,17:30:01,Chatswood Station,17:55:00,M1,18:01:04,18:10:07,5.0,40.10
60,T4,17:31:00,Martin Place Station,17:35:30,M1,17:49:59,18:10:07,6.6,39.12
71,T4,17:37:00,Martin Place Station,17:41:30,M1,17:49:59,18:10:07,6.6,33.12
90,T1,17:30:01,Chatswood Station,17:55:00,M1,18:05:04,18:14:07,5.0,44.10
99,T4,17:31:00,Martin Place Station,17:35:30,M1,17:53:59,18:14:07,6.6,43.12
102,T9,17:34:01,Chatswood Station,17:59:00,M1,18:05:04,18:14:07,5.0,40.10
112,T4,17:37:00,Martin Place Station,17:41:30,M1,17:53:59,18:14:07,6.6,37.12
122,T4,17:41:00,Martin Place Station,17:45:30,M1,17:53:59,18:14:07,6.6,33.12
141,T9,17:36:01,Epping Station,18:05:00,M1,18:12:13,18:15:24,5.5,39.38


In [86]:
def prepare_daily_timetable(travel_date):
    """
    Return scheduled Train, Metro and Light Rail stop events
    operating on the requested date.
    """

    active_services = get_active_services(travel_date)

    daily = timetable[
        timetable["service_id"].isin(active_services)
    ].copy()

    daily = daily.sort_values(
        ["trip_id", "stop_sequence"]
    ).reset_index(drop=True)

    return daily


# Test with our existing date
test_timetable = prepare_daily_timetable("2026-10-28")

print("Scheduled stop events:", len(test_timetable))
print("Unique trips:", test_timetable["trip_id"].nunique())

display(
    test_timetable.groupby("transport_mode")
    .agg(
        trips=("trip_id", "nunique"),
        stop_events=("stop_id", "count")
    )
)

Scheduled stop events: 59541
Unique trips: 3705


,trips,stop_events
transport_mode,,
Light Rail,563,7997
Metro,434,9114
Train,2708,42430


In [87]:
def get_station_departures(
    daily,
    origin,
    departure_time,
    window_minutes=30
):
    """
    Find scheduled departures from an origin station
    within the requested time window.

    Excludes trips that terminate at the origin.
    """

    # 1. Find the station ID
    origin_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == origin,
            "station_id"
        ]
    )

    if not origin_ids:
        raise ValueError(f"Station not found: {origin}")

    # 2. Convert departure time to seconds
    time_string = (
        departure_time
        if len(departure_time) == 8
        else departure_time + ":00"
    )

    requested_seconds = time_to_seconds(time_string)

    # 3. Find departures within the time window
    departures = daily[
        daily["station_id"].isin(origin_ids)
        &
        daily["departure_seconds"].between(
            requested_seconds,
            requested_seconds + window_minutes * 60
        )
    ].copy()

    # 4. Identify the final stop of each trip
    last_sequences = daily.groupby(
        "trip_id"
    )["stop_sequence"].max()

    departures["last_stop_sequence"] = (
        departures["trip_id"].map(last_sequences)
    )

    # 5. Exclude terminating services
    departures = departures[
        departures["stop_sequence"]
        < departures["last_stop_sequence"]
    ].copy()

    # 6. Sort departures
    return departures.sort_values(
        "departure_seconds"
    ).reset_index(drop=True)


# Test our function
test_departures = get_station_departures(
    daily=test_timetable,
    origin="Central Station",
    departure_time="17:30",
    window_minutes=30
)

print("Valid boarding opportunities:", len(test_departures))

display(
    test_departures[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "stop_id"
        ]
    ].head(15)
)

Valid boarding opportunities: 91


,route_short_name,transport_mode,departure_time,stop_id
0,T1,Train,17:30:01,2000336
1,M1,Metro,17:30:02,2000466
2,T4,Train,17:31:00,2000344
3,L2,Light Rail,17:31:35,2000447
4,T4,Train,17:32:00,2000345
5,T8,Train,17:32:00,2000343
6,T2,Train,17:32:00,2000339
7,T2,Train,17:32:01,2000341
8,T8,Train,17:32:01,2000337
9,M1,Metro,17:32:49,2000467


In [88]:
def find_direct_journeys(
    daily,
    origin,
    destination,
    departure_time,
    window_minutes=30
):
    """
    Find direct scheduled journeys between two stations.
    """

    # 1. Get valid origin departures
    departures = get_station_departures(
        daily=daily,
        origin=origin,
        departure_time=departure_time,
        window_minutes=window_minutes
    )

    # 2. Find destination station IDs
    destination_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == destination,
            "station_id"
        ]
    )

    if not destination_ids:
        raise ValueError(f"Destination not found: {destination}")

    # 3. Find destination arrivals
    arrivals = daily[
        daily["station_id"].isin(destination_ids)
    ][
        [
            "trip_id",
            "stop_sequence",
            "arrival_seconds",
            "arrival_time"
        ]
    ].rename(columns={
        "stop_sequence": "destination_sequence",
        "arrival_seconds": "destination_arrival_seconds",
        "arrival_time": "destination_arrival_time"
    })

    # 4. Match origin departures to destination arrivals
    journeys = departures.merge(
        arrivals,
        on="trip_id",
        how="inner"
    )

    # 5. Destination must be after origin
    journeys = journeys[
        journeys["destination_sequence"]
        > journeys["stop_sequence"]
    ].copy()

    # 6. Calculate journey duration
    journeys["journey_duration_min"] = (
        (
            journeys["destination_arrival_seconds"]
            - journeys["departure_seconds"]
        ) / 60
    ).round(2)

    # 7. Add useful output fields
    journeys["journey_type"] = "Direct"
    journeys["transfers"] = 0

    # 8. Remove duplicate boarding/arrival combinations
    journeys = journeys.drop_duplicates(
        subset=[
            "trip_id",
            "stop_id",
            "stop_sequence",
            "destination_sequence"
        ]
    )

    # 9. Rank by earliest arrival
    journeys = journeys.sort_values(
        [
            "destination_arrival_seconds",
            "departure_seconds"
        ]
    ).reset_index(drop=True)

    return journeys


# Test: Central → Macquarie University
direct_test = find_direct_journeys(
    daily=test_timetable,
    origin="Central Station",
    destination="Macquarie University Station",
    departure_time="17:30"
)

print("Direct journeys found:", len(direct_test))

display(
    direct_test[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "destination_arrival_time",
            "journey_duration_min"
        ]
    ].head(10)
)

Direct journeys found: 8


,route_short_name,transport_mode,departure_time,destination_arrival_time,journey_duration_min
0,M1,Metro,17:30:02,17:54:07,24.08
1,M1,Metro,17:34:02,17:58:07,24.08
2,M1,Metro,17:38:02,18:02:07,24.08
3,M1,Metro,17:42:02,18:06:07,24.08
4,M1,Metro,17:46:02,18:10:07,24.08
5,M1,Metro,17:50:02,18:14:07,24.08
6,M1,Metro,17:54:02,18:18:07,24.08
7,M1,Metro,17:58:02,18:22:07,24.08


In [89]:
def find_one_transfer_journeys(
    daily,
    origin,
    destination,
    departure_time,
    window_minutes=30,
    connection_window_minutes=30
):
    """Find scheduled journeys requiring one platform transfer."""

    departures = get_station_departures(
        daily, origin, departure_time, window_minutes
    ).copy()

    origin_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == origin, "station_id"
        ]
    )
    destination_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == destination, "station_id"
        ]
    )

    if not destination_ids:
        raise ValueError(f"Destination not found: {destination}")

    # Identify each first-leg boarding event
    departures["boarding_id"] = range(len(departures))

    first_boardings = departures[
        [
            "boarding_id", "trip_id", "stop_id",
            "stop_sequence", "departure_seconds",
            "departure_time", "route_short_name",
            "transport_mode"
        ]
    ].rename(columns={
        "stop_id": "origin_platform",
        "stop_sequence": "origin_sequence",
        "departure_seconds": "origin_departure_seconds",
        "departure_time": "origin_departure_time",
        "route_short_name": "first_line",
        "transport_mode": "first_mode"
    })

    # Find downstream stops on first-leg services
    first_stops = first_boardings.merge(
        daily[
            [
                "trip_id", "stop_id", "station_id",
                "stop_sequence", "arrival_seconds",
                "arrival_time"
            ]
        ],
        on="trip_id",
        how="inner"
    )

    first_stops = first_stops[
        (first_stops["stop_sequence"] > first_stops["origin_sequence"])
        & (~first_stops["station_id"].isin(origin_ids))
        & (~first_stops["station_id"].isin(destination_ids))
    ].copy()

    # Match first-leg arrival platforms to reachable platforms
    connections = first_stops.merge(
        all_platform_transfers[
            ["from_stop_id", "to_stop_id", "minimum_transfer_seconds"]
        ],
        left_on="stop_id",
        right_on="from_stop_id",
        how="inner"
    )

    connections["earliest_boarding_seconds"] = (
        connections["arrival_seconds"]
        + connections["minimum_transfer_seconds"]
    )

    # Find scheduled departures from connecting platforms
    second_departures = daily[
        [
            "trip_id", "stop_id", "stop_sequence",
            "departure_seconds", "departure_time",
            "route_short_name", "transport_mode"
        ]
    ].rename(columns={
        "trip_id": "second_trip_id",
        "stop_id": "second_platform",
        "stop_sequence": "second_boarding_sequence",
        "departure_seconds": "second_departure_seconds",
        "departure_time": "second_departure_time",
        "route_short_name": "second_line",
        "transport_mode": "second_mode"
    })

    connections = connections.merge(
        second_departures,
        left_on="to_stop_id",
        right_on="second_platform",
        how="inner"
    )

    connections = connections[
        (connections["second_trip_id"] != connections["trip_id"])
        & (
            connections["second_departure_seconds"]
            >= connections["earliest_boarding_seconds"]
        )
        & (
            connections["second_departure_seconds"]
            <= connections["arrival_seconds"]
               + connection_window_minutes * 60
        )
    ].copy()

    # Find destination arrivals on second-leg services
    destination_arrivals = daily[
        daily["station_id"].isin(destination_ids)
    ][
        ["trip_id", "stop_sequence", "arrival_seconds", "arrival_time"]
    ].rename(columns={
        "trip_id": "second_trip_id",
        "stop_sequence": "destination_sequence",
        "arrival_seconds": "destination_arrival_seconds",
        "arrival_time": "destination_arrival_time"
    })

    journeys = connections.merge(
        destination_arrivals,
        on="second_trip_id",
        how="inner"
    )

    journeys = journeys[
        journeys["destination_sequence"]
        > journeys["second_boarding_sequence"]
    ].copy()

    journeys["journey_duration_min"] = (
        (
            journeys["destination_arrival_seconds"]
            - journeys["origin_departure_seconds"]
        ) / 60
    ).round(2)

    journeys["journey_type"] = "One transfer"
    journeys["transfers"] = 1

    journeys = journeys.merge(
        station_lookup.rename(
            columns={"stop_name": "transfer_station"}
        ),
        on="station_id",
        how="left"
    )

    return journeys.sort_values(
        ["destination_arrival_seconds", "origin_departure_seconds"]
    ).reset_index(drop=True)


# Test the function
transfer_test = find_one_transfer_journeys(
    daily=test_timetable,
    origin="Central Station",
    destination="Macquarie University Station",
    departure_time="17:30"
)

print("One-transfer candidates:", len(transfer_test))

display(
    transfer_test[
        [
            "first_line",
            "origin_departure_time",
            "transfer_station",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "journey_duration_min"
        ]
    ].head(10)
)

One-transfer candidates: 1049


,first_line,origin_departure_time,transfer_station,second_line,second_departure_time,destination_arrival_time,journey_duration_min
0,M1,17:30:02,Gadigal Station,M1,17:36:01,17:58:07,28.08
1,M1,17:30:02,Martin Place Station,M1,17:37:59,17:58:07,28.08
2,M1,17:30:02,Barangaroo Station,M1,17:40:11,17:58:07,28.08
3,M1,17:30:02,Victoria Cross Station,M1,17:43:09,17:58:07,28.08
4,M1,17:30:02,Crows Nest Station,M1,17:45:16,17:58:07,28.08
5,M1,17:30:02,Chatswood Station,M1,17:49:04,17:58:07,28.08
6,M1,17:30:02,North Ryde Station,M1,17:54:26,17:58:07,28.08
7,M1,17:30:02,Macquarie Park Station,M1,17:56:36,17:58:07,28.08
8,M1,17:30:02,Gadigal Station,M1,17:40:01,18:02:07,32.08
9,M1,17:30:02,Martin Place Station,M1,17:41:59,18:02:07,32.08


In [90]:
def find_multimodal_journeys(
    origin,
    destination,
    travel_date,
    departure_time,
    window_minutes=30,
    max_transfers=1,
    limit=10
):
    """
    Find scheduled multimodal journeys with up to one transfer.

    Supports Train, Metro and Light Rail.
    """

    if max_transfers not in (0, 1):
        raise ValueError("Only 0 or 1 transfer is currently supported.")

    if origin == destination:
        raise ValueError("Origin and destination must be different.")

    daily = prepare_daily_timetable(travel_date)

    requested_time = (
        departure_time if len(departure_time) == 8
        else departure_time + ":00"
    )
    requested_seconds = time_to_seconds(requested_time)

    # 1. Find direct journeys
    direct = find_direct_journeys(
        daily, origin, destination,
        departure_time, window_minutes
    ).copy()

    direct["first_line"] = direct["route_short_name"]
    direct["second_line"] = None
    direct["transfer_station"] = None
    direct["origin_departure_time"] = direct["departure_time"]
    direct["origin_departure_seconds"] = direct["departure_seconds"]

    columns = [
        "journey_type", "first_line", "second_line",
        "origin_departure_time", "origin_departure_seconds",
        "transfer_station", "destination_arrival_time",
        "destination_arrival_seconds", "journey_duration_min",
        "transfers"
    ]

    results = [direct[columns]]

    if max_transfers == 1:

        # 2. Find feasible one-transfer journeys
        transfer = find_one_transfer_journeys(
            daily, origin, destination,
            departure_time, window_minutes
        ).copy()

        # 3. Check if staying on first service is better
        destination_ids = set(
            station_lookup.loc[
                station_lookup["stop_name"] == destination,
                "station_id"
            ]
        )

        first_trip_arrivals = daily[
            daily["station_id"].isin(destination_ids)
        ][
            ["trip_id", "stop_sequence", "arrival_seconds"]
        ].rename(columns={
            "stop_sequence": "first_destination_sequence",
            "arrival_seconds": "first_destination_arrival"
        })

        transfer = transfer.merge(
            first_trip_arrivals,
            on="trip_id",
            how="left"
        )

        unnecessary = (
            (
                transfer["first_destination_sequence"]
                > transfer["origin_sequence"]
            )
            &
            (
                transfer["first_destination_arrival"]
                <= transfer["destination_arrival_seconds"]
            )
        )

        transfer = transfer.loc[~unnecessary].copy()

        results.append(transfer[columns])

    # 4. Combine direct and transfer journeys
    candidates = pd.concat(
        results,
        ignore_index=True
    ).reset_index(drop=True)

    # 5. Remove dominated journeys
    keep = []

    for i, journey in candidates.iterrows():

        alternatives = candidates.drop(index=i)

        dominated = (
            (
                alternatives["origin_departure_seconds"]
                >= journey["origin_departure_seconds"]
            )
            &
            (
                alternatives["destination_arrival_seconds"]
                <= journey["destination_arrival_seconds"]
            )
            &
            (
                alternatives["transfers"]
                <= journey["transfers"]
            )
            &
            (
                (
                    alternatives["origin_departure_seconds"]
                    > journey["origin_departure_seconds"]
                )
                |
                (
                    alternatives["destination_arrival_seconds"]
                    < journey["destination_arrival_seconds"]
                )
                |
                (
                    alternatives["transfers"] < journey["transfers"]
                )
            )
        ).any()

        if not dominated:
            keep.append(i)

    journeys = candidates.loc[keep].copy()

    # 6. Calculate elapsed time from requested departure
    journeys["elapsed_from_request_min"] = (
        (
            journeys["destination_arrival_seconds"]
            - requested_seconds
        ) / 60
    ).round(2)

    # 7. Rank and limit results
    journeys = journeys.sort_values(
        [
            "destination_arrival_seconds",
            "transfers",
            "origin_departure_seconds"
        ],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    return journeys.head(limit)


# Test our complete journey planner
journey_results = find_multimodal_journeys(
    origin="Central Station",
    destination="Macquarie University Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=10
)

display(journey_results)

,journey_type,first_line,second_line,origin_departure_time,origin_departure_seconds,transfer_station,destination_arrival_time,destination_arrival_seconds,journey_duration_min,transfers,elapsed_from_request_min
0,Direct,M1,None,17:30:02,63002,None,17:54:07,64447,24.08,0,24.12
1,Direct,M1,None,17:34:02,63242,None,17:58:07,64687,24.08,0,28.12
2,Direct,M1,None,17:38:02,63482,None,18:02:07,64927,24.08,0,32.12
3,Direct,M1,None,17:42:02,63722,None,18:06:07,65167,24.08,0,36.12
4,Direct,M1,None,17:46:02,63962,None,18:10:07,65407,24.08,0,40.12
5,Direct,M1,None,17:50:02,64202,None,18:14:07,65647,24.08,0,44.12
6,Direct,M1,None,17:54:02,64442,None,18:18:07,65887,24.08,0,48.12
7,Direct,M1,None,17:58:02,64682,None,18:22:07,66127,24.08,0,52.12


In [91]:
# Test a journey requiring a transfer
journey_test_2 = find_multimodal_journeys(
    origin="Macquarie University Station",
    destination="Parramatta Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    window_minutes=30,
    max_transfers=1,
    limit=10
)

print("Journeys found:", len(journey_test_2))

display(
    journey_test_2[
        [
            "journey_type",
            "first_line",
            "second_line",
            "origin_departure_time",
            "transfer_station",
            "destination_arrival_time",
            "journey_duration_min",
            "transfers",
            "elapsed_from_request_min"
        ]
    ]
)

Journeys found: 7


,journey_type,first_line,second_line,origin_departure_time,transfer_station,destination_arrival_time,journey_duration_min,transfers,elapsed_from_request_min
0,One transfer,M1,T1,17:31:54,Central Station,18:29:00,57.1,1,59.0
1,One transfer,M1,T1,17:35:54,Central Station,18:32:00,56.1,1,62.0
2,One transfer,M1,T1,17:39:54,Central Station,18:38:00,58.1,1,68.0
3,One transfer,M1,T1,17:43:54,Central Station,18:41:00,57.1,1,71.0
4,One transfer,M1,T1,17:47:54,Central Station,18:44:00,56.1,1,74.0
5,One transfer,M1,T1,17:55:54,Central Station,18:56:00,60.1,1,86.0
6,One transfer,M1,T1,17:59:54,Central Station,18:59:00,59.1,1,89.0


In [92]:
# Get all feasible one-transfer candidates
epping_test = find_one_transfer_journeys(
    daily=test_timetable,
    origin="Macquarie University Station",
    destination="Parramatta Station",
    departure_time="17:30",
    window_minutes=30
)

print("Total transfer candidates:", len(epping_test))

# Count candidates by interchange station
display(
    epping_test.groupby("transfer_station")
    .size()
    .reset_index(name="journeys")
    .sort_values("journeys", ascending=False)
    .head(20)
)

# Specifically inspect Epping
epping_connections = epping_test[
    epping_test["transfer_station"] == "Epping Station"
].copy()

print("Journeys via Epping:", len(epping_connections))

display(
    epping_connections[
        [
            "first_line",
            "origin_departure_time",
            "arrival_time",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "minimum_transfer_seconds",
            "journey_duration_min"
        ]
    ].head(15)
)

Total transfer candidates: 57


,transfer_station,journeys
0,Central Station,57


Journeys via Epping: 0


,first_line,origin_departure_time,arrival_time,second_line,second_departure_time,destination_arrival_time,minimum_transfer_seconds,journey_duration_min


In [93]:
epping_direct = find_direct_journeys(
    daily=test_timetable,
    origin="Epping Station",
    destination="Parramatta Station",
    departure_time="17:30",
    window_minutes=90
)

print("Direct Epping → Parramatta journeys:", len(epping_direct))

display(
    epping_direct[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "destination_arrival_time",
            "journey_duration_min"
        ]
    ].head(15)
)

Direct Epping → Parramatta journeys: 0


,route_short_name,transport_mode,departure_time,destination_arrival_time,journey_duration_min


In [94]:
# Find train departures from Epping
epping_departures = get_station_departures(
    daily=test_timetable,
    origin="Epping Station",
    departure_time="17:30",
    window_minutes=90
)

epping_trains = epping_departures[
    epping_departures["transport_mode"] == "Train"
].copy()

# Find downstream stations on those trains
reachable_stops = epping_trains[
    ["trip_id", "stop_sequence"]
].merge(
    test_timetable[
        ["trip_id", "station_id", "stop_sequence", "arrival_time"]
    ],
    on="trip_id",
    how="inner",
    suffixes=("_boarding", "_destination")
)

reachable_stops = reachable_stops[
    reachable_stops["stop_sequence_destination"]
    > reachable_stops["stop_sequence_boarding"]
]

# Attach station names
reachable_stops = reachable_stops.merge(
    station_lookup,
    on="station_id",
    how="left"
)

print("Train departures from Epping:", len(epping_trains))
print(
    "Unique downstream stations:",
    reachable_stops["station_id"].nunique()
)

display(
    reachable_stops[
        ["stop_name", "arrival_time"]
    ].drop_duplicates()
    .sort_values("stop_name")
    .head(40)
)

Train departures from Epping: 15
Unique downstream stations: 17


,stop_name,arrival_time
1,Beecroft Station,17:40:42
24,Beecroft Station,17:55:42
47,Beecroft Station,18:10:42
70,Beecroft Station,18:25:42
87,Beecroft Station,18:40:42
104,Beecroft Station,18:55:42
10,Burwood Station,17:56:00
20,Burwood Station,18:06:18
33,Burwood Station,18:11:00
43,Burwood Station,18:21:18


In [95]:
burwood_direct = find_direct_journeys(
    daily=test_timetable,
    origin="Burwood Station",
    destination="Parramatta Station",
    departure_time="17:30",
    window_minutes=120
)

print(
    "Direct Burwood → Parramatta journeys:",
    len(burwood_direct)
)

display(
    burwood_direct[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "destination_arrival_time",
            "journey_duration_min"
        ]
    ].head(15)
)

Direct Burwood → Parramatta journeys: 8


,route_short_name,transport_mode,departure_time,destination_arrival_time,journey_duration_min
0,T2,Train,17:43:18,18:06:30,23.2
1,T2,Train,17:58:18,18:21:30,23.2
2,T2,Train,18:13:18,18:36:30,23.2
3,T2,Train,18:28:18,18:51:30,23.2
4,T2,Train,18:43:18,19:06:30,23.2
5,T2,Train,18:57:12,19:20:30,23.3
6,T2,Train,19:12:12,19:35:30,23.3
7,T2,Train,19:27:12,19:50:30,23.3


In [96]:
# Find Metro → Train journeys via Epping to Burwood
burwood_connections = find_one_transfer_journeys(
    daily=test_timetable,
    origin="Macquarie University Station",
    destination="Burwood Station",
    departure_time="17:30",
    window_minutes=30
)

# Keep only journeys transferring at Epping
via_epping = burwood_connections[
    burwood_connections["transfer_station"] == "Epping Station"
].copy()

# Sort by arrival at Burwood
via_epping = via_epping.sort_values(
    "destination_arrival_seconds"
)

print("Macquarie University → Epping → Burwood:", len(via_epping))

display(
    via_epping[
        [
            "first_line",
            "origin_departure_time",
            "transfer_station",
            "arrival_time",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "journey_duration_min"
        ]
    ].head(15)
)

Macquarie University → Epping → Burwood: 23


,first_line,origin_departure_time,transfer_station,arrival_time,second_line,second_departure_time,destination_arrival_time,journey_duration_min
0,M1,17:30:37,Epping Station,17:33:51,T9,17:42:00,18:06:18,35.68
1,M1,17:30:37,Epping Station,17:33:51,T9,17:53:01,18:11:00,40.38
2,M1,17:34:37,Epping Station,17:37:51,T9,17:53:01,18:11:00,36.38
3,M1,17:38:37,Epping Station,17:41:51,T9,17:53:01,18:11:00,32.38
4,M1,17:42:37,Epping Station,17:45:51,T9,17:53:01,18:11:00,28.38
7,M1,17:30:37,Epping Station,17:33:51,T9,17:57:00,18:21:18,50.68
8,M1,17:34:37,Epping Station,17:37:51,T9,17:57:00,18:21:18,46.68
9,M1,17:38:37,Epping Station,17:41:51,T9,17:57:00,18:21:18,42.68
10,M1,17:42:37,Epping Station,17:45:51,T9,17:57:00,18:21:18,38.68
11,M1,17:46:37,Epping Station,17:49:51,T9,17:57:00,18:21:18,34.68


In [97]:
print("Columns in via_epping:")
print(via_epping.columns.tolist())

print("\nColumns in burwood_connections:")
print(burwood_connections.columns.tolist())

print("\nTransfer-related columns:")
print([
    col for col in burwood_connections.columns
    if "stop_id" in col
    or "transfer" in col
    or "platform" in col
])

Columns in via_epping:
['boarding_id', 'trip_id', 'origin_platform', 'origin_sequence', 'origin_departure_seconds', 'origin_departure_time', 'first_line', 'first_mode', 'stop_id', 'station_id', 'stop_sequence', 'arrival_seconds', 'arrival_time', 'from_stop_id', 'to_stop_id', 'minimum_transfer_seconds', 'earliest_boarding_seconds', 'second_trip_id', 'second_platform', 'second_boarding_sequence', 'second_departure_seconds', 'second_departure_time', 'second_line', 'second_mode', 'destination_sequence', 'destination_arrival_seconds', 'destination_arrival_time', 'journey_duration_min', 'journey_type', 'transfers', 'transfer_station']

Columns in burwood_connections:
['boarding_id', 'trip_id', 'origin_platform', 'origin_sequence', 'origin_departure_seconds', 'origin_departure_time', 'first_line', 'first_mode', 'stop_id', 'station_id', 'stop_sequence', 'arrival_seconds', 'arrival_time', 'from_stop_id', 'to_stop_id', 'minimum_transfer_seconds', 'earliest_boarding_seconds', 'second_trip_id', 's

In [98]:
# Rename columns to make both transfers explicit
burwood_connections = burwood_connections.rename(columns={
    "from_stop_id_x": "epping_from_platform",
    "to_stop_id_x": "epping_to_platform",
    "minimum_transfer_seconds_x": "epping_transfer_seconds",
    "from_stop_id_y": "burwood_from_platform",
    "to_stop_id_y": "burwood_to_platform",
    "minimum_transfer_seconds_y": "burwood_transfer_seconds"
})

# Match T2 departures at the correct Burwood platform
burwood_connections = burwood_connections.merge(
    burwood_t2_departures,
    left_on="burwood_to_platform",
    right_on="third_platform",
    how="inner"
)

# Apply minimum transfer time at Burwood
burwood_connections = burwood_connections[
    burwood_connections["third_departure_seconds"]
    >= (
        burwood_connections["burwood_arrival_seconds"]
        + burwood_connections["burwood_transfer_seconds"]
    )
].copy()

# Match the third train to its arrival at Parramatta
complete_two_transfer = burwood_connections.merge(
    parramatta_arrivals,
    on="third_trip_id",
    how="inner"
)

# Ensure Parramatta comes after Burwood on the third train
complete_two_transfer = complete_two_transfer[
    complete_two_transfer["final_sequence"]
    > complete_two_transfer["third_boarding_sequence"]
].copy()

# Calculate total journey duration
complete_two_transfer["total_minutes"] = (
    complete_two_transfer["final_arrival_seconds"]
    - complete_two_transfer["origin_departure_seconds"]
) / 60

print("Feasible two-transfer journeys:", len(complete_two_transfer))

display(
    complete_two_transfer[
        [
            "origin_departure_time",
            "second_departure_time",
            "destination_arrival_time",
            "third_departure_time",
            "final_arrival_time",
            "burwood_transfer_seconds",
            "total_minutes"
        ]
    ].sort_values(
        "final_arrival_time"
    ).head(15)
)

NameError: name 'burwood_t2_departures' is not defined

In [ ]:
# Identify Parramatta Station
parramatta_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == "Parramatta Station",
        "station_id"
    ]
)

# Get all scheduled arrivals at Parramatta
parramatta_arrivals = test_timetable[
    test_timetable["station_id"].isin(parramatta_ids)
][
    ["trip_id", "stop_sequence", "arrival_seconds", "arrival_time"]
].rename(columns={
    "trip_id": "third_trip_id",
    "stop_sequence": "final_sequence",
    "arrival_seconds": "final_arrival_seconds",
    "arrival_time": "final_arrival_time"
})

print("Parramatta arrival records:", len(parramatta_arrivals))
print("Columns:", parramatta_arrivals.columns.tolist())

display(parramatta_arrivals.head())

Parramatta arrival records: 484
Columns: ['third_trip_id', 'final_sequence', 'final_arrival_seconds', 'final_arrival_time']


,third_trip_id,final_sequence,final_arrival_seconds,final_arrival_time
9114,1--B.1324.165.116.B.8.92000948,1,20611,05:43:31
9195,1--E.1324.165.116.B.8.92003215,26,33630,09:20:30
9196,1--F.1324.165.116.B.8.92003214,1,34111,09:28:31
9277,1--J.1324.165.116.B.8.92000805,26,47130,13:05:30
9278,1--K.1324.165.116.B.8.92001138,1,47611,13:13:31


In [ ]:
# Match Burwood T2 departures to Parramatta arrivals
complete_two_transfer = burwood_connections.merge(
    parramatta_arrivals,
    on="third_trip_id",
    how="inner"
)

# Ensure Parramatta is AFTER Burwood on the same train
complete_two_transfer = complete_two_transfer[
    complete_two_transfer["final_sequence"]
    > complete_two_transfer["third_boarding_sequence"]
].copy()

# Calculate total journey duration
complete_two_transfer["total_minutes"] = (
    complete_two_transfer["final_arrival_seconds"]
    - complete_two_transfer["origin_departure_seconds"]
) / 60

# Show the results
print("Feasible two-transfer journeys:", len(complete_two_transfer))

display(
    complete_two_transfer[
        [
            "origin_departure_time",
            "second_departure_time",
            "destination_arrival_time",
            "third_departure_time",
            "final_arrival_time",
            "burwood_transfer_seconds",
            "total_minutes"
        ]
    ]
    .sort_values("final_arrival_time")
    .head(15)
)

Feasible two-transfer journeys: 283


,origin_departure_time,second_departure_time,destination_arrival_time,third_departure_time,final_arrival_time,burwood_transfer_seconds,total_minutes
21,17:30:37,17:42:00,18:06:18,18:13:18,18:36:30,360.0,65.883333
26,17:30:37,17:42:00,18:06:18,18:28:18,18:51:30,360.0,80.883333
53,17:30:37,17:53:01,18:11:00,18:28:18,18:51:30,390.0,80.883333
80,17:34:37,17:53:01,18:11:00,18:28:18,18:51:30,390.0,76.883333
107,17:38:37,17:53:01,18:11:00,18:28:18,18:51:30,390.0,72.883333
134,17:42:37,17:53:01,18:11:00,18:28:18,18:51:30,390.0,68.883333
160,17:30:37,17:57:00,18:21:18,18:28:18,18:51:30,360.0,80.883333
186,17:34:37,17:57:00,18:21:18,18:28:18,18:51:30,360.0,76.883333
212,17:38:37,17:57:00,18:21:18,18:28:18,18:51:30,360.0,72.883333
238,17:42:37,17:57:00,18:21:18,18:28:18,18:51:30,360.0,68.883333


In [ ]:
# Rebuild Central options
central_options = journey_test_2[
    [
        "first_line",
        "second_line",
        "transfer_station",
        "origin_departure_time",
        "destination_arrival_time",
        "destination_arrival_seconds",
        "journey_duration_min",
        "transfers"
    ]
].copy()

central_options["route"] = (
    central_options["first_line"].astype(str)
    + " → "
    + central_options["second_line"].astype(str)
)

central_options["transfer_path"] = central_options["transfer_station"]


# Rebuild two-transfer options without renaming existing columns
two_transfer_options = complete_two_transfer[
    [
        "origin_departure_time",
        "final_arrival_time",
        "final_arrival_seconds",
        "total_minutes"
    ]
].copy()

two_transfer_options["route"] = "M1 → T9 → T2"
two_transfer_options["transfer_path"] = "Epping → Burwood"
two_transfer_options["transfers"] = 2

two_transfer_options["destination_arrival_time"] = (
    two_transfer_options["final_arrival_time"]
)

two_transfer_options["destination_arrival_seconds"] = (
    two_transfer_options["final_arrival_seconds"]
)

two_transfer_options["journey_duration_min"] = (
    two_transfer_options["total_minutes"]
)


# Select exactly the same columns from both DataFrames
comparison_columns = [
    "route",
    "transfer_path",
    "origin_departure_time",
    "destination_arrival_time",
    "destination_arrival_seconds",
    "journey_duration_min",
    "transfers"
]

comparison = pd.concat(
    [
        central_options[comparison_columns],
        two_transfer_options[comparison_columns]
    ],
    ignore_index=True
)

comparison = comparison.sort_values(
    ["destination_arrival_seconds", "transfers"]
)

print("Total journey combinations:", len(comparison))

display(
    comparison[
        [
            "route",
            "transfer_path",
            "origin_departure_time",
            "destination_arrival_time",
            "journey_duration_min",
            "transfers"
        ]
    ].head(15)
)

Total journey combinations: 290


,route,transfer_path,origin_departure_time,destination_arrival_time,journey_duration_min,transfers
0,M1 → T1,Central Station,17:31:54,18:29:00,57.100000,1
1,M1 → T1,Central Station,17:35:54,18:32:00,56.100000,1
14,M1 → T9 → T2,Epping → Burwood,17:30:37,18:36:30,65.883333,2
2,M1 → T1,Central Station,17:39:54,18:38:00,58.100000,1
3,M1 → T1,Central Station,17:43:54,18:41:00,57.100000,1
4,M1 → T1,Central Station,17:47:54,18:44:00,56.100000,1
19,M1 → T9 → T2,Epping → Burwood,17:30:37,18:51:30,80.883333,2
32,M1 → T9 → T2,Epping → Burwood,17:30:37,18:51:30,80.883333,2
45,M1 → T9 → T2,Epping → Burwood,17:34:37,18:51:30,76.883333,2
58,M1 → T9 → T2,Epping → Burwood,17:38:37,18:51:30,72.883333,2


In [ ]:
# Start with the combined comparison table
ranked = comparison.copy()

# Convert departure times into seconds
ranked["origin_departure_seconds"] = (
    ranked["origin_departure_time"].apply(time_to_seconds)
)

# Remove exact duplicates in the displayed journey information
ranked = ranked.drop_duplicates(
    subset=[
        "route",
        "transfer_path",
        "origin_departure_seconds",
        "destination_arrival_seconds",
        "transfers"
    ]
).reset_index(drop=True)

# Identify Pareto-dominated journeys
departures = ranked["origin_departure_seconds"].to_numpy()
arrivals = ranked["destination_arrival_seconds"].to_numpy()
transfers = ranked["transfers"].to_numpy()

keep = np.ones(len(ranked), dtype=bool)

for i in range(len(ranked)):
    better_or_equal = (
        (departures >= departures[i])
        & (arrivals <= arrivals[i])
        & (transfers <= transfers[i])
    )

    strictly_better = (
        (departures > departures[i])
        | (arrivals < arrivals[i])
        | (transfers < transfers[i])
    )

    better_or_equal[i] = False

    if np.any(better_or_equal & strictly_better):
        keep[i] = False

ranked = ranked.loc[keep].sort_values(
    ["destination_arrival_seconds", "transfers"]
)

print("Original combinations:", len(comparison))
print("Non-dominated journeys:", len(ranked))

display(
    ranked[
        [
            "route",
            "transfer_path",
            "origin_departure_time",
            "destination_arrival_time",
            "journey_duration_min",
            "transfers"
        ]
    ].head(15)
)

Original combinations: 290
Non-dominated journeys: 7


,route,transfer_path,origin_departure_time,destination_arrival_time,journey_duration_min,transfers
0,M1 → T1,Central Station,17:31:54,18:29:00,57.1,1
1,M1 → T1,Central Station,17:35:54,18:32:00,56.1,1
3,M1 → T1,Central Station,17:39:54,18:38:00,58.1,1
4,M1 → T1,Central Station,17:43:54,18:41:00,57.1,1
5,M1 → T1,Central Station,17:47:54,18:44:00,56.1,1
11,M1 → T1,Central Station,17:55:54,18:56:00,60.1,1
12,M1 → T1,Central Station,17:59:54,18:59:00,59.1,1


In [ ]:
def get_valid_connections(
    daily,
    arrival_platform,
    arrival_seconds,
    connection_window_minutes=30
):
    """
    Find valid onward boarding events from an arrival platform.

    Applies platform transfer times and a maximum waiting window.
    """

    # Find reachable boarding platforms
    transfers = all_platform_transfers[
        all_platform_transfers["from_stop_id"] == arrival_platform
    ][
        ["to_stop_id", "minimum_transfer_seconds"]
    ].copy()

    if transfers.empty:
        return pd.DataFrame()

    # Get scheduled departures at reachable platforms
    departures = daily.merge(
        transfers,
        left_on="stop_id",
        right_on="to_stop_id",
        how="inner"
    )

    # Earliest departure after completing the transfer
    departures["earliest_departure"] = (
        arrival_seconds
        + departures["minimum_transfer_seconds"]
    )

    # Keep only feasible departures
    departures = departures[
        (
            departures["departure_seconds"]
            >= departures["earliest_departure"]
        )
        & (
            departures["departure_seconds"]
            <= arrival_seconds + connection_window_minutes * 60
        )
    ].copy()

    return departures.sort_values("departure_seconds")

In [ ]:
# Find T2 boarding events at Burwood
burwood_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == "Burwood Station",
        "station_id"
    ]
)

burwood_t2_departures = test_timetable[
    (test_timetable["station_id"].isin(burwood_ids))
    & (test_timetable["route_short_name"] == "T2")
][
    [
        "trip_id", "stop_id", "stop_sequence",
        "departure_seconds", "departure_time"
    ]
].rename(columns={
    "trip_id": "third_trip_id",
    "stop_id": "third_platform",
    "stop_sequence": "third_boarding_sequence",
    "departure_seconds": "third_departure_seconds",
    "departure_time": "third_departure_time"
})

# Get the T9 arrival platform at Burwood
burwood_t9_arrivals = test_timetable[
    test_timetable["station_id"].isin(burwood_ids)
][
    ["trip_id", "stop_id", "stop_sequence", "arrival_seconds"]
].rename(columns={
    "trip_id": "second_trip_id",
    "stop_id": "burwood_arrival_platform",
    "stop_sequence": "burwood_arrival_sequence",
    "arrival_seconds": "burwood_arrival_seconds"
})

# Match each second-leg trip to its actual arrival platform
second_leg_arrivals = via_epping.merge(
    burwood_t9_arrivals,
    on="second_trip_id",
    how="inner"
)

second_leg_arrivals = second_leg_arrivals[
    second_leg_arrivals["burwood_arrival_sequence"]
    > second_leg_arrivals["second_boarding_sequence"]
].copy()

# Find valid platform-to-platform transfers
burwood_connections = second_leg_arrivals.merge(
    all_platform_transfers[
        ["from_stop_id", "to_stop_id", "minimum_transfer_seconds"]
    ],
    left_on="burwood_arrival_platform",
    right_on="from_stop_id",
    how="inner"
)

# Match T2 departures on reachable platforms
burwood_connections = burwood_connections.merge(
    burwood_t2_departures,
    left_on="to_stop_id",
    right_on="third_platform",
    how="inner"
)

# Check transfer timing
burwood_connections = burwood_connections[
    burwood_connections["third_departure_seconds"]
    >= (
        burwood_connections["burwood_arrival_seconds"]
        + burwood_connections["minimum_transfer_seconds"]
    )
].copy()

# Find T2 arrivals at Parramatta
parramatta_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == "Parramatta Station",
        "station_id"
    ]
)

parramatta_arrivals = test_timetable[
    test_timetable["station_id"].isin(parramatta_ids)
][
    ["trip_id", "stop_sequence", "arrival_seconds", "arrival_time"]
].rename(columns={
    "trip_id": "third_trip_id",
    "stop_sequence": "final_sequence",
    "arrival_seconds": "final_arrival_seconds",
    "arrival_time": "final_arrival_time"
})

complete_two_transfer = burwood_connections.merge(
    parramatta_arrivals,
    on="third_trip_id",
    how="inner"
)

complete_two_transfer = complete_two_transfer[
    complete_two_transfer["final_sequence"]
    > complete_two_transfer["third_boarding_sequence"]
].copy()

complete_two_transfer["total_minutes"] = (
    (
        complete_two_transfer["final_arrival_seconds"]
        - complete_two_transfer["origin_departure_seconds"]
    ) / 60
).round(2)

print("Feasible two-transfer journeys:", len(complete_two_transfer))

display(
    complete_two_transfer[
        [
            "origin_departure_time",
            "second_departure_time",
            "destination_arrival_time",
            "third_departure_time",
            "final_arrival_time",
            "minimum_transfer_seconds",
            "total_minutes"
        ]
    ].sort_values("final_arrival_time").head(15)
)

KeyError: 'to_stop_id'

In [ ]:
# Select our earliest validated two-transfer journey
test_journey = complete_two_transfer.sort_values(
    "final_arrival_seconds"
).iloc[0]

# Get the arrival platform and time at Burwood
arrival_platform = test_journey["burwood_arrival_platform"]
arrival_seconds = test_journey["burwood_arrival_seconds"]

# Find valid onward connections
connections = get_valid_connections(
    daily=test_timetable,
    arrival_platform=arrival_platform,
    arrival_seconds=arrival_seconds,
    connection_window_minutes=30
)

print("Valid onward boarding events:", len(connections))

display(
    connections[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "stop_id",
            "minimum_transfer_seconds"
        ]
    ].head(20)
)

Valid onward boarding events: 15


,route_short_name,transport_mode,departure_time,stop_id,minimum_transfer_seconds
429,T2,Train,18:13:18,213496,360.0
140,T9,Train,18:17:00,213492,300.0
13,T2,Train,18:17:06,213496,360.0
436,T2,Train,18:18:36,213495,330.0
365,T3,Train,18:20:30,213496,360.0
201,T9,Train,18:21:48,213493,120.0
383,T2,Train,18:22:30,213495,330.0
362,T2,Train,18:24:18,213496,360.0
22,T9,Train,18:26:30,213491,330.0
542,T2,Train,18:28:18,213496,360.0


In [99]:
# STEP 55 — Find which onward connections reach Parramatta

# 1. Identify Parramatta Station
parramatta_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == "Parramatta Station",
        "station_id"
    ]
)

# 2. Get scheduled arrivals at Parramatta
parramatta_stops = test_timetable[
    test_timetable["station_id"].isin(parramatta_ids)
][
    [
        "trip_id",
        "stop_sequence",
        "arrival_time",
        "arrival_seconds"
    ]
].copy()

# Rename columns BEFORE merging to avoid naming collisions
parramatta_stops = parramatta_stops.rename(columns={
    "stop_sequence": "destination_sequence",
    "arrival_time": "destination_arrival_time",
    "arrival_seconds": "destination_arrival_seconds"
})

# 3. Prepare valid onward boarding events
boarding_connections = connections[
    [
        "trip_id",
        "route_short_name",
        "transport_mode",
        "stop_id",
        "stop_sequence",
        "departure_time",
        "departure_seconds",
        "minimum_transfer_seconds"
    ]
].copy()

boarding_connections = boarding_connections.rename(columns={
    "stop_id": "boarding_platform",
    "stop_sequence": "boarding_sequence"
})

# 4. Match boarding events to Parramatta arrivals
onward_journeys = boarding_connections.merge(
    parramatta_stops,
    on="trip_id",
    how="inner",
    validate="many_to_many"
)

# 5. Ensure Parramatta comes AFTER the boarding station
onward_journeys = onward_journeys[
    onward_journeys["destination_sequence"]
    > onward_journeys["boarding_sequence"]
].copy()

# 6. Calculate onward travel duration
onward_journeys["onward_duration_min"] = (
    onward_journeys["destination_arrival_seconds"]
    - onward_journeys["departure_seconds"]
) / 60

# 7. Sort by earliest arrival
onward_journeys = onward_journeys.sort_values(
    "destination_arrival_seconds"
)

# 8. Display results
print("Valid onward boarding events:", len(connections))
print("Connections reaching Parramatta:", len(onward_journeys))

display(
    onward_journeys[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "destination_arrival_time",
            "onward_duration_min",
            "minimum_transfer_seconds"
        ]
    ].head(20)
)

TypeError: list indices must be integers or slices, not list

In [100]:
def find_onward_journeys(
    daily,
    arrival_platform,
    arrival_seconds,
    destination,
    connection_window_minutes=30,
    exclude_trip_ids=None
):
    """
    Find journeys from an arrival platform to a destination
    using one additional scheduled service.
    """

    # 1. Find feasible onward boarding events
    connections = get_valid_connections(
        daily=daily,
        arrival_platform=arrival_platform,
        arrival_seconds=arrival_seconds,
        connection_window_minutes=connection_window_minutes
    )

    if connections.empty:
        return pd.DataFrame()

    # 2. Exclude trips already used in the journey
    if exclude_trip_ids is not None:
        connections = connections[
            ~connections["trip_id"].isin(exclude_trip_ids)
        ].copy()

    if connections.empty:
        return pd.DataFrame()

    # 3. Find destination station IDs
    destination_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == destination,
            "station_id"
        ]
    )

    if not destination_ids:
        raise ValueError(f"Destination not found: {destination}")

    # 4. Get destination arrival events
    destination_stops = daily[
        daily["station_id"].isin(destination_ids)
    ][
        [
            "trip_id",
            "stop_sequence",
            "arrival_time",
            "arrival_seconds"
        ]
    ].rename(columns={
        "stop_sequence": "destination_sequence",
        "arrival_time": "destination_arrival_time",
        "arrival_seconds": "destination_arrival_seconds"
    })

    # 5. Prepare boarding events with explicit names
    boarding = connections[
        [
            "trip_id",
            "route_short_name",
            "transport_mode",
            "stop_id",
            "stop_sequence",
            "departure_time",
            "departure_seconds",
            "minimum_transfer_seconds"
        ]
    ].rename(columns={
        "stop_id": "boarding_platform",
        "stop_sequence": "boarding_sequence"
    })

    # 6. Match boarding trips to destination arrivals
    journeys = boarding.merge(
        destination_stops,
        on="trip_id",
        how="inner"
    )

    # 7. Ensure the destination occurs after boarding
    journeys = journeys[
        journeys["destination_sequence"]
        > journeys["boarding_sequence"]
    ].copy()

    # 8. Calculate travel duration
    journeys["onward_duration_min"] = (
        journeys["destination_arrival_seconds"]
        - journeys["departure_seconds"]
    ) / 60

    return journeys.sort_values(
        "destination_arrival_seconds"
    ).reset_index(drop=True)

In [101]:
# STEP 57 — Test reusable onward journey function

test_journey = complete_two_transfer.sort_values(
    "final_arrival_seconds"
).iloc[0]

# Find onward journeys from Burwood to Parramatta
test_onward = find_onward_journeys(
    daily=test_timetable,
    arrival_platform=test_journey["burwood_arrival_platform"],
    arrival_seconds=test_journey["burwood_arrival_seconds"],
    destination="Parramatta Station",
    connection_window_minutes=30,
    exclude_trip_ids=[
        test_journey["trip_id"],
        test_journey["second_trip_id"]
    ]
)

print("Valid onward journeys:", len(test_onward))

display(
    test_onward[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "destination_arrival_time",
            "onward_duration_min",
            "minimum_transfer_seconds"
        ]
    ]
)

NameError: name 'complete_two_transfer' is not defined

In [ ]:
# STEP 58 — Discover possible second interchange stations

# 1. Select the earliest validated two-transfer journey
test_journey = complete_two_transfer.sort_values(
    "final_arrival_seconds"
).iloc[0]

# 2. Identify the second train (T9)
second_trip_id = test_journey["second_trip_id"]
boarding_sequence = test_journey["second_boarding_sequence"]

# 3. Find all downstream stops on the second train
downstream_stops = test_timetable[
    (test_timetable["trip_id"] == second_trip_id)
    & (test_timetable["stop_sequence"] > boarding_sequence)
].copy()

# 4. Prepare a clean station lookup
# Rename BEFORE merging to avoid column naming conflicts
station_names = station_lookup[
    ["station_id", "stop_name"]
].drop_duplicates(
    subset=["station_id"]
).rename(columns={
    "stop_name": "interchange_station"
})

# 5. Attach official station names
downstream_stops = downstream_stops.merge(
    station_names,
    on="station_id",
    how="left",
    validate="many_to_one"
)

# 6. Sort stops by arrival time
downstream_stops = downstream_stops.sort_values(
    "arrival_seconds"
).reset_index(drop=True)

# 7. Display results
print("Second train:", second_trip_id)
print("Possible downstream interchange stops:", len(downstream_stops))

display(
    downstream_stops[
        [
            "interchange_station",
            "arrival_time",
            "stop_id",
            "arrival_seconds"
        ]
    ]
)

Second train: 125R.1324.165.16.A.8.92001037
Possible downstream interchange stops: 11


,interchange_station,arrival_time,stop_id,arrival_seconds
0,Eastwood Station,17:45:00,2122231,63900
1,Denistone Station,17:47:42,2114171,64062
2,West Ryde Station,17:49:30,2114191,64170
3,Meadowbank Station,17:51:42,2114181,64302
4,Rhodes Station,17:53:48,213891,64428
5,Concord West Station,17:56:30,213881,64590
6,North Strathfield Station,17:58:30,2137171,64710
7,Strathfield Station,18:03:00,2135235,64980
8,Burwood Station,18:06:18,213493,65178
9,Redfern Station,18:15:42,2015133,65742


In [ ]:
# STEP 59 — Find onward journeys from every downstream station

all_onward_options = []

# Trips already used in the journey
used_trips = [
    test_journey["trip_id"],
    test_journey["second_trip_id"]
]

for _, stop in downstream_stops.iterrows():

    # Find valid onward journeys to Parramatta
    onward = find_onward_journeys(
        daily=test_timetable,
        arrival_platform=stop["stop_id"],
        arrival_seconds=stop["arrival_seconds"],
        destination="Parramatta Station",
        connection_window_minutes=30,
        exclude_trip_ids=used_trips
    )

    if onward.empty:
        continue

    # Record the interchange station
    onward["second_interchange"] = stop["interchange_station"]

    # Record arrival time at the interchange
    onward["interchange_arrival_time"] = stop["arrival_time"]

    # Calculate total journey duration from Macquarie University
    onward["total_journey_min"] = (
        onward["destination_arrival_seconds"]
        - test_journey["origin_departure_seconds"]
    ) / 60

    all_onward_options.append(onward)

# Combine results
if all_onward_options:

    discovered_routes = pd.concat(
        all_onward_options,
        ignore_index=True
    )

    discovered_routes = discovered_routes.sort_values(
        "destination_arrival_seconds"
    ).reset_index(drop=True)

    print(
        "Total feasible onward connections:",
        len(discovered_routes)
    )

    print(
        "Interchange stations with valid connections:",
        discovered_routes["second_interchange"].nunique()
    )

    display(
        discovered_routes[
            [
                "second_interchange",
                "interchange_arrival_time",
                "route_short_name",
                "departure_time",
                "destination_arrival_time",
                "total_journey_min",
                "minimum_transfer_seconds"
            ]
        ].head(20)
    )

else:
    discovered_routes = pd.DataFrame()
    print("No feasible onward connections found.")

Total feasible onward connections: 21
Interchange stations with valid connections: 4


,second_interchange,interchange_arrival_time,route_short_name,departure_time,destination_arrival_time,total_journey_min,minimum_transfer_seconds
0,Strathfield Station,18:03:00,T1,18:11:24,18:23:00,52.383333,300.0
1,Strathfield Station,18:03:00,T1,18:17:24,18:29:00,58.383333,300.0
2,Strathfield Station,18:03:00,T1,18:20:24,18:32:00,61.383333,300.0
3,Strathfield Station,18:03:00,T2,18:16:00,18:36:30,65.883333,384.0
4,Burwood Station,18:06:18,T2,18:13:18,18:36:30,65.883333,360.0
5,Strathfield Station,18:03:00,T1,18:26:24,18:38:00,67.383333,300.0
6,Strathfield Station,18:03:00,T1,18:29:24,18:41:00,70.383333,300.0
7,Redfern Station,18:15:42,T1,18:21:36,18:44:00,73.383333,312.0
8,Strathfield Station,18:03:00,T1,18:32:24,18:44:00,73.383333,300.0
9,Burwood Station,18:06:18,T2,18:28:18,18:51:30,80.883333,360.0


In [ ]:
# STEP 60 — Validate the fastest discovered two-transfer journey

fastest = discovered_routes.sort_values(
    "destination_arrival_seconds"
).iloc[0]

# Confirm that the third train is different from the first two
distinct_trips = (
    fastest["trip_id"] not in [
        test_journey["trip_id"],
        test_journey["second_trip_id"]
    ]
)

# Check first transfer: Epping
epping_transfer_valid = (
    test_journey["second_departure_seconds"]
    >= (
        test_journey["arrival_seconds"]
        + test_journey["epping_transfer_seconds"]
    )
)

# Check second transfer: Strathfield
strathfield_transfer_valid = (
    fastest["departure_seconds"]
    >= (
        fastest["interchange_arrival_time"]
        and time_to_seconds(fastest["interchange_arrival_time"])
        + fastest["minimum_transfer_seconds"]
    )
)

# Confirm arrival is after final departure
destination_valid = (
    fastest["destination_arrival_seconds"]
    > fastest["departure_seconds"]
)

print("Three distinct trips:", distinct_trips)
print("Epping transfer valid:", epping_transfer_valid)
print("Strathfield transfer valid:", strathfield_transfer_valid)
print("Destination timing valid:", destination_valid)

print("\nFastest discovered journey:")
print("Second interchange:", fastest["second_interchange"])
print("Final train:", fastest["route_short_name"])
print("Final departure:", fastest["departure_time"])
print("Parramatta arrival:", fastest["destination_arrival_time"])
print("Total duration (minutes):", round(fastest["total_journey_min"], 2))

Three distinct trips: True
Epping transfer valid: True
Strathfield transfer valid: True
Destination timing valid: True

Fastest discovered journey:
Second interchange: Strathfield Station
Final train: T1
Final departure: 18:11:24
Parramatta arrival: 18:23:00
Total duration (minutes): 52.38


In [ ]:
# STEP 61 — Reusable downstream stop discovery

def get_downstream_stops(daily, trip_id, boarding_sequence):
    """
    Return all stops visited after boarding a given trip.

    These stops can be potential destinations or interchanges.
    """

    downstream = daily.loc[
        (daily["trip_id"] == trip_id)
        & (daily["stop_sequence"] > boarding_sequence),
        [
            "trip_id",
            "stop_id",
            "station_id",
            "stop_sequence",
            "arrival_time",
            "arrival_seconds"
        ]
    ].copy()

    # Explicitly name the station lookup column
    names = station_lookup[
        ["station_id", "stop_name"]
    ].drop_duplicates("station_id").rename(
        columns={"stop_name": "station_name"}
    )

    downstream = downstream.merge(
        names,
        on="station_id",
        how="left",
        validate="many_to_one"
    )

    return downstream.sort_values(
        ["stop_sequence", "arrival_seconds"]
    ).reset_index(drop=True)


# Test on our validated T9 journey
test_downstream = get_downstream_stops(
    daily=test_timetable,
    trip_id=test_journey["second_trip_id"],
    boarding_sequence=test_journey["second_boarding_sequence"]
)

print("Downstream stops:", len(test_downstream))

display(
    test_downstream[
        ["station_name", "arrival_time", "stop_id"]
    ]
)

Downstream stops: 11


,station_name,arrival_time,stop_id
0,Eastwood Station,17:45:00,2122231
1,Denistone Station,17:47:42,2114171
2,West Ryde Station,17:49:30,2114191
3,Meadowbank Station,17:51:42,2114181
4,Rhodes Station,17:53:48,213891
5,Concord West Station,17:56:30,213881
6,North Strathfield Station,17:58:30,2137171
7,Strathfield Station,18:03:00,2135235
8,Burwood Station,18:06:18,213493
9,Redfern Station,18:15:42,2015133


In [ ]:
# STEP 62A — Find first-transfer journeys to Strathfield

first_transfer_candidates = find_one_transfer_journeys(
    daily=test_timetable,
    origin="Macquarie University Station",
    destination="Strathfield Station",
    departure_time="17:30",
    window_minutes=30,
    connection_window_minutes=30
)

print("First-transfer candidates:", len(first_transfer_candidates))

display(
    first_transfer_candidates[
        [
            "first_line",
            "origin_departure_time",
            "transfer_station",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "journey_duration_min"
        ]
    ].sort_values(
        "destination_arrival_time"
    ).head(15)
)

First-transfer candidates: 143


,first_line,origin_departure_time,transfer_station,second_line,second_departure_time,destination_arrival_time,journey_duration_min
0,M1,17:30:37,Epping Station,T9,17:42:00,18:03:00,32.38
1,M1,17:30:37,Epping Station,T9,17:53:01,18:07:48,37.18
2,M1,17:34:37,Epping Station,T9,17:53:01,18:07:48,33.18
3,M1,17:38:37,Epping Station,T9,17:53:01,18:07:48,29.18
4,M1,17:42:37,Epping Station,T9,17:53:01,18:07:48,25.18
5,M1,17:31:54,Central Station,T1,18:04:01,18:16:30,44.60
6,M1,17:30:37,Epping Station,T9,17:57:00,18:18:00,47.38
7,M1,17:34:37,Epping Station,T9,17:57:00,18:18:00,43.38
8,M1,17:38:37,Epping Station,T9,17:57:00,18:18:00,39.38
9,M1,17:42:37,Epping Station,T9,17:57:00,18:18:00,35.38


In [ ]:
# STEP 62B — Extend one-transfer journeys to a third service

# Keep unique second-leg arrival events
candidates = (
    first_transfer_candidates
    .sort_values("destination_arrival_seconds")
    .drop_duplicates(
        subset=[
            "second_trip_id",
            "destination_arrival_seconds"
        ]
    )
    .head(20)
    .copy()
)

all_three_leg_journeys = []

for _, journey in candidates.iterrows():

    # Identify the actual arrival platform at Strathfield
    arrival_events = test_timetable[
        (test_timetable["trip_id"] == journey["second_trip_id"])
        & (test_timetable["station_id"].isin(
            station_lookup.loc[
                station_lookup["stop_name"] == "Strathfield Station",
                "station_id"
            ]
        ))
        & (
            test_timetable["stop_sequence"]
            == journey["destination_sequence"]
        )
    ]

    for _, arrival in arrival_events.iterrows():

        # Find a valid third service to Parramatta
        onward = find_onward_journeys(
            daily=test_timetable,
            arrival_platform=arrival["stop_id"],
            arrival_seconds=arrival["arrival_seconds"],
            destination="Parramatta Station",
            connection_window_minutes=30,
            exclude_trip_ids=[
                journey["trip_id"],
                journey["second_trip_id"]
            ]
        )

        if onward.empty:
            continue

        onward = onward.copy()

        onward["first_line"] = journey["first_line"]
        onward["second_line"] = journey["second_line"]
        onward["first_transfer"] = journey["transfer_station"]
        onward["second_transfer"] = "Strathfield Station"
        onward["origin_departure_time"] = journey["origin_departure_time"]
        onward["origin_departure_seconds"] = journey["origin_departure_seconds"]

        onward["total_journey_min"] = (
            onward["destination_arrival_seconds"]
            - onward["origin_departure_seconds"]
        ) / 60

        all_three_leg_journeys.append(onward)

if all_three_leg_journeys:

    three_leg_results = pd.concat(
        all_three_leg_journeys,
        ignore_index=True
    )

    three_leg_results["route"] = (
        three_leg_results["first_line"].astype(str)
        + " → "
        + three_leg_results["second_line"].astype(str)
        + " → "
        + three_leg_results["route_short_name"].astype(str)
    )

    three_leg_results = three_leg_results.sort_values(
        "destination_arrival_seconds"
    )

    print("Three-leg journey combinations:", len(three_leg_results))

    display(
        three_leg_results[
            [
                "route",
                "first_transfer",
                "second_transfer",
                "origin_departure_time",
                "departure_time",
                "destination_arrival_time",
                "total_journey_min"
            ]
        ].head(15)
    )

else:
    three_leg_results = pd.DataFrame()
    print("No valid three-leg journeys found.")

Three-leg journey combinations: 115


,route,first_transfer,second_transfer,origin_departure_time,departure_time,destination_arrival_time,total_journey_min
0,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:11:24,18:23:00,52.383333
8,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,58.383333
1,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,58.383333
2,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:20:24,18:32:00,61.383333
9,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:20:24,18:32:00,61.383333
14,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:20:24,18:32:00,60.100000
3,M1 → T9 → T2,Epping Station,Strathfield Station,17:30:37,18:16:00,18:36:30,65.883333
4,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:26:24,18:38:00,67.383333
43,M1 → T9 → T1,Central Station,Strathfield Station,17:35:54,18:26:24,18:38:00,62.100000
10,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:26:24,18:38:00,67.383333


In [ ]:
# STEP 63 — Automatically discover second interchange stations

# Start with 5 unique second-leg trips
sample_candidates = (
    first_transfer_candidates
    .sort_values("destination_arrival_seconds")
    .drop_duplicates(subset=["second_trip_id"])
    .head(5)
    .copy()
)

all_auto_routes = []

for _, journey in sample_candidates.iterrows():

    # Find every downstream station on the second train
    downstream = get_downstream_stops(
        daily=test_timetable,
        trip_id=journey["second_trip_id"],
        boarding_sequence=journey["second_boarding_sequence"]
    )

    for _, stop in downstream.iterrows():

        # Avoid transferring at the origin or first interchange
        if stop["station_name"] in [
            "Macquarie University Station",
            journey["transfer_station"]
        ]:
            continue

        # Find onward services to Parramatta
        onward = find_onward_journeys(
            daily=test_timetable,
            arrival_platform=stop["stop_id"],
            arrival_seconds=stop["arrival_seconds"],
            destination="Parramatta Station",
            connection_window_minutes=30,
            exclude_trip_ids=[
                journey["trip_id"],
                journey["second_trip_id"]
            ]
        )

        if onward.empty:
            continue

        onward = onward.copy()

        # Store the full journey information
        onward["first_line"] = journey["first_line"]
        onward["second_line"] = journey["second_line"]
        onward["first_transfer"] = journey["transfer_station"]
        onward["second_transfer"] = stop["station_name"]
        onward["origin_departure_time"] = journey["origin_departure_time"]
        onward["origin_departure_seconds"] = journey["origin_departure_seconds"]

        onward["total_journey_min"] = (
            onward["destination_arrival_seconds"]
            - onward["origin_departure_seconds"]
        ) / 60

        all_auto_routes.append(onward)

if all_auto_routes:

    auto_routes = pd.concat(
        all_auto_routes,
        ignore_index=True
    )

    auto_routes["route"] = (
        auto_routes["first_line"].astype(str)
        + " → "
        + auto_routes["second_line"].astype(str)
        + " → "
        + auto_routes["route_short_name"].astype(str)
    )

    auto_routes = auto_routes.sort_values(
        "destination_arrival_seconds"
    )

    print("Automatically discovered journeys:", len(auto_routes))
    print(
        "Second interchange stations:",
        auto_routes["second_transfer"].nunique()
    )

    display(
        auto_routes[
            [
                "route",
                "first_transfer",
                "second_transfer",
                "origin_departure_time",
                "departure_time",
                "destination_arrival_time",
                "total_journey_min"
            ]
        ].head(20)
    )

else:
    auto_routes = pd.DataFrame()
    print("No two-transfer journeys found.")

Automatically discovered journeys: 98
Second interchange stations: 14


,route,first_transfer,second_transfer,origin_departure_time,departure_time,destination_arrival_time,total_journey_min
0,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:11:24,18:23:00,52.383333
1,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,58.383333
21,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,58.383333
43,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:20:24,18:32:00,60.100000
36,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:09:36,18:32:00,60.100000
22,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:20:24,18:32:00,61.383333
2,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:20:24,18:32:00,61.383333
8,M1 → T9 → T2,Epping Station,Burwood Station,17:30:37,18:13:18,18:36:30,65.883333
3,M1 → T9 → T2,Epping Station,Strathfield Station,17:30:37,18:16:00,18:36:30,65.883333
37,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:15:36,18:38:00,66.100000


In [ ]:
# STEP 64 — Deduplicate journeys using actual trip IDs

print("Original journey combinations:", len(auto_routes))

# Inspect available journey identifiers
print("\nAvailable trip-related columns:")
print([
    col for col in auto_routes.columns
    if "trip" in col.lower()
])

# Check which trip identifiers we have
display(
    auto_routes[
        [
            "route",
            "first_transfer",
            "second_transfer",
            "origin_departure_time",
            "destination_arrival_time",
            "trip_id"
        ]
    ].head(10)
)

Original journey combinations: 98

Available trip-related columns:
['trip_id']


,route,first_transfer,second_transfer,origin_departure_time,destination_arrival_time,trip_id
0,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:23:00,197F.1324.165.16.B.8.92001727
1,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:29:00,145N.1324.165.16.A.8.92001340
21,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:29:00,145N.1324.165.16.A.8.92001340
43,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:32:00,165F.1324.165.124.A.8.92002689
36,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:32:00,165F.1324.165.124.A.8.92002689
22,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:32:00,165F.1324.165.124.A.8.92002689
2,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:32:00,165F.1324.165.124.A.8.92002689
8,M1 → T9 → T2,Epping Station,Burwood Station,17:30:37,18:36:30,78-P.1324.165.16.A.8.92002733
3,M1 → T9 → T2,Epping Station,Strathfield Station,17:30:37,18:36:30,78-P.1324.165.16.A.8.92002733
37,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:38:00,155N.1324.165.16.A.8.92002106


In [ ]:
# STEP 65 — Rebuild automatically discovered journeys with all trip IDs

sample_candidates = (
    first_transfer_candidates
    .sort_values("destination_arrival_seconds")
    .drop_duplicates(subset=["second_trip_id"])
    .head(5)
    .copy()
)

all_auto_routes = []

for _, journey in sample_candidates.iterrows():

    downstream = get_downstream_stops(
        daily=test_timetable,
        trip_id=journey["second_trip_id"],
        boarding_sequence=journey["second_boarding_sequence"]
    )

    for _, stop in downstream.iterrows():

        if stop["station_name"] in [
            "Macquarie University Station",
            journey["transfer_station"]
        ]:
            continue

        onward = find_onward_journeys(
            daily=test_timetable,
            arrival_platform=stop["stop_id"],
            arrival_seconds=stop["arrival_seconds"],
            destination="Parramatta Station",
            connection_window_minutes=30,
            exclude_trip_ids=[
                journey["trip_id"],
                journey["second_trip_id"]
            ]
        )

        if onward.empty:
            continue

        onward = onward.copy()

        # Preserve all three trip identifiers
        onward["first_trip_id"] = journey["trip_id"]
        onward["second_trip_id"] = journey["second_trip_id"]
        onward["third_trip_id"] = onward["trip_id"]

        onward["first_line"] = journey["first_line"]
        onward["second_line"] = journey["second_line"]

        onward["first_transfer"] = journey["transfer_station"]
        onward["second_transfer"] = stop["station_name"]

        onward["origin_departure_time"] = journey["origin_departure_time"]
        onward["origin_departure_seconds"] = journey["origin_departure_seconds"]

        onward["total_journey_min"] = (
            onward["destination_arrival_seconds"]
            - onward["origin_departure_seconds"]
        ) / 60

        all_auto_routes.append(onward)

if all_auto_routes:

    auto_routes = pd.concat(
        all_auto_routes,
        ignore_index=True
    )

    auto_routes["route"] = (
        auto_routes["first_line"].astype(str)
        + " → "
        + auto_routes["second_line"].astype(str)
        + " → "
        + auto_routes["route_short_name"].astype(str)
    )

    # Deduplicate using the complete itinerary
    unique_routes = (
        auto_routes
        .drop_duplicates(
            subset=[
                "first_trip_id",
                "second_trip_id",
                "third_trip_id",
                "first_transfer",
                "second_transfer",
                "origin_departure_seconds",
                "departure_seconds"
            ]
        )
        .sort_values(
            ["destination_arrival_seconds", "origin_departure_seconds"]
        )
        .reset_index(drop=True)
    )

    print("Original combinations:", len(auto_routes))
    print("Unique itineraries:", len(unique_routes))
    print("Duplicates removed:", len(auto_routes) - len(unique_routes))

    display(
        unique_routes[
            [
                "route",
                "first_transfer",
                "second_transfer",
                "origin_departure_time",
                "departure_time",
                "destination_arrival_time",
                "total_journey_min"
            ]
        ].head(15)
    )

else:
    auto_routes = pd.DataFrame()
    unique_routes = pd.DataFrame()
    print("No two-transfer journeys found.")

Original combinations: 98
Unique itineraries: 98
Duplicates removed: 0


,route,first_transfer,second_transfer,origin_departure_time,departure_time,destination_arrival_time,total_journey_min
0,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:11:24,18:23:00,52.383333
1,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,58.383333
2,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,58.383333
3,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:20:24,18:32:00,61.383333
4,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:20:24,18:32:00,61.383333
5,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:09:36,18:32:00,60.100000
6,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:20:24,18:32:00,60.100000
7,M1 → T9 → T2,Epping Station,Strathfield Station,17:30:37,18:16:00,18:36:30,65.883333
8,M1 → T9 → T2,Epping Station,Burwood Station,17:30:37,18:13:18,18:36:30,65.883333
9,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:26:24,18:38:00,67.383333


In [ ]:
# STEP 66 — Investigate apparently duplicate journeys

# Find journeys with identical passenger-facing details
passenger_columns = [
    "route",
    "first_transfer",
    "second_transfer",
    "origin_departure_time",
    "departure_time",
    "destination_arrival_time"
]

duplicate_mask = unique_routes.duplicated(
    subset=passenger_columns,
    keep=False
)

apparent_duplicates = unique_routes.loc[
    duplicate_mask
].copy()

print("Total unique itineraries:", len(unique_routes))
print("Apparently duplicate rows:", len(apparent_duplicates))

print(
    "Distinct passenger-facing journeys:",
    len(unique_routes.drop_duplicates(subset=passenger_columns))
)

# Compare underlying trip IDs
display(
    apparent_duplicates[
        passenger_columns + [
            "first_trip_id",
            "second_trip_id",
            "third_trip_id"
        ]
    ]
    .sort_values(passenger_columns)
    .head(15)
)

Total unique itineraries: 98
Apparently duplicate rows: 39
Distinct passenger-facing journeys: 75


,route,first_transfer,second_transfer,origin_departure_time,departure_time,destination_arrival_time,first_trip_id,second_trip_id,third_trip_id
38,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:30:01,18:56:00,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037,107N.1324.165.16.T.8.92001465
39,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:30:01,18:56:00,0160-006-154-013:1000,130N.1324.165.16.A.8.92003126,107N.1324.165.16.T.8.92001465
46,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:33:01,18:59:00,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037,143N.1324.165.16.A.8.92000923
47,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:33:01,18:59:00,0160-006-154-013:1000,130N.1324.165.16.A.8.92003126,143N.1324.165.16.A.8.92000923
61,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:45:01,19:11:00,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037,157P.1324.165.16.A.8.92002747
62,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:45:01,19:11:00,0160-006-154-013:1000,130N.1324.165.16.A.8.92003126,157P.1324.165.16.A.8.92002747
64,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:45:01,19:11:00,0160-006-154-013:1000,191J.1324.165.16.B.8.92003392,157P.1324.165.16.A.8.92002747
69,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:48:01,19:14:00,0160-006-154-013:1000,130N.1324.165.16.A.8.92003126,195P.1324.165.16.B.8.92001854
71,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:48:01,19:14:00,0160-006-154-013:1000,191J.1324.165.16.B.8.92003392,195P.1324.165.16.B.8.92001854
1,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037,145N.1324.165.16.A.8.92001340


In [ ]:
# STEP 67 — Compare apparently identical journeys

# Select the two journeys from rows 1 and 2
comparison_rows = unique_routes.loc[[1, 2]].copy()

results = []

for _, journey in comparison_rows.iterrows():

    # Find the middle train's arrival at Strathfield
    middle_arrival = test_timetable[
        (test_timetable["trip_id"] == journey["second_trip_id"])
        & (
            test_timetable["station_id"].isin(
                station_lookup.loc[
                    station_lookup["stop_name"] == "Strathfield Station",
                    "station_id"
                ]
            )
        )
    ].sort_values("arrival_seconds")

    if middle_arrival.empty:
        continue

    arrival = middle_arrival.iloc[0]

    results.append({
        "second_trip_id": journey["second_trip_id"],
        "middle_arrival_time": arrival["arrival_time"],
        "third_departure_time": journey["departure_time"],
        "waiting_minutes": round(
            (
                journey["departure_seconds"]
                - arrival["arrival_seconds"]
            ) / 60,
            2
        ),
        "final_arrival": journey["destination_arrival_time"]
    })

transfer_comparison = pd.DataFrame(results)

display(transfer_comparison)

,second_trip_id,middle_arrival_time,third_departure_time,waiting_minutes,final_arrival
0,125R.1324.165.16.A.8.92001037,18:03:00,18:17:24,14.4,18:29:00
1,130N.1324.165.16.A.8.92003126,18:07:48,18:17:24,9.6,18:29:00


In [ ]:
# STEP 68 — Calculate second-transfer connection quality

ranked_routes = unique_routes.copy()

# Get actual arrival events for each second-leg trip
middle_arrivals = test_timetable[
    [
        "trip_id",
        "stop_id",
        "station_id",
        "arrival_seconds",
        "arrival_time"
    ]
].rename(columns={
    "trip_id": "second_trip_id",
    "stop_id": "second_arrival_platform",
    "arrival_seconds": "second_arrival_seconds",
    "arrival_time": "second_arrival_time"
})

# Match the second-leg arrival to the third-leg boarding connection
ranked_routes = ranked_routes.merge(
    middle_arrivals,
    left_on=[
        "second_trip_id",
        "boarding_platform"
    ],
    right_on=[
        "second_trip_id",
        "second_arrival_platform"
    ],
    how="left"
) if False else ranked_routes

# Use the existing transfer information from the onward search
# Recover arrival time by matching each trip to its second interchange
station_ids = station_lookup.rename(columns={
    "stop_name": "second_transfer"
})

middle_arrivals = middle_arrivals.merge(
    station_ids[["station_id", "second_transfer"]],
    on="station_id",
    how="left"
)

ranked_routes = ranked_routes.merge(
    middle_arrivals[
        [
            "second_trip_id",
            "second_transfer",
            "second_arrival_platform",
            "second_arrival_seconds",
            "second_arrival_time"
        ]
    ],
    on=["second_trip_id", "second_transfer"],
    how="left"
)

# Calculate connection quality
ranked_routes["waiting_minutes"] = (
    ranked_routes["departure_seconds"]
    - ranked_routes["second_arrival_seconds"]
) / 60

ranked_routes["spare_connection_minutes"] = (
    ranked_routes["departure_seconds"]
    - ranked_routes["second_arrival_seconds"]
    - ranked_routes["minimum_transfer_seconds"]
) / 60

# Only retain valid connections
ranked_routes = ranked_routes[
    ranked_routes["spare_connection_minutes"] >= 0
].copy()

print("Journeys with valid second transfers:", len(ranked_routes))

display(
    ranked_routes[
        [
            "route",
            "first_transfer",
            "second_transfer",
            "second_arrival_time",
            "departure_time",
            "waiting_minutes",
            "spare_connection_minutes",
            "destination_arrival_time"
        ]
    ]
    .sort_values("destination_arrival_time")
    .head(15)
)

Journeys with valid second transfers: 98


,route,first_transfer,second_transfer,second_arrival_time,departure_time,waiting_minutes,spare_connection_minutes,destination_arrival_time
0,M1 → T9 → T1,Epping Station,Strathfield Station,18:03:00,18:11:24,8.4,3.4,18:23:00
1,M1 → T9 → T1,Epping Station,Strathfield Station,18:03:00,18:17:24,14.4,9.4,18:29:00
2,M1 → T9 → T1,Epping Station,Strathfield Station,18:07:48,18:17:24,9.6,2.2,18:29:00
3,M1 → T9 → T1,Epping Station,Strathfield Station,18:03:00,18:20:24,17.4,12.4,18:32:00
4,M1 → T9 → T1,Epping Station,Strathfield Station,18:07:48,18:20:24,12.6,5.2,18:32:00
5,M1 → T1 → T1,Central Station,Redfern Station,18:06:06,18:09:36,3.5,1.5,18:32:00
6,M1 → T1 → T1,Central Station,Strathfield Station,18:16:30,18:20:24,3.9,1.9,18:32:00
7,M1 → T9 → T2,Epping Station,Strathfield Station,18:03:00,18:16:00,13.0,6.6,18:36:30
8,M1 → T9 → T2,Epping Station,Burwood Station,18:06:18,18:13:18,7.0,1.0,18:36:30
9,M1 → T9 → T1,Epping Station,Strathfield Station,18:03:00,18:26:24,23.4,18.4,18:38:00


In [ ]:
# STEP 69 — Rank journeys by arrival time and transfer practicality

journey_ranking = ranked_routes.copy()

# Keep only journeys with valid connection times
journey_ranking = journey_ranking[
    journey_ranking["spare_connection_minutes"] >= 0
].copy()

# Calculate journey duration
journey_ranking["journey_duration_min"] = (
    journey_ranking["destination_arrival_seconds"]
    - journey_ranking["origin_departure_seconds"]
) / 60

# Categorise second-transfer connection comfort
def connection_category(spare_minutes):
    if spare_minutes < 2:
        return "Tight"
    elif spare_minutes < 5:
        return "Moderate"
    else:
        return "Comfortable"

journey_ranking["connection_quality"] = (
    journey_ranking["spare_connection_minutes"]
    .apply(connection_category)
)

# Remove passenger-facing duplicates, keeping the option
# with the greatest spare connection time
passenger_columns = [
    "route",
    "first_transfer",
    "second_transfer",
    "origin_departure_time",
    "departure_time",
    "destination_arrival_time"
]

journey_ranking = (
    journey_ranking
    .sort_values("spare_connection_minutes", ascending=False)
    .drop_duplicates(subset=passenger_columns)
)

# Rank by earliest arrival, then duration,
# then connection buffer
journey_ranking = (
    journey_ranking
    .sort_values(
        [
            "destination_arrival_seconds",
            "journey_duration_min",
            "spare_connection_minutes"
        ],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

print("Ranked passenger-facing journeys:", len(journey_ranking))

display(
    journey_ranking[
        [
            "route",
            "first_transfer",
            "second_transfer",
            "origin_departure_time",
            "destination_arrival_time",
            "journey_duration_min",
            "spare_connection_minutes",
            "connection_quality"
        ]
    ].head(15)
)

Ranked passenger-facing journeys: 75


,route,first_transfer,second_transfer,origin_departure_time,destination_arrival_time,journey_duration_min,spare_connection_minutes,connection_quality
0,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:23:00,52.383333,3.4,Moderate
1,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:29:00,58.383333,9.4,Comfortable
2,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:32:00,60.100000,1.9,Tight
3,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:32:00,60.100000,1.5,Tight
4,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:32:00,61.383333,12.4,Comfortable
5,M1 → T9 → T2,Epping Station,Strathfield Station,17:30:37,18:36:30,65.883333,6.6,Comfortable
6,M1 → T9 → T2,Epping Station,Burwood Station,17:30:37,18:36:30,65.883333,1.0,Tight
7,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:38:00,66.100000,7.9,Comfortable
8,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:38:00,66.100000,7.5,Comfortable
9,M1 → T9 → T1,Central Station,Strathfield Station,17:31:54,18:38:00,66.100000,1.0,Tight


In [ ]:
# STEP 70 — Calculate first-transfer connection buffers

# Build a lookup of the first transfer
first_transfer_lookup = first_transfer_candidates[
    [
        "trip_id",
        "second_trip_id",
        "origin_departure_seconds",
        "arrival_seconds",
        "second_departure_seconds",
        "from_stop_id",
        "to_stop_id",
        "minimum_transfer_seconds"
    ]
].copy()

first_transfer_lookup = first_transfer_lookup.rename(
    columns={
        "trip_id": "first_trip_id",
        "arrival_seconds": "first_transfer_arrival_seconds",
        "second_departure_seconds": "first_transfer_departure_seconds",
        "minimum_transfer_seconds": "first_minimum_transfer_seconds"
    }
)

# Calculate waiting and spare connection time
first_transfer_lookup["first_waiting_minutes"] = (
    first_transfer_lookup["first_transfer_departure_seconds"]
    - first_transfer_lookup["first_transfer_arrival_seconds"]
) / 60

first_transfer_lookup["first_spare_minutes"] = (
    first_transfer_lookup["first_transfer_departure_seconds"]
    - first_transfer_lookup["first_transfer_arrival_seconds"]
    - first_transfer_lookup["first_minimum_transfer_seconds"]
) / 60

# Preserve separate first-transfer alternatives
first_transfer_lookup = (
    first_transfer_lookup
    .sort_values("first_spare_minutes", ascending=False)
    .drop_duplicates(
        subset=[
            "first_trip_id",
            "second_trip_id",
            "origin_departure_seconds"
        ]
    )
)

# Merge into ranked journeys
journey_ranking = journey_ranking.merge(
    first_transfer_lookup[
        [
            "first_trip_id",
            "second_trip_id",
            "origin_departure_seconds",
            "first_waiting_minutes",
            "first_spare_minutes"
        ]
    ],
    on=[
        "first_trip_id",
        "second_trip_id",
        "origin_departure_seconds"
    ],
    how="left",
    validate="many_to_one"
)

# Overall journey comfort is determined by the tighter transfer
journey_ranking["minimum_spare_minutes"] = journey_ranking[
    ["first_spare_minutes", "spare_connection_minutes"]
].min(axis=1)

def overall_connection_quality(spare):
    if pd.isna(spare):
        return "Unknown"
    if spare < 2:
        return "Tight"
    elif spare < 5:
        return "Moderate"
    return "Comfortable"

journey_ranking["overall_connection_quality"] = (
    journey_ranking["minimum_spare_minutes"]
    .apply(overall_connection_quality)
)

print(
    "Journeys with first-transfer data:",
    journey_ranking["first_spare_minutes"].notna().sum()
)

print("Total ranked journeys:", len(journey_ranking))

display(
    journey_ranking[
        [
            "route",
            "first_transfer",
            "second_transfer",
            "destination_arrival_time",
            "first_spare_minutes",
            "spare_connection_minutes",
            "minimum_spare_minutes",
            "overall_connection_quality"
        ]
    ].head(15)
)

Journeys with first-transfer data: 75
Total ranked journeys: 75


,route,first_transfer,second_transfer,destination_arrival_time,first_spare_minutes,spare_connection_minutes,minimum_spare_minutes,overall_connection_quality
0,M1 → T9 → T1,Epping Station,Strathfield Station,18:23:00,2.650000,3.4,2.650000,Moderate
1,M1 → T9 → T1,Epping Station,Strathfield Station,18:29:00,2.650000,9.4,2.650000,Moderate
2,M1 → T1 → T1,Central Station,Strathfield Station,18:32:00,1.766667,1.9,1.766667,Tight
3,M1 → T1 → T1,Central Station,Redfern Station,18:32:00,1.766667,1.5,1.500000,Tight
4,M1 → T9 → T1,Epping Station,Strathfield Station,18:32:00,2.650000,12.4,2.650000,Moderate
5,M1 → T9 → T2,Epping Station,Strathfield Station,18:36:30,2.650000,6.6,2.650000,Moderate
6,M1 → T9 → T2,Epping Station,Burwood Station,18:36:30,2.650000,1.0,1.000000,Tight
7,M1 → T1 → T1,Central Station,Strathfield Station,18:38:00,1.766667,7.9,1.766667,Tight
8,M1 → T1 → T1,Central Station,Redfern Station,18:38:00,1.766667,7.5,1.766667,Tight
9,M1 → T9 → T1,Central Station,Strathfield Station,18:38:00,4.366667,1.0,1.000000,Tight


In [ ]:
# STEP 71 — Combine one-transfer and two-transfer journeys

# One-transfer journeys
one_transfer = journey_test_2.copy()

one_transfer_results = pd.DataFrame({
    "route": (
        one_transfer["first_line"].astype(str)
        + " → "
        + one_transfer["second_line"].astype(str)
    ),
    "first_transfer": one_transfer["transfer_station"],
    "second_transfer": None,
    "origin_departure_time": one_transfer["origin_departure_time"],
    "destination_arrival_time": one_transfer["destination_arrival_time"],
    "origin_departure_seconds": one_transfer["origin_departure_seconds"],
    "destination_arrival_seconds": one_transfer["destination_arrival_seconds"],
    "journey_duration_min": one_transfer["journey_duration_min"],
    "transfers": 1,
    "connection_quality": "Not yet evaluated"
})

# Two-transfer journeys
two_transfer_results = pd.DataFrame({
    "route": journey_ranking["route"],
    "first_transfer": journey_ranking["first_transfer"],
    "second_transfer": journey_ranking["second_transfer"],
    "origin_departure_time": journey_ranking["origin_departure_time"],
    "destination_arrival_time": journey_ranking["destination_arrival_time"],
    "origin_departure_seconds": journey_ranking["origin_departure_seconds"],
    "destination_arrival_seconds": journey_ranking["destination_arrival_seconds"],
    "journey_duration_min": journey_ranking["journey_duration_min"],
    "transfers": 2,
    "connection_quality": journey_ranking["overall_connection_quality"]
})

# Combine both types
combined_journeys = pd.concat(
    [one_transfer_results, two_transfer_results],
    ignore_index=True
)

# Sort by arrival time, then fewer transfers
combined_journeys = (
    combined_journeys
    .sort_values(
        [
            "destination_arrival_seconds",
            "transfers",
            "journey_duration_min"
        ]
    )
    .reset_index(drop=True)
)

print("Total combined journey options:", len(combined_journeys))

display(
    combined_journeys[
        [
            "route",
            "first_transfer",
            "second_transfer",
            "origin_departure_time",
            "destination_arrival_time",
            "journey_duration_min",
            "transfers",
            "connection_quality"
        ]
    ].head(15)
)

Total combined journey options: 82


,route,first_transfer,second_transfer,origin_departure_time,destination_arrival_time,journey_duration_min,transfers,connection_quality
0,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:23:00,52.383333,2,Moderate
1,M1 → T1,Central Station,None,17:31:54,18:29:00,57.100000,1,Not yet evaluated
2,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:29:00,58.383333,2,Moderate
3,M1 → T1,Central Station,None,17:35:54,18:32:00,56.100000,1,Not yet evaluated
4,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:32:00,60.100000,2,Tight
5,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:32:00,60.100000,2,Tight
6,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:32:00,61.383333,2,Moderate
7,M1 → T9 → T2,Epping Station,Strathfield Station,17:30:37,18:36:30,65.883333,2,Moderate
8,M1 → T9 → T2,Epping Station,Burwood Station,17:30:37,18:36:30,65.883333,2,Tight
9,M1 → T1,Central Station,None,17:39:54,18:38:00,58.100000,1,Not yet evaluated


In [ ]:
# STEP 72 — Generate passenger-friendly journey recommendations

def recommend_journeys(journeys):

    if journeys.empty:
        return pd.DataFrame()

    df = journeys.copy()

    recommendations = []

    # 1. Fastest arrival
    fastest = df.sort_values(
        [
            "destination_arrival_seconds",
            "transfers",
            "journey_duration_min"
        ]
    ).iloc[0].copy()

    fastest["recommendation"] = "Fastest arrival"
    recommendations.append(fastest)

    # 2. Fewest transfers
    simplest = df.sort_values(
        [
            "transfers",
            "destination_arrival_seconds",
            "journey_duration_min"
        ]
    ).iloc[0].copy()

    simplest["recommendation"] = "Fewest transfers"
    recommendations.append(simplest)

    # 3. Most comfortable evaluated connection
    comfort_rank = {
        "Comfortable": 3,
        "Moderate": 2,
        "Tight": 1
    }

    evaluated = df[
        df["connection_quality"].isin(comfort_rank)
    ].copy()

    if not evaluated.empty:

        evaluated["comfort_score"] = (
            evaluated["connection_quality"]
            .map(comfort_rank)
        )

        comfortable = evaluated.sort_values(
            [
                "comfort_score",
                "destination_arrival_seconds",
                "journey_duration_min"
            ],
            ascending=[False, True, True]
        ).iloc[0].copy()

        comfortable["recommendation"] = "Most comfortable evaluated"
        recommendations.append(comfortable)

    result = pd.DataFrame(recommendations)

    columns = [
        "recommendation",
        "route",
        "first_transfer",
        "second_transfer",
        "origin_departure_time",
        "destination_arrival_time",
        "journey_duration_min",
        "transfers",
        "connection_quality"
    ]

    return result[columns].reset_index(drop=True)


# Test our recommendation engine
recommendations = recommend_journeys(combined_journeys)

display(recommendations)

,recommendation,route,first_transfer,second_transfer,origin_departure_time,destination_arrival_time,journey_duration_min,transfers,connection_quality
0,Fastest arrival,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:23:00,52.383333,2,Moderate
1,Fewest transfers,M1 → T1,Central Station,NaN,17:31:54,18:29:00,57.100000,1,Not yet evaluated
2,Most comfortable evaluated,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:56:00,85.383333,2,Comfortable


In [ ]:
# STEP 74 — Recommend distinct journeys with accurate labels

def recommend_journeys_v3(journeys, max_extra_minutes=15):

    if journeys.empty:
        return pd.DataFrame()

    df = journeys.copy().reset_index(drop=True)
    recommendations = []
    selected_indices = set()

    # 1. Fastest arrival
    fastest_idx = df.sort_values(
        ["destination_arrival_seconds", "transfers", "journey_duration_min"]
    ).index[0]

    fastest = df.loc[fastest_idx].copy()
    fastest["recommendation"] = "Fastest arrival"

    recommendations.append(fastest)
    selected_indices.add(fastest_idx)

    # 2. Fewest transfers (prefer a distinct journey)
    remaining = df.drop(index=list(selected_indices))

    if not remaining.empty:
        simplest_idx = remaining.sort_values(
            ["transfers", "destination_arrival_seconds"]
        ).index[0]

        simplest = df.loc[simplest_idx].copy()
        simplest["recommendation"] = "Fewest transfers"

        recommendations.append(simplest)
        selected_indices.add(simplest_idx)

    # 3. Best evaluated alternative within 15 minutes
    latest_arrival = (
        fastest["destination_arrival_seconds"]
        + max_extra_minutes * 60
    )

    remaining = df.drop(index=list(selected_indices))

    alternatives = remaining[
        (remaining["destination_arrival_seconds"] <= latest_arrival)
        & (remaining["connection_quality"].isin(
            ["Comfortable", "Moderate", "Tight"]
        ))
    ].copy()

    if not alternatives.empty:

        comfort_rank = {
            "Comfortable": 3,
            "Moderate": 2,
            "Tight": 1
        }

        alternatives["comfort_score"] = (
            alternatives["connection_quality"].map(comfort_rank)
        )

        alternative_idx = alternatives.sort_values(
            [
                "comfort_score",
                "destination_arrival_seconds",
                "transfers"
            ],
            ascending=[False, True, True]
        ).index[0]

        alternative = df.loc[alternative_idx].copy()

        if alternative["connection_quality"] == "Comfortable":
            alternative["recommendation"] = "Comfortable alternative"
        else:
            alternative["recommendation"] = "Best evaluated alternative"

        recommendations.append(alternative)

    result = pd.DataFrame(recommendations)

    columns = [
        "recommendation",
        "route",
        "first_transfer",
        "second_transfer",
        "origin_departure_time",
        "destination_arrival_time",
        "journey_duration_min",
        "transfers",
        "connection_quality"
    ]

    return result[columns].reset_index(drop=True)


recommendations_v3 = recommend_journeys_v3(
    combined_journeys,
    max_extra_minutes=15
)

display(recommendations_v3)

,recommendation,route,first_transfer,second_transfer,origin_departure_time,destination_arrival_time,journey_duration_min,transfers,connection_quality
0,Fastest arrival,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:23:00,52.383333,2,Moderate
1,Fewest transfers,M1 → T1,Central Station,NaN,17:31:54,18:29:00,57.100000,1,Not yet evaluated
2,Best evaluated alternative,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:29:00,58.383333,2,Moderate


In [ ]:
# STEP 75A — Discover reachable stations without a destination

def get_reachable_first_leg_stops(
    daily,
    origin,
    departure_time,
    window_minutes=30
):
    # Find available departures from the origin
    departures = get_station_departures(
        daily=daily,
        origin=origin,
        departure_time=departure_time,
        window_minutes=window_minutes
    ).copy()

    if departures.empty:
        return pd.DataFrame()

    # Give boarding-event columns explicit names
    boarding = departures[
        ["trip_id", "stop_sequence", "departure_seconds"]
    ].rename(columns={
        "stop_sequence": "boarding_sequence",
        "departure_seconds": "origin_departure_seconds"
    })

    # Find downstream stops on each boarded trip
    reachable = boarding.merge(
        daily[
            [
                "trip_id",
                "stop_id",
                "station_id",
                "stop_sequence",
                "arrival_time",
                "arrival_seconds"
            ]
        ],
        on="trip_id",
        how="inner"
    )

    # Only include stops after boarding
    reachable = reachable[
        reachable["stop_sequence"]
        > reachable["boarding_sequence"]
    ].copy()

    # Add station names
    names = station_lookup[
        ["station_id", "stop_name"]
    ].drop_duplicates("station_id").rename(
        columns={"stop_name": "station_name"}
    )

    reachable = reachable.merge(
        names,
        on="station_id",
        how="left",
        validate="many_to_one"
    )

    return reachable.sort_values(
        ["origin_departure_seconds", "arrival_seconds"]
    ).reset_index(drop=True)


# Test
reachable_first_leg = get_reachable_first_leg_stops(
    daily=test_timetable,
    origin="Macquarie University Station",
    departure_time="17:30",
    window_minutes=30
)

print("Reachable first-leg stop events:", len(reachable_first_leg))
print(
    "Unique reachable stations:",
    reachable_first_leg["station_name"].nunique()
)

display(
    reachable_first_leg[
        [
            "station_name",
            "arrival_time",
            "trip_id"
        ]
    ].head(20)
)

Reachable first-leg stop events: 160
Unique reachable stations: 20


,station_name,arrival_time,trip_id
0,Epping Station,17:33:51,0160-006-154-013:1000
1,Cherrybrook Station,17:39:38,0160-006-154-013:1000
2,Castle Hill Station,17:42:22,0160-006-154-013:1000
3,Hills Showground Station,17:44:55,0160-006-154-013:1000
4,Norwest Station,17:47:30,0160-006-154-013:1000
5,Bella Vista Station,17:50:10,0160-006-154-013:1000
6,Kellyville Station,17:52:45,0160-006-154-013:1000
7,Rouse Hill Station,17:55:40,0160-006-154-013:1000
8,Tallawong Station,17:58:17,0160-006-154-013:1000
9,Macquarie Park Station,17:33:27,0160-006-106-014:1000


In [ ]:
# STEP 75B — Discover first-transfer connections automatically

all_first_connections = []

for _, arrival in reachable_first_leg.iterrows():

    connections = get_valid_connections(
        daily=test_timetable,
        arrival_platform=arrival["stop_id"],
        arrival_seconds=arrival["arrival_seconds"],
        connection_window_minutes=30
    )

    if connections.empty:
        continue

    # Exclude continuing on the same service
    connections = connections[
        connections["trip_id"] != arrival["trip_id"]
    ].copy()

    # Exclude boarding events at terminal stops
    last_sequences = test_timetable.groupby("trip_id")[
        "stop_sequence"
    ].max()

    connections = connections[
        connections["stop_sequence"]
        < connections["trip_id"].map(last_sequences)
    ].copy()

    if connections.empty:
        continue

    # Preserve both journey legs
    connections["first_trip_id"] = arrival["trip_id"]
    connections["first_arrival_station"] = arrival["station_name"]
    connections["first_arrival_platform"] = arrival["stop_id"]
    connections["first_arrival_seconds"] = arrival["arrival_seconds"]
    connections["origin_departure_seconds"] = arrival[
        "origin_departure_seconds"
    ]

    connections["second_trip_id"] = connections["trip_id"]

    connections["first_spare_minutes"] = (
        connections["departure_seconds"]
        - connections["first_arrival_seconds"]
        - connections["minimum_transfer_seconds"]
    ) / 60

    all_first_connections.append(connections)

if all_first_connections:

    first_connections = pd.concat(
        all_first_connections,
        ignore_index=True
    )

    print("Valid first-transfer connections:", len(first_connections))

    print(
        "Unique interchange stations:",
        first_connections["first_arrival_station"].nunique()
    )

    print(
        "Unique second-leg services:",
        first_connections["second_trip_id"].nunique()
    )

    display(
        first_connections[
            [
                "first_arrival_station",
                "first_trip_id",
                "second_trip_id",
                "departure_time",
                "first_spare_minutes"
            ]
        ].head(20)
    )

else:
    first_connections = pd.DataFrame()
    print("No valid first-transfer connections found.")

Valid first-transfer connections: 2694
Unique interchange stations: 20
Unique second-leg services: 191


,first_arrival_station,first_trip_id,second_trip_id,departure_time,first_spare_minutes
0,Epping Station,0160-006-154-013:1000,0160-006-155-013:1000,17:38:36,2.750000
1,Epping Station,0160-006-154-013:1000,0160-006-131-004:1000,17:40:13,1.366667
2,Epping Station,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037,17:42:00,2.650000
3,Epping Station,0160-006-154-013:1000,0160-006-152-015:1000,17:42:36,6.750000
4,Epping Station,0160-006-154-013:1000,0160-006-107-014:1000,17:44:13,5.366667
5,Epping Station,0160-006-154-013:1000,0160-006-112-013:1000,17:46:36,10.750000
6,Epping Station,0160-006-154-013:1000,0160-006-108-014:1000,17:48:13,9.366667
7,Epping Station,0160-006-154-013:1000,0160-006-128-005:1000,17:50:36,14.750000
8,Epping Station,0160-006-154-013:1000,156E.1324.165.16.A.8.92001489,17:51:00,11.650000
9,Epping Station,0160-006-154-013:1000,0160-006-123-012:1000,17:52:13,13.366667


In [ ]:
# STEP 75E — Expand all first-transfer connections

# Keep the columns needed for second-leg expansion
connections = first_connections[
    [
        "first_trip_id",
        "second_trip_id",
        "first_arrival_station",
        "origin_departure_seconds",
        "departure_seconds",
        "stop_sequence",
        "first_spare_minutes"
    ]
].copy()

connections = connections.rename(columns={
    "departure_seconds": "second_departure_seconds",
    "stop_sequence": "second_boarding_sequence",
    "first_arrival_station": "first_transfer"
})

# Remove exact duplicate boarding connections
connections = connections.drop_duplicates()

# Prepare downstream timetable events
downstream_stops = test_timetable[
    [
        "trip_id",
        "station_id",
        "stop_id",
        "stop_sequence",
        "arrival_time",
        "arrival_seconds"
    ]
].rename(columns={
    "trip_id": "second_trip_id"
})

# Expand every second-leg trip to its downstream stops
second_leg_reachable_full = connections.merge(
    downstream_stops,
    on="second_trip_id",
    how="inner"
)

# Only keep stops AFTER boarding the second service
second_leg_reachable_full = second_leg_reachable_full[
    second_leg_reachable_full["stop_sequence"]
    > second_leg_reachable_full["second_boarding_sequence"]
].copy()

# Add station names
station_names = (
    station_lookup[
        ["station_id", "stop_name"]
    ]
    .drop_duplicates("station_id")
    .rename(columns={"stop_name": "station_name"})
)

second_leg_reachable_full = second_leg_reachable_full.merge(
    station_names,
    on="station_id",
    how="left",
    validate="many_to_one"
)

# Check destination reachability
parramatta_full = second_leg_reachable_full[
    second_leg_reachable_full["station_name"]
    == "Parramatta Station"
].copy()

print(
    "Total second-leg reachable events:",
    len(second_leg_reachable_full)
)

print(
    "Unique second-leg services:",
    second_leg_reachable_full["second_trip_id"].nunique()
)

print(
    "Unique reachable stations:",
    second_leg_reachable_full["station_name"].nunique()
)

print(
    "One-transfer arrivals at Parramatta:",
    len(parramatta_full)
)

if not parramatta_full.empty:

    parramatta_full["journey_duration_min"] = (
        parramatta_full["arrival_seconds"]
        - parramatta_full["origin_departure_seconds"]
    ) / 60

    display(
        parramatta_full[
            [
                "first_transfer",
                "arrival_time",
                "journey_duration_min",
                "first_spare_minutes"
            ]
        ]
        .sort_values("journey_duration_min")
        .head(10)
    )

Total second-leg reachable events: 28490
Unique second-leg services: 191
Unique reachable stations: 187
One-transfer arrivals at Parramatta: 57


,first_transfer,arrival_time,journey_duration_min,first_spare_minutes
17009,Central Station,18:44:00,56.1,0.766667
6196,Central Station,18:32:00,56.1,0.766667
2497,Central Station,18:29:00,57.1,1.766667
13437,Central Station,18:41:00,57.1,1.766667
9824,Central Station,18:38:00,58.1,2.766667
27476,Central Station,18:59:00,59.1,2.766667
13573,Central Station,18:44:00,60.1,4.766667
24071,Central Station,18:56:00,60.1,3.766667
2641,Central Station,18:32:00,60.1,4.766667
9940,Central Station,18:41:00,61.1,5.766667


In [ ]:
# STEP 75D — Identify journeys reaching the destination after one transfer

destination = "Parramatta Station"

# Find second-leg stops matching the destination
one_transfer_arrivals = second_leg_reachable[
    second_leg_reachable["station_name"] == destination
].copy()

if one_transfer_arrivals.empty:
    print(
        f"No one-transfer journeys to {destination} "
        "found in the current sample."
    )
else:
    # Calculate total journey duration
    one_transfer_arrivals["journey_duration_min"] = (
        one_transfer_arrivals["arrival_seconds"]
        - one_transfer_arrivals["origin_departure_seconds"]
    ) / 60

    # Add route labels
    route_lookup = (
        test_timetable[
            ["trip_id", "route_short_name"]
        ]
        .drop_duplicates("trip_id")
        .set_index("trip_id")["route_short_name"]
    )

    one_transfer_arrivals["first_line"] = (
        one_transfer_arrivals["first_trip_id"].map(route_lookup)
    )

    one_transfer_arrivals["second_line"] = (
        one_transfer_arrivals["second_trip_id"].map(route_lookup)
    )

    one_transfer_arrivals["route"] = (
        one_transfer_arrivals["first_line"].astype(str)
        + " → "
        + one_transfer_arrivals["second_line"].astype(str)
    )

    display(
        one_transfer_arrivals[
            [
                "route",
                "first_transfer",
                "arrival_time",
                "journey_duration_min",
                "first_spare_minutes"
            ]
        ].sort_values("journey_duration_min").head(10)
    )

print(
    "Second-leg services explored:",
    second_leg_reachable["second_trip_id"].nunique()
)

print(
    "Unique second-leg stations explored:",
    second_leg_reachable["station_name"].nunique()
)

No one-transfer journeys to Parramatta Station found in the current sample.
Second-leg services explored: 16
Unique second-leg stations explored: 37


In [ ]:
# STEP 75F — Evaluate one-transfer journeys

one_transfer_results = parramatta_full.copy()

# Calculate journey duration
one_transfer_results["journey_duration_min"] = (
    one_transfer_results["arrival_seconds"]
    - one_transfer_results["origin_departure_seconds"]
) / 60

# Classify connection quality
def classify_connection(spare_minutes):
    if spare_minutes < 2:
        return "Tight"
    elif spare_minutes < 5:
        return "Moderate"
    else:
        return "Comfortable"


one_transfer_results["connection_quality"] = (
    one_transfer_results["first_spare_minutes"]
    .apply(classify_connection)
)

# Add route labels
trip_routes = (
    test_timetable[
        ["trip_id", "route_short_name"]
    ]
    .drop_duplicates("trip_id")
    .set_index("trip_id")["route_short_name"]
)

one_transfer_results["first_line"] = (
    one_transfer_results["first_trip_id"].map(trip_routes)
)

one_transfer_results["second_line"] = (
    one_transfer_results["second_trip_id"].map(trip_routes)
)

one_transfer_results["route"] = (
    one_transfer_results["first_line"].astype(str)
    + " → "
    + one_transfer_results["second_line"].astype(str)
)

# Recover origin departure time
def seconds_to_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


one_transfer_results["origin_departure_time"] = (
    one_transfer_results["origin_departure_seconds"]
    .apply(seconds_to_time)
)

# Remove duplicate itinerary records
one_transfer_results = (
    one_transfer_results
    .drop_duplicates(
        subset=[
            "first_trip_id",
            "second_trip_id",
            "origin_departure_seconds",
            "arrival_seconds"
        ]
    )
    .sort_values(
        ["arrival_seconds", "journey_duration_min"]
    )
    .reset_index(drop=True)
)

print("Valid one-transfer journeys:", len(one_transfer_results))

display(
    one_transfer_results[
        [
            "route",
            "first_transfer",
            "origin_departure_time",
            "arrival_time",
            "journey_duration_min",
            "first_spare_minutes",
            "connection_quality"
        ]
    ].head(15)
)

Valid one-transfer journeys: 54


,route,first_transfer,origin_departure_time,arrival_time,journey_duration_min,first_spare_minutes,connection_quality
0,M1 → T1,Central Station,17:31:54,18:29:00,57.1,1.766667,Tight
1,M1 → T1,Central Station,17:35:54,18:32:00,56.1,0.766667,Tight
2,M1 → T1,Central Station,17:31:54,18:32:00,60.1,4.766667,Moderate
3,M1 → T1,Central Station,17:39:54,18:38:00,58.1,2.766667,Moderate
4,M1 → T1,Central Station,17:35:54,18:38:00,62.1,6.766667,Comfortable
5,M1 → T1,Central Station,17:31:54,18:38:00,66.1,10.766667,Comfortable
6,M1 → T1,Central Station,17:43:54,18:41:00,57.1,1.766667,Tight
7,M1 → T1,Central Station,17:39:54,18:41:00,61.1,5.766667,Comfortable
8,M1 → T1,Central Station,17:35:54,18:41:00,65.1,9.766667,Comfortable
9,M1 → T1,Central Station,17:31:54,18:41:00,69.1,13.766667,Comfortable


In [ ]:
# STEP 76A — Discover potential second-transfer stations

origin_station = "Macquarie University Station"
destination_station = "Parramatta Station"

second_transfer_candidates = second_leg_reachable_full[
    (
        second_leg_reachable_full["station_name"]
        != origin_station
    )
    &
    (
        second_leg_reachable_full["station_name"]
        != destination_station
    )
    &
    (
        second_leg_reachable_full["station_name"]
        != second_leg_reachable_full["first_transfer"]
    )
].copy()

# Avoid processing identical arrival events more than once
second_transfer_candidates = (
    second_transfer_candidates
    .drop_duplicates(
        subset=[
            "first_trip_id",
            "second_trip_id",
            "origin_departure_seconds",
            "stop_id",
            "arrival_seconds"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Potential second-transfer arrival events:",
    len(second_transfer_candidates)
)

print(
    "Unique potential second-transfer stations:",
    second_transfer_candidates["station_name"].nunique()
)

# Check that Strathfield was discovered automatically
print(
    "Strathfield discovered:",
    (
        second_transfer_candidates["station_name"]
        == "Strathfield Station"
    ).any()
)

display(
    second_transfer_candidates[
        [
            "first_transfer",
            "station_name",
            "arrival_time",
            "first_spare_minutes",
            "second_trip_id"
        ]
    ]
    .sort_values("arrival_time")
    .head(20)
)

Potential second-transfer arrival events: 12062
Unique potential second-transfer stations: 185
Strathfield discovered: True


,first_transfer,station_name,arrival_time,first_spare_minutes,second_trip_id
391,Macquarie Park Station,North Ryde Station,17:39:35,2.500000,0160-006-120-012:1000
410,Macquarie Park Station,North Ryde Station,17:43:35,6.500000,0160-006-121-012:1000
1967,Macquarie Park Station,North Ryde Station,17:43:35,2.500000,0160-006-121-012:1000
0,Epping Station,Cherrybrook Station,17:43:38,2.750000,0160-006-155-013:1000
515,North Ryde Station,Macquarie Park Station,17:44:06,2.850000,0160-006-128-005:1000
19,Epping Station,Eastwood Station,17:45:00,2.650000,125R.1324.165.16.A.8.92001037
392,Macquarie Park Station,Chatswood Station,17:45:10,2.500000,0160-006-120-012:1000
8,Epping Station,Macquarie Park Station,17:45:27,1.366667,0160-006-131-004:1000
401,Macquarie Park Station,Epping Station,17:45:51,3.150000,0160-006-112-013:1000
1,Epping Station,Castle Hill Station,17:46:22,2.750000,0160-006-155-013:1000


In [ ]:
# STEP 76B — Inspect potential second-transfer stations

print(
    "Total second-transfer candidates:",
    len(second_transfer_candidates)
)

print(
    "Unique second-transfer stations:",
    second_transfer_candidates["station_name"].nunique()
)

strathfield_candidates = second_transfer_candidates[
    second_transfer_candidates["station_name"] == "Strathfield Station"
].copy()

print(
    "Strathfield transfer candidates:",
    len(strathfield_candidates)
)

# Inspect the best Strathfield candidates
if not strathfield_candidates.empty:

    strathfield_candidates["journey_elapsed_min"] = (
        strathfield_candidates["arrival_seconds"]
        - strathfield_candidates["origin_departure_seconds"]
    ) / 60

    display(
        strathfield_candidates[
            [
                "first_transfer",
                "station_name",
                "arrival_time",
                "journey_elapsed_min",
                "first_spare_minutes",
                "first_trip_id",
                "second_trip_id"
            ]
        ]
        .sort_values("arrival_time")
        .head(15)
    )

else:
    print("No Strathfield candidates found.")

Total second-transfer candidates: 12062
Unique second-transfer stations: 185
Strathfield transfer candidates: 136


,first_transfer,station_name,arrival_time,journey_elapsed_min,first_spare_minutes,first_trip_id,second_trip_id
26,Epping Station,Strathfield Station,18:03:00,32.383333,2.650000,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037
96,Epping Station,Strathfield Station,18:07:48,37.183333,13.666667,0160-006-154-013:1000,130N.1324.165.16.A.8.92003126
1647,Epping Station,Strathfield Station,18:07:48,33.183333,9.666667,0160-006-155-013:1000,130N.1324.165.16.A.8.92003126
3231,Epping Station,Strathfield Station,18:07:48,29.183333,5.666667,0160-006-152-015:1000,130N.1324.165.16.A.8.92003126
4758,Epping Station,Strathfield Station,18:07:48,25.183333,1.666667,0160-006-112-013:1000,130N.1324.165.16.A.8.92003126
888,Central Station,Strathfield Station,18:16:30,44.600000,1.766667,0160-006-106-014:1000,145N.1324.165.16.A.8.92001340
125,Epping Station,Strathfield Station,18:18:00,47.383333,17.650000,0160-006-154-013:1000,191J.1324.165.16.B.8.92003392
1676,Epping Station,Strathfield Station,18:18:00,43.383333,13.650000,0160-006-155-013:1000,191J.1324.165.16.B.8.92003392
3260,Epping Station,Strathfield Station,18:18:00,39.383333,9.650000,0160-006-152-015:1000,191J.1324.165.16.B.8.92003392
4787,Epping Station,Strathfield Station,18:18:00,35.383333,5.650000,0160-006-112-013:1000,191J.1324.165.16.B.8.92003392


In [ ]:
# STEP 76C — Validate the third-leg connection

# Select the earliest Strathfield candidate
candidate = (
    strathfield_candidates
    .sort_values("arrival_seconds")
    .iloc[0]
)

print("First transfer:", candidate["first_transfer"])
print("Second transfer:", candidate["station_name"])
print("Arrival at second transfer:", candidate["arrival_time"])

# Find valid services after arriving at Strathfield
third_connections = get_valid_connections(
    daily=test_timetable,
    arrival_platform=candidate["stop_id"],
    arrival_seconds=candidate["arrival_seconds"],
    connection_window_minutes=30
)

# Exclude services already used
third_connections = third_connections[
    ~third_connections["trip_id"].isin([
        candidate["first_trip_id"],
        candidate["second_trip_id"]
    ])
].copy()

print("Valid third-leg boarding options:", len(third_connections))

# Find services that continue to Parramatta
third_leg_journeys = []

for _, connection in third_connections.iterrows():

    downstream = get_downstream_stops(
        daily=test_timetable,
        trip_id=connection["trip_id"],
        boarding_sequence=connection["stop_sequence"]
    )

    destination_stops = downstream[
        downstream["station_name"] == "Parramatta Station"
    ]

    for _, destination_stop in destination_stops.iterrows():

        third_leg_journeys.append({
            "third_trip_id": connection["trip_id"],
            "third_departure_time": connection["departure_time"],
            "destination_arrival_time": destination_stop["arrival_time"],
            "journey_duration_min": (
                destination_stop["arrival_seconds"]
                - candidate["origin_departure_seconds"]
            ) / 60,
            "second_spare_minutes": (
                connection["departure_seconds"]
                - candidate["arrival_seconds"]
                - connection["minimum_transfer_seconds"]
            ) / 60
        })

third_leg_results = pd.DataFrame(third_leg_journeys)

print("Third-leg journeys reaching Parramatta:", len(third_leg_results))

if not third_leg_results.empty:
    display(
        third_leg_results
        .sort_values("destination_arrival_time")
        .head(10)
    )

First transfer: Epping Station
Second transfer: Strathfield Station
Arrival at second transfer: 18:03:00
Valid third-leg boarding options: 25
Third-leg journeys reaching Parramatta: 8


,third_trip_id,third_departure_time,destination_arrival_time,journey_duration_min,second_spare_minutes
0,197F.1324.165.16.B.8.92001727,18:11:24,18:23:00,52.383333,3.4
2,145N.1324.165.16.A.8.92001340,18:17:24,18:29:00,58.383333,9.4
3,165F.1324.165.124.A.8.92002689,18:20:24,18:32:00,61.383333,12.4
1,78-P.1324.165.16.A.8.92002733,18:16:00,18:36:30,65.883333,6.6
4,155N.1324.165.16.A.8.92002106,18:26:24,18:38:00,67.383333,18.4
5,111U.1324.165.16.T.8.92004513,18:29:24,18:41:00,70.383333,21.4
7,154N.1324.165.16.A.8.92001114,18:32:24,18:44:00,73.383333,24.4
6,92-P.1324.165.60.A.8.92004162,18:31:00,18:51:30,80.883333,21.6


In [ ]:
# STEP 76D — Vectorised third-leg journey discovery

def find_third_leg_journeys(
    daily,
    candidates,
    destination,
    max_connection_minutes=30
):
    if candidates.empty:
        return pd.DataFrame()

    # Prepare potential third-leg boarding events
    boarding = daily[
        [
            "trip_id",
            "stop_id",
            "stop_sequence",
            "departure_time",
            "departure_seconds"
        ]
    ].rename(columns={
        "trip_id": "third_trip_id",
        "stop_id": "third_boarding_platform",
        "stop_sequence": "third_boarding_sequence",
        "departure_time": "third_departure_time",
        "departure_seconds": "third_departure_seconds"
    })

    # Platform transfer rules
    transfers = all_platform_transfers[
        [
            "from_stop_id",
            "to_stop_id",
            "minimum_transfer_seconds"
        ]
    ].drop_duplicates()

    # Match second-leg arrivals to transfer pathways
    connections = candidates.merge(
        transfers,
        left_on="stop_id",
        right_on="from_stop_id",
        how="inner"
    )

    # Match pathways to third-leg departures
    connections = connections.merge(
        boarding,
        left_on="to_stop_id",
        right_on="third_boarding_platform",
        how="inner"
    )

    # Keep only valid connection times
    earliest_boarding = (
        connections["arrival_seconds"]
        + connections["minimum_transfer_seconds"]
    )

    latest_boarding = (
        connections["arrival_seconds"]
        + max_connection_minutes * 60
    )

    connections = connections[
        (connections["third_departure_seconds"] >= earliest_boarding)
        & (connections["third_departure_seconds"] <= latest_boarding)
        & (connections["third_trip_id"] != connections["first_trip_id"])
        & (connections["third_trip_id"] != connections["second_trip_id"])
    ].copy()

    # Prepare destination arrivals
    destination_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == destination,
            "station_id"
        ]
    )

    arrivals = daily[
        daily["station_id"].isin(destination_ids)
    ][
        [
            "trip_id",
            "stop_sequence",
            "arrival_time",
            "arrival_seconds"
        ]
    ].rename(columns={
        "trip_id": "third_trip_id",
        "stop_sequence": "destination_sequence",
        "arrival_time": "destination_arrival_time",
        "arrival_seconds": "destination_arrival_seconds"
    })

    # Find third-leg services reaching destination
    results = connections.merge(
        arrivals,
        on="third_trip_id",
        how="inner"
    )

    # Destination must occur after boarding
    results = results[
        results["destination_sequence"]
        > results["third_boarding_sequence"]
    ].copy()

    results["journey_duration_min"] = (
        results["destination_arrival_seconds"]
        - results["origin_departure_seconds"]
    ) / 60

    results["second_spare_minutes"] = (
        results["third_departure_seconds"]
        - results["arrival_seconds"]
        - results["minimum_transfer_seconds"]
    ) / 60

    return results.sort_values(
        ["destination_arrival_seconds", "journey_duration_min"]
    ).reset_index(drop=True)


# Test only Strathfield candidates first
third_leg_test = find_third_leg_journeys(
    daily=test_timetable,
    candidates=strathfield_candidates,
    destination="Parramatta Station"
)

print("Third-leg journeys found:", len(third_leg_test))

display(
    third_leg_test[
        [
            "first_transfer",
            "station_name",
            "third_departure_time",
            "destination_arrival_time",
            "journey_duration_min",
            "first_spare_minutes",
            "second_spare_minutes"
        ]
    ].head(15)
)

Third-leg journeys found: 731


,first_transfer,station_name,third_departure_time,destination_arrival_time,journey_duration_min,first_spare_minutes,second_spare_minutes
0,Epping Station,Strathfield Station,18:11:24,18:23:00,52.383333,2.650000,3.4
1,Epping Station,Strathfield Station,18:17:24,18:29:00,46.383333,1.666667,2.2
2,Epping Station,Strathfield Station,18:17:24,18:29:00,50.383333,5.666667,2.2
3,Epping Station,Strathfield Station,18:17:24,18:29:00,54.383333,9.666667,2.2
4,Epping Station,Strathfield Station,18:17:24,18:29:00,58.383333,2.650000,9.4
5,Epping Station,Strathfield Station,18:17:24,18:29:00,58.383333,13.666667,2.2
6,Epping Station,Strathfield Station,18:20:24,18:32:00,49.383333,1.666667,5.2
7,Epping Station,Strathfield Station,18:20:24,18:32:00,53.383333,5.666667,5.2
8,Epping Station,Strathfield Station,18:20:24,18:32:00,57.383333,9.666667,5.2
9,Central Station,Strathfield Station,18:20:24,18:32:00,60.100000,1.766667,1.9


In [ ]:
# STEP 76E — Search all second-transfer stations

import time

start_time = time.perf_counter()

# Run the vectorised search across every candidate
all_two_transfer_journeys = find_third_leg_journeys(
    daily=test_timetable,
    candidates=second_transfer_candidates,
    destination="Parramatta Station",
    max_connection_minutes=30
)

# Keep journeys lasting no more than 120 minutes
all_two_transfer_journeys = all_two_transfer_journeys[
    (all_two_transfer_journeys["journey_duration_min"] > 0)
    & (all_two_transfer_journeys["journey_duration_min"] <= 120)
].copy()

# Remove duplicate itinerary records
all_two_transfer_journeys = (
    all_two_transfer_journeys
    .drop_duplicates(
        subset=[
            "first_trip_id",
            "second_trip_id",
            "third_trip_id",
            "origin_departure_seconds",
            "destination_arrival_seconds"
        ]
    )
    .sort_values(
        ["destination_arrival_seconds", "journey_duration_min"]
    )
    .reset_index(drop=True)
)

elapsed = time.perf_counter() - start_time

print(f"Search completed in {elapsed:.2f} seconds")
print("Two-transfer journeys found:", len(all_two_transfer_journeys))

print(
    "Unique first-transfer stations:",
    all_two_transfer_journeys["first_transfer"].nunique()
)

print(
    "Unique second-transfer stations:",
    all_two_transfer_journeys["station_name"].nunique()
)

display(
    all_two_transfer_journeys[
        [
            "first_transfer",
            "station_name",
            "destination_arrival_time",
            "journey_duration_min",
            "first_spare_minutes",
            "second_spare_minutes"
        ]
    ].head(15)
)

Search completed in 1.17 seconds
Two-transfer journeys found: 2935
Unique first-transfer stations: 11
Unique second-transfer stations: 17


,first_transfer,station_name,destination_arrival_time,journey_duration_min,first_spare_minutes,second_spare_minutes
0,Epping Station,Strathfield Station,18:23:00,52.383333,2.650000,3.400000
1,Epping Station,Strathfield Station,18:29:00,46.383333,1.666667,2.200000
2,Epping Station,Strathfield Station,18:29:00,50.383333,5.666667,2.200000
3,Epping Station,Strathfield Station,18:29:00,54.383333,9.666667,2.200000
4,Epping Station,Strathfield Station,18:29:00,58.383333,2.650000,9.400000
5,Epping Station,Strathfield Station,18:29:00,58.383333,13.666667,2.200000
6,Epping Station,Strathfield Station,18:32:00,49.383333,1.666667,5.200000
7,Epping Station,Strathfield Station,18:32:00,53.383333,5.666667,5.200000
8,Epping Station,Strathfield Station,18:32:00,57.383333,9.666667,5.200000
9,Macquarie Park Station,Central Station,18:32:00,60.100000,2.500000,0.766667


In [ ]:
# STEP 77A — Pareto filter across one- and two-transfer journeys

# Standardise one-transfer journeys
one = one_transfer_results[
    [
        "origin_departure_seconds",
        "arrival_seconds",
        "journey_duration_min",
        "first_transfer",
        "connection_quality"
    ]
].copy()

one = one.rename(columns={
    "arrival_seconds": "destination_arrival_seconds"
})

one["second_transfer"] = None
one["transfers"] = 1

# Standardise two-transfer journeys
two = all_two_transfer_journeys[
    [
        "origin_departure_seconds",
        "destination_arrival_seconds",
        "journey_duration_min",
        "first_transfer",
        "station_name"
    ]
].copy()

two = two.rename(columns={
    "station_name": "second_transfer"
})

two["transfers"] = 2
two["connection_quality"] = "Not yet classified"

# Combine both
all_journeys = pd.concat(
    [one, two],
    ignore_index=True
)

# Remove exact duplicates
all_journeys = all_journeys.drop_duplicates()

# Pareto dominance filtering
def pareto_filter(journeys):

    df = journeys.reset_index(drop=True).copy()

    keep = []

    for i, journey in df.iterrows():

        dominated = (
            (df["origin_departure_seconds"]
             >= journey["origin_departure_seconds"])
            &
            (df["destination_arrival_seconds"]
             <= journey["destination_arrival_seconds"])
            &
            (df["transfers"] <= journey["transfers"])
            &
            (
                (df["origin_departure_seconds"]
                 > journey["origin_departure_seconds"])
                |
                (df["destination_arrival_seconds"]
                 < journey["destination_arrival_seconds"])
                |
                (df["transfers"] < journey["transfers"])
            )
        ).any()

        keep.append(not dominated)

    return df[keep].sort_values(
        ["destination_arrival_seconds", "transfers"]
    ).reset_index(drop=True)


pareto_journeys = pareto_filter(all_journeys)

print("Journeys before filtering:", len(all_journeys))
print("Journeys after filtering:", len(pareto_journeys))

display(
    pareto_journeys[
        [
            "first_transfer",
            "second_transfer",
            "journey_duration_min",
            "transfers",
            "connection_quality"
        ]
    ].head(20)
)

Journeys before filtering: 1028
Journeys after filtering: 11


,first_transfer,second_transfer,journey_duration_min,transfers,connection_quality
0,Epping Station,Strathfield Station,52.383333,2,Not yet classified
1,Central Station,None,57.100000,1,Tight
2,Epping Station,Strathfield Station,46.383333,2,Not yet classified
3,Central Station,None,56.100000,1,Tight
4,Central Station,None,58.100000,1,Moderate
5,Epping Station,Strathfield Station,51.383333,2,Not yet classified
6,Central Station,None,57.100000,1,Tight
7,Central Station,None,56.100000,1,Tight
8,Epping Station,Strathfield Station,45.383333,2,Not yet classified
9,Central Station,None,60.100000,1,Moderate


In [ ]:
# STEP 77B — Evaluate connection quality for two-transfer journeys

two_transfer_quality = all_two_transfer_journeys.copy()

# The weakest connection determines overall quality
two_transfer_quality["minimum_spare_minutes"] = (
    two_transfer_quality[
        ["first_spare_minutes", "second_spare_minutes"]
    ].min(axis=1)
)

two_transfer_quality["connection_quality"] = (
    two_transfer_quality["minimum_spare_minutes"]
    .apply(classify_connection)
)

# Match quality back to the selected two-transfer journeys
quality_lookup = (
    two_transfer_quality[
        [
            "origin_departure_seconds",
            "destination_arrival_seconds",
            "first_transfer",
            "station_name",
            "journey_duration_min",
            "connection_quality",
            "minimum_spare_minutes"
        ]
    ]
    .rename(columns={
        "station_name": "second_transfer"
    })
    .drop_duplicates(
        subset=[
            "origin_departure_seconds",
            "destination_arrival_seconds",
            "first_transfer",
            "second_transfer",
            "journey_duration_min"
        ]
    )
)

# Preserve the one-transfer quality labels
pareto_journeys_v2 = pareto_journeys.merge(
    quality_lookup,
    on=[
        "origin_departure_seconds",
        "destination_arrival_seconds",
        "first_transfer",
        "second_transfer",
        "journey_duration_min"
    ],
    how="left",
    suffixes=("", "_evaluated")
)

pareto_journeys_v2["connection_quality"] = (
    pareto_journeys_v2["connection_quality_evaluated"]
    .fillna(pareto_journeys_v2["connection_quality"])
)

pareto_journeys_v2 = pareto_journeys_v2.drop(
    columns=["connection_quality_evaluated"]
)

# Format departure and arrival times
pareto_journeys_v2["departure_time"] = (
    pareto_journeys_v2["origin_departure_seconds"]
    .apply(seconds_to_time)
)

pareto_journeys_v2["arrival_time"] = (
    pareto_journeys_v2["destination_arrival_seconds"]
    .apply(seconds_to_time)
)

display(
    pareto_journeys_v2[
        [
            "departure_time",
            "arrival_time",
            "first_transfer",
            "second_transfer",
            "journey_duration_min",
            "transfers",
            "minimum_spare_minutes",
            "connection_quality"
        ]
    ]
)

,departure_time,arrival_time,first_transfer,second_transfer,journey_duration_min,transfers,minimum_spare_minutes,connection_quality
0,17:30:37,18:23:00,Epping Station,Strathfield Station,52.383333,2,2.650000,Moderate
1,17:31:54,18:29:00,Central Station,None,57.100000,1,NaN,Tight
2,17:42:37,18:29:00,Epping Station,Strathfield Station,46.383333,2,1.666667,Tight
3,17:35:54,18:32:00,Central Station,None,56.100000,1,NaN,Tight
4,17:39:54,18:38:00,Central Station,None,58.100000,1,NaN,Moderate
5,17:46:37,18:38:00,Epping Station,Strathfield Station,51.383333,2,1.650000,Tight
6,17:43:54,18:41:00,Central Station,None,57.100000,1,NaN,Tight
7,17:47:54,18:44:00,Central Station,None,56.100000,1,NaN,Tight
8,17:58:37,18:44:00,Epping Station,Strathfield Station,45.383333,2,0.666667,Tight
9,17:55:54,18:56:00,Central Station,None,60.100000,1,NaN,Moderate


In [ ]:
# STEP 77C — Comfort-aware Pareto filtering

# Start from all journeys, before the previous Pareto filter
comfort_journeys = all_journeys.copy()

# Calculate overall connection buffer for two-transfer journeys
two_quality = all_two_transfer_journeys.copy()

two_quality["minimum_spare_minutes"] = two_quality[
    ["first_spare_minutes", "second_spare_minutes"]
].min(axis=1)

# Build a lookup, preserving the best buffer for matching itineraries
two_lookup = (
    two_quality.groupby(
        [
            "origin_departure_seconds",
            "destination_arrival_seconds",
            "first_transfer",
            "station_name",
            "journey_duration_min"
        ],
        as_index=False
    )["minimum_spare_minutes"]
    .max()
    .rename(columns={"station_name": "second_transfer"})
)

# Add two-transfer connection buffers
comfort_journeys = comfort_journeys.merge(
    two_lookup,
    on=[
        "origin_departure_seconds",
        "destination_arrival_seconds",
        "first_transfer",
        "second_transfer",
        "journey_duration_min"
    ],
    how="left"
)

# Add one-transfer connection buffers
one_lookup = (
    one_transfer_results.groupby(
        [
            "origin_departure_seconds",
            "arrival_seconds",
            "first_transfer",
            "journey_duration_min"
        ],
        as_index=False
    )["first_spare_minutes"]
    .max()
    .rename(columns={
        "arrival_seconds": "destination_arrival_seconds",
        "first_spare_minutes": "one_transfer_spare"
    })
)

comfort_journeys = comfort_journeys.merge(
    one_lookup,
    on=[
        "origin_departure_seconds",
        "destination_arrival_seconds",
        "first_transfer",
        "journey_duration_min"
    ],
    how="left"
)

comfort_journeys["minimum_spare_minutes"] = (
    comfort_journeys["minimum_spare_minutes"]
    .fillna(comfort_journeys["one_transfer_spare"])
)

comfort_journeys["connection_quality"] = (
    comfort_journeys["minimum_spare_minutes"]
    .apply(
        lambda x: classify_connection(x)
        if pd.notna(x) else "Unknown"
    )
)

# Convert comfort labels into numerical scores
comfort_scores = {
    "Unknown": 0,
    "Tight": 1,
    "Moderate": 2,
    "Comfortable": 3
}

comfort_journeys["comfort_score"] = (
    comfort_journeys["connection_quality"].map(comfort_scores)
)

# Vectorised dominance check using NumPy
import numpy as np

def comfort_pareto_filter(df):

    data = df.reset_index(drop=True).copy()

    departure = data["origin_departure_seconds"].to_numpy()
    arrival = data["destination_arrival_seconds"].to_numpy()
    transfers = data["transfers"].to_numpy()
    comfort = data["comfort_score"].to_numpy()

    keep = np.ones(len(data), dtype=bool)

    for i in range(len(data)):

        dominates = (
            (departure >= departure[i])
            & (arrival <= arrival[i])
            & (transfers <= transfers[i])
            & (comfort >= comfort[i])
            & (
                (departure > departure[i])
                | (arrival < arrival[i])
                | (transfers < transfers[i])
                | (comfort > comfort[i])
            )
        )

        if dominates.any():
            keep[i] = False

    return data.loc[keep].sort_values(
        ["destination_arrival_seconds", "transfers"]
    ).reset_index(drop=True)


comfort_pareto_journeys = comfort_pareto_filter(comfort_journeys)

comfort_pareto_journeys["departure_time"] = (
    comfort_pareto_journeys["origin_departure_seconds"]
    .apply(seconds_to_time)
)

comfort_pareto_journeys["arrival_time"] = (
    comfort_pareto_journeys["destination_arrival_seconds"]
    .apply(seconds_to_time)
)

print("Journeys before filtering:", len(comfort_journeys))
print("Journeys after comfort-aware filtering:", len(comfort_pareto_journeys))

display(
    comfort_pareto_journeys[
        [
            "departure_time",
            "arrival_time",
            "first_transfer",
            "second_transfer",
            "transfers",
            "minimum_spare_minutes",
            "connection_quality"
        ]
    ].head(25)
)

Journeys before filtering: 1028
Journeys after comfort-aware filtering: 25


,departure_time,arrival_time,first_transfer,second_transfer,transfers,minimum_spare_minutes,connection_quality
0,17:30:37,18:23:00,Epping Station,Strathfield Station,2,2.650000,Moderate
1,17:31:54,18:29:00,Central Station,None,1,1.766667,Tight
2,17:42:37,18:29:00,Epping Station,Strathfield Station,2,1.666667,Tight
3,17:38:37,18:29:00,Epping Station,Strathfield Station,2,2.200000,Moderate
4,17:35:54,18:32:00,Central Station,None,1,0.766667,Tight
5,17:31:54,18:32:00,Central Station,None,1,4.766667,Moderate
6,17:38:37,18:32:00,Epping Station,Strathfield Station,2,5.200000,Comfortable
7,17:39:54,18:38:00,Central Station,None,1,2.766667,Moderate
8,17:35:54,18:38:00,Central Station,None,1,6.766667,Comfortable
9,17:46:37,18:38:00,Epping Station,Strathfield Station,2,1.650000,Tight


In [ ]:
# STEP 78 — Passenger-friendly journey recommendations

def recommend_best_journeys(journeys, max_extra_minutes=15):

    if journeys.empty:
        return pd.DataFrame()

    df = journeys.copy().reset_index(drop=True)

    selected = []
    used = set()

    # 1. Fastest arrival
    fastest_idx = df.sort_values(
        ["destination_arrival_seconds", "transfers"]
    ).index[0]

    fastest = df.loc[fastest_idx].copy()
    fastest["recommendation"] = "Fastest arrival"

    selected.append(fastest)
    used.add(fastest_idx)

    # 2. Fewest transfers
    remaining = df.drop(index=list(used))

    if not remaining.empty:

        simple_idx = remaining.sort_values(
            ["transfers", "destination_arrival_seconds"]
        ).index[0]

        simple = df.loc[simple_idx].copy()
        simple["recommendation"] = "Fewest transfers"

        selected.append(simple)
        used.add(simple_idx)

    # 3. Comfortable alternative
    latest_arrival = (
        fastest["destination_arrival_seconds"]
        + max_extra_minutes * 60
    )

    remaining = df.drop(index=list(used))

    comfortable = remaining[
        (remaining["destination_arrival_seconds"] <= latest_arrival)
        & (remaining["connection_quality"] == "Comfortable")
    ].copy()

    if not comfortable.empty:

        # Prefer fewer transfers, then earlier arrival
        comfort_idx = comfortable.sort_values(
            ["transfers", "destination_arrival_seconds"]
        ).index[0]

        comfort = df.loc[comfort_idx].copy()
        comfort["recommendation"] = "Comfortable alternative"

        selected.append(comfort)

    result = pd.DataFrame(selected)

    columns = [
        "recommendation",
        "departure_time",
        "arrival_time",
        "first_transfer",
        "second_transfer",
        "journey_duration_min",
        "transfers",
        "minimum_spare_minutes",
        "connection_quality"
    ]

    return result[columns].reset_index(drop=True)


final_recommendations = recommend_best_journeys(
    comfort_pareto_journeys,
    max_extra_minutes=15
)

display(final_recommendations)

,recommendation,departure_time,arrival_time,first_transfer,second_transfer,journey_duration_min,transfers,minimum_spare_minutes,connection_quality
0,Fastest arrival,17:30:37,18:23:00,Epping Station,Strathfield Station,52.383333,2,2.650000,Moderate
1,Fewest transfers,17:31:54,18:29:00,Central Station,NaN,57.100000,1,1.766667,Tight
2,Comfortable alternative,17:35:54,18:38:00,Central Station,NaN,62.100000,1,6.766667,Comfortable


In [ ]:
# STEP 79A — Exclude tight connections from recommendations

def filter_reliable_journeys(journeys, min_spare_minutes=2):

    reliable = journeys[
        (journeys["minimum_spare_minutes"] >= min_spare_minutes)
        & (journeys["connection_quality"].isin(
            ["Moderate", "Comfortable"]
        ))
    ].copy()

    return reliable.reset_index(drop=True)


reliable_journeys = filter_reliable_journeys(
    comfort_pareto_journeys,
    min_spare_minutes=2
)

print("Journeys before filtering:", len(comfort_pareto_journeys))
print("Reliable journeys:", len(reliable_journeys))
print(
    "Tight or unevaluated journeys excluded:",
    len(comfort_pareto_journeys) - len(reliable_journeys)
)

reliable_recommendations = recommend_best_journeys(
    reliable_journeys,
    max_extra_minutes=15
)

display(reliable_recommendations)

Journeys before filtering: 25
Reliable journeys: 18
Tight or unevaluated journeys excluded: 7


,recommendation,departure_time,arrival_time,first_transfer,second_transfer,journey_duration_min,transfers,minimum_spare_minutes,connection_quality
0,Fastest arrival,17:30:37,18:23:00,Epping Station,Strathfield Station,52.383333,2,2.650000,Moderate
1,Fewest transfers,17:31:54,18:32:00,Central Station,NaN,60.100000,1,4.766667,Moderate
2,Comfortable alternative,17:35:54,18:38:00,Central Station,NaN,62.100000,1,6.766667,Comfortable


In [ ]:
# STEP 79B — Reusable reliability filtering

def filter_reliable_journeys_v2(
    journeys,
    min_spare_minutes=2
):
    if journeys.empty:
        return journeys.copy()

    df = journeys.copy()

    # Direct journeys do not require connection buffers
    direct = df["transfers"] == 0

    # Transfer journeys must meet the reliability threshold
    reliable_transfers = (
        (df["transfers"] > 0)
        & (df["minimum_spare_minutes"] >= min_spare_minutes)
    )

    filtered = df[
        direct | reliable_transfers
    ].copy()

    return filtered.sort_values(
        [
            "destination_arrival_seconds",
            "transfers",
            "journey_duration_min"
        ]
    ).reset_index(drop=True)


# Test against existing results
reliable_journeys_v2 = filter_reliable_journeys_v2(
    comfort_pareto_journeys,
    min_spare_minutes=2
)

print("Journeys before:", len(comfort_pareto_journeys))
print("Journeys after:", len(reliable_journeys_v2))

print(
    "Tight connections remaining:",
    (
        (reliable_journeys_v2["transfers"] > 0)
        & (reliable_journeys_v2["minimum_spare_minutes"] < 2)
    ).sum()
)

display(
    reliable_journeys_v2[
        [
            "departure_time",
            "arrival_time",
            "first_transfer",
            "second_transfer",
            "transfers",
            "minimum_spare_minutes",
            "connection_quality"
        ]
    ].head(15)
)

Journeys before: 25
Journeys after: 18
Tight connections remaining: 0


,departure_time,arrival_time,first_transfer,second_transfer,transfers,minimum_spare_minutes,connection_quality
0,17:30:37,18:23:00,Epping Station,Strathfield Station,2,2.650000,Moderate
1,17:38:37,18:29:00,Epping Station,Strathfield Station,2,2.200000,Moderate
2,17:31:54,18:32:00,Central Station,None,1,4.766667,Moderate
3,17:38:37,18:32:00,Epping Station,Strathfield Station,2,5.200000,Comfortable
4,17:39:54,18:38:00,Central Station,None,1,2.766667,Moderate
5,17:35:54,18:38:00,Central Station,None,1,6.766667,Comfortable
6,17:42:37,18:38:00,Epping Station,Strathfield Station,2,3.400000,Moderate
7,17:39:54,18:41:00,Central Station,None,1,5.766667,Comfortable
8,17:42:37,18:41:00,Epping Station,Strathfield Station,2,5.650000,Comfortable
9,17:43:54,18:44:00,Central Station,None,1,4.766667,Moderate


In [ ]:
# STEP 80A — Reliability-first journey ranking

# Start from all candidate journeys
all_reliable_candidates = filter_reliable_journeys_v2(
    comfort_journeys,
    min_spare_minutes=2
)

# Apply Pareto filtering only after excluding tight connections
reliable_pareto = comfort_pareto_filter(
    all_reliable_candidates
)

# Format departure and arrival times
reliable_pareto["departure_time"] = (
    reliable_pareto["origin_departure_seconds"]
    .apply(seconds_to_time)
)

reliable_pareto["arrival_time"] = (
    reliable_pareto["destination_arrival_seconds"]
    .apply(seconds_to_time)
)

print("Original candidates:", len(comfort_journeys))
print("Reliable candidates:", len(all_reliable_candidates))
print("After Pareto filtering:", len(reliable_pareto))

print(
    "Tight connections remaining:",
    (
        (reliable_pareto["transfers"] > 0)
        & (reliable_pareto["minimum_spare_minutes"] < 2)
    ).sum()
)

display(
    reliable_pareto[
        [
            "departure_time",
            "arrival_time",
            "first_transfer",
            "second_transfer",
            "transfers",
            "minimum_spare_minutes",
            "connection_quality"
        ]
    ].head(20)
)

Original candidates: 1028
Reliable candidates: 902
After Pareto filtering: 18
Tight connections remaining: 0


,departure_time,arrival_time,first_transfer,second_transfer,transfers,minimum_spare_minutes,connection_quality
0,17:30:37,18:23:00,Epping Station,Strathfield Station,2,2.650000,Moderate
1,17:38:37,18:29:00,Epping Station,Strathfield Station,2,2.200000,Moderate
2,17:31:54,18:32:00,Central Station,None,1,4.766667,Moderate
3,17:38:37,18:32:00,Epping Station,Strathfield Station,2,5.200000,Comfortable
4,17:39:54,18:38:00,Central Station,None,1,2.766667,Moderate
5,17:35:54,18:38:00,Central Station,None,1,6.766667,Comfortable
6,17:42:37,18:38:00,Epping Station,Strathfield Station,2,3.400000,Moderate
7,17:39:54,18:41:00,Central Station,None,1,5.766667,Comfortable
8,17:42:37,18:41:00,Epping Station,Strathfield Station,2,5.650000,Comfortable
9,17:43:54,18:44:00,Central Station,None,1,4.766667,Moderate


In [ ]:
# STEP 80B — Require at least 5 minutes of spare transfer time

MIN_SPARE_MINUTES = 5

reliable_candidates_5min = filter_reliable_journeys_v2(
    comfort_journeys,
    min_spare_minutes=MIN_SPARE_MINUTES
)

reliable_pareto_5min = comfort_pareto_filter(
    reliable_candidates_5min
)

reliable_pareto_5min["departure_time"] = (
    reliable_pareto_5min["origin_departure_seconds"]
    .apply(seconds_to_time)
)

reliable_pareto_5min["arrival_time"] = (
    reliable_pareto_5min["destination_arrival_seconds"]
    .apply(seconds_to_time)
)

print("Original candidates:", len(comfort_journeys))
print("Reliable candidates:", len(reliable_candidates_5min))
print("After Pareto filtering:", len(reliable_pareto_5min))

final_recommendations_5min = recommend_best_journeys(
    reliable_pareto_5min,
    max_extra_minutes=15
)

display(final_recommendations_5min)

Original candidates: 1028
Reliable candidates: 717
After Pareto filtering: 8


,recommendation,departure_time,arrival_time,first_transfer,second_transfer,journey_duration_min,transfers,minimum_spare_minutes,connection_quality
0,Fastest arrival,17:38:37,18:32:00,Epping Station,Strathfield Station,53.383333,2,5.200000,Comfortable
1,Fewest transfers,17:35:54,18:38:00,Central Station,NaN,62.100000,1,6.766667,Comfortable
2,Comfortable alternative,17:39:54,18:41:00,Central Station,NaN,61.100000,1,5.766667,Comfortable


In [102]:
# STEP 82A — Inspect GTFS-Realtime trip updates

import json
from pathlib import Path

# Find project root
BASE_DIR = Path.cwd()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

# Locate latest downloaded train updates
data_dir = BASE_DIR / "data" / "raw" / "train_trip_updates"

files = sorted(data_dir.glob("*.json"))

if not files:
    raise FileNotFoundError("No train trip updates found.")

latest_file = files[-1]

print("Loading:", latest_file.name)

with open(latest_file, "r", encoding="utf-8") as f:
    realtime_data = json.load(f)

entities = realtime_data.get("entity", [])

print("Total entities:", len(entities))

# Inspect first entity containing trip updates
trip_updates = [
    entity["trip_update"]
    for entity in entities
    if "trip_update" in entity
]

print("Trip updates:", len(trip_updates))

if trip_updates:
    example = trip_updates[0]

    print("\nTrip information:")
    print(json.dumps(example.get("trip", {}), indent=2))

    print("\nFirst 3 stop updates:")
    print(
        json.dumps(
            example.get("stop_time_update", [])[:3],
            indent=2
        )
    )

Loading: train_trip_updates_20261028_123122_794180.json
Total entities: 474
Trip updates: 474

Trip information:
{
  "trip_id": "Y418.1324.165.16.D.10.92003563",
  "schedule_relationship": "SCHEDULED",
  "route_id": "RTTA_REV"
}

First 3 stop updates:
[]


In [103]:
# STEP 82B — Inspect available delay information

import pandas as pd
import json

records = []

for trip in trip_updates:
    stop_updates = trip.get("stop_time_update", [])

    records.append({
        "trip_id": trip.get("trip", {}).get("trip_id"),
        "route_id": trip.get("trip", {}).get("route_id"),
        "stop_updates": len(stop_updates),
        "has_arrival_delay": any(
            "delay" in stop.get("arrival", {})
            for stop in stop_updates
        ),
        "has_departure_delay": any(
            "delay" in stop.get("departure", {})
            for stop in stop_updates
        ),
        "has_predicted_time": any(
            "time" in stop.get("arrival", {})
            or "time" in stop.get("departure", {})
            for stop in stop_updates
        )
    })

delay_summary = pd.DataFrame(records)

print("Total trips:", len(delay_summary))
print("Trips with stop updates:", (delay_summary["stop_updates"] > 0).sum())
print("Trips with arrival delays:", delay_summary["has_arrival_delay"].sum())
print("Trips with departure delays:", delay_summary["has_departure_delay"].sum())
print("Trips with predicted times:", delay_summary["has_predicted_time"].sum())

print("\nSample trips with stop updates:")
display(delay_summary[delay_summary["stop_updates"] > 0].head(10))

# Inspect an example containing stop updates
example_with_stops = next(
    (trip for trip in trip_updates if trip.get("stop_time_update")),
    None
)

if example_with_stops:
    print("\nExample stop update:")
    print(json.dumps(example_with_stops["stop_time_update"][:2], indent=2))
else:
    print("\nNo stop-level updates found in this snapshot.")

Total trips: 474
Trips with stop updates: 289
Trips with arrival delays: 248
Trips with departure delays: 195
Trips with predicted times: 113

Sample trips with stop updates:


,trip_id,route_id,stop_updates,has_arrival_delay,has_departure_delay,has_predicted_time
1,158S.1324.165.16.A.8.92000289,NSN_2k,19,True,True,False
2,138T.1324.165.16.A.8.92004873,WST_1a,1,True,True,False
3,521X.1324.165.24.A.8.92002793,CMB_1a,10,True,True,False
5,569N.1324.165.124.A.8.92004294,CMB_1a,27,True,True,False
11,536M.1324.165.16.A.8.92000424,CMB_2a,2,True,True,False
15,W709.1324.165.16.D.6.92230754,BMT_1,2,True,True,False
16,14-N.1324.165.16.B.8.92004343,IWL_1a,1,True,True,False
17,130V.1324.165.16.A.8.92003397,NTH_2a,13,True,True,False
22,W757.1324.165.16.D.6.92004310,BMT_1,2,True,True,False
23,95-M.1324.165.16.A.8.92000294,IWL_2b,1,True,True,False



Example stop update:
[
  {
    "arrival": {
      "delay": 223
    },
    "departure": {
      "delay": 223
    },
    "stop_id": "2077311",
    "schedule_relationship": "SCHEDULED"
  },
  {
    "arrival": {
      "delay": 223
    },
    "departure": {
      "delay": 223
    },
    "stop_id": "2076241",
    "schedule_relationship": "SCHEDULED"
  }
]


In [104]:
# STEP 82C — Extract stop-level train delays

delay_records = []

for trip_update in trip_updates:

    trip_info = trip_update.get("trip", {})
    trip_id = trip_info.get("trip_id")
    route_id = trip_info.get("route_id")

    for stop in trip_update.get("stop_time_update", []):

        arrival = stop.get("arrival", {})
        departure = stop.get("departure", {})

        arrival_delay = arrival.get("delay")
        departure_delay = departure.get("delay")

        delay_records.append({
            "trip_id": trip_id,
            "route_id": route_id,
            "stop_id": stop.get("stop_id"),
            "stop_sequence": stop.get("stop_sequence"),
            "arrival_delay_min": (
                arrival_delay / 60
                if arrival_delay is not None else None
            ),
            "departure_delay_min": (
                departure_delay / 60
                if departure_delay is not None else None
            )
        })

train_delays = pd.DataFrame(delay_records)

print("Total stop-level updates:", len(train_delays))
print("Arrival delays available:", train_delays["arrival_delay_min"].notna().sum())
print("Departure delays available:", train_delays["departure_delay_min"].notna().sum())

print("\nArrival delay statistics (minutes):")
display(train_delays["arrival_delay_min"].describe().round(2))

print("\nSample train delays:")
display(train_delays.head(10))

Total stop-level updates: 2408
Arrival delays available: 1980
Departure delays available: 1903

Arrival delay statistics (minutes):


count    1980.00
mean        1.01
std         2.79
min         0.00
25%         0.00
50%         0.00
75%         0.75
max        47.77
Name: arrival_delay_min, dtype: float64


Sample train delays:


,trip_id,route_id,stop_id,stop_sequence,arrival_delay_min,departure_delay_min
0,158S.1324.165.16.A.8.92000289,NSN_2k,2077311,None,3.716667,3.716667
1,158S.1324.165.16.A.8.92000289,NSN_2k,2076241,None,3.716667,3.716667
2,158S.1324.165.16.A.8.92000289,NSN_2k,2074191,None,3.716667,3.716667
3,158S.1324.165.16.A.8.92000289,NSN_2k,2074181,None,3.716667,3.716667
4,158S.1324.165.16.A.8.92000289,NSN_2k,2073161,None,3.716667,3.716667
5,158S.1324.165.16.A.8.92000289,NSN_2k,207261,None,3.716667,3.716667
6,158S.1324.165.16.A.8.92000289,NSN_2k,207191,None,3.716667,3.716667
7,158S.1324.165.16.A.8.92000289,NSN_2k,2070141,None,3.716667,3.716667
8,158S.1324.165.16.A.8.92000289,NSN_2k,2069131,None,3.716667,3.716667
9,158S.1324.165.16.A.8.92000289,NSN_2k,2067141,None,3.716667,3.216667


In [105]:
# STEP 82D — Add station names to real-time delays

from pathlib import Path
import pandas as pd

stops_path = (
    BASE_DIR
    / "data"
    / "raw"
    / "gtfs_static"
    / "gtfs_static"
    / "stops.txt"
)

stops_df = pd.read_csv(
    stops_path,
    dtype={"stop_id": str},
    low_memory=False
)

# Keep the columns we need
station_lookup = (
    stops_df[["stop_id", "stop_name"]]
    .drop_duplicates(subset=["stop_id"])
)

# Match identifier types before joining
train_delays["stop_id"] = train_delays["stop_id"].astype("string")
station_lookup["stop_id"] = station_lookup["stop_id"].astype("string")

# Join delay observations to station names
train_delays_named = train_delays.merge(
    station_lookup,
    on="stop_id",
    how="left",
    validate="many_to_one"
)

print("Total stop-level records:", len(train_delays_named))
print(
    "Records with station names:",
    train_delays_named["stop_name"].notna().sum()
)

print(
    "Station matching rate:",
    f"{train_delays_named['stop_name'].notna().mean():.1%}"
)

display(
    train_delays_named[
        [
            "trip_id",
            "route_id",
            "stop_name",
            "arrival_delay_min",
            "departure_delay_min"
        ]
    ].head(15)
)

Total stop-level records: 2408
Records with station names: 2400
Station matching rate: 99.7%


,trip_id,route_id,stop_name,arrival_delay_min,departure_delay_min
0,158S.1324.165.16.A.8.92000289,NSN_2k,"Waitara Station, Platform 1",3.716667,3.716667
1,158S.1324.165.16.A.8.92000289,NSN_2k,"Wahroonga Station, Platform 1",3.716667,3.716667
2,158S.1324.165.16.A.8.92000289,NSN_2k,"Warrawee Station, Platform 1",3.716667,3.716667
3,158S.1324.165.16.A.8.92000289,NSN_2k,"Turramurra Station, Platform 1",3.716667,3.716667
4,158S.1324.165.16.A.8.92000289,NSN_2k,"Pymble Station, Platform 1",3.716667,3.716667
5,158S.1324.165.16.A.8.92000289,NSN_2k,"Gordon Station, Platform 1",3.716667,3.716667
6,158S.1324.165.16.A.8.92000289,NSN_2k,"Killara Station, Platform 1",3.716667,3.716667
7,158S.1324.165.16.A.8.92000289,NSN_2k,"Lindfield Station, Platform 1",3.716667,3.716667
8,158S.1324.165.16.A.8.92000289,NSN_2k,"Roseville Station, Platform 1",3.716667,3.716667
9,158S.1324.165.16.A.8.92000289,NSN_2k,"Chatswood Station, Platform 1",3.716667,3.216667


In [106]:
# STEP 82E — Check real-time trip IDs against GTFS Static

import pandas as pd

trips_path = (
    BASE_DIR
    / "data"
    / "raw"
    / "gtfs_static"
    / "gtfs_static"
    / "trips.txt"
)

static_trips = pd.read_csv(
    trips_path,
    dtype={"trip_id": str, "route_id": str},
    low_memory=False
)

realtime_trip_ids = set(
    train_delays_named["trip_id"].dropna().astype(str)
)

static_trip_ids = set(
    static_trips["trip_id"].dropna().astype(str)
)

matched_trip_ids = realtime_trip_ids & static_trip_ids

print("Unique realtime trip IDs:", len(realtime_trip_ids))
print("Unique static trip IDs:", len(static_trip_ids))
print("Matching trip IDs:", len(matched_trip_ids))

print(
    "Trip ID matching rate:",
    f"{len(matched_trip_ids) / len(realtime_trip_ids):.1%}"
    if realtime_trip_ids else "N/A"
)

print("\nSample realtime trip IDs:")
print(list(sorted(realtime_trip_ids))[:5])

print("\nSample static trip IDs:")
print(list(sorted(static_trip_ids))[:5])

Unique realtime trip IDs: 289
Unique static trip IDs: 211419
Matching trip IDs: 177
Trip ID matching rate: 61.2%

Sample realtime trip IDs:
['10-S.1324.165.16.B.8.92002978', '10-T.1324.165.16.B.8.92002977', '101U.1324.165.16.T.8.92003690', '103R.1324.165.24.T.8.92003207', '107S.1324.165.16.T.8.92001820']

Sample static trip IDs:
['0160-006-101-002:1000', '0160-006-101-003:1000', '0160-006-101-004:1000', '0160-006-101-005:1000', '0160-006-101-006:1000']


In [107]:
# STEP 82F — Investigate unmatched real-time trip IDs

unmatched_trip_ids = realtime_trip_ids - static_trip_ids

trip_match_analysis = (
    train_delays_named[
        ["trip_id", "route_id"]
    ]
    .drop_duplicates()
    .copy()
)

trip_match_analysis["matched"] = (
    trip_match_analysis["trip_id"].isin(static_trip_ids)
)

print("Trip matching by route:")

route_summary = (
    trip_match_analysis
    .groupby("route_id", dropna=False)
    .agg(
        total_trips=("trip_id", "count"),
        matched_trips=("matched", "sum")
    )
    .reset_index()
)

route_summary["match_rate"] = (
    route_summary["matched_trips"]
    / route_summary["total_trips"]
    * 100
).round(1)

display(
    route_summary.sort_values(
        "total_trips",
        ascending=False
    ).head(20)
)

print("\nSample unmatched real-time trip IDs:")
for trip_id in sorted(unmatched_trip_ids)[:15]:
    print(trip_id)

print("\nSample matching real-time trip IDs:")
for trip_id in sorted(matched_trip_ids)[:10]:
    print(trip_id)

Trip matching by route:


,route_id,total_trips,matched_trips,match_rate
39,RTTA_REV,72,0,0.0
7,CCN_1c,16,16,100.0
45,SHL_1e,12,0,0.0
32,NSN_1a,12,12,100.0
36,NTH_2a,11,9,81.8
53,WST_1a,10,10,100.0
35,NTH_1a,10,10,100.0
3,BMT_1,8,7,87.5
10,CMB_1a,8,7,87.5
26,IWL_1a,7,7,100.0



Sample unmatched real-time trip IDs:
101U.1324.165.16.T.8.92003690
103R.1324.165.24.T.8.92003207
109L.1324.165.116.T.8.92000552
112M.1324.165.16.T.8.92330370
122X.1324.165.16.A.8.92001801
128U.1324.165.16.A.8.92003660
133V.1324.165.16.A.8.92004824
141X.1324.165.16.A.8.92003451
141Y.1324.165.16.A.8.92004461
142W.1324.165.16.A.8.92005089
145U.1324.165.16.A.8.92003953
151T.1324.165.16.A.8.92003849
157T.1324.165.16.A.8.92003055
157U.1324.165.16.A.8.92003054
162R.1324.165.16.A.8.92001444

Sample matching real-time trip IDs:
10-S.1324.165.16.B.8.92002978
10-T.1324.165.16.B.8.92002977
107S.1324.165.16.T.8.92001820
109K.1324.165.116.T.8.92000553
11-X.1324.165.16.B.8.92002653
110J.1324.165.112.T.8.92003331
111W.1324.165.16.T.8.92258953
120V.1324.165.16.A.8.92002737
123W.1324.165.16.A.8.92000519
124Y.1324.165.16.A.8.92002519


In [108]:
# STEP 82G — Inspect transport modes in GTFS Static

routes_path = (
    BASE_DIR
    / "data/raw/gtfs_static/gtfs_static/routes.txt"
)

routes_df = pd.read_csv(
    routes_path,
    dtype={"route_id": str},
    low_memory=False
)

print("Route types in static GTFS:")

display(
    routes_df.groupby("route_type")
    .agg(
        route_count=("route_id", "nunique")
    )
    .reset_index()
    .sort_values("route_type")
)

print("\nSample routes:")

display(
    routes_df[
        ["route_id", "route_short_name", "route_long_name", "route_type"]
    ].head(20)
)

Route types in static GTFS:


,route_type,route_count
0,2,17
1,4,24
2,106,27
3,204,163
4,205,139
5,401,1
6,700,1168
7,712,8647
8,714,45
9,900,6



Sample routes:


,route_id,route_short_name,route_long_name,route_type
0,1-SC0-1-sj2-1,SC01,Kiama to Bomaderry,700
1,1-SH1-0-sj2-2,SH100,Moss Vale to Goulburn,700
2,11-400-1-sj2-1,4001,Penrith Station to McCarthy College,712
3,11-400-2-sj2-1,4002,Swallow Dr opp Skylark Cres to Caroline Chisho...,712
4,11-400-3-sj2-1,4003,Caroline Chisholm College to York PS via Regen...,712
5,11-400-5-sj2-1,4005,Weir Rd after Seventh St to Holy Family PS,712
6,11-400-8-sj2-1,4008,Bennett Rd after Shepherd St to Penrith Christ...,712
7,11-400-9-sj2-1,4009,Xavier College to Henry Fulton PS,712
8,11-401-0-sj2-1,4010,Weir Rd after Seventh St to Caroline Chisholm ...,712
9,11-401-1-sj2-1,4011,Glenmore Park HS to St Dominic's College,712


In [109]:
# STEP 82H — Inspect rail, metro and light rail routes

selected_types = [0, 106, 204, 205, 401, 900]

rail_routes = routes_df[
    routes_df["route_type"].isin(selected_types)
].copy()

print("Total candidate rail, metro and light rail routes:", len(rail_routes))

display(
    rail_routes[
        ["route_id", "route_short_name", "route_long_name", "route_type"]
    ]
    .sort_values(["route_type", "route_short_name"])
    .groupby("route_type", group_keys=False)
    .head(8)
)

Total candidate rail, metro and light rail routes: 336


,route_id,route_short_name,route_long_name,route_type
9503,76-223-sj2-1,223,223,106
9504,76-224-sj2-1,224,224,106
9517,76-243-sj2-1,243,243,106
9518,76-244-sj2-1,244,244,106
9538,76-31-sj2-1,31,31,106
9551,76-32-sj2-1,32,32,106
9568,76-33-sj2-1,33,33,106
9576,76-34-sj2-1,34,34,106
9479,76-135-sj2-1,135,Broadmeadow to Taree Manning Mall,204
9480,76-136-sj2-1,136,Taree Manning Mall to Broadmeadow,204


In [110]:
# STEP 82I — Identify Sydney Trains, Metro and Light Rail

import pandas as pd

routes_df["route_short_name"] = (
    routes_df["route_short_name"].astype("string")
)

# Sydney Trains: T1, T2, T3, etc.
sydney_trains = routes_df[
    routes_df["route_short_name"].str.fullmatch(
        r"T\d+", na=False
    )
].copy()

# Sydney Metro
metro = routes_df[
    routes_df["route_short_name"].eq("M1")
].copy()

# Light Rail: L1, L2, L3, L4, etc.
light_rail = routes_df[
    routes_df["route_short_name"].str.fullmatch(
        r"L\d+|NLR|LX", na=False
    )
].copy()

for name, df in [
    ("Sydney Trains", sydney_trains),
    ("Sydney Metro", metro),
    ("Light Rail", light_rail)
]:
    print(f"\n{name}: {len(df)} routes")

    display(
        df[
            [
                "route_id",
                "route_short_name",
                "route_long_name",
                "route_type"
            ]
        ]
    )


Sydney Trains: 10 routes


,route_id,route_short_name,route_long_name,route_type
2320,2-T1-N-sj2-1,T1,T1 North Shore and Western Line,2
2321,2-T1-W-sj2-1,T1,T1 North Shore and Western Line,2
2322,2-T2-sj2-1,T2,T2 Leppington and Inner West Line,2
2323,2-T3-sj2-1,T3,T3 Liverpool and Inner West Line,2
2324,2-T4-sj2-1,T4,T4 Eastern Suburbs and Illawarra Line,2
2325,2-T5-sj2-1,T5,T5 Cumberland Line,2
2326,2-T6-sj2-1,T6,T6 Lidcombe and Bankstown Line,2
2327,2-T7-sj2-1,T7,T7 Olympic Park Line,2
2328,2-T8-sj2-1,T8,T8 Airport and South Line,2
2329,2-T9-sj2-1,T9,T9 Northern Line,2



Sydney Metro: 1 routes


,route_id,route_short_name,route_long_name,route_type
3272,3-M1-sj2-1,M1,M1 Metro North West and Bankstown Line,401



Light Rail: 6 routes


,route_id,route_short_name,route_long_name,route_type
10153,78-L1-sj2-1,L1,L1 Dulwich Hill Line,900
10154,78-L2-sj2-1,L2,L2 Randwick Line,900
10155,78-L3-sj2-1,L3,L3 Kingsford Line,900
10156,78-L4-sj2-1,L4,L4 Westmead and Carlingford Line,900
10157,78-LX-sj2-1,LX,LX Special Event Services,900
10158,78-NLR-sj2-1,NLR,NLR,900


In [111]:
# STEP 82J — Create unified rail transport routes

sydney_trains = sydney_trains.copy()
metro = metro.copy()
light_rail = light_rail.copy()

# Label each transport mode
sydney_trains["transport_mode"] = "Sydney Trains"
metro["transport_mode"] = "Metro"
light_rail["transport_mode"] = "Light Rail"

# Combine all three
unified_routes = pd.concat(
    [sydney_trains, metro, light_rail],
    ignore_index=True
)

# Keep relevant columns
unified_routes = unified_routes[
    [
        "route_id",
        "route_short_name",
        "route_long_name",
        "route_type",
        "transport_mode"
    ]
]

print("Total routes:", len(unified_routes))

print("\nRoutes by transport mode:")
display(
    unified_routes.groupby("transport_mode")
    .agg(route_count=("route_id", "nunique"))
    .reset_index()
)

print("\nUnified routes:")
display(unified_routes)

Total routes: 17

Routes by transport mode:


,transport_mode,route_count
0,Light Rail,6
1,Metro,1
2,Sydney Trains,10



Unified routes:


,route_id,route_short_name,route_long_name,route_type,transport_mode
0,2-T1-N-sj2-1,T1,T1 North Shore and Western Line,2,Sydney Trains
1,2-T1-W-sj2-1,T1,T1 North Shore and Western Line,2,Sydney Trains
2,2-T2-sj2-1,T2,T2 Leppington and Inner West Line,2,Sydney Trains
3,2-T3-sj2-1,T3,T3 Liverpool and Inner West Line,2,Sydney Trains
4,2-T4-sj2-1,T4,T4 Eastern Suburbs and Illawarra Line,2,Sydney Trains
5,2-T5-sj2-1,T5,T5 Cumberland Line,2,Sydney Trains
6,2-T6-sj2-1,T6,T6 Lidcombe and Bankstown Line,2,Sydney Trains
7,2-T7-sj2-1,T7,T7 Olympic Park Line,2,Sydney Trains
8,2-T8-sj2-1,T8,T8 Airport and South Line,2,Sydney Trains
9,2-T9-sj2-1,T9,T9 Northern Line,2,Sydney Trains


In [112]:
# STEP 82K — Create unified scheduled trips

import pandas as pd

trips_path = (
    BASE_DIR
    / "data/raw/gtfs_static/gtfs_static/trips.txt"
)

# Load static trips
static_trips = pd.read_csv(
    trips_path,
    dtype={
        "trip_id": str,
        "route_id": str,
        "service_id": str
    },
    low_memory=False
)

# Join trips with our selected transport routes
unified_trips = static_trips.merge(
    unified_routes[
        [
            "route_id",
            "route_short_name",
            "transport_mode"
        ]
    ],
    on="route_id",
    how="inner",
    validate="many_to_one"
)

print("Total scheduled trips:", len(unified_trips))

print("\nScheduled trips by transport mode:")
display(
    unified_trips.groupby("transport_mode")
    .agg(
        scheduled_trips=("trip_id", "nunique"),
        routes=("route_id", "nunique")
    )
    .reset_index()
)

print("\nSample scheduled trips:")
display(
    unified_trips[
        [
            "trip_id",
            "route_short_name",
            "transport_mode",
            "service_id"
        ]
    ].head(10)
)

print("\nDuplicate trip IDs:", unified_trips["trip_id"].duplicated().sum())

Total scheduled trips: 63645

Scheduled trips by transport mode:


,transport_mode,scheduled_trips,routes
0,Light Rail,12980,6
1,Metro,2367,1
2,Sydney Trains,48298,10



Sample scheduled trips:


,trip_id,route_short_name,transport_mode,service_id
0,101A.2010.103.128.T.8.91040144,T1,Sydney Trains,TA+cc000+7
1,101A.2010.103.2.T.8.91040144,T1,Sydney Trains,TA+nn+7
2,101D.2010.103.128.T.8.91040148,T1,Sydney Trains,TA+cc000+7
3,101D.2010.103.2.T.8.91040148,T1,Sydney Trains,TA+nn+7
4,101E.2010.103.128.T.8.91040150,T1,Sydney Trains,TA+cc000+7
5,101E.2010.103.2.T.8.91040150,T1,Sydney Trains,TA+nn+7
6,101H.2010.103.128.T.8.91040156,T1,Sydney Trains,TA+cc000+7
7,101H.2010.103.2.T.8.91040156,T1,Sydney Trains,TA+nn+7
8,101J.2010.103.128.T.8.91040158,T1,Sydney Trains,TA+cc000+7
9,101J.2010.103.2.T.8.91040158,T1,Sydney Trains,TA+nn+7



Duplicate trip IDs: 0


In [113]:
# STEP 82L — Create unified scheduled stop timetable

import pandas as pd

stop_times_path = (
    BASE_DIR
    / "data/raw/gtfs_static/gtfs_static/stop_times.txt"
)

# Load stop times
static_stop_times = pd.read_csv(
    stop_times_path,
    dtype={
        "trip_id": str,
        "stop_id": str
    },
    low_memory=False
)

# Keep only trips for our three transport modes
unified_timetable = static_stop_times.merge(
    unified_trips[
        [
            "trip_id",
            "route_short_name",
            "transport_mode",
            "service_id"
        ]
    ],
    on="trip_id",
    how="inner",
    validate="many_to_one"
)

# Add station names
unified_timetable = unified_timetable.merge(
    station_lookup,
    on="stop_id",
    how="left",
    validate="many_to_one"
)

print("Total scheduled stop records:", len(unified_timetable))

print("\nStop records by transport mode:")
display(
    unified_timetable.groupby("transport_mode")
    .agg(
        stop_records=("stop_id", "count"),
        unique_trips=("trip_id", "nunique"),
        unique_stops=("stop_id", "nunique")
    )
    .reset_index()
)

print("\nSample timetable:")
display(
    unified_timetable[
        [
            "transport_mode",
            "route_short_name",
            "trip_id",
            "stop_name",
            "arrival_time",
            "departure_time",
            "stop_sequence"
        ]
    ].head(15)
)

print(
    "\nStation matching rate:",
    f"{unified_timetable['stop_name'].notna().mean():.1%}"
)

Total scheduled stop records: 1013610

Stop records by transport mode:


,transport_mode,stop_records,unique_trips,unique_stops
0,Light Rail,185037,12980,126
1,Metro,68749,2367,62
2,Sydney Trains,759824,48298,466



Sample timetable:


,transport_mode,route_short_name,trip_id,stop_name,arrival_time,departure_time,stop_sequence
0,Sydney Trains,T9,101A.1324.165.16.T.8.92005031,"Gordon Station, Platform 2",05:02:00,05:02:00,1
1,Sydney Trains,T9,101A.1324.165.16.T.8.92005031,"Killara Station, Platform 1",05:03:42,05:04:12,2
2,Sydney Trains,T9,101A.1324.165.16.T.8.92005031,"Lindfield Station, Platform 1",05:05:48,05:06:18,3
3,Sydney Trains,T9,101A.1324.165.16.T.8.92005031,"Roseville Station, Platform 1",05:08:06,05:08:36,4
4,Sydney Trains,T9,101A.1324.165.16.T.8.92005031,"Chatswood Station, Platform 1",05:11:00,05:12:00,5
5,Sydney Trains,T9,101A.1324.165.16.T.8.92005031,"Artarmon Station, Platform 1",05:14:00,05:14:30,6
6,Sydney Trains,T9,101A.1324.165.16.T.8.92005031,"St Leonards Station, Platform 2",05:16:36,05:17:06,7
7,Sydney Trains,T9,101A.1324.165.16.T.8.92005031,"Wollstonecraft Station, Platform 1",05:19:00,05:19:30,8
8,Sydney Trains,T9,101A.1324.165.16.T.8.92005031,"Waverton Station, Platform 1",05:21:12,05:21:42,9
9,Sydney Trains,T9,101A.1324.165.16.T.8.92005031,"North Sydney Station, Platform 1",05:24:00,05:25:00,10



Station matching rate: 100.0%


In [114]:
# STEP 82M — Match real-time train delays to scheduled stops

# Keep the fields needed for matching
realtime_delays = train_delays_named[
    [
        "trip_id",
        "stop_id",
        "arrival_delay_min",
        "departure_delay_min"
    ]
].copy()

# Remove duplicate trip-stop pairs if any
print(
    "Duplicate realtime trip-stop pairs:",
    realtime_delays.duplicated(["trip_id", "stop_id"]).sum()
)

# Check coverage before merging
static_keys = unified_timetable[
    ["trip_id", "stop_id"]
].drop_duplicates()

realtime_keys = realtime_delays[
    ["trip_id", "stop_id"]
].drop_duplicates()

matching_keys = realtime_keys.merge(
    static_keys,
    on=["trip_id", "stop_id"],
    how="inner"
)

print("Realtime stop records:", len(realtime_keys))
print("Matching trip-stop pairs:", len(matching_keys))

print(
    "Realtime stop matching rate:",
    f"{len(matching_keys) / len(realtime_keys):.1%}"
    if len(realtime_keys) else "N/A"
)

print("\nMatching delay examples:")

display(
    realtime_delays.merge(
        unified_timetable[
            [
                "trip_id",
                "stop_id",
                "transport_mode",
                "route_short_name",
                "stop_name",
                "arrival_time",
                "departure_time"
            ]
        ],
        on=["trip_id", "stop_id"],
        how="inner"
    )[
        [
            "transport_mode",
            "route_short_name",
            "stop_name",
            "arrival_time",
            "arrival_delay_min",
            "departure_delay_min"
        ]
    ].head(15)
)

Duplicate realtime trip-stop pairs: 0
Realtime stop records: 2408
Matching trip-stop pairs: 1523
Realtime stop matching rate: 63.2%

Matching delay examples:


,transport_mode,route_short_name,stop_name,arrival_time,arrival_delay_min,departure_delay_min
0,Sydney Trains,T1,"Waitara Station, Platform 1",23:28:48,3.716667,3.716667
1,Sydney Trains,T1,"Wahroonga Station, Platform 1",23:31:00,3.716667,3.716667
2,Sydney Trains,T1,"Warrawee Station, Platform 1",23:33:00,3.716667,3.716667
3,Sydney Trains,T1,"Turramurra Station, Platform 1",23:35:00,3.716667,3.716667
4,Sydney Trains,T1,"Pymble Station, Platform 1",23:37:42,3.716667,3.716667
5,Sydney Trains,T1,"Gordon Station, Platform 1",23:40:30,3.716667,3.716667
6,Sydney Trains,T1,"Killara Station, Platform 1",23:42:42,3.716667,3.716667
7,Sydney Trains,T1,"Lindfield Station, Platform 1",23:44:48,3.716667,3.716667
8,Sydney Trains,T1,"Roseville Station, Platform 1",23:47:06,3.716667,3.716667
9,Sydney Trains,T1,"Chatswood Station, Platform 1",23:50:00,3.716667,3.216667


In [115]:
# STEP 82N — Calculate predicted stop times

import pandas as pd

# Convert GTFS HH:MM:SS to seconds since service-day midnight
def gtfs_to_seconds(time_value):
    if pd.isna(time_value):
        return None

    hours, minutes, seconds = map(int, str(time_value).split(":"))

    return hours * 3600 + minutes * 60 + seconds


# Convert seconds back to GTFS HH:MM:SS
def seconds_to_gtfs(total_seconds):
    if pd.isna(total_seconds):
        return None

    total_seconds = int(round(total_seconds))

    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


# Join only matched real-time train stops
matched_realtime = realtime_delays.merge(
    unified_timetable[
        [
            "trip_id",
            "stop_id",
            "transport_mode",
            "route_short_name",
            "stop_name",
            "arrival_time",
            "departure_time"
        ]
    ],
    on=["trip_id", "stop_id"],
    how="inner",
    validate="one_to_one"
)

# Convert scheduled times to seconds
matched_realtime["scheduled_arrival_sec"] = (
    matched_realtime["arrival_time"].map(gtfs_to_seconds)
)

matched_realtime["scheduled_departure_sec"] = (
    matched_realtime["departure_time"].map(gtfs_to_seconds)
)

# Calculate predictions where direct delay values are available
matched_realtime["predicted_arrival_sec"] = (
    matched_realtime["scheduled_arrival_sec"]
    + matched_realtime["arrival_delay_min"] * 60
)

matched_realtime["predicted_departure_sec"] = (
    matched_realtime["scheduled_departure_sec"]
    + matched_realtime["departure_delay_min"] * 60
)

matched_realtime["predicted_arrival"] = (
    matched_realtime["predicted_arrival_sec"].map(seconds_to_gtfs)
)

matched_realtime["predicted_departure"] = (
    matched_realtime["predicted_departure_sec"].map(seconds_to_gtfs)
)

print("Matched stop records:", len(matched_realtime))
print(
    "Arrival predictions calculated:",
    matched_realtime["predicted_arrival"].notna().sum()
)
print(
    "Departure predictions calculated:",
    matched_realtime["predicted_departure"].notna().sum()
)

display(
    matched_realtime[
        [
            "route_short_name",
            "stop_name",
            "arrival_time",
            "arrival_delay_min",
            "predicted_arrival",
            "departure_time",
            "departure_delay_min",
            "predicted_departure"
        ]
    ].head(15)
)

Matched stop records: 1523
Arrival predictions calculated: 1201
Departure predictions calculated: 1182


,route_short_name,stop_name,arrival_time,arrival_delay_min,predicted_arrival,departure_time,departure_delay_min,predicted_departure
0,T1,"Waitara Station, Platform 1",23:28:48,3.716667,23:32:31,23:29:18,3.716667,23:33:01
1,T1,"Wahroonga Station, Platform 1",23:31:00,3.716667,23:34:43,23:31:30,3.716667,23:35:13
2,T1,"Warrawee Station, Platform 1",23:33:00,3.716667,23:36:43,23:33:30,3.716667,23:37:13
3,T1,"Turramurra Station, Platform 1",23:35:00,3.716667,23:38:43,23:35:30,3.716667,23:39:13
4,T1,"Pymble Station, Platform 1",23:37:42,3.716667,23:41:25,23:38:12,3.716667,23:41:55
5,T1,"Gordon Station, Platform 1",23:40:30,3.716667,23:44:13,23:41:00,3.716667,23:44:43
6,T1,"Killara Station, Platform 1",23:42:42,3.716667,23:46:25,23:43:12,3.716667,23:46:55
7,T1,"Lindfield Station, Platform 1",23:44:48,3.716667,23:48:31,23:45:18,3.716667,23:49:01
8,T1,"Roseville Station, Platform 1",23:47:06,3.716667,23:50:49,23:47:36,3.716667,23:51:19
9,T1,"Chatswood Station, Platform 1",23:50:00,3.716667,23:53:43,23:51:00,3.216667,23:54:13


In [116]:
# STEP 82O — Check realtime coverage by transport mode

# Unique trips with directly reported delays
realtime_trip_ids = set(
    matched_realtime.loc[
        matched_realtime[
            ["arrival_delay_min", "departure_delay_min"]
        ].notna().any(axis=1),
        "trip_id"
    ]
)

coverage = (
    unified_trips[
        ["trip_id", "transport_mode"]
    ]
    .drop_duplicates()
    .assign(
        has_realtime=lambda df: df["trip_id"].isin(realtime_trip_ids)
    )
    .groupby("transport_mode")
    .agg(
        scheduled_trips=("trip_id", "count"),
        trips_with_realtime=("has_realtime", "sum")
    )
    .reset_index()
)

coverage["coverage_pct"] = (
    coverage["trips_with_realtime"]
    / coverage["scheduled_trips"]
    * 100
).round(2)

display(coverage)

,transport_mode,scheduled_trips,trips_with_realtime,coverage_pct
0,Light Rail,12980,0,0.00
1,Metro,2367,0,0.00
2,Sydney Trains,48298,126,0.26


In [117]:
# STEP 82P — Inspect Metro GTFS-Realtime trip updates

import json
from pathlib import Path

metro_dir = BASE_DIR / "data/raw/metro_trip_updates"

metro_files = sorted(metro_dir.glob("*.json"))

if not metro_files:
    print("No Metro realtime JSON files found.")
else:
    metro_file = metro_files[-1]

    with open(metro_file, "r", encoding="utf-8") as f:
        metro_data = json.load(f)

    metro_entities = metro_data.get("entity", [])

    metro_updates = [
        entity.get("trip_update", entity.get("tripUpdate"))
        for entity in metro_entities
        if entity.get("trip_update", entity.get("tripUpdate")) is not None
    ]

    print("File:", metro_file.name)
    print("Total entities:", len(metro_entities))
    print("Trip updates:", len(metro_updates))

    if metro_updates:
        example = metro_updates[0]

        print("\nTrip information:")
        print(json.dumps(example.get("trip", {}), indent=2))

        stop_updates = example.get(
            "stop_time_update",
            example.get("stopTimeUpdate", [])
        )

        print("\nFirst 3 stop updates:")
        print(json.dumps(stop_updates[:3], indent=2))

        print("\nTrips containing stop updates:")
        print(sum(
            bool(u.get("stop_time_update", u.get("stopTimeUpdate", [])))
            for u in metro_updates
        ))

File: metro_trip_updates_20261028_123131_289550.json
Total entities: 27
Trip updates: 27

Trip information:
{
  "trip_id": "0160-006-120-018:1000",
  "start_time": "23:30:00",
  "start_date": "20261028",
  "schedule_relationship": "SCHEDULED",
  "route_id": "SMNW_M1",
  "direction_id": 1
}

First 3 stop updates:
[
  {
    "stop_sequence": 1,
    "departure": {
      "delay": 53,
      "time": "1793190653"
    },
    "stop_id": "2155269",
    "schedule_relationship": "SCHEDULED"
  },
  {
    "stop_sequence": 2,
    "arrival": {
      "delay": 53,
      "time": "1793190776"
    },
    "departure": {
      "delay": 53,
      "time": "1793190806"
    },
    "stop_id": "2155267",
    "schedule_relationship": "SCHEDULED"
  },
  {
    "stop_sequence": 3,
    "arrival": {
      "delay": 53,
      "time": "1793190945"
    },
    "departure": {
      "delay": 53,
      "time": "1793190975"
    },
    "stop_id": "2155265",
    "schedule_relationship": "SCHEDULED"
  }
]

Trips containing stop upda

In [118]:
# STEP 82Q — Match Metro realtime trips and stops to GTFS Static

metro_records = []

for update in metro_updates:
    trip = update.get("trip", {})

    stops = update.get(
        "stop_time_update",
        update.get("stopTimeUpdate", [])
    )

    for stop in stops:
        metro_records.append({
            "trip_id": trip.get("trip_id", trip.get("tripId")),
            "route_id": trip.get("route_id", trip.get("routeId")),
            "start_date": trip.get("start_date", trip.get("startDate")),
            "stop_id": str(stop.get("stop_id", stop.get("stopId"))),
            "arrival_delay_sec": stop.get("arrival", {}).get("delay"),
            "departure_delay_sec": stop.get("departure", {}).get("delay"),
            "arrival_timestamp": stop.get("arrival", {}).get("time"),
            "departure_timestamp": stop.get("departure", {}).get("time")
        })

metro_realtime_df = pd.DataFrame(metro_records)

metro_static = unified_timetable[
    unified_timetable["transport_mode"] == "Metro"
].copy()

metro_static["stop_id"] = metro_static["stop_id"].astype(str)

metro_trip_ids = set(metro_realtime_df["trip_id"].dropna())
static_metro_trip_ids = set(metro_static["trip_id"].dropna())

matching_trips = metro_trip_ids & static_metro_trip_ids

matching_stops = metro_realtime_df[
    ["trip_id", "stop_id"]
].drop_duplicates().merge(
    metro_static[["trip_id", "stop_id"]].drop_duplicates(),
    on=["trip_id", "stop_id"],
    how="inner"
)

print("Realtime Metro trips:", len(metro_trip_ids))
print("Matching Metro trip IDs:", len(matching_trips))
print("Trip matching rate:",
      f"{len(matching_trips) / len(metro_trip_ids):.1%}"
      if metro_trip_ids else "N/A")

print("\nRealtime Metro stop updates:", len(metro_realtime_df))
print("Matching trip-stop pairs:", len(matching_stops))

print("\nSample Metro realtime data:")
display(metro_realtime_df.head(5))

Realtime Metro trips: 27
Matching Metro trip IDs: 27
Trip matching rate: 100.0%

Realtime Metro stop updates: 567
Matching trip-stop pairs: 558

Sample Metro realtime data:


,trip_id,route_id,start_date,stop_id,arrival_delay_sec,departure_delay_sec,arrival_timestamp,departure_timestamp
0,0160-006-120-018:1000,SMNW_M1,20261028,2155269,NaN,53.0,NaN,1793190653
1,0160-006-120-018:1000,SMNW_M1,20261028,2155267,53.0,53.0,1793190776,1793190806
2,0160-006-120-018:1000,SMNW_M1,20261028,2155265,53.0,53.0,1793190945,1793190975
3,0160-006-120-018:1000,SMNW_M1,20261028,2153402,53.0,53.0,1793191095,1793191125
4,0160-006-120-018:1000,SMNW_M1,20261028,2153404,53.0,53.0,1793191252,1793191282


In [119]:
# STEP 82R — Calculate Metro predicted stop times

# Join Metro realtime data to static timetable
metro_predictions = metro_realtime_df.merge(
    metro_static[
        [
            "trip_id",
            "stop_id",
            "route_short_name",
            "stop_name",
            "arrival_time",
            "departure_time"
        ]
    ],
    on=["trip_id", "stop_id"],
    how="inner",
    validate="one_to_one"
)

metro_predictions["transport_mode"] = "Metro"

# Convert GTFS scheduled times to seconds
metro_predictions["scheduled_arrival_sec"] = (
    metro_predictions["arrival_time"].map(gtfs_to_seconds)
)

metro_predictions["scheduled_departure_sec"] = (
    metro_predictions["departure_time"].map(gtfs_to_seconds)
)

# Convert delay seconds to minutes
metro_predictions["arrival_delay_min"] = (
    metro_predictions["arrival_delay_sec"] / 60
)

metro_predictions["departure_delay_min"] = (
    metro_predictions["departure_delay_sec"] / 60
)

# Calculate predicted times using direct delay values
metro_predictions["predicted_arrival_sec"] = (
    metro_predictions["scheduled_arrival_sec"]
    + metro_predictions["arrival_delay_sec"]
)

metro_predictions["predicted_departure_sec"] = (
    metro_predictions["scheduled_departure_sec"]
    + metro_predictions["departure_delay_sec"]
)

metro_predictions["predicted_arrival"] = (
    metro_predictions["predicted_arrival_sec"].map(seconds_to_gtfs)
)

metro_predictions["predicted_departure"] = (
    metro_predictions["predicted_departure_sec"].map(seconds_to_gtfs)
)

print("Matched Metro stop records:", len(metro_predictions))

print(
    "Arrival predictions:",
    metro_predictions["predicted_arrival"].notna().sum()
)

print(
    "Departure predictions:",
    metro_predictions["predicted_departure"].notna().sum()
)

print("\nMetro arrival delay statistics (minutes):")
display(metro_predictions["arrival_delay_min"].describe().round(2))

print("\nSample Metro predictions:")
display(
    metro_predictions[
        [
            "route_short_name",
            "stop_name",
            "arrival_time",
            "arrival_delay_min",
            "predicted_arrival",
            "departure_time",
            "departure_delay_min",
            "predicted_departure"
        ]
    ].head(10)
)

Matched Metro stop records: 558
Arrival predictions: 531
Departure predictions: 540

Metro arrival delay statistics (minutes):


count    531.00
mean       2.93
std        2.90
min        0.00
25%        0.00
50%        1.88
75%        6.62
max        7.42
Name: arrival_delay_min, dtype: float64


Sample Metro predictions:


,route_short_name,stop_name,arrival_time,arrival_delay_min,predicted_arrival,departure_time,departure_delay_min,predicted_departure
0,M1,"Tallawong Station, Platform 2",23:30:00,NaN,NaN,23:30:00,0.883333,23:30:53
1,M1,"Rouse Hill Station, Platform 1",23:32:03,0.883333,23:32:56,23:32:33,0.883333,23:33:26
2,M1,"Kellyville Station, Platform 1",23:34:52,0.883333,23:35:45,23:35:22,0.883333,23:36:15
3,M1,"Bella Vista Station, Platform 1",23:37:22,0.883333,23:38:15,23:37:52,0.883333,23:38:45
4,M1,"Norwest Station, Platform 1",23:39:59,0.883333,23:40:52,23:40:29,0.883333,23:41:22
5,M1,"Hills Showground Station, Platform 1",23:42:39,0.883333,23:43:32,23:43:09,0.883333,23:44:02
6,M1,"Castle Hill Station, Platform 1",23:45:10,0.883333,23:46:03,23:45:40,0.883333,23:46:33
7,M1,"Cherrybrook Station, Platform 1",23:47:55,0.883333,23:48:48,23:48:25,0.883333,23:49:18
8,M1,"Epping Station, Platform 5",23:53:28,0.883333,23:54:21,23:54:13,0.883333,23:55:06
9,M1,"Macquarie University Station, Platform 1",23:57:24,0.883333,23:58:17,23:57:54,0.883333,23:58:47


In [121]:

from pathlib import Path
import json

# Find available Light Rail realtime JSON files
light_rail_files = sorted(
    Path("data/raw").glob("*light*rail*/*.json")
)

print("Light Rail files found:", len(light_rail_files))

for file in light_rail_files:
    print(file)

# Inspect the first file
if light_rail_files:
    with open(light_rail_files[0], "r") as f:
        light_rail_data = json.load(f)

    print("\nTop-level keys:", list(light_rail_data.keys()))

    entities = light_rail_data.get("entity", [])

    print("Number of entities:", len(entities))

    if entities:
        print("\nFirst entity:")
        print(json.dumps(entities[0], indent=2)[:3000])


Light Rail files found: 0


In [122]:

# STEP 82S.1 — List available realtime data folders

raw_dir = BASE_DIR / "data" / "raw"

print("Available data folders:\n")

for folder in sorted(raw_dir.iterdir()):
    if folder.is_dir():
        json_files = list(folder.glob("*.json"))
        print(f"{folder.name}: {len(json_files)} JSON files")


Available data folders:

gtfs_static: 0 JSON files
lightrail_cbd_southeast: 1 JSON files
lightrail_innerwest: 1 JSON files
lightrail_newcastle: 1 JSON files
lightrail_parramatta: 1 JSON files
metro_trip_updates: 1 JSON files
road_incidents: 3 JSON files
run_summaries: 4 JSON files
traffic_stations: 3 JSON files
train_alerts: 3 JSON files
train_positions: 3 JSON files
train_trip_updates: 3 JSON files


In [123]:

# STEP 82S.2 — Inspect all four Light Rail realtime feeds

light_rail_folders = [
    "lightrail_cbd_southeast",
    "lightrail_innerwest",
    "lightrail_newcastle",
    "lightrail_parramatta"
]

light_rail_data = {}

for folder_name in light_rail_folders:
    folder = raw_dir / folder_name
    files = sorted(folder.glob("*.json"))

    if not files:
        print(f"{folder_name}: No JSON files found")
        continue

    with open(files[-1], "r") as f:
        data = json.load(f)

    light_rail_data[folder_name] = data

    entities = data.get("entity", [])
    trip_updates = [
        e.get("trip_update", e.get("tripUpdate"))
        for e in entities
        if e.get("trip_update", e.get("tripUpdate"))
    ]

    print(f"\n--- {folder_name} ---")
    print("Total entities:", len(entities))
    print("Trip updates:", len(trip_updates))

    if trip_updates:
        sample = trip_updates[0]
        print("Sample trip:", sample.get("trip", {}))
        print(
            "Sample stop update:",
            sample.get(
                "stop_time_update",
                sample.get("stopTimeUpdate", [])
            )[:1]
        )



--- lightrail_cbd_southeast ---
Total entities: 46
Trip updates: 46
Sample trip: {'trip_id': '47197-10812:1001', 'start_time': '23:44:00', 'start_date': '20261028', 'schedule_relationship': 'SCHEDULED', 'route_id': '1001_L2', 'direction_id': 0}
Sample stop update: [{'stop_sequence': 1, 'departure': {'delay': 0, 'time': '1793191440'}, 'stop_id': '2000449', 'schedule_relationship': 'SCHEDULED'}]

--- lightrail_innerwest ---
Total entities: 16
Trip updates: 16
Sample trip: {'trip_id': '48886-1309', 'start_time': '00:31:00', 'start_date': '20261029', 'schedule_relationship': 'SCHEDULED', 'route_id': 'IWLR-191', 'direction_id': 0}
Sample stop update: [{'stop_sequence': 1, 'arrival': {'time': '1793194260'}, 'departure': {'time': '1793194260'}, 'stop_id': '2000257', 'schedule_relationship': 'SCHEDULED'}]

--- lightrail_newcastle ---
Total entities: 14
Trip updates: 14
Sample trip: {'trip_id': '71676.261026.31.2330', 'start_time': '23:30:00', 'start_date': '20261028', 'schedule_relationship':

In [124]:

# STEP 82T — Combine Light Rail realtime stop updates

light_rail_records = []

for network, data in light_rail_data.items():
    for entity in data.get("entity", []):
        update = entity.get("trip_update", entity.get("tripUpdate"))

        if not update:
            continue

        trip = update.get("trip", {})

        stops = update.get(
            "stop_time_update",
            update.get("stopTimeUpdate", [])
        )

        for stop in stops:
            arrival = stop.get("arrival", {})
            departure = stop.get("departure", {})

            light_rail_records.append({
                "network": network,
                "trip_id": trip.get("trip_id"),
                "route_id": trip.get("route_id"),
                "start_date": trip.get("start_date"),
                "stop_id": stop.get("stop_id"),
                "stop_sequence": stop.get("stop_sequence"),
                "arrival_delay_sec": arrival.get("delay"),
                "departure_delay_sec": departure.get("delay"),
                "arrival_timestamp": arrival.get("time"),
                "departure_timestamp": departure.get("time")
            })

light_rail_realtime_df = pd.DataFrame(light_rail_records)

print("Total Light Rail stop updates:", len(light_rail_realtime_df))

print("\nStop updates by network:")
display(light_rail_realtime_df.groupby("network").size())

print("\nAvailable realtime information:")
display(
    light_rail_realtime_df.groupby("network")[
        [
            "arrival_delay_sec",
            "departure_delay_sec",
            "arrival_timestamp",
            "departure_timestamp"
        ]
    ].count()
)

print("\nSample records:")
display(light_rail_realtime_df.head(10))


Total Light Rail stop updates: 1409

Stop updates by network:


network
lightrail_cbd_southeast    660
lightrail_innerwest        368
lightrail_newcastle         84
lightrail_parramatta       297
dtype: int64


Available realtime information:


,arrival_delay_sec,departure_delay_sec,arrival_timestamp,departure_timestamp
network,,,,
lightrail_cbd_southeast,615,614,615,614
lightrail_innerwest,230,230,368,368
lightrail_newcastle,70,70,30,32
lightrail_parramatta,278,278,278,278



Sample records:


,network,trip_id,route_id,start_date,stop_id,stop_sequence,arrival_delay_sec,departure_delay_sec,arrival_timestamp,departure_timestamp
0,lightrail_cbd_southeast,47197-10812:1001,1001_L2,20261028,2000449,1,NaN,0.0,NaN,1793191440
1,lightrail_cbd_southeast,47197-10812:1001,1001_L2,20261028,2000453,2,-22.0,-22.0,1793191548,1793191553
2,lightrail_cbd_southeast,47197-10812:1001,1001_L2,20261028,2000454,3,3.0,3.0,1793191638,1793191643
3,lightrail_cbd_southeast,47197-10812:1001,1001_L2,20261028,2000456,4,39.0,39.0,1793191784,1793191789
4,lightrail_cbd_southeast,47197-10812:1001,1001_L2,20261028,2000458,5,67.0,67.0,1793191877,1793191907
5,lightrail_cbd_southeast,47197-10812:1001,1001_L2,20261028,2000443,6,113.0,113.0,1793192068,1793192073
6,lightrail_cbd_southeast,47197-10812:1001,1001_L2,20261028,2000445,7,104.0,104.0,1793192204,1793192209
7,lightrail_cbd_southeast,47197-10812:1001,1001_L2,20261028,2000447,8,145.0,145.0,1793192350,1793192400
8,lightrail_cbd_southeast,47197-10812:1001,1001_L2,20261028,2010108,9,181.0,181.0,1793192566,1793192571
9,lightrail_cbd_southeast,47197-10812:1001,1001_L2,20261028,2021101,10,237.0,237.0,1793192787,1793192837


In [125]:

# STEP 82U — Check Light Rail realtime-to-static matching

light_rail_static = unified_timetable[
    unified_timetable["route_short_name"].isin(
        ["L1", "L2", "L3", "L4", "LX", "NLR"]
    )
].copy()

static_trip_ids = set(light_rail_static["trip_id"].astype(str))
realtime_trip_ids = set(light_rail_realtime_df["trip_id"].astype(str))

matched_trip_ids = realtime_trip_ids & static_trip_ids

print("Realtime Light Rail trips:", len(realtime_trip_ids))
print("Matching static trip IDs:", len(matched_trip_ids))
print(
    "Trip matching rate:",
    round(len(matched_trip_ids) / len(realtime_trip_ids) * 100, 1),
    "%"
)

# Check matching trip-stop pairs
static_pairs = set(
    zip(
        light_rail_static["trip_id"].astype(str),
        light_rail_static["stop_id"].astype(str)
    )
)

realtime_pairs = list(
    zip(
        light_rail_realtime_df["trip_id"].astype(str),
        light_rail_realtime_df["stop_id"].astype(str)
    )
)

matching_pairs = sum(
    pair in static_pairs for pair in realtime_pairs
)

print("\nRealtime stop updates:", len(realtime_pairs))
print("Matching trip-stop pairs:", matching_pairs)
print(
    "Stop matching rate:",
    round(matching_pairs / len(realtime_pairs) * 100, 1),
    "%"
)

print("\nTrip matching by network:")
display(
    light_rail_realtime_df.assign(
        trip_matched=lambda df: df["trip_id"].astype(str).isin(static_trip_ids)
    ).groupby("network")["trip_matched"].agg(["sum", "count", "mean"])
    .round(3)
)


Realtime Light Rail trips: 95
Matching static trip IDs: 91
Trip matching rate: 95.8 %

Realtime stop updates: 1409
Matching trip-stop pairs: 1353
Stop matching rate: 96.0 %

Trip matching by network:


,sum,count,mean
network,,,
lightrail_cbd_southeast,604,660,0.915
lightrail_innerwest,368,368,1.000
lightrail_newcastle,84,84,1.000
lightrail_parramatta,297,297,1.000


In [126]:

# STEP 82V — Light Rail predicted times

import numpy as np
import pandas as pd

light_rail_predictions = light_rail_realtime_df.merge(
    light_rail_static[
        ["trip_id", "stop_id", "route_short_name", "stop_name",
         "arrival_time", "departure_time"]
    ],
    on=["trip_id", "stop_id"],
    how="inner",
    validate="one_to_one"
)

light_rail_predictions["transport_mode"] = "Light Rail"

# Convert scheduled GTFS times into seconds
for event in ["arrival", "departure"]:
    light_rail_predictions[f"scheduled_{event}_sec"] = (
        light_rail_predictions[f"{event}_time"].map(gtfs_to_seconds)
    )

    # Convert reported delay to numeric seconds
    reported_delay = pd.to_numeric(
        light_rail_predictions[f"{event}_delay_sec"],
        errors="coerce"
    )

    # Convert Unix timestamp to Sydney local datetime
    predicted_datetime = pd.to_datetime(
        light_rail_predictions[f"{event}_timestamp"],
        unit="s",
        utc=True,
        errors="coerce"
    ).dt.tz_convert("Australia/Sydney")

    # Service date is important for trips after midnight
    service_date = pd.to_datetime(
        light_rail_predictions["start_date"],
        format="%Y%m%d",
        errors="coerce"
    )

    local_midnight = (
        service_date.dt.tz_localize("Australia/Sydney")
    )

    timestamp_seconds = (
        predicted_datetime - local_midnight
    ).dt.total_seconds()

    # Recover missing delays from absolute timestamps
    recovered_delay = (
        timestamp_seconds
        - light_rail_predictions[f"scheduled_{event}_sec"]
    )

    light_rail_predictions[f"{event}_delay_sec"] = (
        reported_delay.fillna(recovered_delay)
    )

    light_rail_predictions[f"{event}_delay_min"] = (
        light_rail_predictions[f"{event}_delay_sec"] / 60
    )

    light_rail_predictions[f"predicted_{event}_sec"] = (
        light_rail_predictions[f"scheduled_{event}_sec"]
        + light_rail_predictions[f"{event}_delay_sec"]
    )

    light_rail_predictions[f"predicted_{event}"] = (
        light_rail_predictions[f"predicted_{event}_sec"]
        .map(seconds_to_gtfs)
    )

print("Matched Light Rail stop records:", len(light_rail_predictions))

print("\nAvailable predictions:")
print(
    "Arrival:",
    light_rail_predictions["predicted_arrival"].notna().sum()
)
print(
    "Departure:",
    light_rail_predictions["predicted_departure"].notna().sum()
)

print("\nArrival delay statistics by network (minutes):")
display(
    light_rail_predictions.groupby("network")["arrival_delay_min"]
    .agg(["count", "mean", "median", "max"])
    .round(2)
)

print("\nSample predictions:")
display(
    light_rail_predictions[
        ["network", "route_short_name", "stop_name",
         "arrival_time", "arrival_delay_min", "predicted_arrival"]
    ].head(10)
)


Matched Light Rail stop records: 1353

Available predictions:
Arrival: 1141
Departure: 1140

Arrival delay statistics by network (minutes):


,count,mean,median,max
network,,,,
lightrail_cbd_southeast,563,2.06,1.75,7.70
lightrail_innerwest,230,0.03,0.00,1.48
lightrail_newcastle,70,-0.03,-0.17,1.35
lightrail_parramatta,278,1.88,2.12,3.60



Sample predictions:


,network,route_short_name,stop_name,arrival_time,arrival_delay_min,predicted_arrival
0,lightrail_cbd_southeast,L2,Circular Quay Light Rail,23:44:00,NaN,NaN
1,lightrail_cbd_southeast,L2,Bridge Street Light Rail,23:46:10,-0.366667,23:45:48
2,lightrail_cbd_southeast,L2,Wynyard Light Rail,23:47:15,0.050000,23:47:18
3,lightrail_cbd_southeast,L2,QVB Light Rail,23:49:05,0.650000,23:49:44
4,lightrail_cbd_southeast,L2,Town Hall Light Rail,23:50:10,1.116667,23:51:17
5,lightrail_cbd_southeast,L2,Chinatown Light Rail,23:52:35,1.883333,23:54:28
6,lightrail_cbd_southeast,L2,Haymarket Light Rail,23:55:00,1.733333,23:56:44
7,lightrail_cbd_southeast,L2,Central Chalmers Street Light Rail,23:56:45,2.416667,23:59:10
8,lightrail_cbd_southeast,L2,Surry Hills Light Rail,23:59:45,3.016667,24:02:46
9,lightrail_cbd_southeast,L2,Moore Park Light Rail,24:02:30,3.950000,24:06:27


In [127]:

# STEP 82W — Diagnose missing Light Rail predictions

diagnostics = light_rail_predictions.copy()

diagnostics["has_timestamp"] = (
    diagnostics["arrival_timestamp"].notna()
)

diagnostics["has_reported_delay"] = (
    light_rail_realtime_df.merge(
        light_rail_static[["trip_id", "stop_id"]],
        on=["trip_id", "stop_id"],
        how="inner"
    )["arrival_delay_sec"].notna().values
)

diagnostics["has_prediction"] = (
    diagnostics["predicted_arrival"].notna()
)

print("Prediction availability by network:")

display(
    diagnostics.groupby("network")[
        ["has_timestamp", "has_reported_delay", "has_prediction"]
    ].sum()
)

print("\nExamples with timestamp but no prediction:")

display(
    diagnostics.loc[
        diagnostics["has_timestamp"]
        & ~diagnostics["has_prediction"],
        [
            "network", "trip_id", "start_date",
            "stop_name", "arrival_time",
            "arrival_timestamp", "arrival_delay_sec"
        ]
    ].head(10)
)


Prediction availability by network:


,has_timestamp,has_reported_delay,has_prediction
network,,,
lightrail_cbd_southeast,563,563,563
lightrail_innerwest,368,230,230
lightrail_newcastle,30,70,70
lightrail_parramatta,278,278,278



Examples with timestamp but no prediction:


,network,trip_id,start_date,stop_name,arrival_time,arrival_timestamp,arrival_delay_sec
604,lightrail_innerwest,48886-1309,20261029,Central Grand Concourse Light Rail,24:31:00,1793194260,NaN
605,lightrail_innerwest,48886-1309,20261029,Capitol Square Light Rail,24:34:08,1793194448,NaN
606,lightrail_innerwest,48886-1309,20261029,Paddy's Markets Light Rail,24:36:06,1793194566,NaN
607,lightrail_innerwest,48886-1309,20261029,Exhibition Centre Light Rail,24:39:00,1793194740,NaN
608,lightrail_innerwest,48886-1309,20261029,Convention Light Rail,24:40:30,1793194830,NaN
609,lightrail_innerwest,48886-1309,20261029,Pyrmont Bay Light Rail,24:41:47,1793194907,NaN
610,lightrail_innerwest,48886-1309,20261029,The Star Light Rail,24:43:00,1793194980,NaN
611,lightrail_innerwest,48886-1309,20261029,John Street Square Light Rail,24:44:30,1793195070,NaN
612,lightrail_innerwest,48886-1309,20261029,Bank Street Light Rail,24:45:45,1793195145,NaN
613,lightrail_innerwest,48886-1309,20261029,Wentworth Park Light Rail,24:47:15,1793195235,NaN


In [128]:

# STEP 82X — Check Inner West timestamps and service dates

innerwest_missing = light_rail_predictions[
    (light_rail_predictions["network"] == "lightrail_innerwest")
    & (light_rail_predictions["predicted_arrival"].isna())
    & (light_rail_predictions["arrival_timestamp"].notna())
].copy()

innerwest_missing["timestamp_sydney"] = pd.to_datetime(
    innerwest_missing["arrival_timestamp"],
    unit="s",
    utc=True,
    errors="coerce"
).dt.tz_convert("Australia/Sydney")

innerwest_missing["scheduled_seconds"] = (
    innerwest_missing["arrival_time"].map(gtfs_to_seconds)
)

print("Missing Inner West predictions:", len(innerwest_missing))

display(
    innerwest_missing[
        [
            "trip_id",
            "start_date",
            "arrival_time",
            "scheduled_seconds",
            "arrival_timestamp",
            "timestamp_sydney"
        ]
    ].head(15)
)

print("\nScheduled time conversion failures:")
print(innerwest_missing["scheduled_seconds"].isna().sum())

print("\nTimestamp conversion failures:")
print(innerwest_missing["timestamp_sydney"].isna().sum())


Missing Inner West predictions: 138


,trip_id,start_date,arrival_time,scheduled_seconds,arrival_timestamp,timestamp_sydney
604,48886-1309,20261029,24:31:00,88260,1793194260,NaT
605,48886-1309,20261029,24:34:08,88448,1793194448,NaT
606,48886-1309,20261029,24:36:06,88566,1793194566,NaT
607,48886-1309,20261029,24:39:00,88740,1793194740,NaT
608,48886-1309,20261029,24:40:30,88830,1793194830,NaT
609,48886-1309,20261029,24:41:47,88907,1793194907,NaT
610,48886-1309,20261029,24:43:00,88980,1793194980,NaT
611,48886-1309,20261029,24:44:30,89070,1793195070,NaT
612,48886-1309,20261029,24:45:45,89145,1793195145,NaT
613,48886-1309,20261029,24:47:15,89235,1793195235,NaT



Scheduled time conversion failures:
0

Timestamp conversion failures:
138


In [129]:

# STEP 82Y — Recover missing Light Rail predictions

service_date = pd.to_datetime(
    light_rail_predictions["start_date"],
    format="%Y%m%d",
    errors="coerce"
).dt.tz_localize("Australia/Sydney")

for event in ["arrival", "departure"]:

    # Convert Unix timestamp strings into numbers first
    timestamp_numeric = pd.to_numeric(
        light_rail_predictions[f"{event}_timestamp"],
        errors="coerce"
    )

    predicted_datetime = pd.to_datetime(
        timestamp_numeric,
        unit="s",
        utc=True,
        errors="coerce"
    ).dt.tz_convert("Australia/Sydney")

    # Seconds relative to the GTFS service date
    timestamp_seconds = (
        predicted_datetime - service_date
    ).dt.total_seconds()

    recovered_delay = (
        timestamp_seconds
        - light_rail_predictions[f"scheduled_{event}_sec"]
    )

    # Preserve reported delays; use timestamps when delay is missing
    reported_delay = pd.to_numeric(
        light_rail_predictions[f"{event}_delay_sec"],
        errors="coerce"
    )

    light_rail_predictions[f"{event}_delay_sec"] = (
        reported_delay.fillna(recovered_delay)
    )

    light_rail_predictions[f"{event}_delay_min"] = (
        light_rail_predictions[f"{event}_delay_sec"] / 60
    )

    light_rail_predictions[f"predicted_{event}_sec"] = (
        light_rail_predictions[f"scheduled_{event}_sec"]
        + light_rail_predictions[f"{event}_delay_sec"]
    )

    light_rail_predictions[f"predicted_{event}"] = (
        light_rail_predictions[f"predicted_{event}_sec"]
        .map(seconds_to_gtfs)
    )

print("Arrival predictions:",
      light_rail_predictions["predicted_arrival"].notna().sum())

print("Departure predictions:",
      light_rail_predictions["predicted_departure"].notna().sum())

print("\nInner West delay statistics:")
display(
    light_rail_predictions[
        light_rail_predictions["network"] == "lightrail_innerwest"
    ]["arrival_delay_min"].describe().round(2)
)

print("\nPreviously missing Inner West predictions:")
display(
    light_rail_predictions.loc[
        [604, 605, 606, 607, 608],
        ["stop_name", "arrival_time",
         "arrival_delay_min", "predicted_arrival"]
    ]
)


Arrival predictions: 1279
Departure predictions: 1278

Inner West delay statistics:


count     368.00
mean     -539.98
std       698.10
min     -1440.00
25%     -1440.00
50%        -0.22
75%         0.17
max         1.48
Name: arrival_delay_min, dtype: float64


Previously missing Inner West predictions:


,stop_name,arrival_time,arrival_delay_min,predicted_arrival
604,Central Grand Concourse Light Rail,24:31:00,-1440.0,00:31:00
605,Capitol Square Light Rail,24:34:08,-1440.0,00:34:08
606,Paddy's Markets Light Rail,24:36:06,-1440.0,00:36:06
607,Exhibition Centre Light Rail,24:39:00,-1440.0,00:39:00
608,Convention Light Rail,24:40:30,-1440.0,00:40:30


In [130]:

# STEP 82Z — Correct exact 24-hour date offsets

for event in ["arrival", "departure"]:

    delay_col = f"{event}_delay_sec"

    # Identify records that appear exactly one day early
    one_day_offset = (
        light_rail_predictions[delay_col]
        .sub(-86400)
        .abs()
        .le(60)
    )

    print(
        f"{event.title()} records with ~24-hour offset:",
        one_day_offset.sum()
    )

    # Correct only the suspected date-offset records
    light_rail_predictions.loc[
        one_day_offset, delay_col
    ] += 86400

    # Recalculate delay and predicted time
    light_rail_predictions[f"{event}_delay_min"] = (
        light_rail_predictions[delay_col] / 60
    )

    light_rail_predictions[f"predicted_{event}_sec"] = (
        light_rail_predictions[f"scheduled_{event}_sec"]
        + light_rail_predictions[delay_col]
    )

    light_rail_predictions[f"predicted_{event}"] = (
        light_rail_predictions[f"predicted_{event}_sec"]
        .map(seconds_to_gtfs)
    )

print("\nUpdated Inner West arrival delay statistics:")

display(
    light_rail_predictions.loc[
        light_rail_predictions["network"] == "lightrail_innerwest",
        "arrival_delay_min"
    ].describe().round(2)
)

print("\nCorrected examples:")

display(
    light_rail_predictions.loc[
        [604, 605, 606, 607, 608],
        [
            "stop_name",
            "arrival_time",
            "arrival_delay_min",
            "predicted_arrival"
        ]
    ]
)


Arrival records with ~24-hour offset: 138
Departure records with ~24-hour offset: 138

Updated Inner West arrival delay statistics:


count    368.00
mean       0.02
std        0.56
min       -2.62
25%       -0.07
50%        0.00
75%        0.17
max        1.48
Name: arrival_delay_min, dtype: float64


Corrected examples:


,stop_name,arrival_time,arrival_delay_min,predicted_arrival
604,Central Grand Concourse Light Rail,24:31:00,0.0,24:31:00
605,Capitol Square Light Rail,24:34:08,0.0,24:34:08
606,Paddy's Markets Light Rail,24:36:06,0.0,24:36:06
607,Exhibition Centre Light Rail,24:39:00,0.0,24:39:00
608,Convention Light Rail,24:40:30,0.0,24:40:30


In [131]:

# STEP 83A — Validate Inner West midnight corrections

sample = light_rail_predictions.loc[
    [604, 605, 606, 607, 608]
].copy()

sample["realtime_datetime_sydney"] = pd.to_datetime(
    pd.to_numeric(sample["arrival_timestamp"], errors="coerce"),
    unit="s",
    utc=True
).dt.tz_convert("Australia/Sydney")

sample["gtfs_service_date"] = pd.to_datetime(
    sample["start_date"],
    format="%Y%m%d"
)

sample["scheduled_datetime"] = (
    sample["gtfs_service_date"]
    + pd.to_timedelta(sample["scheduled_arrival_sec"], unit="s")
)

display(
    sample[
        [
            "trip_id",
            "start_date",
            "arrival_time",
            "scheduled_datetime",
            "realtime_datetime_sydney",
            "arrival_delay_min",
            "predicted_arrival"
        ]
    ]
)


,trip_id,start_date,arrival_time,scheduled_datetime,realtime_datetime_sydney,arrival_delay_min,predicted_arrival
604,48886-1309,20261029,24:31:00,2026-10-30 00:31:00,2026-10-29 00:31:00+11:00,0.0,24:31:00
605,48886-1309,20261029,24:34:08,2026-10-30 00:34:08,2026-10-29 00:34:08+11:00,0.0,24:34:08
606,48886-1309,20261029,24:36:06,2026-10-30 00:36:06,2026-10-29 00:36:06+11:00,0.0,24:36:06
607,48886-1309,20261029,24:39:00,2026-10-30 00:39:00,2026-10-29 00:39:00+11:00,0.0,24:39:00
608,48886-1309,20261029,24:40:30,2026-10-30 00:40:30,2026-10-29 00:40:30+11:00,0.0,24:40:30


In [132]:

# STEP 83B — Examine realtime date alignment by trip

innerwest = light_rail_predictions[
    light_rail_predictions["network"] == "lightrail_innerwest"
].copy()

# Convert timestamps to Sydney local time
innerwest["realtime_datetime"] = pd.to_datetime(
    pd.to_numeric(innerwest["arrival_timestamp"], errors="coerce"),
    unit="s",
    utc=True,
    errors="coerce"
).dt.tz_convert("Australia/Sydney")

# Calculate the actual scheduled calendar datetime
service_dates = pd.to_datetime(
    innerwest["start_date"],
    format="%Y%m%d",
    errors="coerce"
)

innerwest["scheduled_datetime"] = (
    service_dates
    + pd.to_timedelta(innerwest["scheduled_arrival_sec"], unit="s")
).dt.tz_localize("Australia/Sydney")

# Compare realtime timestamp with scheduled datetime
innerwest["raw_difference_min"] = (
    innerwest["realtime_datetime"]
    - innerwest["scheduled_datetime"]
).dt.total_seconds() / 60

print("Difference between realtime and scheduled times:")
display(
    innerwest["raw_difference_min"].describe().round(2)
)

print("\nDifferences grouped by trip:")
display(
    innerwest.groupby(["trip_id", "start_date"]).agg(
        stop_updates=("stop_id", "count"),
        min_difference=("raw_difference_min", "min"),
        median_difference=("raw_difference_min", "median"),
        max_difference=("raw_difference_min", "max")
    ).round(2)
)


Difference between realtime and scheduled times:


count     368.00
mean     -539.98
std       698.10
min     -1440.00
25%     -1440.00
50%        -0.22
75%         0.17
max         1.48
Name: raw_difference_min, dtype: float64


Differences grouped by trip:


,,stop_updates,min_difference,median_difference,max_difference
trip_id,start_date,,,,
48886-1294,20261028,23,-1.57,0.23,1.03
48886-1295,20261028,23,-2.62,-1.10,0.53
48886-1296,20261028,23,-0.32,0.13,1.48
48886-1297,20261028,23,-1.20,0.17,1.13
48886-1298,20261028,23,-0.27,-0.10,0.98
48886-1299,20261028,23,-0.03,0.78,1.23
48886-1300,20261028,23,-0.23,-0.10,0.23
48886-1301,20261028,23,0.20,0.32,0.45
48886-1302,20261028,23,-0.17,-0.05,0.08


In [133]:

# STEP 83C — Flag date-mismatched Light Rail predictions

# Use the original, uncorrected timestamp differences
# to identify potential service-date mismatches.

light_rail_predictions["realtime_quality"] = "valid"

# Identify Inner West trips with an exact one-day mismatch
mismatched_trips = set(
    innerwest.loc[
        innerwest["raw_difference_min"].between(-1441, -1439),
        "trip_id"
    ]
)

mismatch_mask = (
    light_rail_predictions["network"].eq("lightrail_innerwest")
    & light_rail_predictions["trip_id"].isin(mismatched_trips)
)

light_rail_predictions.loc[
    mismatch_mask, "realtime_quality"
] = "date_mismatch"

# Don't use unverified realtime predictions for these trips
for column in [
    "predicted_arrival",
    "predicted_departure",
    "predicted_arrival_sec",
    "predicted_departure_sec"
]:
    light_rail_predictions.loc[mismatch_mask, column] = np.nan

print("Realtime quality summary:")
display(
    light_rail_predictions.groupby(
        ["network", "realtime_quality"]
    ).size()
)

print("\nUsable arrival predictions:",
      light_rail_predictions["predicted_arrival"].notna().sum())

print("Usable departure predictions:",
      light_rail_predictions["predicted_departure"].notna().sum())

print("\nFlagged trips:")
print(sorted(mismatched_trips))


Realtime quality summary:


network                  realtime_quality
lightrail_cbd_southeast  valid               604
lightrail_innerwest      date_mismatch       138
                         valid               230
lightrail_newcastle      valid                84
lightrail_parramatta     valid               297
dtype: int64


Usable arrival predictions: 1141
Usable departure predictions: 1140

Flagged trips:
['48886-1304', '48886-1305', '48886-1306', '48886-1307', '48886-1308', '48886-1309']


In [134]:

# STEP 83D — Compare the three realtime datasets

datasets = {
    "Sydney Trains": matched_realtime,
    "Metro": metro_predictions,
    "Light Rail": light_rail_predictions
}

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print("Rows:", len(df))
    print("Columns:", sorted(df.columns.tolist()))



--- Sydney Trains ---
Rows: 1523
Columns: ['arrival_delay_min', 'arrival_time', 'departure_delay_min', 'departure_time', 'predicted_arrival', 'predicted_arrival_sec', 'predicted_departure', 'predicted_departure_sec', 'route_short_name', 'scheduled_arrival_sec', 'scheduled_departure_sec', 'stop_id', 'stop_name', 'transport_mode', 'trip_id']

--- Metro ---
Rows: 558
Columns: ['arrival_delay_min', 'arrival_delay_sec', 'arrival_time', 'arrival_timestamp', 'departure_delay_min', 'departure_delay_sec', 'departure_time', 'departure_timestamp', 'predicted_arrival', 'predicted_arrival_sec', 'predicted_departure', 'predicted_departure_sec', 'route_id', 'route_short_name', 'scheduled_arrival_sec', 'scheduled_departure_sec', 'start_date', 'stop_id', 'stop_name', 'transport_mode', 'trip_id']

--- Light Rail ---
Rows: 1353
Columns: ['arrival_delay_min', 'arrival_delay_sec', 'arrival_time', 'arrival_timestamp', 'departure_delay_min', 'departure_delay_sec', 'departure_time', 'departure_timestamp', 'n

In [135]:

# STEP 83E — Combine all realtime transport predictions

common_columns = [
    "trip_id",
    "stop_id",
    "stop_name",
    "route_short_name",
    "transport_mode",
    "arrival_time",
    "departure_time",
    "scheduled_arrival_sec",
    "scheduled_departure_sec",
    "arrival_delay_min",
    "departure_delay_min",
    "predicted_arrival",
    "predicted_departure",
    "predicted_arrival_sec",
    "predicted_departure_sec",
    "realtime_quality"
]

# Make copies to preserve our original datasets
trains = matched_realtime.copy()
metro = metro_predictions.copy()
light_rail = light_rail_predictions.copy()

# Standardise transport mode names
trains["transport_mode"] = "Sydney Trains"
metro["transport_mode"] = "Metro"
light_rail["transport_mode"] = "Light Rail"

# Add quality flags where missing
trains["realtime_quality"] = "valid"
metro["realtime_quality"] = "valid"

# Combine all three datasets
unified_realtime = pd.concat(
    [
        trains[common_columns],
        metro[common_columns],
        light_rail[common_columns]
    ],
    ignore_index=True
)

print("Total realtime stop records:", len(unified_realtime))

print("\nRecords by transport mode:")
display(
    unified_realtime.groupby("transport_mode").size()
)

print("\nRealtime quality:")
display(
    unified_realtime.groupby(
        ["transport_mode", "realtime_quality"]
    ).size()
)

print("\nAvailable predictions:")
print("Arrival:", unified_realtime["predicted_arrival"].notna().sum())
print("Departure:", unified_realtime["predicted_departure"].notna().sum())

print("\nSample:")
display(unified_realtime.head(5))


Total realtime stop records: 3434

Records by transport mode:


transport_mode
Light Rail       1353
Metro             558
Sydney Trains    1523
dtype: int64


Realtime quality:


transport_mode  realtime_quality
Light Rail      date_mismatch        138
                valid               1215
Metro           valid                558
Sydney Trains   valid               1523
dtype: int64


Available predictions:
Arrival: 2873
Departure: 2862

Sample:


,trip_id,stop_id,stop_name,route_short_name,transport_mode,arrival_time,departure_time,scheduled_arrival_sec,scheduled_departure_sec,arrival_delay_min,departure_delay_min,predicted_arrival,predicted_departure,predicted_arrival_sec,predicted_departure_sec,realtime_quality
0,158S.1324.165.16.A.8.92000289,2077311,"Waitara Station, Platform 1",T1,Sydney Trains,23:28:48,23:29:18,84528,84558,3.716667,3.716667,23:32:31,23:33:01,84751.0,84781.0,valid
1,158S.1324.165.16.A.8.92000289,2076241,"Wahroonga Station, Platform 1",T1,Sydney Trains,23:31:00,23:31:30,84660,84690,3.716667,3.716667,23:34:43,23:35:13,84883.0,84913.0,valid
2,158S.1324.165.16.A.8.92000289,2074191,"Warrawee Station, Platform 1",T1,Sydney Trains,23:33:00,23:33:30,84780,84810,3.716667,3.716667,23:36:43,23:37:13,85003.0,85033.0,valid
3,158S.1324.165.16.A.8.92000289,2074181,"Turramurra Station, Platform 1",T1,Sydney Trains,23:35:00,23:35:30,84900,84930,3.716667,3.716667,23:38:43,23:39:13,85123.0,85153.0,valid
4,158S.1324.165.16.A.8.92000289,2073161,"Pymble Station, Platform 1",T1,Sydney Trains,23:37:42,23:38:12,85062,85092,3.716667,3.716667,23:41:25,23:41:55,85285.0,85315.0,valid


In [136]:

# STEP 83F — Create effective arrival and departure times

unified_realtime["effective_arrival_sec"] = (
    unified_realtime["predicted_arrival_sec"]
    .where(
        unified_realtime["realtime_quality"].eq("valid")
    )
    .fillna(unified_realtime["scheduled_arrival_sec"])
)

unified_realtime["effective_departure_sec"] = (
    unified_realtime["predicted_departure_sec"]
    .where(
        unified_realtime["realtime_quality"].eq("valid")
    )
    .fillna(unified_realtime["scheduled_departure_sec"])
)

# Identify whether the effective time uses realtime or schedule
unified_realtime["arrival_source"] = (
    unified_realtime["predicted_arrival_sec"]
    .notna()
    & unified_realtime["realtime_quality"].eq("valid")
).map({True: "realtime", False: "scheduled"})

unified_realtime["departure_source"] = (
    unified_realtime["predicted_departure_sec"]
    .notna()
    & unified_realtime["realtime_quality"].eq("valid")
).map({True: "realtime", False: "scheduled"})

print("Arrival time sources:")
display(
    unified_realtime.groupby(
        ["transport_mode", "arrival_source"]
    ).size()
)

print("\nDeparture time sources:")
display(
    unified_realtime.groupby(
        ["transport_mode", "departure_source"]
    ).size()
)

print("\nSample effective journey times:")
display(
    unified_realtime[
        [
            "transport_mode",
            "stop_name",
            "arrival_time",
            "predicted_arrival",
            "effective_arrival_sec",
            "arrival_source"
        ]
    ].head(10)
)


Arrival time sources:


transport_mode  arrival_source
Light Rail      realtime          1141
                scheduled          212
Metro           realtime           531
                scheduled           27
Sydney Trains   realtime          1201
                scheduled          322
dtype: int64


Departure time sources:


transport_mode  departure_source
Light Rail      realtime            1140
                scheduled            213
Metro           realtime             540
                scheduled             18
Sydney Trains   realtime            1182
                scheduled            341
dtype: int64


Sample effective journey times:


,transport_mode,stop_name,arrival_time,predicted_arrival,effective_arrival_sec,arrival_source
0,Sydney Trains,"Waitara Station, Platform 1",23:28:48,23:32:31,84751.0,realtime
1,Sydney Trains,"Wahroonga Station, Platform 1",23:31:00,23:34:43,84883.0,realtime
2,Sydney Trains,"Warrawee Station, Platform 1",23:33:00,23:36:43,85003.0,realtime
3,Sydney Trains,"Turramurra Station, Platform 1",23:35:00,23:38:43,85123.0,realtime
4,Sydney Trains,"Pymble Station, Platform 1",23:37:42,23:41:25,85285.0,realtime
5,Sydney Trains,"Gordon Station, Platform 1",23:40:30,23:44:13,85453.0,realtime
6,Sydney Trains,"Killara Station, Platform 1",23:42:42,23:46:25,85585.0,realtime
7,Sydney Trains,"Lindfield Station, Platform 1",23:44:48,23:48:31,85711.0,realtime
8,Sydney Trains,"Roseville Station, Platform 1",23:47:06,23:50:49,85849.0,realtime
9,Sydney Trains,"Chatswood Station, Platform 1",23:50:00,23:53:43,86023.0,realtime


In [137]:

# STEP 83G — Validate unique realtime trip-stop records

duplicate_mask = unified_realtime.duplicated(
    subset=["trip_id", "stop_id"],
    keep=False
)

duplicates = unified_realtime[duplicate_mask]

print("Total realtime stop records:", len(unified_realtime))
print("Duplicate trip-stop records:", len(duplicates))

print("\nUnique realtime trips:")
display(
    unified_realtime.groupby("transport_mode")["trip_id"]
    .nunique()
)

print("\nMissing effective times:")
print(
    "Arrival:",
    unified_realtime["effective_arrival_sec"].isna().sum()
)
print(
    "Departure:",
    unified_realtime["effective_departure_sec"].isna().sum()
)

if not duplicates.empty:
    print("\nDuplicate examples:")
    display(
        duplicates[
            ["transport_mode", "trip_id", "stop_id", "stop_name"]
        ].head(10)
    )


Total realtime stop records: 3434
Duplicate trip-stop records: 0

Unique realtime trips:


transport_mode
Light Rail        91
Metro             27
Sydney Trains    136
Name: trip_id, dtype: int64


Missing effective times:
Arrival: 0
Departure: 0


In [138]:

# STEP 83H — Merge realtime predictions into static timetable

realtime_columns = [
    "trip_id",
    "stop_id",
    "effective_arrival_sec",
    "effective_departure_sec",
    "arrival_source",
    "departure_source",
    "realtime_quality"
]

# Keep static timetable intact
journey_timetable = unified_timetable.copy()

# Standardise join keys
journey_timetable["trip_id"] = journey_timetable["trip_id"].astype(str)
journey_timetable["stop_id"] = journey_timetable["stop_id"].astype(str)

realtime_lookup = unified_realtime[realtime_columns].copy()
realtime_lookup["trip_id"] = realtime_lookup["trip_id"].astype(str)
realtime_lookup["stop_id"] = realtime_lookup["stop_id"].astype(str)

# Merge realtime observations onto static timetable
journey_timetable = journey_timetable.merge(
    realtime_lookup,
    on=["trip_id", "stop_id"],
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Static timetable rows:", len(unified_timetable))
print("Merged timetable rows:", len(journey_timetable))

print("\nMerge results:")
display(journey_timetable["_merge"].value_counts())

print("\nRealtime-enriched rows by transport mode:")
display(
    journey_timetable.loc[
        journey_timetable["_merge"] == "both"
    ].groupby("transport_mode").size()
)

print("\nSample matched records:")
display(
    journey_timetable.loc[
        journey_timetable["_merge"] == "both",
        [
            "trip_id",
            "stop_name",
            "arrival_time",
            "effective_arrival_sec",
            "arrival_source"
        ]
    ].head(10)
)


Static timetable rows: 1013610
Merged timetable rows: 1013610

Merge results:


_merge
left_only     1010176
both             3434
right_only          0
Name: count, dtype: int64


Realtime-enriched rows by transport mode:


transport_mode
Light Rail       1353
Metro             558
Sydney Trains    1523
dtype: int64


Sample matched records:


,trip_id,stop_name,arrival_time,effective_arrival_sec,arrival_source
17703,107S.1324.165.16.T.8.92001820,"Schofields Station, Platform 1",22:36:01,81361.0,scheduled
17704,107S.1324.165.16.T.8.92001820,"Quakers Hill Station, Platform 1",22:38:48,81528.0,scheduled
17705,107S.1324.165.16.T.8.92001820,"Marayong Station, Platform 1",22:42:00,81720.0,scheduled
17706,107S.1324.165.16.T.8.92001820,"Blacktown Station, Platform 1",22:45:00,81900.0,scheduled
19351,109K.1324.165.116.T.8.92000553,"Richmond Station, Platform 2",24:22:01,87723.0,realtime
19352,109K.1324.165.116.T.8.92000553,"East Richmond Station, Platform 1",24:23:30,87810.0,realtime
19353,109K.1324.165.116.T.8.92000553,"Clarendon Station, Platform 1",24:27:00,88020.0,realtime
19354,109K.1324.165.116.T.8.92000553,"Windsor Station, Platform 1",24:31:00,88260.0,realtime
19355,109K.1324.165.116.T.8.92000553,"Mulgrave Station, Platform 1",24:34:00,88440.0,realtime
19356,109K.1324.165.116.T.8.92000553,"Vineyard Station, Platform 1",24:37:30,88650.0,realtime


In [139]:

# STEP 83I — Inspect service-date information

print("Journey timetable columns:")
print(sorted(journey_timetable.columns.tolist()))

print("\nTrip table columns:")
print(sorted(unified_trips.columns.tolist()))

print("\nCalendar table columns:")
print(sorted(calendar.columns.tolist()))

print("\nCalendar dates table columns:")
print(sorted(calendar_dates.columns.tolist()))

print("\nSample trip records:")
display(unified_trips.head(3))


Journey timetable columns:
['_merge', 'arrival_source', 'arrival_time', 'departure_source', 'departure_time', 'drop_off_type', 'effective_arrival_sec', 'effective_departure_sec', 'pickup_type', 'realtime_quality', 'route_short_name', 'service_id', 'shape_dist_traveled', 'stop_headsign', 'stop_id', 'stop_name', 'stop_note', 'stop_sequence', 'timepoint', 'transport_mode', 'trip_id']

Trip table columns:
['bikes_allowed', 'block_id', 'direction_id', 'route_direction', 'route_id', 'route_short_name', 'service_id', 'shape_id', 'transport_mode', 'trip_headsign', 'trip_id', 'trip_note', 'wheelchair_accessible']

Calendar table columns:
['end_date', 'friday', 'monday', 'saturday', 'service_id', 'start_date', 'sunday', 'thursday', 'tuesday', 'wednesday']

Calendar dates table columns:
['date', 'exception_type', 'service_id']

Sample trip records:


,route_id,service_id,trip_id,shape_id,trip_headsign,direction_id,block_id,wheelchair_accessible,route_direction,trip_note,bikes_allowed,route_short_name,transport_mode
0,2-T1-N-sj2-1,TA+cc000+7,101A.2010.103.128.T.8.91040144,2-T1-N-sj2-1.13.R,Penrith Via Gordon,1,7+3112,1,Berowra to City via Gordon,NaN,NaN,T1,Sydney Trains
1,2-T1-N-sj2-1,TA+nn+7,101A.2010.103.2.T.8.91040144,2-T1-N-sj2-1.13.R,Penrith Via Gordon,1,7+3113,1,Berowra to City via Gordon,NaN,NaN,T1,Sydney Trains
2,2-T1-N-sj2-1,TA+cc000+7,101D.2010.103.128.T.8.91040148,2-T1-N-sj2-1.23.H,Lindfield,0,7+1345,1,City to Berowra via Gordon,NaN,NaN,T1,Sydney Trains


In [140]:

# STEP 83J — Inspect existing daily timetable function

import inspect

print(inspect.getsource(prepare_daily_timetable))


def prepare_daily_timetable(travel_date):
    """
    Return scheduled Train, Metro and Light Rail stop events
    operating on the requested date.
    """

    active_services = get_active_services(travel_date)

    daily = timetable[
        timetable["service_id"].isin(active_services)
    ].copy()

    daily = daily.sort_values(
        ["trip_id", "stop_sequence"]
    ).reset_index(drop=True)

    return daily



In [141]:

# STEP 83K — Build a date-aware realtime timetable

import pandas as pd

REALTIME_SNAPSHOT_DATE = pd.Timestamp("2026-10-28").date()

def prepare_realtime_timetable(travel_date):
    travel_date = pd.Timestamp(travel_date).date()

    # Start with the existing date-filtered timetable
    daily = prepare_daily_timetable(travel_date).copy()

    # Convert scheduled GTFS times to seconds
    daily["scheduled_arrival_sec"] = daily["arrival_time"].map(gtfs_to_seconds)
    daily["scheduled_departure_sec"] = daily["departure_time"].map(gtfs_to_seconds)

    # Default to scheduled times
    daily["effective_arrival_sec"] = daily["scheduled_arrival_sec"]
    daily["effective_departure_sec"] = daily["scheduled_departure_sec"]

    daily["arrival_source"] = "scheduled"
    daily["departure_source"] = "scheduled"

    # Never apply a snapshot to an unrelated travel date
    if travel_date != REALTIME_SNAPSHOT_DATE:
        return daily

    # Keep only valid realtime predictions
    lookup = unified_realtime.loc[
        unified_realtime["realtime_quality"].eq("valid"),
        [
            "trip_id",
            "stop_id",
            "predicted_arrival_sec",
            "predicted_departure_sec"
        ]
    ].copy()

    # Standardise join keys
    daily["trip_id"] = daily["trip_id"].astype(str)
    daily["stop_id"] = daily["stop_id"].astype(str)

    lookup["trip_id"] = lookup["trip_id"].astype(str)
    lookup["stop_id"] = lookup["stop_id"].astype(str)

    # Attach realtime predictions to the selected day
    daily = daily.merge(
        lookup,
        on=["trip_id", "stop_id"],
        how="left",
        validate="many_to_one"
    )

    for event in ["arrival", "departure"]:
        predicted = daily[f"predicted_{event}_sec"]

        daily[f"effective_{event}_sec"] = predicted.fillna(
            daily[f"scheduled_{event}_sec"]
        )

        daily[f"{event}_source"] = predicted.notna().map(
            {True: "realtime", False: "scheduled"}
        )

    return daily


# Test on the snapshot date and a different date
for date in ["2026-10-28", "2026-10-29"]:
    result = prepare_realtime_timetable(date)

    print(f"\nDate: {date}")
    print("Timetable rows:", len(result))
    print("Realtime arrivals:", (result["arrival_source"] == "realtime").sum())
    print("Scheduled arrivals:", (result["arrival_source"] == "scheduled").sum())


TypeError: strptime() argument 1 must be str, not datetime.date

In [142]:

# STEP 83K — Build a date-aware realtime timetable

import pandas as pd

REALTIME_SNAPSHOT_DATE = pd.Timestamp("2026-10-28").date()

def prepare_realtime_timetable(travel_date):

    # Convert input into a consistent date format
    travel_date = pd.Timestamp(travel_date).date()

    # Our existing function expects a string, not datetime.date
    date_string = travel_date.strftime("%Y-%m-%d")

    # Step 1 — Get scheduled services for the requested date
    daily = prepare_daily_timetable(date_string).copy()

    # Step 2 — Convert GTFS scheduled times into seconds
    daily["scheduled_arrival_sec"] = (
        daily["arrival_time"].map(gtfs_to_seconds)
    )

    daily["scheduled_departure_sec"] = (
        daily["departure_time"].map(gtfs_to_seconds)
    )

    # Step 3 — Default to scheduled times
    daily["effective_arrival_sec"] = daily["scheduled_arrival_sec"]
    daily["effective_departure_sec"] = daily["scheduled_departure_sec"]

    daily["arrival_source"] = "scheduled"
    daily["departure_source"] = "scheduled"

    # Step 4 — Only use realtime data on the snapshot date
    if travel_date != REALTIME_SNAPSHOT_DATE:
        return daily

    # Step 5 — Select valid realtime predictions
    lookup = unified_realtime.loc[
        unified_realtime["realtime_quality"] == "valid",
        [
            "trip_id",
            "stop_id",
            "predicted_arrival_sec",
            "predicted_departure_sec"
        ]
    ].copy()

    # Step 6 — Standardise matching columns
    daily["trip_id"] = daily["trip_id"].astype(str)
    daily["stop_id"] = daily["stop_id"].astype(str)

    lookup["trip_id"] = lookup["trip_id"].astype(str)
    lookup["stop_id"] = lookup["stop_id"].astype(str)

    # Step 7 — Merge realtime predictions with scheduled services
    daily = daily.merge(
        lookup,
        on=["trip_id", "stop_id"],
        how="left",
        validate="many_to_one"
    )

    # Step 8 — Use realtime when available, otherwise scheduled
    for event in ["arrival", "departure"]:

        predicted = daily[f"predicted_{event}_sec"]

        daily[f"effective_{event}_sec"] = (
            predicted.fillna(daily[f"scheduled_{event}_sec"])
        )

        daily[f"{event}_source"] = predicted.notna().map({
            True: "realtime",
            False: "scheduled"
        })

    return daily


# STEP 9 — Test the function

for date in ["2026-10-28", "2026-10-29"]:

    result = prepare_realtime_timetable(date)

    print(f"\nDate: {date}")
    print("Timetable rows:", len(result))

    print(
        "Realtime arrivals:",
        (result["arrival_source"] == "realtime").sum()
    )

    print(
        "Scheduled arrivals:",
        (result["arrival_source"] == "scheduled").sum()
    )



Date: 2026-10-28
Timetable rows: 59541
Realtime arrivals: 2295
Scheduled arrivals: 57246

Date: 2026-10-29
Timetable rows: 59597
Realtime arrivals: 0
Scheduled arrivals: 59597


In [143]:

# STEP 84A — Inspect existing journey search functions

import inspect

functions_to_review = [
    find_direct_journeys,
    find_one_transfer_journeys,
    find_third_leg_journeys
]

for func in functions_to_review:
    print(f"\n{'=' * 60}")
    print(f"FUNCTION: {func.__name__}")
    print(f"{'=' * 60}")
    print(inspect.getsource(func))


NameError: name 'find_third_leg_journeys' is not defined

In [144]:

# STEP 84A.1 — Find and inspect available journey functions

import inspect

function_names = [
    "find_direct_journeys",
    "find_one_transfer_journeys",
    "find_third_leg_journeys",
    "pareto_filter"
]

for name in function_names:
    func = globals().get(name)

    if callable(func):
        print(f"\n{'=' * 60}")
        print(f"FUNCTION: {name}")
        print(f"{'=' * 60}")

        try:
            print(inspect.getsource(func))
        except (OSError, TypeError):
            print("Function exists, but source code is unavailable.")
    else:
        print(f"\n{name}: NOT DEFINED")



FUNCTION: find_direct_journeys
def find_direct_journeys(
    daily,
    origin,
    destination,
    departure_time,
    window_minutes=30
):
    """
    Find direct scheduled journeys between two stations.
    """

    # 1. Get valid origin departures
    departures = get_station_departures(
        daily=daily,
        origin=origin,
        departure_time=departure_time,
        window_minutes=window_minutes
    )

    # 2. Find destination station IDs
    destination_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == destination,
            "station_id"
        ]
    )

    if not destination_ids:
        raise ValueError(f"Destination not found: {destination}")

    # 3. Find destination arrivals
    arrivals = daily[
        daily["station_id"].isin(destination_ids)
    ][
        [
            "trip_id",
            "stop_sequence",
            "arrival_seconds",
            "arrival_time"
        ]
    ].rename(columns={
        "stop_sequence":

In [145]:

# STEP 84A.2 — Export existing journey functions

import inspect
from pathlib import Path

function_names = [
    "get_station_departures",
    "find_direct_journeys",
    "find_one_transfer_journeys",
    "find_third_leg_journeys",
    "pareto_filter"
]

output = []

for name in function_names:
    func = globals().get(name)

    if callable(func):
        try:
            source = inspect.getsource(func)
        except (OSError, TypeError):
            source = f"{name}: source unavailable"
    else:
        source = f"{name}: NOT DEFINED"

    output.append(f"\n{'=' * 60}\n{name}\n{'=' * 60}\n{source}")

file_path = Path("routing_functions.txt")
file_path.write_text("\n".join(output), encoding="utf-8")

print("Saved to:", file_path.resolve())
print("File size:", file_path.stat().st_size, "bytes")


Saved to: /Users/joanne/Desktop/Traffic NSW project/notebooks/routing_functions.txt
File size: 9275 bytes


In [146]:

# STEP 84B — Make realtime timetable compatible with routing functions

realtime_daily = prepare_realtime_timetable("2026-10-28").copy()

# Preserve original scheduled times for comparison
realtime_daily["original_arrival_seconds"] = (
    realtime_daily["scheduled_arrival_sec"]
)

realtime_daily["original_departure_seconds"] = (
    realtime_daily["scheduled_departure_sec"]
)

# Use predicted times where available, scheduled times otherwise
realtime_daily["arrival_seconds"] = (
    realtime_daily["effective_arrival_sec"]
)

realtime_daily["departure_seconds"] = (
    realtime_daily["effective_departure_sec"]
)

# Validate that routing-required columns are present
required_columns = [
    "trip_id",
    "stop_id",
    "station_id",
    "stop_sequence",
    "arrival_seconds",
    "departure_seconds",
    "arrival_source",
    "departure_source"
]

missing = [
    col for col in required_columns
    if col not in realtime_daily.columns
]

print("Missing required columns:", missing)
print("Total timetable rows:", len(realtime_daily))

print("\nArrival sources:")
print(realtime_daily["arrival_source"].value_counts())

print("\nDeparture sources:")
print(realtime_daily["departure_source"].value_counts())

print("\nSample realtime predictions:")
display(
    realtime_daily.loc[
        realtime_daily["arrival_source"] == "realtime",
        [
            "stop_name",
            "arrival_time",
            "arrival_seconds",
            "original_arrival_seconds",
            "arrival_source"
        ]
    ].head(10)
)


Missing required columns: []
Total timetable rows: 59541

Arrival sources:
arrival_source
scheduled    57246
realtime      2295
Name: count, dtype: int64

Departure sources:
departure_source
scheduled    57257
realtime      2284
Name: count, dtype: int64

Sample realtime predictions:


KeyError: "['stop_name'] not in index"

In [147]:

# STEP 84B — Prepare realtime timetable for routing

realtime_daily = prepare_realtime_timetable("2026-10-28").copy()

# Keep original scheduled times
realtime_daily["original_arrival_seconds"] = (
    realtime_daily["scheduled_arrival_sec"]
)

realtime_daily["original_departure_seconds"] = (
    realtime_daily["scheduled_departure_sec"]
)

# Use realtime predictions where available
realtime_daily["arrival_seconds"] = (
    realtime_daily["effective_arrival_sec"]
)

realtime_daily["departure_seconds"] = (
    realtime_daily["effective_departure_sec"]
)

# Check required routing columns
required_columns = [
    "trip_id",
    "stop_id",
    "station_id",
    "stop_sequence",
    "arrival_seconds",
    "departure_seconds",
    "arrival_source",
    "departure_source"
]

missing = [
    col for col in required_columns
    if col not in realtime_daily.columns
]

print("Missing required columns:", missing)
print("Total timetable rows:", len(realtime_daily))

print("\nArrival sources:")
print(realtime_daily["arrival_source"].value_counts())

print("\nDeparture sources:")
print(realtime_daily["departure_source"].value_counts())

# Display sample predictions using available columns
sample_columns = [
    "trip_id",
    "stop_id",
    "arrival_time",
    "arrival_seconds",
    "original_arrival_seconds",
    "arrival_source"
]

print("\nSample realtime predictions:")

display(
    realtime_daily.loc[
        realtime_daily["arrival_source"] == "realtime",
        sample_columns
    ].head(10)
)


Missing required columns: []
Total timetable rows: 59541

Arrival sources:
arrival_source
scheduled    57246
realtime      2295
Name: count, dtype: int64

Departure sources:
departure_source
scheduled    57257
realtime      2284
Name: count, dtype: int64

Sample realtime predictions:


,trip_id,stop_id,arrival_time,arrival_seconds,original_arrival_seconds,arrival_source
379,0160-006-101-020:1000,2155267,22:32:03,81557.0,81123,realtime
380,0160-006-101-020:1000,2155265,22:34:52,81726.0,81292,realtime
381,0160-006-101-020:1000,2153402,22:37:22,81876.0,81442,realtime
382,0160-006-101-020:1000,2153404,22:39:59,82033.0,81599,realtime
383,0160-006-101-020:1000,2154264,22:42:39,82193.0,81759,realtime
384,0160-006-101-020:1000,2154262,22:45:10,82344.0,81910,realtime
385,0160-006-101-020:1000,2126159,22:47:55,82509.0,82075,realtime
386,0160-006-101-020:1000,2121225,22:53:28,82819.0,82408,realtime
387,0160-006-101-020:1000,2113351,22:57:24,83033.0,82644,realtime
388,0160-006-101-020:1000,2113341,22:59:27,83179.0,82767,realtime


In [149]:

# STEP 84C.1 — Check station lookup and timetable columns

print("STATION LOOKUP COLUMNS:")
print(station_lookup.columns.tolist())

print("\nREALTIME TIMETABLE COLUMNS:")
print(realtime_daily.columns.tolist())

print("\nSTATION LOOKUP SAMPLE:")
display(station_lookup.head(5))

print("\nMACQUARIE UNIVERSITY MATCHES:")
display(
    station_lookup[
        station_lookup.astype(str)
        .apply(
            lambda col: col.str.contains(
                "Macquarie University",
                case=False,
                na=False
            )
        )
        .any(axis=1)
    ].head(10)
)


STATION LOOKUP COLUMNS:
['stop_id', 'stop_name']

REALTIME TIMETABLE COLUMNS:
['trip_id', 'stop_id', 'arrival_time', 'departure_time', 'stop_sequence', 'route_short_name', 'transport_mode', 'service_id', 'arrival_seconds', 'departure_seconds', 'station_id', 'scheduled_arrival_sec', 'scheduled_departure_sec', 'effective_arrival_sec', 'effective_departure_sec', 'arrival_source', 'departure_source', 'predicted_arrival_sec', 'predicted_departure_sec', 'original_arrival_seconds', 'original_departure_seconds']

STATION LOOKUP SAMPLE:


,stop_id,stop_name
0,2533211,"Kiama Station, Platform 1"
1,2533212,"Kiama Station, Platform 2"
2,253330,Kiama Station
3,253330_CCP1,CCP Railway Pde N
4,253330_CCP2,CCP Railway Pde S



MACQUARIE UNIVERSITY MATCHES:


,stop_id,stop_name
98,211310,Macquarie University Station
112,211316,"Macquarie University Campus, University Av, St..."
118,2113213,"Macquarie University Station, Waterloo Rd, Sta..."
121,2113223,"Macquarie University Station, Waterloo Rd, Sta..."
126,2113249,"Macquarie University Campus, University Av, St..."
127,2113301,"Macquarie University Station, Herring Rd, Stand F"
130,2113318,"Macquarie University Station, Waterloo Rd, Sta..."
131,2113319,"Macquarie University Station, Herring Rd, Stand D"
135,2113351,"Macquarie University Station, Platform 1"
136,2113352,"Macquarie University Station, Platform 2"


In [150]:

# STEP 84C.2 — Fix station_lookup for journey routing

# Preserve the original lookup
original_station_lookup = station_lookup.copy()

# Get platform-to-station mappings from the timetable
platform_mapping = (
    realtime_daily[
        ["stop_id", "station_id"]
    ]
    .drop_duplicates()
    .copy()
)

# Standardise IDs for matching
platform_mapping["stop_id"] = platform_mapping["stop_id"].astype(str)

original_station_lookup["stop_id"] = (
    original_station_lookup["stop_id"].astype(str)
)

# Add station_id to the existing lookup
station_lookup = original_station_lookup.merge(
    platform_mapping,
    on="stop_id",
    how="left",
    validate="one_to_one"
)

# Add station-level rows where the station ID
# also appears as a stop ID in the original lookup
station_names = original_station_lookup.rename(
    columns={
        "stop_id": "station_id",
        "stop_name": "station_name"
    }
)

station_lookup = station_lookup.merge(
    station_names,
    on="station_id",
    how="left",
    validate="many_to_one"
)

# Use station-level names when available
station_lookup["stop_name"] = (
    station_lookup["station_name"]
    .fillna(station_lookup["stop_name"])
)

# Keep the columns expected by routing functions
station_lookup = station_lookup[
    ["stop_id", "stop_name", "station_id"]
]

print("Updated station_lookup columns:")
print(station_lookup.columns.tolist())

print("\nMacquarie University station mappings:")
display(
    station_lookup[
        station_lookup["stop_name"] == "Macquarie University Station"
    ].head(15)
)

print("\nMissing station IDs:")
print(station_lookup["station_id"].isna().sum())


Updated station_lookup columns:
['stop_id', 'stop_name', 'station_id']

Macquarie University station mappings:


,stop_id,stop_name,station_id
98,211310,Macquarie University Station,NaN
135,2113351,Macquarie University Station,211310
136,2113352,Macquarie University Station,211310



Missing station IDs:
170767


In [151]:

# STEP 84C.3 — Investigate station mapping quality

print("Total lookup rows:", len(station_lookup))
print("Unique stop IDs:", station_lookup["stop_id"].nunique())
print("Duplicate stop IDs:", station_lookup["stop_id"].duplicated().sum())

print("\nMissing station IDs:", station_lookup["station_id"].isna().sum())

# Check whether missing mappings affect actual timetable stops
timetable_stop_ids = set(realtime_daily["stop_id"].astype(str))

missing_mappings = station_lookup[
    station_lookup["station_id"].isna()
].copy()

missing_mappings["used_in_timetable"] = (
    missing_mappings["stop_id"].isin(timetable_stop_ids)
)

print("\nMissing mappings used in today's timetable:")
print(missing_mappings["used_in_timetable"].sum())

print("\nSample missing mappings:")
display(missing_mappings.head(10))

print("\nTimetable station mapping:")
print(
    "Missing station IDs in realtime_daily:",
    realtime_daily["station_id"].isna().sum()
)


Total lookup rows: 171285
Unique stop IDs: 171285
Duplicate stop IDs: 0

Missing station IDs: 170767

Missing mappings used in today's timetable:
0

Sample missing mappings:


,stop_id,stop_name,station_id,used_in_timetable
0,2533211,"Kiama Station, Platform 1",NaN,False
1,2533212,"Kiama Station, Platform 2",NaN,False
2,253330,Kiama Station,NaN,False
3,253330_CCP1,CCP Railway Pde N,NaN,False
4,253330_CCP2,CCP Railway Pde S,NaN,False
5,253330_DP1,STR/LFT To P1/2,NaN,False
6,253330_ENT1,ENT Railway Pde,NaN,False
7,253330_LKR1,LKR Railway Pde,NaN,False
8,253330_TXI1,TXI Railway Pde,NaN,False
9,253331,"Kiama Station, Railway Pde",NaN,False



Timetable station mapping:
Missing station IDs in realtime_daily: 0


In [152]:

# STEP 84C.4 — Build clean station lookup for routing

# Use the original GTFS stop names
stop_names = original_station_lookup[
    ["stop_id", "stop_name"]
].copy()

stop_names["stop_id"] = stop_names["stop_id"].astype(str)

# Get all station/platform relationships from the timetable
routing_stations = realtime_daily[
    ["stop_id", "station_id"]
].drop_duplicates().copy()

routing_stations["stop_id"] = routing_stations["stop_id"].astype(str)
routing_stations["station_id"] = routing_stations["station_id"].astype(str)

# Find the parent station name
parent_names = stop_names.rename(
    columns={
        "stop_id": "station_id",
        "stop_name": "stop_name"
    }
)

# Match each platform to its parent station
station_lookup = routing_stations.merge(
    parent_names,
    on="station_id",
    how="left",
    validate="many_to_one"
)

# Verify mapping quality
print("Routing station lookup rows:", len(station_lookup))
print("Missing station names:", station_lookup["stop_name"].isna().sum())
print("Duplicate platform IDs:", station_lookup["stop_id"].duplicated().sum())

print("\nMacquarie University:")
display(
    station_lookup[
        station_lookup["stop_name"] == "Macquarie University Station"
    ]
)

print("\nParramatta:")
display(
    station_lookup[
        station_lookup["stop_name"] == "Parramatta Station"
    ].head(10)
)


Routing station lookup rows: 518
Missing station names: 0
Duplicate platform IDs: 0

Macquarie University:


,stop_id,station_id,stop_name
9,2113351,211310,Macquarie University Station
31,2113352,211310,Macquarie University Station



Parramatta:


,stop_id,station_id,stop_name
42,2150414,215020,Parramatta Station
264,2150411,215020,Parramatta Station
282,2150412,215020,Parramatta Station
371,2150413,215020,Parramatta Station


In [153]:

# STEP 84D — Retry direct journey search

origin = "Macquarie University Station"
destination = "Parramatta Station"

direct_results = find_direct_journeys(
    daily=realtime_daily,
    origin=origin,
    destination=destination,
    departure_time="17:30",
    window_minutes=60
)

print("Direct journeys found:", len(direct_results))

if not direct_results.empty:

    display(
        direct_results[
            [
                "trip_id",
                "departure_time",
                "destination_arrival_time",
                "departure_seconds",
                "destination_arrival_seconds",
                "journey_duration_min",
                "transport_mode"
            ]
        ].head(10)
    )

else:
    print("No direct journeys found.")
    print("This route may require one or more transfers.")


Direct journeys found: 0
No direct journeys found.
This route may require one or more transfers.


In [154]:

# STEP 84E — Test one-transfer journeys with realtime timetable

one_transfer_results = find_one_transfer_journeys(
    daily=realtime_daily,
    origin="Macquarie University Station",
    destination="Parramatta Station",
    departure_time="17:30",
    window_minutes=60,
    connection_window_minutes=45
)

print("One-transfer journeys found:", len(one_transfer_results))

if not one_transfer_results.empty:

    columns = [
        "trip_id",
        "second_trip_id",
        "transfer_station",
        "origin_departure_seconds",
        "arrival_seconds",
        "second_departure_seconds",
        "destination_arrival_seconds",
        "journey_duration_min"
    ]

    display(
        one_transfer_results[
            [col for col in columns if col in one_transfer_results.columns]
        ].head(10)
    )

else:
    print("No one-transfer journeys found.")


One-transfer journeys found: 3078


,trip_id,second_trip_id,transfer_station,origin_departure_seconds,arrival_seconds,second_departure_seconds,destination_arrival_seconds,journey_duration_min
0,0160-006-106-014:1000,145N.1324.165.16.A.8.92001340,Central Station,63114.0,64569.0,65041.0,66540.0,57.1
1,0160-006-106-014:1000,145N.1324.165.16.A.8.92001340,Central Station,63114.0,64569.0,65041.0,66540.0,57.1
2,0160-006-106-014:1000,145N.1324.165.16.A.8.92001340,Central Station,63114.0,64569.0,65041.0,66540.0,57.1
3,0160-006-106-014:1000,145N.1324.165.16.A.8.92001340,Central Station,63114.0,64569.0,65041.0,66540.0,57.1
4,0160-006-106-014:1000,145N.1324.165.16.A.8.92001340,Central Station,63114.0,64569.0,65041.0,66540.0,57.1
5,0160-006-106-014:1000,145N.1324.165.16.A.8.92001340,Central Station,63114.0,64569.0,65041.0,66540.0,57.1
6,0160-006-106-014:1000,145N.1324.165.16.A.8.92001340,Central Station,63114.0,64569.0,65041.0,66540.0,57.1
7,0160-006-106-014:1000,145N.1324.165.16.A.8.92001340,Central Station,63114.0,64569.0,65041.0,66540.0,57.1
8,0160-006-106-014:1000,145N.1324.165.16.A.8.92001340,Central Station,63114.0,64569.0,65041.0,66540.0,57.1
9,0160-006-106-014:1000,145N.1324.165.16.A.8.92001340,Central Station,63114.0,64569.0,65041.0,66540.0,57.1


In [155]:

# STEP 84F — Format journey results for readability

def format_transport_time(seconds):
    if pd.isna(seconds):
        return None

    seconds = int(seconds)

    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60

    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


readable_journeys = one_transfer_results.copy()

time_columns = {
    "origin_departure_seconds": "Departure",
    "arrival_seconds": "Transfer arrival",
    "second_departure_seconds": "Transfer departure",
    "destination_arrival_seconds": "Destination arrival"
}

for original, readable in time_columns.items():
    readable_journeys[readable] = (
        readable_journeys[original].map(format_transport_time)
    )

# Calculate the actual available transfer time
readable_journeys["Transfer minutes"] = (
    (
        readable_journeys["second_departure_seconds"]
        - readable_journeys["arrival_seconds"]
    ) / 60
).round(2)

# Remove repeated journey combinations for display
readable_journeys = readable_journeys.drop_duplicates(
    subset=[
        "trip_id",
        "second_trip_id",
        "station_id",
        "origin_departure_seconds",
        "second_departure_seconds"
    ]
)

display_columns = [
    "transfer_station",
    "Departure",
    "Transfer arrival",
    "Transfer departure",
    "Destination arrival",
    "Transfer minutes",
    "journey_duration_min"
]

print("Unique journey combinations:", len(readable_journeys))

display(
    readable_journeys[display_columns].head(10)
)


Unique journey combinations: 162


,transfer_station,Departure,Transfer arrival,Transfer departure,Destination arrival,Transfer minutes,journey_duration_min
0,Central Station,17:31:54,17:56:09,18:04:01,18:29:00,7.87,57.1
19,Central Station,17:31:54,17:56:09,18:07:01,18:32:00,10.87,60.1
38,Central Station,17:35:54,18:00:09,18:07:01,18:32:00,6.87,56.1
57,Central Station,17:31:54,17:56:09,18:13:01,18:38:00,16.87,66.1
76,Central Station,17:35:54,18:00:09,18:13:01,18:38:00,12.87,62.1
95,Central Station,17:39:54,18:04:09,18:13:01,18:38:00,8.87,58.1
114,Central Station,17:31:54,17:56:09,18:16:01,18:41:00,19.87,69.1
133,Central Station,17:35:54,18:00:09,18:16:01,18:41:00,15.87,65.1
152,Central Station,17:39:54,18:04:09,18:16:01,18:41:00,11.87,61.1
171,Central Station,17:43:54,18:08:09,18:16:01,18:41:00,7.87,57.1


In [156]:

# STEP 84G — Validate realtime transfer connections

validated_journeys = one_transfer_results.copy()

# Actual time available between services
validated_journeys["transfer_available_sec"] = (
    validated_journeys["second_departure_seconds"]
    - validated_journeys["arrival_seconds"]
)

# Require at least 5 minutes or the platform-specific minimum
validated_journeys["required_transfer_sec"] = (
    validated_journeys["minimum_transfer_seconds"]
    .clip(lower=300)
)

# Keep only connections with sufficient transfer time
validated_journeys = validated_journeys[
    validated_journeys["transfer_available_sec"]
    >= validated_journeys["required_transfer_sec"]
].copy()

# Remove repeated journey combinations for display
validated_journeys = validated_journeys.drop_duplicates(
    subset=[
        "trip_id",
        "second_trip_id",
        "station_id",
        "origin_departure_seconds",
        "second_departure_seconds"
    ]
)

# Display readable times
for original, readable in {
    "origin_departure_seconds": "Departure",
    "arrival_seconds": "Transfer arrival",
    "second_departure_seconds": "Transfer departure",
    "destination_arrival_seconds": "Destination arrival"
}.items():
    validated_journeys[readable] = (
        validated_journeys[original].map(format_transport_time)
    )

validated_journeys["Transfer minutes"] = (
    validated_journeys["transfer_available_sec"] / 60
).round(2)

validated_journeys["Required minutes"] = (
    validated_journeys["required_transfer_sec"] / 60
).round(2)

print("Original journey combinations:", len(readable_journeys))
print("Valid journey combinations:", len(validated_journeys))

display(
    validated_journeys[
        [
            "transfer_station",
            "Departure",
            "Transfer arrival",
            "Transfer departure",
            "Destination arrival",
            "Transfer minutes",
            "Required minutes",
            "journey_duration_min"
        ]
    ].head(10)
)


Original journey combinations: 162
Valid journey combinations: 162


,transfer_station,Departure,Transfer arrival,Transfer departure,Destination arrival,Transfer minutes,Required minutes,journey_duration_min
0,Central Station,17:31:54,17:56:09,18:04:01,18:29:00,7.87,6.1,57.1
19,Central Station,17:31:54,17:56:09,18:07:01,18:32:00,10.87,6.1,60.1
38,Central Station,17:35:54,18:00:09,18:07:01,18:32:00,6.87,6.1,56.1
57,Central Station,17:31:54,17:56:09,18:13:01,18:38:00,16.87,6.1,66.1
76,Central Station,17:35:54,18:00:09,18:13:01,18:38:00,12.87,6.1,62.1
95,Central Station,17:39:54,18:04:09,18:13:01,18:38:00,8.87,6.1,58.1
114,Central Station,17:31:54,17:56:09,18:16:01,18:41:00,19.87,6.1,69.1
133,Central Station,17:35:54,18:00:09,18:16:01,18:41:00,15.87,6.1,65.1
152,Central Station,17:39:54,18:04:09,18:16:01,18:41:00,11.87,6.1,61.1
171,Central Station,17:43:54,18:08:09,18:16:01,18:41:00,7.87,6.1,57.1


In [157]:

# STEP 84H — Analyse transfer stations and fastest journeys

print("TRANSFER STATIONS")
print(
    validated_journeys["transfer_station"]
    .value_counts()
    .to_string()
)

print("\nFASTEST 10 JOURNEYS")

fastest_journeys = (
    validated_journeys
    .sort_values([
        "journey_duration_min",
        "destination_arrival_seconds"
    ])
    .head(10)
)

display(
    fastest_journeys[
        [
            "transfer_station",
            "first_mode",
            "second_mode",
            "Departure",
            "Transfer arrival",
            "Transfer departure",
            "Destination arrival",
            "Transfer minutes",
            "journey_duration_min"
        ]
    ]
)

print("\nTRANSFER VALIDATION")
print(
    "Connections below required transfer time:",
    (
        validated_journeys["transfer_available_sec"]
        < validated_journeys["required_transfer_sec"]
    ).sum()
)


TRANSFER STATIONS
transfer_station
Central Station    162

FASTEST 10 JOURNEYS


,transfer_station,first_mode,second_mode,Departure,Transfer arrival,Transfer departure,Destination arrival,Transfer minutes,journey_duration_min
38,Central Station,Metro,Train,17:35:54,18:00:09,18:07:01,18:32:00,6.87,56.1
266,Central Station,Metro,Train,17:47:54,18:12:09,18:19:01,18:44:00,6.87,56.1
0,Central Station,Metro,Train,17:31:54,17:56:09,18:04:01,18:29:00,7.87,57.1
171,Central Station,Metro,Train,17:43:54,18:08:09,18:16:01,18:41:00,7.87,57.1
95,Central Station,Metro,Train,17:39:54,18:04:09,18:13:01,18:38:00,8.87,58.1
1045,Central Station,Metro,Train,18:15:54,18:40:09,18:48:01,19:14:00,7.87,58.1
1482,Central Station,Metro,Train,18:27:54,18:52:09,19:00:01,19:26:00,7.87,58.1
570,Central Station,Metro,Train,17:59:54,18:24:09,18:33:01,18:59:00,8.87,59.1
855,Central Station,Metro,Train,18:11:54,18:36:09,18:45:01,19:11:00,8.87,59.1
19,Central Station,Metro,Train,17:31:54,17:56:09,18:07:01,18:32:00,10.87,60.1



TRANSFER VALIDATION
Connections below required transfer time: 0


In [158]:

# STEP 84I — Inspect transfer network coverage

print("Total platform transfer connections:", len(all_platform_transfers))

print("\nTransfer table columns:")
print(all_platform_transfers.columns.tolist())

# Add station names to transfer endpoints
transfer_check = all_platform_transfers.copy()

platform_names = station_lookup[
    ["stop_id", "station_id", "stop_name"]
].copy()

transfer_check["from_stop_id"] = transfer_check["from_stop_id"].astype(str)
transfer_check["to_stop_id"] = transfer_check["to_stop_id"].astype(str)

transfer_check = transfer_check.merge(
    platform_names.rename(columns={
        "stop_id": "from_stop_id",
        "stop_name": "from_station"
    })[["from_stop_id", "from_station"]],
    on="from_stop_id",
    how="left"
)

transfer_check = transfer_check.merge(
    platform_names.rename(columns={
        "stop_id": "to_stop_id",
        "stop_name": "to_station"
    })[["to_stop_id", "to_station"]],
    on="to_stop_id",
    how="left"
)

print("\nTransfer connections by station:")
print(
    transfer_check["from_station"]
    .value_counts()
    .head(20)
    .to_string()
)

print("\nEpping transfers:")
display(
    transfer_check[
        transfer_check["from_station"].str.contains(
            "Epping", case=False, na=False
        )
    ].head(10)
)

print("\nStrathfield transfers:")
display(
    transfer_check[
        transfer_check["from_station"].str.contains(
            "Strathfield", case=False, na=False
        )
    ].head(10)
)


Total platform transfer connections: 2676

Transfer table columns:
['station_id', 'from_stop_id', 'to_stop_id', 'from_mode', 'to_mode', 'pathway_seconds', 'pathway_minutes', 'transfer_buffer_seconds', 'minimum_transfer_seconds', 'minimum_transfer_minutes']

Transfer connections by station:
from_station
Central Station         551
Redfern Station         132
Town Hall Station        64
Strathfield Station      64
Blacktown Station        49
Wynyard Station          36
Sydenham Station         36
Burwood Station          36
Lidcombe Station         36
Homebush Station         30
Circular Quay            25
Epping Station           25
Ashfield Station         25
Hornsby Station          20
Martin Place Station     16
Tempe Station            16
North Sydney Station     16
Chatswood Station        16
Denistone Station        16
Eastwood Station         16

Epping transfers:


,station_id,from_stop_id,to_stop_id,from_mode,to_mode,pathway_seconds,pathway_minutes,transfer_buffer_seconds,minimum_transfer_seconds,minimum_transfer_minutes,from_station,to_station
1320,212110,2121221,2121222,Train,Train,60.0,1.0,180,240.0,4.0,Epping Station,Epping Station
1321,212110,2121221,2121223,Train,Train,102.0,1.7,180,282.0,4.7,Epping Station,Epping Station
1322,212110,2121221,2121225,Train,Metro,90.0,1.5,240,330.0,5.5,Epping Station,Epping Station
1323,212110,2121221,2121226,Train,Metro,90.0,1.5,240,330.0,5.5,Epping Station,Epping Station
1324,212110,2121222,2121221,Train,Train,60.0,1.0,180,240.0,4.0,Epping Station,Epping Station
1325,212110,2121222,2121223,Train,Train,102.0,1.7,180,282.0,4.7,Epping Station,Epping Station
1326,212110,2121222,2121225,Train,Metro,90.0,1.5,240,330.0,5.5,Epping Station,Epping Station
1327,212110,2121222,2121226,Train,Metro,90.0,1.5,240,330.0,5.5,Epping Station,Epping Station
1328,212110,2121223,2121221,Train,Train,102.0,1.7,180,282.0,4.7,Epping Station,Epping Station
1329,212110,2121223,2121222,Train,Train,102.0,1.7,180,282.0,4.7,Epping Station,Epping Station



Strathfield transfers:


,station_id,from_stop_id,to_stop_id,from_mode,to_mode,pathway_seconds,pathway_minutes,transfer_buffer_seconds,minimum_transfer_seconds,minimum_transfer_minutes,from_station,to_station
1420,213510,2135231,2135232,Train,Train,120.0,2.0,180,300.0,5.0,Strathfield Station,Strathfield Station
1421,213510,2135231,2135233,Train,Train,204.0,3.4,180,384.0,6.4,Strathfield Station,Strathfield Station
1422,213510,2135231,2135234,Train,Train,204.0,3.4,180,384.0,6.4,Strathfield Station,Strathfield Station
1423,213510,2135231,2135235,Train,Train,264.0,4.4,180,444.0,7.4,Strathfield Station,Strathfield Station
1424,213510,2135231,2135236,Train,Train,264.0,4.4,180,444.0,7.4,Strathfield Station,Strathfield Station
1425,213510,2135231,2135237,Train,Train,324.0,5.4,180,504.0,8.4,Strathfield Station,Strathfield Station
1426,213510,2135231,2135238,Train,Train,324.0,5.4,180,504.0,8.4,Strathfield Station,Strathfield Station
1427,213510,2135232,2135231,Train,Train,120.0,2.0,180,300.0,5.0,Strathfield Station,Strathfield Station
1428,213510,2135232,2135233,Train,Train,204.0,3.4,180,384.0,6.4,Strathfield Station,Strathfield Station
1429,213510,2135232,2135234,Train,Train,204.0,3.4,180,384.0,6.4,Strathfield Station,Strathfield Station


In [159]:

# STEP 84J — Verify services for a two-transfer journey

route_stations = [
    "Macquarie University Station",
    "Epping Station",
    "Strathfield Station",
    "Parramatta Station"
]

for station in route_stations:

    station_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == station,
            "station_id"
        ]
    )

    station_services = realtime_daily[
        realtime_daily["station_id"].isin(station_ids)
    ]

    print(f"\n{station}")
    print("Station IDs:", station_ids)
    print("Stop events:", len(station_services))
    print("Unique trips:", station_services["trip_id"].nunique())

    print("Transport modes:")
    print(station_services["transport_mode"].value_counts().to_string())



Macquarie University Station
Station IDs: {'211310'}
Stop events: 434
Unique trips: 434
Transport modes:
transport_mode
Metro    434

Epping Station
Station IDs: {'212110'}
Stop events: 624
Unique trips: 624
Transport modes:
transport_mode
Metro    434
Train    190

Strathfield Station
Station IDs: {'213510'}
Stop events: 861
Unique trips: 861
Transport modes:
transport_mode
Train    861

Parramatta Station
Station IDs: {'215020'}
Stop events: 484
Unique trips: 484
Transport modes:
transport_mode
Train    484


In [160]:

# STEP 85A — Check services connecting the test route

route_stations = [
    "Macquarie University Station",
    "Epping Station",
    "Strathfield Station",
    "Parramatta Station"
]

station_ids = {}

for station in route_stations:
    ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == station,
            "station_id"
        ]
    )
    station_ids[station] = ids

for origin, destination in zip(
    route_stations[:-1],
    route_stations[1:]
):
    origin_trips = set(
        realtime_daily.loc[
            realtime_daily["station_id"].isin(station_ids[origin]),
            "trip_id"
        ]
    )

    destination_trips = set(
        realtime_daily.loc[
            realtime_daily["station_id"].isin(station_ids[destination]),
            "trip_id"
        ]
    )

    shared_trips = origin_trips & destination_trips

    # Verify correct travel direction using stop sequence
    leg_stops = realtime_daily[
        realtime_daily["trip_id"].isin(shared_trips)
        & realtime_daily["station_id"].isin(
            station_ids[origin] | station_ids[destination]
        )
    ]

    origin_events = leg_stops[
        leg_stops["station_id"].isin(station_ids[origin])
    ][["trip_id", "stop_sequence"]].rename(
        columns={"stop_sequence": "origin_sequence"}
    )

    destination_events = leg_stops[
        leg_stops["station_id"].isin(station_ids[destination])
    ][["trip_id", "stop_sequence"]].rename(
        columns={"stop_sequence": "destination_sequence"}
    )

    valid_pairs = origin_events.merge(
        destination_events,
        on="trip_id"
    )

    valid_pairs = valid_pairs[
        valid_pairs["destination_sequence"]
        > valid_pairs["origin_sequence"]
    ]

    print(f"\n{origin} → {destination}")
    print("Valid directional trips:", valid_pairs["trip_id"].nunique())



Macquarie University Station → Epping Station
Valid directional trips: 217

Epping Station → Strathfield Station
Valid directional trips: 96

Strathfield Station → Parramatta Station
Valid directional trips: 202


In [161]:

# STEP 85B — Test a complete two-transfer journey

import pandas as pd

route = [
    "Macquarie University Station",
    "Epping Station",
    "Strathfield Station",
    "Parramatta Station"
]

def get_leg_options(origin, destination):
    origin_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"].eq(origin), "station_id"
        ]
    )
    destination_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"].eq(destination), "station_id"
        ]
    )

    starts = realtime_daily[
        realtime_daily["station_id"].isin(origin_ids)
    ][["trip_id", "stop_id", "stop_sequence", "departure_seconds"]].rename(
        columns={
            "stop_id": "boarding_platform",
            "stop_sequence": "boarding_sequence",
            "departure_seconds": "depart_sec"
        }
    )

    ends = realtime_daily[
        realtime_daily["station_id"].isin(destination_ids)
    ][["trip_id", "stop_id", "stop_sequence", "arrival_seconds"]].rename(
        columns={
            "stop_id": "arrival_platform",
            "stop_sequence": "arrival_sequence",
            "arrival_seconds": "arrive_sec"
        }
    )

    options = starts.merge(ends, on="trip_id")
    return options[
        (options["arrival_sequence"] > options["boarding_sequence"])
        & (options["arrive_sec"] >= options["depart_sec"])
    ].copy()


legs = [
    get_leg_options(route[i], route[i + 1])
    for i in range(3)
]

# Limit first departures to 17:30–18:00
legs[0] = legs[0][
    legs[0]["depart_sec"].between(
        gtfs_to_seconds("17:30:00"),
        gtfs_to_seconds("18:00:00")
    )
]

# Prepare platform transfer requirements
transfer_rules = all_platform_transfers[
    ["from_stop_id", "to_stop_id", "minimum_transfer_seconds"]
].copy()

transfer_rules["from_stop_id"] = transfer_rules["from_stop_id"].astype(str)
transfer_rules["to_stop_id"] = transfer_rules["to_stop_id"].astype(str)
transfer_rules["required_sec"] = transfer_rules[
    "minimum_transfer_seconds"
].clip(lower=300)

# Connect successive legs using exact platform IDs
candidates = legs[0].rename(columns={
    "trip_id": "trip_1",
    "boarding_platform": "board_1",
    "arrival_platform": "arrive_1",
    "depart_sec": "depart_1",
    "arrive_sec": "arrival_1"
})

for i in [1, 2]:
    next_leg = legs[i].rename(columns={
        "trip_id": f"trip_{i+1}",
        "boarding_platform": f"board_{i+1}",
        "arrival_platform": f"arrive_{i+1}",
        "depart_sec": f"depart_{i+1}",
        "arrive_sec": f"arrival_{i+1}"
    })

    rules = transfer_rules.rename(columns={
        "from_stop_id": f"arrive_{i}",
        "to_stop_id": f"board_{i+1}",
        "required_sec": f"required_{i}"
    })[
        [f"arrive_{i}", f"board_{i+1}", f"required_{i}"]
    ]

    candidates[f"arrive_{i}"] = candidates[f"arrive_{i}"].astype(str)
    next_leg[f"board_{i+1}"] = next_leg[f"board_{i+1}"].astype(str)

    candidates = candidates.merge(rules, on=f"arrive_{i}")
    candidates = candidates.merge(next_leg, on=f"board_{i+1}")

    candidates = candidates[
        (candidates[f"trip_{i}"] != candidates[f"trip_{i+1}"])
        & (
            candidates[f"depart_{i+1}"]
            >= candidates[f"arrival_{i}"] + candidates[f"required_{i}"]
        )
        & (
            candidates[f"depart_{i+1}"]
            <= candidates[f"arrival_{i}"] + 45 * 60
        )
    ].copy()

# Prevent reusing a previous trip
candidates = candidates[
    candidates["trip_1"] != candidates["trip_3"]
].copy()

# Calculate final arrival and journey duration
candidates["journey_duration_min"] = (
    (candidates["arrival_3"] - candidates["depart_1"]) / 60
)

candidates = candidates.sort_values(
    ["arrival_3", "journey_duration_min"]
).drop_duplicates(
    subset=["trip_1", "trip_2", "trip_3"]
)

print("Valid two-transfer journeys:", len(candidates))

if not candidates.empty:
    best = candidates.iloc[0]

    print("\nEarliest-arriving candidate:")
    print("Departure:", format_transport_time(best["depart_1"]))
    print("Epping arrival:", format_transport_time(best["arrival_1"]))
    print("Epping departure:", format_transport_time(best["depart_2"]))
    print("Strathfield arrival:", format_transport_time(best["arrival_2"]))
    print("Strathfield departure:", format_transport_time(best["depart_3"]))
    print("Parramatta arrival:", format_transport_time(best["arrival_3"]))
    print("Duration:", round(best["journey_duration_min"], 2), "minutes")
    print(
        "Transfer buffers:",
        round((best["depart_2"] - best["arrival_1"]) / 60, 2),
        "and",
        round((best["depart_3"] - best["arrival_2"]) / 60, 2),
        "minutes"
    )


Valid two-transfer journeys: 288

Earliest-arriving candidate:
Departure: 17:30:37
Epping arrival: 17:33:51
Epping departure: 17:42:00
Strathfield arrival: 18:03:00
Strathfield departure: 18:11:24
Parramatta arrival: 18:23:00
Duration: 52.38 minutes
Transfer buffers: 8.15 and 8.4 minutes


In [162]:

# STEP 85C — Calculate transfer safety margins

two_transfer_results = candidates.copy()

for i in [1, 2]:
    two_transfer_results[f"available_transfer_{i}_min"] = (
        two_transfer_results[f"depart_{i+1}"]
        - two_transfer_results[f"arrival_{i}"]
    ) / 60

    two_transfer_results[f"required_transfer_{i}_min"] = (
        two_transfer_results[f"required_{i}"] / 60
    )

    two_transfer_results[f"spare_transfer_{i}_min"] = (
        two_transfer_results[f"available_transfer_{i}_min"]
        - two_transfer_results[f"required_transfer_{i}_min"]
    )

two_transfer_results["minimum_spare_minutes"] = (
    two_transfer_results[
        ["spare_transfer_1_min", "spare_transfer_2_min"]
    ].min(axis=1)
)

two_transfer_results["Departure"] = (
    two_transfer_results["depart_1"].map(format_transport_time)
)

two_transfer_results["Arrival"] = (
    two_transfer_results["arrival_3"].map(format_transport_time)
)

print("Total valid journeys:", len(two_transfer_results))

display(
    two_transfer_results[
        [
            "Departure",
            "Arrival",
            "journey_duration_min",
            "available_transfer_1_min",
            "required_transfer_1_min",
            "spare_transfer_1_min",
            "available_transfer_2_min",
            "required_transfer_2_min",
            "spare_transfer_2_min",
            "minimum_spare_minutes"
        ]
    ].head(10).round(2)
)


Total valid journeys: 288


,Departure,Arrival,journey_duration_min,available_transfer_1_min,required_transfer_1_min,spare_transfer_1_min,available_transfer_2_min,required_transfer_2_min,spare_transfer_2_min,minimum_spare_minutes
5176,17:30:37,18:23:00,52.38,8.15,5.5,2.65,8.4,5.0,3.4,2.65
2280,17:42:37,18:29:00,46.38,7.17,5.5,1.67,9.6,7.4,2.2,1.67
4704,17:38:37,18:29:00,50.38,11.17,5.5,5.67,9.6,7.4,2.2,2.20
6522,17:34:37,18:29:00,54.38,15.17,5.5,9.67,9.6,7.4,2.2,2.20
5108,17:30:37,18:29:00,58.38,8.15,5.5,2.65,14.4,5.0,9.4,2.65
5714,17:30:37,18:29:00,58.38,19.17,5.5,13.67,9.6,7.4,2.2,2.20
2317,17:42:37,18:32:00,49.38,7.17,5.5,1.67,12.6,7.4,5.2,1.67
4741,17:38:37,18:32:00,53.38,11.17,5.5,5.67,12.6,7.4,5.2,5.20
6559,17:34:37,18:32:00,57.38,15.17,5.5,9.67,12.6,7.4,5.2,5.20
5145,17:30:37,18:32:00,61.38,8.15,5.5,2.65,17.4,5.0,12.4,2.65


In [163]:

# STEP 85D — Check backup services for the fastest journey

best = two_transfer_results.sort_values("arrival_3").iloc[0]

for transfer_number, station_name in [
    (1, "Epping Station"),
    (2, "Strathfield Station")
]:
    arrival = best[f"arrival_{transfer_number}"]
    planned_departure = best[f"depart_{transfer_number + 1}"]
    planned_trip = best[f"trip_{transfer_number + 1}"]

    # Identify the next destination on the planned route
    next_station = (
        "Strathfield Station"
        if transfer_number == 1
        else "Parramatta Station"
    )

    # Reuse our directional leg search
    alternatives = get_leg_options(station_name, next_station)

    # Apply platform-specific transfer rules from the actual arrival platform
    arrival_platform = str(best[f"arrive_{transfer_number}"])

    rules = transfer_rules[
        transfer_rules["from_stop_id"].eq(arrival_platform)
    ][["to_stop_id", "required_sec"]]

    alternatives["boarding_platform"] = (
        alternatives["boarding_platform"].astype(str)
    )

    alternatives = alternatives.merge(
        rules,
        left_on="boarding_platform",
        right_on="to_stop_id"
    )

    # Backup must depart after the planned service,
    # and allow enough time to reach its boarding platform
    alternatives = alternatives[
        (alternatives["trip_id"] != planned_trip)
        & (alternatives["depart_sec"] > planned_departure)
        & (
            alternatives["depart_sec"]
            >= arrival + alternatives["required_sec"]
        )
    ].sort_values(["depart_sec", "arrive_sec"])

    print(f"\n{station_name} → {next_station}")
    print("Planned departure:", format_transport_time(planned_departure))

    if alternatives.empty:
        print("No backup service found.")
    else:
        backup = alternatives.iloc[0]

        print("Next backup departure:", format_transport_time(backup["depart_sec"]))
        print("Backup arrival:", format_transport_time(backup["arrive_sec"]))
        print(
            "Additional wait:",
            round((backup["depart_sec"] - planned_departure) / 60, 2),
            "minutes"
        )



Epping Station → Strathfield Station
Planned departure: 17:42:00
Next backup departure: 17:53:01
Backup arrival: 18:07:48
Additional wait: 11.02 minutes

Strathfield Station → Parramatta Station
Planned departure: 18:11:24
Next backup departure: 18:16:00
Backup arrival: 18:36:30
Additional wait: 4.6 minutes


In [164]:

# STEP 85E — Simulate a missed connection at Epping

best = two_transfer_results.sort_values(
    ["arrival_3", "journey_duration_min"]
).iloc[0]

# Get all directional Epping → Strathfield services
backup_legs = get_leg_options(
    "Epping Station",
    "Strathfield Station"
)

# Get all directional Strathfield → Parramatta services
final_legs = get_leg_options(
    "Strathfield Station",
    "Parramatta Station"
)

# Apply transfer rules at Epping
epping_rules = transfer_rules[
    transfer_rules["from_stop_id"].eq(str(best["arrive_1"]))
][["to_stop_id", "required_sec"]].copy()

backup_legs["boarding_platform"] = (
    backup_legs["boarding_platform"].astype(str)
)

backup_legs = backup_legs.merge(
    epping_rules,
    left_on="boarding_platform",
    right_on="to_stop_id"
)

# Only consider services after the missed planned train
backup_legs = backup_legs[
    (backup_legs["depart_sec"] > best["depart_2"])
    & (
        backup_legs["depart_sec"]
        >= best["arrival_1"] + backup_legs["required_sec"]
    )
].copy()

# Match Strathfield arrival platforms to onward departure platforms
strathfield_rules = transfer_rules[
    ["from_stop_id", "to_stop_id", "required_sec"]
].copy()

backup_legs["arrival_platform"] = (
    backup_legs["arrival_platform"].astype(str)
)

final_legs["boarding_platform"] = (
    final_legs["boarding_platform"].astype(str)
)

recovery = backup_legs.merge(
    strathfield_rules,
    left_on="arrival_platform",
    right_on="from_stop_id",
    suffixes=("_epping", "_strathfield")
)

recovery = recovery.merge(
    final_legs,
    left_on="to_stop_id",
    right_on="boarding_platform",
    suffixes=("_backup", "_final")
)

# Enforce transfer time at Strathfield
recovery = recovery[
    (recovery["trip_id_backup"] != recovery["trip_id_final"])
    & (
        recovery["depart_sec_final"]
        >= recovery["arrive_sec_backup"]
        + recovery["required_sec_strathfield"]
    )
].copy()

recovery = recovery.sort_values(
    ["arrive_sec_final", "depart_sec_backup"]
)

print("Valid recovery journeys:", len(recovery))
print(
    "Original Parramatta arrival:",
    format_transport_time(best["arrival_3"])
)

if not recovery.empty:
    recovered = recovery.iloc[0]

    print(
        "Backup departure from Epping:",
        format_transport_time(recovered["depart_sec_backup"])
    )
    print(
        "Arrival at Strathfield:",
        format_transport_time(recovered["arrive_sec_backup"])
    )
    print(
        "Onward departure:",
        format_transport_time(recovered["depart_sec_final"])
    )
    print(
        "Recovered Parramatta arrival:",
        format_transport_time(recovered["arrive_sec_final"])
    )
    print(
        "Missed-connection arrival penalty:",
        round(
            (recovered["arrive_sec_final"] - best["arrival_3"]) / 60,
            2
        ),
        "minutes"
    )
else:
    print("No feasible recovery journey found.")


KeyError: 'to_stop_id'

In [165]:

# STEP 85E — Corrected missed-connection simulation at Epping

best = two_transfer_results.sort_values(
    ["arrival_3", "journey_duration_min"]
).iloc[0]

# 1. Find services for both remaining legs
backup_legs = get_leg_options(
    "Epping Station", "Strathfield Station"
)

final_legs = get_leg_options(
    "Strathfield Station", "Parramatta Station"
)

# 2. Apply platform transfer requirements at Epping
epping_rules = transfer_rules.rename(columns={
    "from_stop_id": "epping_from",
    "to_stop_id": "epping_to",
    "required_sec": "epping_required"
})

epping_rules = epping_rules[
    epping_rules["epping_from"].eq(str(best["arrive_1"]))
]

backup_legs["boarding_platform"] = (
    backup_legs["boarding_platform"].astype(str)
)

backup_legs = backup_legs.merge(
    epping_rules[["epping_to", "epping_required"]],
    left_on="boarding_platform",
    right_on="epping_to"
)

# Only use services departing after the missed train
backup_legs = backup_legs[
    (backup_legs["depart_sec"] > best["depart_2"])
    & (
        backup_legs["depart_sec"]
        >= best["arrival_1"] + backup_legs["epping_required"]
    )
].copy()

# 3. Apply transfer requirements at Strathfield
strathfield_rules = transfer_rules.rename(columns={
    "from_stop_id": "strathfield_from",
    "to_stop_id": "strathfield_to",
    "required_sec": "strathfield_required"
})

backup_legs["arrival_platform"] = (
    backup_legs["arrival_platform"].astype(str)
)

final_legs["boarding_platform"] = (
    final_legs["boarding_platform"].astype(str)
)

recovery = backup_legs.merge(
    strathfield_rules[
        ["strathfield_from", "strathfield_to", "strathfield_required"]
    ],
    left_on="arrival_platform",
    right_on="strathfield_from"
)

recovery = recovery.merge(
    final_legs,
    left_on="strathfield_to",
    right_on="boarding_platform",
    suffixes=("_backup", "_final")
)

# 4. Validate the onward connection
recovery = recovery[
    (recovery["trip_id_backup"] != recovery["trip_id_final"])
    & (
        recovery["depart_sec_final"]
        >= recovery["arrive_sec_backup"]
        + recovery["strathfield_required"]
    )
].copy()

# 5. Find the earliest recovered arrival
recovery = recovery.sort_values(
    ["arrive_sec_final", "depart_sec_backup"]
)

print("Valid recovery journeys:", len(recovery))
print(
    "Original Parramatta arrival:",
    format_transport_time(best["arrival_3"])
)

if not recovery.empty:
    recovered = recovery.iloc[0]

    print(
        "Backup departure from Epping:",
        format_transport_time(recovered["depart_sec_backup"])
    )
    print(
        "Arrival at Strathfield:",
        format_transport_time(recovered["arrive_sec_backup"])
    )
    print(
        "Onward departure from Strathfield:",
        format_transport_time(recovered["depart_sec_final"])
    )
    print(
        "Recovered Parramatta arrival:",
        format_transport_time(recovered["arrive_sec_final"])
    )
    print(
        "Missed-connection arrival penalty:",
        round(
            (recovered["arrive_sec_final"] - best["arrival_3"]) / 60,
            2
        ),
        "minutes"
    )
else:
    print("No feasible recovery journey found.")


Valid recovery journeys: 597
Original Parramatta arrival: 18:23:00
Backup departure from Epping: 17:53:01
Arrival at Strathfield: 18:07:48
Onward departure from Strathfield: 18:17:24
Recovered Parramatta arrival: 18:29:00
Missed-connection arrival penalty: 6.0 minutes


In [166]:

# STEP 86A — Discover first-transfer stations automatically

origin = "Macquarie University Station"
destination = "Parramatta Station"
departure_time = "17:30:00"

# Find departures from the origin
origin_departures = get_station_departures(
    realtime_daily,
    origin,
    departure_time,
    window_minutes=60
)

# Find every downstream stop on those trips
first_leg_stops = origin_departures[
    ["trip_id", "stop_sequence", "departure_seconds"]
].rename(columns={
    "stop_sequence": "origin_sequence",
    "departure_seconds": "origin_departure_seconds"
}).merge(
    realtime_daily[
        [
            "trip_id",
            "station_id",
            "stop_id",
            "stop_sequence",
            "arrival_seconds"
        ]
    ],
    on="trip_id"
)

# Keep stations reached after boarding
first_leg_stops = first_leg_stops[
    first_leg_stops["stop_sequence"]
    > first_leg_stops["origin_sequence"]
].copy()

# Map station IDs to station names
station_names = (
    station_lookup[["station_id", "stop_name"]]
    .drop_duplicates(subset=["station_id"])
    .rename(columns={"stop_name": "transfer_station"})
)

first_leg_stops = first_leg_stops.merge(
    station_names,
    on="station_id",
    how="left"
)

# Exclude the origin and destination
first_leg_stops = first_leg_stops[
    ~first_leg_stops["transfer_station"].isin(
        [origin, destination]
    )
]

# Summarise potential transfer stations
transfer_candidates = (
    first_leg_stops.groupby("transfer_station")
    .agg(
        available_arrivals=("trip_id", "nunique"),
        earliest_arrival=("arrival_seconds", "min")
    )
    .reset_index()
    .sort_values("earliest_arrival")
)

transfer_candidates["earliest_arrival_time"] = (
    transfer_candidates["earliest_arrival"]
    .map(format_transport_time)
)

print(
    "Potential first-transfer stations:",
    len(transfer_candidates)
)

display(
    transfer_candidates[
        [
            "transfer_station",
            "available_arrivals",
            "earliest_arrival_time"
        ]
    ].head(30)
)


Potential first-transfer stations: 20


,transfer_station,available_arrivals,earliest_arrival_time
11,Macquarie Park Station,15,17:33:27
7,Epping Station,15,17:33:51
13,North Ryde Station,15,17:35:35
5,Cherrybrook Station,15,17:39:38
4,Chatswood Station,15,17:41:10
2,Castle Hill Station,15,17:42:22
9,Hills Showground Station,15,17:44:55
6,Crows Nest Station,15,17:45:02
18,Victoria Cross Station,15,17:47:11
14,Norwest Station,15,17:47:30


In [167]:

# STEP 86B — Validate first-transfer opportunities

# Keep the first-leg arrival events
first_arrivals = first_leg_stops[
    [
        "trip_id",
        "station_id",
        "stop_id",
        "arrival_seconds",
        "origin_departure_seconds"
    ]
].drop_duplicates().copy()

first_arrivals["stop_id"] = first_arrivals["stop_id"].astype(str)

# Platform-specific transfer rules
platform_rules = all_platform_transfers[
    [
        "station_id",
        "from_stop_id",
        "to_stop_id",
        "minimum_transfer_seconds"
    ]
].copy()

platform_rules["station_id"] = platform_rules["station_id"].astype(str)
platform_rules["from_stop_id"] = platform_rules["from_stop_id"].astype(str)
platform_rules["to_stop_id"] = platform_rules["to_stop_id"].astype(str)

platform_rules["required_transfer_seconds"] = (
    platform_rules["minimum_transfer_seconds"]
    .clip(lower=300)
)

first_arrivals["station_id"] = first_arrivals["station_id"].astype(str)

# Match arrival platforms to possible boarding platforms
first_transfer_options = first_arrivals.merge(
    platform_rules,
    left_on=["station_id", "stop_id"],
    right_on=["station_id", "from_stop_id"],
    how="inner"
)

# Calculate earliest possible departure after walking
first_transfer_options["earliest_next_departure"] = (
    first_transfer_options["arrival_seconds"]
    + first_transfer_options["required_transfer_seconds"]
)

# Add station names for inspection
first_transfer_options = first_transfer_options.merge(
    station_names.assign(
        station_id=lambda df: df["station_id"].astype(str)
    ),
    on="station_id",
    how="left"
)

summary = (
    first_transfer_options.groupby("transfer_station")
    .agg(
        possible_connections=("to_stop_id", "count"),
        earliest_ready=("earliest_next_departure", "min")
    )
    .reset_index()
    .sort_values("earliest_ready")
)

summary["earliest_ready_time"] = (
    summary["earliest_ready"]
    .map(format_transport_time)
)

print("Stations with platform transfer options:", len(summary))

display(
    summary[
        [
            "transfer_station",
            "possible_connections",
            "earliest_ready_time"
        ]
    ]
)


Stations with platform transfer options: 20


,transfer_station,possible_connections,earliest_ready_time
11,Macquarie Park Station,30,17:38:27
7,Epping Station,75,17:38:51
13,North Ryde Station,30,17:40:35
5,Cherrybrook Station,30,17:44:38
4,Chatswood Station,60,17:46:10
2,Castle Hill Station,30,17:47:22
9,Hills Showground Station,30,17:49:55
6,Crows Nest Station,30,17:50:02
18,Victoria Cross Station,30,17:52:11
14,Norwest Station,30,17:52:30


In [168]:

# STEP 86C — Discover valid second-leg departures

# Prepare departure events from the timetable
second_departures = realtime_daily[
    [
        "trip_id",
        "stop_id",
        "station_id",
        "stop_sequence",
        "departure_seconds"
    ]
].copy()

second_departures["stop_id"] = (
    second_departures["stop_id"].astype(str)
)

# Rename first-leg and second-leg fields
first_options = first_transfer_options.rename(columns={
    "trip_id": "first_trip_id",
    "stop_id": "first_arrival_platform",
    "arrival_seconds": "first_arrival_seconds"
})

second_departures = second_departures.rename(columns={
    "trip_id": "second_trip_id",
    "stop_id": "second_boarding_platform",
    "station_id": "second_station_id",
    "stop_sequence": "second_boarding_sequence",
    "departure_seconds": "second_departure_seconds"
})

# Match platforms to actual departures
second_leg_candidates = first_options.merge(
    second_departures,
    left_on="to_stop_id",
    right_on="second_boarding_platform",
    how="inner"
)

# Require different services and feasible transfer times
second_leg_candidates = second_leg_candidates[
    (
        second_leg_candidates["first_trip_id"]
        != second_leg_candidates["second_trip_id"]
    )
    & (
        second_leg_candidates["second_departure_seconds"]
        >= second_leg_candidates["earliest_next_departure"]
    )
    & (
        second_leg_candidates["second_departure_seconds"]
        <= second_leg_candidates["first_arrival_seconds"] + 45 * 60
    )
].copy()

# Exclude boarding at a trip's final stop
last_sequences = (
    realtime_daily.groupby("trip_id")["stop_sequence"]
    .max()
)

second_leg_candidates = second_leg_candidates[
    second_leg_candidates["second_boarding_sequence"]
    < second_leg_candidates["second_trip_id"].map(last_sequences)
].copy()

# Deduplicate equivalent transfer choices
second_leg_candidates = second_leg_candidates.drop_duplicates(
    subset=[
        "first_trip_id",
        "first_arrival_platform",
        "second_trip_id",
        "second_boarding_platform",
        "first_arrival_seconds",
        "second_departure_seconds"
    ]
)

print("Valid first-to-second leg connections:", len(second_leg_candidates))

print("\nTransfer stations with connecting services:")

display(
    second_leg_candidates.groupby("transfer_station")
    .agg(
        connections=("second_trip_id", "count"),
        unique_second_services=("second_trip_id", "nunique")
    )
    .sort_values("connections", ascending=False)
    .head(20)
)


Valid first-to-second leg connections: 7319

Transfer stations with connecting services:


,connections,unique_second_services
transfer_station,,
Central Station,1501,215
Chatswood Station,536,84
Martin Place Station,533,85
Epping Station,389,63
Barangaroo Station,299,47
Macquarie Park Station,299,47
Victoria Cross Station,299,47
North Ryde Station,299,47
Crows Nest Station,299,47


In [169]:

# STEP 86D — Discover second-transfer stations automatically

# Find all downstream stops on the second-leg services
second_leg_stops = second_leg_candidates[
    [
        "first_trip_id",
        "second_trip_id",
        "transfer_station",
        "origin_departure_seconds",
        "second_boarding_sequence",
        "second_departure_seconds"
    ]
].rename(columns={
    "transfer_station": "first_transfer_station"
}).merge(
    realtime_daily[
        [
            "trip_id",
            "station_id",
            "stop_id",
            "stop_sequence",
            "arrival_seconds"
        ]
    ].rename(columns={
        "trip_id": "second_trip_id",
        "station_id": "second_arrival_station_id",
        "stop_id": "second_arrival_platform",
        "stop_sequence": "second_arrival_sequence",
        "arrival_seconds": "second_arrival_seconds"
    }),
    on="second_trip_id",
    how="inner"
)

# Keep only downstream stops with valid travel times
second_leg_stops = second_leg_stops[
    (
        second_leg_stops["second_arrival_sequence"]
        > second_leg_stops["second_boarding_sequence"]
    )
    & (
        second_leg_stops["second_arrival_seconds"]
        >= second_leg_stops["second_departure_seconds"]
    )
].copy()

# Add station names
second_leg_stops["second_arrival_station_id"] = (
    second_leg_stops["second_arrival_station_id"].astype(str)
)

second_station_names = station_names.rename(columns={
    "station_id": "second_arrival_station_id",
    "transfer_station": "second_transfer_station"
}).copy()

second_station_names["second_arrival_station_id"] = (
    second_station_names["second_arrival_station_id"].astype(str)
)

second_leg_stops = second_leg_stops.merge(
    second_station_names,
    on="second_arrival_station_id",
    how="left"
)

# Exclude repeated transfer stations and the origin/destination
second_leg_stops = second_leg_stops[
    (
        second_leg_stops["second_transfer_station"]
        != second_leg_stops["first_transfer_station"]
    )
    & (
        ~second_leg_stops["second_transfer_station"].isin(
            [origin, destination]
        )
    )
].copy()

# Summarise possible second-transfer stations
second_transfer_summary = (
    second_leg_stops.groupby("second_transfer_station")
    .agg(
        possible_arrivals=("second_trip_id", "count"),
        unique_services=("second_trip_id", "nunique"),
        earliest_arrival=("second_arrival_seconds", "min")
    )
    .reset_index()
    .sort_values("possible_arrivals", ascending=False)
)

second_transfer_summary["earliest_arrival_time"] = (
    second_transfer_summary["earliest_arrival"]
    .map(format_transport_time)
)

print(
    "Potential second-transfer stations:",
    len(second_transfer_summary)
)

display(
    second_transfer_summary[
        [
            "second_transfer_station",
            "possible_arrivals",
            "unique_services",
            "earliest_arrival_time"
        ]
    ].head(25)
)

print("\nStrathfield check:")
display(
    second_transfer_summary[
        second_transfer_summary["second_transfer_station"]
        == "Strathfield Station"
    ]
)


Potential second-transfer stations: 185


,second_transfer_station,possible_arrivals,unique_services,earliest_arrival_time
27,Central Station,2937,114,18:04:09
155,Tallawong Station,2840,36,18:06:17
140,Rouse Hill Station,2781,51,18:03:40
154,Sydenham Station,2752,45,18:10:19
75,Kellyville Station,2728,52,18:00:45
97,Martin Place Station,2724,73,18:00:25
10,Bella Vista Station,2679,53,17:58:10
28,Chatswood Station,2653,72,17:49:10
117,Norwest Station,2644,55,17:55:30
52,Epping Station,2621,68,17:45:51



Strathfield check:


,second_transfer_station,possible_arrivals,unique_services,earliest_arrival,earliest_arrival_time
150,Strathfield Station,386,53,64980.0,18:03:00


In [170]:

# STEP 86E — Connect second-transfer stations to the destination

# Keep second-leg arrival events and journey history
second_arrivals = second_leg_stops.copy()

second_arrivals["second_arrival_platform"] = (
    second_arrivals["second_arrival_platform"].astype(str)
)

# Match second-leg arrival platforms to possible third-leg platforms
third_transfer_rules = platform_rules[
    [
        "from_stop_id",
        "to_stop_id",
        "required_transfer_seconds"
    ]
].rename(columns={
    "from_stop_id": "second_arrival_platform",
    "to_stop_id": "third_boarding_platform",
    "required_transfer_seconds": "second_required_transfer"
})

third_transfer_options = second_arrivals.merge(
    third_transfer_rules,
    on="second_arrival_platform",
    how="inner"
)

# Find all departures from the third boarding platform
third_departures = realtime_daily[
    [
        "trip_id",
        "stop_id",
        "stop_sequence",
        "departure_seconds"
    ]
].rename(columns={
    "trip_id": "third_trip_id",
    "stop_id": "third_departure_platform",
    "stop_sequence": "third_boarding_sequence",
    "departure_seconds": "third_departure_seconds"
}).copy()

third_departures["third_departure_platform"] = (
    third_departures["third_departure_platform"].astype(str)
)

third_leg_candidates = third_transfer_options.merge(
    third_departures,
    left_on="third_boarding_platform",
    right_on="third_departure_platform",
    how="inner"
)

# Validate the second transfer
third_leg_candidates = third_leg_candidates[
    (
        third_leg_candidates["third_trip_id"]
        != third_leg_candidates["second_trip_id"]
    )
    & (
        third_leg_candidates["third_trip_id"]
        != third_leg_candidates["first_trip_id"]
    )
    & (
        third_leg_candidates["third_departure_seconds"]
        >= third_leg_candidates["second_arrival_seconds"]
        + third_leg_candidates["second_required_transfer"]
    )
    & (
        third_leg_candidates["third_departure_seconds"]
        <= third_leg_candidates["second_arrival_seconds"] + 45 * 60
    )
].copy()

# Identify destination arrivals
destination_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"].eq(destination),
        "station_id"
    ].astype(str)
)

destination_arrivals = realtime_daily[
    realtime_daily["station_id"].astype(str).isin(destination_ids)
][
    ["trip_id", "stop_sequence", "arrival_seconds"]
].rename(columns={
    "trip_id": "third_trip_id",
    "stop_sequence": "destination_sequence",
    "arrival_seconds": "destination_arrival_seconds"
})

# Keep third services that reach the destination downstream
complete_journeys = third_leg_candidates.merge(
    destination_arrivals,
    on="third_trip_id",
    how="inner"
)

complete_journeys = complete_journeys[
    (
        complete_journeys["destination_sequence"]
        > complete_journeys["third_boarding_sequence"]
    )
    & (
        complete_journeys["destination_arrival_seconds"]
        >= complete_journeys["third_departure_seconds"]
    )
].copy()

# Remove journeys that return to an earlier transfer station
complete_journeys = complete_journeys[
    complete_journeys["first_transfer_station"]
    != complete_journeys["second_transfer_station"]
]

# Calculate journey duration
complete_journeys["journey_duration_min"] = (
    complete_journeys["destination_arrival_seconds"]
    - complete_journeys["origin_departure_seconds"]
) / 60

complete_journeys = complete_journeys.sort_values(
    ["destination_arrival_seconds", "journey_duration_min"]
)

print("Complete two-transfer journey candidates:", len(complete_journeys))

if not complete_journeys.empty:
    display(
        complete_journeys[
            [
                "first_transfer_station",
                "second_transfer_station",
                "origin_departure_seconds",
                "destination_arrival_seconds",
                "journey_duration_min"
            ]
        ].head(10)
    )

    print("\nEarliest-arriving journey:")
    best = complete_journeys.iloc[0]

    print("First transfer:", best["first_transfer_station"])
    print("Second transfer:", best["second_transfer_station"])
    print(
        "Departure:",
        format_transport_time(best["origin_departure_seconds"])
    )
    print(
        "Arrival:",
        format_transport_time(best["destination_arrival_seconds"])
    )
    print(
        "Duration:",
        round(best["journey_duration_min"], 2),
        "minutes"
    )


Complete two-transfer journey candidates: 55746


,first_transfer_station,second_transfer_station,origin_departure_seconds,destination_arrival_seconds,journey_duration_min
63,Epping Station,Strathfield Station,63037.0,66180.0,52.383333
21420,Epping Station,Strathfield Station,63757.0,66540.0,46.383333
14484,Epping Station,Strathfield Station,63517.0,66540.0,50.383333
7408,Epping Station,Strathfield Station,63277.0,66540.0,54.383333
59,Epping Station,Strathfield Station,63037.0,66540.0,58.383333
207,Epping Station,Strathfield Station,63037.0,66540.0,58.383333
21423,Epping Station,Strathfield Station,63757.0,66720.0,49.383333
14487,Epping Station,Strathfield Station,63517.0,66720.0,53.383333
7411,Epping Station,Strathfield Station,63277.0,66720.0,57.383333
62,Epping Station,Strathfield Station,63037.0,66720.0,61.383333



Earliest-arriving journey:
First transfer: Epping Station
Second transfer: Strathfield Station
Departure: 17:30:37
Arrival: 18:23:00
Duration: 52.38 minutes


In [171]:

# STEP 86F — Clean and validate completed two-transfer journeys

clean_journeys = complete_journeys.copy()

# Remove duplicate service combinations
journey_keys = [
    "first_trip_id",
    "second_trip_id",
    "third_trip_id",
    "origin_departure_seconds",
    "second_departure_seconds",
    "third_departure_seconds",
    "destination_arrival_seconds"
]

clean_journeys = clean_journeys.drop_duplicates(
    subset=journey_keys
)

# Validate chronological order
clean_journeys = clean_journeys[
    (
        clean_journeys["second_departure_seconds"]
        >= clean_journeys["origin_departure_seconds"]
    )
    & (
        clean_journeys["third_departure_seconds"]
        >= clean_journeys["second_departure_seconds"]
    )
    & (
        clean_journeys["destination_arrival_seconds"]
        >= clean_journeys["third_departure_seconds"]
    )
].copy()

# Calculate first-transfer safety margin
clean_journeys["first_spare_minutes"] = (
    clean_journeys["second_departure_seconds"]
    - clean_journeys["first_arrival_seconds"]
    - clean_journeys["required_transfer_seconds"]
) / 60

# Calculate second-transfer safety margin
clean_journeys["second_spare_minutes"] = (
    clean_journeys["third_departure_seconds"]
    - clean_journeys["second_arrival_seconds"]
    - clean_journeys["second_required_transfer"]
) / 60

# Require both transfers to be feasible
clean_journeys = clean_journeys[
    (clean_journeys["first_spare_minutes"] >= 0)
    & (clean_journeys["second_spare_minutes"] >= 0)
].copy()

clean_journeys["minimum_spare_minutes"] = clean_journeys[
    ["first_spare_minutes", "second_spare_minutes"]
].min(axis=1)

clean_journeys = clean_journeys.sort_values(
    ["destination_arrival_seconds", "journey_duration_min"]
)

# Format times for inspection
clean_journeys["departure_time"] = (
    clean_journeys["origin_departure_seconds"]
    .map(format_transport_time)
)

clean_journeys["arrival_time"] = (
    clean_journeys["destination_arrival_seconds"]
    .map(format_transport_time)
)

print("Original candidate records:", len(complete_journeys))
print("Cleaned journey records:", len(clean_journeys))

display(
    clean_journeys[
        [
            "first_transfer_station",
            "second_transfer_station",
            "departure_time",
            "arrival_time",
            "journey_duration_min",
            "first_spare_minutes",
            "second_spare_minutes",
            "minimum_spare_minutes"
        ]
    ].head(10).round(2)
)


KeyError: 'first_arrival_seconds'

In [172]:

# STEP 86F — Fix first-leg arrival column name

print("First-leg arrival columns:")
print([
    col for col in complete_journeys.columns
    if "arrival" in col
])

# Standardise the column name
if "first_arrival_seconds" not in complete_journeys.columns:
    if "arrival_seconds" in complete_journeys.columns:
        complete_journeys = complete_journeys.rename(
            columns={"arrival_seconds": "first_arrival_seconds"}
        )
    else:
        raise KeyError(
            "Cannot find the first-leg arrival time. "
            "Check the columns printed above."
        )

print("\nColumn fixed:", "first_arrival_seconds" in complete_journeys.columns)


First-leg arrival columns:
['second_arrival_station_id', 'second_arrival_platform', 'second_arrival_sequence', 'second_arrival_seconds', 'destination_arrival_seconds']


KeyError: 'Cannot find the first-leg arrival time. Check the columns printed above.'

In [173]:

# ============================================================
# STEP 86F — Clean and validate two-transfer journeys
# ============================================================

import pandas as pd

clean_journeys = complete_journeys.copy()

print("Original candidate records:", len(clean_journeys))

# ------------------------------------------------------------
# 1. Recover first-leg arrival time
# ------------------------------------------------------------

# Step 86D did not retain first-leg arrival time.
# Recover it using the first service and its arrival platform.

first_leg_arrivals = realtime_daily[
    [
        "trip_id",
        "stop_id",
        "arrival_seconds"
    ]
].copy()

first_leg_arrivals["stop_id"] = (
    first_leg_arrivals["stop_id"].astype(str)
)

first_leg_arrivals = first_leg_arrivals.rename(columns={
    "trip_id": "first_trip_id",
    "stop_id": "first_arrival_platform",
    "arrival_seconds": "first_arrival_seconds"
})

# The first transfer platform is retained in
# first_transfer_options as 'first_arrival_platform'.
# Reconnect it using the first trip and second trip.

first_transfer_details = first_transfer_options[
    [
        "first_trip_id",
        "first_arrival_platform",
        "second_boarding_platform"
    ]
].copy() if "second_boarding_platform" in first_transfer_options.columns else None

# Step 86C contains the complete first-transfer information.
first_transfer_details = second_leg_candidates[
    [
        "first_trip_id",
        "second_trip_id",
        "origin_departure_seconds",
        "first_arrival_platform",
        "first_arrival_seconds",
        "required_transfer_seconds",
        "second_boarding_platform",
        "second_departure_seconds"
    ]
].drop_duplicates().copy()

# Reconnect first-transfer information to completed journeys.
# Include the service departure times to avoid mismatching
# different transfer combinations.

merge_keys = [
    "first_trip_id",
    "second_trip_id",
    "origin_departure_seconds",
    "second_departure_seconds"
]

# Avoid duplicated columns during merge
columns_to_restore = [
    "first_arrival_platform",
    "first_arrival_seconds",
    "required_transfer_seconds",
    "second_boarding_platform"
]

existing_columns = [
    c for c in columns_to_restore
    if c in clean_journeys.columns
]

clean_journeys = clean_journeys.drop(
    columns=existing_columns
)

clean_journeys = clean_journeys.merge(
    first_transfer_details,
    on=merge_keys,
    how="inner"
)

print("Records after restoring first transfer:", len(clean_journeys))

# ------------------------------------------------------------
# 2. Remove duplicate service and platform combinations
# ------------------------------------------------------------

journey_keys = [
    "first_trip_id",
    "second_trip_id",
    "third_trip_id",
    "origin_departure_seconds",
    "first_arrival_platform",
    "second_boarding_platform",
    "second_arrival_platform",
    "third_boarding_platform",
    "second_departure_seconds",
    "third_departure_seconds",
    "destination_arrival_seconds"
]

clean_journeys = clean_journeys.drop_duplicates(
    subset=journey_keys
)

# ------------------------------------------------------------
# 3. Validate chronological order
# ------------------------------------------------------------

clean_journeys = clean_journeys[
    (
        clean_journeys["first_arrival_seconds"]
        >= clean_journeys["origin_departure_seconds"]
    )
    & (
        clean_journeys["second_departure_seconds"]
        >= clean_journeys["first_arrival_seconds"]
    )
    & (
        clean_journeys["second_arrival_seconds"]
        >= clean_journeys["second_departure_seconds"]
    )
    & (
        clean_journeys["third_departure_seconds"]
        >= clean_journeys["second_arrival_seconds"]
    )
    & (
        clean_journeys["destination_arrival_seconds"]
        >= clean_journeys["third_departure_seconds"]
    )
].copy()

# ------------------------------------------------------------
# 4. Calculate transfer safety margins
# ------------------------------------------------------------

# Ensure a minimum of 5 minutes at each transfer

clean_journeys["first_required_seconds"] = (
    clean_journeys["required_transfer_seconds"]
    .clip(lower=300)
)

clean_journeys["second_required_seconds"] = (
    clean_journeys["second_required_transfer"]
    .clip(lower=300)
)

clean_journeys["first_spare_minutes"] = (
    clean_journeys["second_departure_seconds"]
    - clean_journeys["first_arrival_seconds"]
    - clean_journeys["first_required_seconds"]
) / 60

clean_journeys["second_spare_minutes"] = (
    clean_journeys["third_departure_seconds"]
    - clean_journeys["second_arrival_seconds"]
    - clean_journeys["second_required_seconds"]
) / 60

# Keep only journeys with feasible transfers

clean_journeys = clean_journeys[
    (clean_journeys["first_spare_minutes"] >= 0)
    & (clean_journeys["second_spare_minutes"] >= 0)
].copy()

# Minimum spare time across both transfers

clean_journeys["minimum_spare_minutes"] = (
    clean_journeys[
        ["first_spare_minutes", "second_spare_minutes"]
    ].min(axis=1)
)

# ------------------------------------------------------------
# 5. Calculate journey duration
# ------------------------------------------------------------

clean_journeys["journey_duration_min"] = (
    clean_journeys["destination_arrival_seconds"]
    - clean_journeys["origin_departure_seconds"]
) / 60

# Sort by earliest arrival, then shortest journey duration

clean_journeys = clean_journeys.sort_values(
    [
        "destination_arrival_seconds",
        "journey_duration_min"
    ]
).reset_index(drop=True)

# ------------------------------------------------------------
# 6. Format departure and arrival times
# ------------------------------------------------------------

clean_journeys["departure_time"] = (
    clean_journeys["origin_departure_seconds"]
    .map(format_transport_time)
)

clean_journeys["arrival_time"] = (
    clean_journeys["destination_arrival_seconds"]
    .map(format_transport_time)
)

# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("\nOriginal candidate records:", len(complete_journeys))
print("Cleaned journey records:", len(clean_journeys))

display(
    clean_journeys[
        [
            "first_transfer_station",
            "second_transfer_station",
            "departure_time",
            "arrival_time",
            "journey_duration_min",
            "first_spare_minutes",
            "second_spare_minutes",
            "minimum_spare_minutes"
        ]
    ].head(10).round(2)
)

# ------------------------------------------------------------
# 8. Show earliest-arriving journey
# ------------------------------------------------------------

if not clean_journeys.empty:

    best_journey = clean_journeys.iloc[0]

    print("\nEARLIEST-ARRIVING TWO-TRANSFER JOURNEY")
    print("-------------------------------------")

    print("Departure:", best_journey["departure_time"])

    print(
        "First transfer:",
        best_journey["first_transfer_station"]
    )

    print(
        "Second transfer:",
        best_journey["second_transfer_station"]
    )

    print("Arrival:", best_journey["arrival_time"])

    print(
        "Journey duration:",
        round(best_journey["journey_duration_min"], 2),
        "minutes"
    )

    print(
        "First transfer spare time:",
        round(best_journey["first_spare_minutes"], 2),
        "minutes"
    )

    print(
        "Second transfer spare time:",
        round(best_journey["second_spare_minutes"], 2),
        "minutes"
    )

else:
    print("No valid two-transfer journeys found.")


Original candidate records: 55746
Records after restoring first transfer: 55746

Original candidate records: 55746
Cleaned journey records: 55746


,first_transfer_station,second_transfer_station,departure_time,arrival_time,journey_duration_min,first_spare_minutes,second_spare_minutes,minimum_spare_minutes
0,Epping Station,Strathfield Station,17:30:37,18:23:00,52.38,2.65,3.4,2.65
1,Epping Station,Strathfield Station,17:42:37,18:29:00,46.38,1.67,2.2,1.67
2,Epping Station,Strathfield Station,17:38:37,18:29:00,50.38,5.67,2.2,2.20
3,Epping Station,Strathfield Station,17:34:37,18:29:00,54.38,9.67,2.2,2.20
4,Epping Station,Strathfield Station,17:30:37,18:29:00,58.38,2.65,9.4,2.65
5,Epping Station,Strathfield Station,17:30:37,18:29:00,58.38,13.67,2.2,2.20
6,Epping Station,Strathfield Station,17:42:37,18:32:00,49.38,1.67,5.2,1.67
7,Epping Station,Strathfield Station,17:38:37,18:32:00,53.38,5.67,5.2,5.20
8,Epping Station,Strathfield Station,17:34:37,18:32:00,57.38,9.67,5.2,5.20
9,Epping Station,Strathfield Station,17:30:37,18:32:00,61.38,2.65,12.4,2.65



EARLIEST-ARRIVING TWO-TRANSFER JOURNEY
-------------------------------------
Departure: 17:30:37
First transfer: Epping Station
Second transfer: Strathfield Station
Arrival: 18:23:00
Journey duration: 52.38 minutes
First transfer spare time: 2.65 minutes
Second transfer spare time: 3.4 minutes


In [174]:

# STEP 87A — Compare journeys across 0, 1 and 2 transfers

origin = "Macquarie University Station"
destination = "Parramatta Station"
departure_time = "17:30:00"

# 0 transfers
direct_results = find_direct_journeys(
    realtime_daily,
    origin,
    destination,
    departure_time,
    window_minutes=60
)

# 1 transfer
one_transfer_results = find_one_transfer_journeys(
    realtime_daily,
    origin,
    destination,
    departure_time,
    window_minutes=60,
    connection_window_minutes=45
)

# 2 transfers
two_transfer_results = clean_journeys.copy()

print("Direct journeys:", len(direct_results))
print("One-transfer candidates:", len(one_transfer_results))
print("Two-transfer candidates:", len(two_transfer_results))

# Compare earliest destination arrival times
summary = []

if not direct_results.empty:
    print("\nDirect journey columns:")
    print(direct_results.columns.tolist())

if not one_transfer_results.empty:
    print("\nOne-transfer journey columns:")
    print(one_transfer_results.columns.tolist())

if not two_transfer_results.empty:
    summary.append({
        "transfers": 2,
        "earliest_arrival": format_transport_time(
            two_transfer_results["destination_arrival_seconds"].min()
        ),
        "minimum_duration_min": round(
            two_transfer_results["journey_duration_min"].min(), 2
        )
    })

display(pd.DataFrame(summary))


Direct journeys: 0
One-transfer candidates: 3078
Two-transfer candidates: 55746

One-transfer journey columns:
['boarding_id', 'trip_id', 'origin_platform', 'origin_sequence', 'origin_departure_seconds', 'origin_departure_time', 'first_line', 'first_mode', 'stop_id_x', 'station_id', 'stop_sequence', 'arrival_seconds', 'arrival_time', 'from_stop_id', 'to_stop_id', 'minimum_transfer_seconds', 'earliest_boarding_seconds', 'second_trip_id', 'second_platform', 'second_boarding_sequence', 'second_departure_seconds', 'second_departure_time', 'second_line', 'second_mode', 'destination_sequence', 'destination_arrival_seconds', 'destination_arrival_time', 'journey_duration_min', 'journey_type', 'transfers', 'stop_id_y', 'transfer_station']


,transfers,earliest_arrival,minimum_duration_min
0,2,18:23:00,45.38


In [175]:

# STEP 87B — Combine and compare 0, 1 and 2-transfer journeys

import pandas as pd

standard_columns = [
    "journey_type",
    "transfers",
    "origin_departure_seconds",
    "destination_arrival_seconds",
    "journey_duration_min",
    "first_transfer_station",
    "second_transfer_station",
    "minimum_spare_minutes"
]

journey_frames = []

# Direct journeys
if not direct_results.empty:
    direct = direct_results.copy()
    direct["journey_type"] = "Direct"
    direct["transfers"] = 0
    direct["first_transfer_station"] = None
    direct["second_transfer_station"] = None
    direct["minimum_spare_minutes"] = None

    journey_frames.append(direct[standard_columns])

# One-transfer journeys
if not one_transfer_results.empty:
    one = one_transfer_results.copy()

    one["journey_type"] = "One transfer"
    one["transfers"] = 1
    one["first_transfer_station"] = one["transfer_station"]
    one["second_transfer_station"] = None

    # Spare time beyond the required platform transfer
    one["minimum_spare_minutes"] = (
        one["second_departure_seconds"]
        - one["arrival_seconds"]
        - one["minimum_transfer_seconds"].clip(lower=300)
    ) / 60

    # Apply the same minimum transfer rule as two-transfer journeys
    one = one[one["minimum_spare_minutes"] >= 0].copy()

    journey_frames.append(one[standard_columns])

# Two-transfer journeys
if not two_transfer_results.empty:
    two = two_transfer_results.copy()
    two["journey_type"] = "Two transfers"
    two["transfers"] = 2

    journey_frames.append(two[standard_columns])

# Combine everything
all_journeys = pd.concat(
    journey_frames,
    ignore_index=True
)

# Sort by earliest arrival, then fewer transfers, then duration
all_journeys = all_journeys.sort_values(
    [
        "destination_arrival_seconds",
        "transfers",
        "journey_duration_min"
    ]
).reset_index(drop=True)

all_journeys["departure_time"] = (
    all_journeys["origin_departure_seconds"]
    .map(format_transport_time)
)

all_journeys["arrival_time"] = (
    all_journeys["destination_arrival_seconds"]
    .map(format_transport_time)
)

print("Total valid journey candidates:", len(all_journeys))

print("\nEarliest arrival by journey type:")

comparison = (
    all_journeys.groupby(["journey_type", "transfers"])
    .agg(
        candidates=("journey_type", "size"),
        earliest_arrival_seconds=("destination_arrival_seconds", "min"),
        shortest_duration_min=("journey_duration_min", "min")
    )
    .reset_index()
    .sort_values("earliest_arrival_seconds")
)

comparison["earliest_arrival"] = (
    comparison["earliest_arrival_seconds"]
    .map(format_transport_time)
)

display(
    comparison[
        [
            "journey_type",
            "transfers",
            "candidates",
            "earliest_arrival",
            "shortest_duration_min"
        ]
    ].round(2)
)

print("\nTop 10 earliest-arriving journeys:")

display(
    all_journeys[
        [
            "journey_type",
            "departure_time",
            "first_transfer_station",
            "second_transfer_station",
            "arrival_time",
            "journey_duration_min",
            "minimum_spare_minutes"
        ]
    ].head(10).round(2)
)

print("\nRECOMMENDED JOURNEY — EARLIEST ARRIVAL")

best = all_journeys.iloc[0]

print("Journey type:", best["journey_type"])
print("Departure:", best["departure_time"])
print("First transfer:", best["first_transfer_station"])
print("Second transfer:", best["second_transfer_station"])
print("Arrival:", best["arrival_time"])
print("Duration:", round(best["journey_duration_min"], 2), "minutes")


Total valid journey candidates: 58824

Earliest arrival by journey type:


,journey_type,transfers,candidates,earliest_arrival,shortest_duration_min
1,Two transfers,2,55746,18:23:00,45.38
0,One transfer,1,3078,18:29:00,56.10



Top 10 earliest-arriving journeys:


,journey_type,departure_time,first_transfer_station,second_transfer_station,arrival_time,journey_duration_min,minimum_spare_minutes
0,Two transfers,17:30:37,Epping Station,Strathfield Station,18:23:00,52.38,2.65
1,One transfer,17:31:54,Central Station,None,18:29:00,57.10,1.77
2,One transfer,17:31:54,Central Station,None,18:29:00,57.10,1.77
3,One transfer,17:31:54,Central Station,None,18:29:00,57.10,1.77
4,One transfer,17:31:54,Central Station,None,18:29:00,57.10,1.77
5,One transfer,17:31:54,Central Station,None,18:29:00,57.10,1.77
6,One transfer,17:31:54,Central Station,None,18:29:00,57.10,1.77
7,One transfer,17:31:54,Central Station,None,18:29:00,57.10,1.77
8,One transfer,17:31:54,Central Station,None,18:29:00,57.10,1.77
9,One transfer,17:31:54,Central Station,None,18:29:00,57.10,1.77



RECOMMENDED JOURNEY — EARLIEST ARRIVAL
Journey type: Two transfers
Departure: 17:30:37
First transfer: Epping Station
Second transfer: Strathfield Station
Arrival: 18:23:00
Duration: 52.38 minutes


In [176]:

# STEP 87C — Deduplicate journeys using actual service combinations

# Rebuild a combined table while retaining service identifiers
frames = []

if not direct_results.empty:
    direct = direct_results.copy()
    direct["journey_type"] = "Direct"
    direct["transfers"] = 0
    direct["first_transfer_station"] = None
    direct["second_transfer_station"] = None
    direct["minimum_spare_minutes"] = float("nan")
    direct["first_trip_id"] = direct["trip_id"]
    direct["second_trip_id"] = None
    direct["third_trip_id"] = None
    frames.append(direct)

if not one_transfer_results.empty:
    one = one_transfer_results.copy()
    one["journey_type"] = "One transfer"
    one["transfers"] = 1
    one["first_trip_id"] = one["trip_id"]
    one["third_trip_id"] = None
    one["first_transfer_station"] = one["transfer_station"]
    one["second_transfer_station"] = None
    one["minimum_spare_minutes"] = (
        one["second_departure_seconds"]
        - one["arrival_seconds"]
        - one["minimum_transfer_seconds"].clip(lower=300)
    ) / 60
    one = one[one["minimum_spare_minutes"] >= 0]
    frames.append(one)

if not clean_journeys.empty:
    two = clean_journeys.copy()
    two["journey_type"] = "Two transfers"
    two["transfers"] = 2
    frames.append(two)

cols = [
    "journey_type", "transfers",
    "first_trip_id", "second_trip_id", "third_trip_id",
    "origin_departure_seconds", "destination_arrival_seconds",
    "journey_duration_min",
    "first_transfer_station", "second_transfer_station",
    "minimum_spare_minutes"
]

journeys_deduplicated = pd.concat(
    [frame.reindex(columns=cols) for frame in frames],
    ignore_index=True
)

before = len(journeys_deduplicated)

journeys_deduplicated = journeys_deduplicated.drop_duplicates(
    subset=[
        "transfers",
        "first_trip_id", "second_trip_id", "third_trip_id",
        "origin_departure_seconds", "destination_arrival_seconds"
    ]
).sort_values(
    ["destination_arrival_seconds", "transfers", "journey_duration_min"]
).reset_index(drop=True)

journeys_deduplicated["departure_time"] = (
    journeys_deduplicated["origin_departure_seconds"]
    .map(format_transport_time)
)

journeys_deduplicated["arrival_time"] = (
    journeys_deduplicated["destination_arrival_seconds"]
    .map(format_transport_time)
)

print("Before deduplication:", before)
print("After deduplication:", len(journeys_deduplicated))
print("Duplicates removed:", before - len(journeys_deduplicated))

display(
    journeys_deduplicated[
        [
            "journey_type", "departure_time",
            "first_transfer_station", "second_transfer_station",
            "arrival_time", "journey_duration_min",
            "minimum_spare_minutes"
        ]
    ].head(15).round(2)
)


Before deduplication: 58824
After deduplication: 14748
Duplicates removed: 44076


,journey_type,departure_time,first_transfer_station,second_transfer_station,arrival_time,journey_duration_min,minimum_spare_minutes
0,Two transfers,17:30:37,Epping Station,Strathfield Station,18:23:00,52.38,2.65
1,One transfer,17:31:54,Central Station,None,18:29:00,57.10,1.77
2,Two transfers,17:42:37,Epping Station,Strathfield Station,18:29:00,46.38,1.67
3,Two transfers,17:38:37,Epping Station,Strathfield Station,18:29:00,50.38,2.20
4,Two transfers,17:34:37,Epping Station,Strathfield Station,18:29:00,54.38,2.20
5,Two transfers,17:30:37,Epping Station,Strathfield Station,18:29:00,58.38,2.65
6,Two transfers,17:30:37,Epping Station,Strathfield Station,18:29:00,58.38,2.20
7,One transfer,17:35:54,Central Station,None,18:32:00,56.10,0.77
8,One transfer,17:31:54,Central Station,None,18:32:00,60.10,4.77
9,Two transfers,17:42:37,Epping Station,Strathfield Station,18:32:00,49.38,1.67


In [177]:

# STEP 87D — Identify non-dominated journey alternatives

import numpy as np
import pandas as pd

candidates = journeys_deduplicated.copy()

# Sort to consider later departures first
candidates = candidates.sort_values(
    [
        "origin_departure_seconds",
        "destination_arrival_seconds",
        "transfers"
    ],
    ascending=[False, True, True]
).reset_index(drop=True)

# A journey is dominated if another journey:
# - departs at the same time or later
# - arrives at the same time or earlier
# - uses the same or fewer transfers
# - is strictly better on at least one criterion

keep = np.ones(len(candidates), dtype=bool)

for transfer_count in sorted(candidates["transfers"].unique()):

    group_indices = candidates.index[
        candidates["transfers"] == transfer_count
    ]

    for idx in group_indices:

        departure = candidates.at[idx, "origin_departure_seconds"]
        arrival = candidates.at[idx, "destination_arrival_seconds"]

        competitors = candidates[
            (candidates["origin_departure_seconds"] >= departure)
            & (candidates["destination_arrival_seconds"] <= arrival)
            & (candidates["transfers"] <= transfer_count)
        ]

        strictly_better = competitors[
            (competitors["origin_departure_seconds"] > departure)
            | (competitors["destination_arrival_seconds"] < arrival)
            | (competitors["transfers"] < transfer_count)
        ]

        if not strictly_better.empty:
            keep[idx] = False

non_dominated = candidates.loc[keep].copy()

non_dominated = non_dominated.sort_values(
    [
        "destination_arrival_seconds",
        "transfers",
        "origin_departure_seconds"
    ],
    ascending=[True, True, False]
).reset_index(drop=True)

print("Deduplicated candidates:", len(candidates))
print("Non-dominated alternatives:", len(non_dominated))

display(
    non_dominated[
        [
            "journey_type",
            "departure_time",
            "first_transfer_station",
            "second_transfer_station",
            "arrival_time",
            "journey_duration_min",
            "minimum_spare_minutes"
        ]
    ].head(20).round(2)
)


Deduplicated candidates: 14748
Non-dominated alternatives: 16


,journey_type,departure_time,first_transfer_station,second_transfer_station,arrival_time,journey_duration_min,minimum_spare_minutes
0,Two transfers,17:30:37,Epping Station,Strathfield Station,18:23:00,52.38,2.65
1,One transfer,17:31:54,Central Station,None,18:29:00,57.10,1.77
2,Two transfers,17:42:37,Epping Station,Strathfield Station,18:29:00,46.38,1.67
3,One transfer,17:35:54,Central Station,None,18:32:00,56.10,0.77
4,One transfer,17:39:54,Central Station,None,18:38:00,58.10,2.77
5,Two transfers,17:46:37,Epping Station,Strathfield Station,18:38:00,51.38,1.65
6,One transfer,17:43:54,Central Station,None,18:41:00,57.10,1.77
7,One transfer,17:47:54,Central Station,None,18:44:00,56.10,0.77
8,Two transfers,17:58:37,Epping Station,Strathfield Station,18:44:00,45.38,0.67
9,One transfer,17:55:54,Central Station,None,18:56:00,60.10,3.77


In [178]:

# STEP 88A — Reusable journey recommendation function

import pandas as pd

def recommend_journey(
    journeys,
    earliest_departure=None,
    latest_departure=None
):
    """
    Select the earliest-arriving feasible journey.

    Parameters
    ----------
    journeys : pd.DataFrame
        Standardised journey candidates.

    earliest_departure : str, optional
        Earliest allowed departure, e.g. "17:30:00".

    latest_departure : str, optional
        Latest allowed departure, e.g. "18:30:00".

    Returns
    -------
    pd.Series or None
        Recommended journey, or None if no journeys qualify.
    """

    if journeys.empty:
        return None

    options = journeys.copy()

    # Apply departure-time filters
    if earliest_departure is not None:
        start_seconds = gtfs_to_seconds(earliest_departure)

        options = options[
            options["origin_departure_seconds"] >= start_seconds
        ]

    if latest_departure is not None:
        end_seconds = gtfs_to_seconds(latest_departure)

        options = options[
            options["origin_departure_seconds"] <= end_seconds
        ]

    if options.empty:
        return None

    # Rank by earliest arrival, fewer transfers, shorter duration
    options = options.sort_values(
        [
            "destination_arrival_seconds",
            "transfers",
            "journey_duration_min"
        ],
        ascending=[True, True, True]
    )

    return options.iloc[0]


# Test using our deduplicated journeys
recommended = recommend_journey(
    journeys_deduplicated,
    earliest_departure="17:30:00",
    latest_departure="18:30:00"
)

if recommended is None:
    print("No suitable journey found.")
else:
    print("RECOMMENDED JOURNEY")
    print("-------------------")
    print("Type:", recommended["journey_type"])
    print(
        "Departure:",
        format_transport_time(
            recommended["origin_departure_seconds"]
        )
    )
    print("First transfer:", recommended["first_transfer_station"])
    print("Second transfer:", recommended["second_transfer_station"])
    print(
        "Arrival:",
        format_transport_time(
            recommended["destination_arrival_seconds"]
        )
    )
    print(
        "Duration:",
        round(recommended["journey_duration_min"], 2),
        "minutes"
    )
    print(
        "Minimum spare transfer time:",
        round(recommended["minimum_spare_minutes"], 2),
        "minutes"
    )


RECOMMENDED JOURNEY
-------------------
Type: Two transfers
Departure: 17:30:37
First transfer: Epping Station
Second transfer: Strathfield Station
Arrival: 18:23:00
Duration: 52.38 minutes
Minimum spare transfer time: 2.65 minutes


In [179]:

# STEP 88C — Inspect platform information for the recommended journey

# Get the full recommended two-transfer journey
best = clean_journeys.iloc[0]

print("RECOMMENDED JOURNEY")
print("===================")

print("First service:", best["first_trip_id"])
print("Second service:", best["second_trip_id"])
print("Third service:", best["third_trip_id"])

print("\nTRANSFER DETAILS")
print("----------------")

print("First transfer:", best["first_transfer_station"])
print("First arrival platform ID:", best["first_arrival_platform"])
print("Second boarding platform ID:", best["second_boarding_platform"])

print("\nSecond transfer:", best["second_transfer_station"])
print("Second arrival platform ID:", best["second_arrival_platform"])
print("Third boarding platform ID:", best["third_boarding_platform"])

print("\nSTATION LOOKUP COLUMNS")
print(station_lookup.columns.tolist())

print("\nGTFS PLATFORM RECORDS")

platform_ids = [
    best["first_arrival_platform"],
    best["second_boarding_platform"],
    best["second_arrival_platform"],
    best["third_boarding_platform"]
]

display(
    station_lookup[
        station_lookup["stop_id"].astype(str).isin(
            [str(p) for p in platform_ids]
        )
    ]
)


RECOMMENDED JOURNEY
First service: 0160-006-154-013:1000
Second service: 125R.1324.165.16.A.8.92001037
Third service: 197F.1324.165.16.B.8.92001727

TRANSFER DETAILS
----------------
First transfer: Epping Station
First arrival platform ID: 2121226
Second boarding platform ID: 2121221

Second transfer: Strathfield Station
Second arrival platform ID: 2135235
Third boarding platform ID: 2135236

STATION LOOKUP COLUMNS
['stop_id', 'station_id', 'stop_name']

GTFS PLATFORM RECORDS


,stop_id,station_id,stop_name
32,2121226,212110,Epping Station
184,2135236,213510,Strathfield Station
205,2121221,212110,Epping Station
220,2135235,213510,Strathfield Station


In [180]:

# STEP 88D — Find platform labels in existing GTFS data

import pandas as pd

# Look for existing DataFrames containing GTFS stop information
possible_tables = []

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        columns = set(value.columns)

        if "stop_id" in columns and (
            "platform_code" in columns
            or "parent_station" in columns
            or "stop_name" in columns
        ):
            possible_tables.append({
                "dataframe": name,
                "rows": len(value),
                "columns": ", ".join(value.columns)
            })

print("Potential GTFS stop tables:")
display(pd.DataFrame(possible_tables))

# Inspect the relevant platform IDs in any table
# containing a platform_code field

platform_ids = {
    "2121226",
    "2121221",
    "2135235",
    "2135236"
}

for item in possible_tables:
    name = item["dataframe"]
    df = globals()[name]

    if "platform_code" in df.columns:
        matches = df[
            df["stop_id"].astype(str).isin(platform_ids)
        ]

        if not matches.empty:
            print(f"\nPlatform details found in: {name}")
            display(matches)


Potential GTFS stop tables:


,dataframe,rows,columns
0,selected_stations,2,"stop_id, stop_code, stop_name, stop_lat, stop_..."
1,train_platforms,28,"stop_id, stop_code, stop_name, stop_lat, stop_..."
2,t8_stops,21,"trip_id, arrival_time, departure_time, stop_id..."
3,transport_stops,568,"stop_id, stop_code, stop_name, stop_lat, stop_..."
4,parent_stations,78053,"stop_id, stop_name"
5,multimodal_stations_named,9,"station_id, transport_modes, platform_count, m..."
6,central_platforms,29,"stop_id, stop_name, transport_mode"
7,group,4,"stop_id, stop_code, stop_name, stop_lat, stop_..."
8,duplicates,0,"trip_id, stop_id, stop_name, route_short_name,..."
9,downstream_stops,19,"trip_id, stop_id, arrival_time, departure_time..."



Platform details found in: transport_stops


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code,station_id,transport_mode
108,2121221,2121221,"Epping Station, Platform 1",-33.77301778,151.08232828,NaN,212110,1,Level 0,1,212110,Train
112,2121226,2121226,"Epping Station, Platform 6",-33.77295652,151.08200169,NaN,212110,1,Level -2,6,212110,Metro
123,2135235,2135235,"Strathfield Station, Platform 5",-33.87152049,151.09396515,NaN,213510,1,Level 0,5,213510,Train
124,2135236,2135236,"Strathfield Station, Platform 6",-33.87156727,151.09391337,NaN,213510,1,Level 0,6,213510,Train



Platform details found in: stops_df


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code
14119,2121221,2121221.0,"Epping Station, Platform 1",-33.773018,151.082328,NaN,212110,1.0,Level 0,1
14123,2121226,2121226.0,"Epping Station, Platform 6",-33.772957,151.082002,NaN,212110,1.0,Level -2,6
14231,2135235,2135235.0,"Strathfield Station, Platform 5",-33.871520,151.093965,NaN,213510,1.0,Level 0,5
14232,2135236,2135236.0,"Strathfield Station, Platform 6",-33.871567,151.093913,NaN,213510,1.0,Level 0,6


In [181]:

# STEP 88E — Build passenger-facing journey details

import pandas as pd

best = clean_journeys.iloc[0]

# Platform lookup from original GTFS data
platform_lookup = (
    transport_stops[
        ["stop_id", "platform_code", "transport_mode", "stop_name"]
    ]
    .drop_duplicates(subset=["stop_id"])
    .copy()
)

platform_lookup["stop_id"] = platform_lookup["stop_id"].astype(str)
platform_lookup = platform_lookup.set_index("stop_id")

def get_platform_details(stop_id):
    stop_id = str(stop_id)

    if stop_id not in platform_lookup.index:
        return {"platform": "Unknown", "mode": "Unknown"}

    record = platform_lookup.loc[stop_id]

    platform = record["platform_code"]
    mode = record["transport_mode"]

    return {
        "platform": (
            str(platform)
            if pd.notna(platform)
            else "Unknown"
        ),
        "mode": (
            str(mode)
            if pd.notna(mode)
            else "Unknown"
        )
    }


# Find the first service's origin boarding platform
origin_stop = realtime_daily[
    (realtime_daily["trip_id"] == best["first_trip_id"])
    & (realtime_daily["station_id"].astype(str).isin(
        station_lookup.loc[
            station_lookup["stop_name"] == origin,
            "station_id"
        ].astype(str)
    ))
    & (
        realtime_daily["departure_seconds"]
        == best["origin_departure_seconds"]
    )
]

# Find the third service's destination arrival platform
destination_stop = realtime_daily[
    (realtime_daily["trip_id"] == best["third_trip_id"])
    & (realtime_daily["station_id"].astype(str).isin(
        station_lookup.loc[
            station_lookup["stop_name"] == destination,
            "station_id"
        ].astype(str)
    ))
    & (
        realtime_daily["arrival_seconds"]
        == best["destination_arrival_seconds"]
    )
]

origin_platform_id = (
    origin_stop.iloc[0]["stop_id"]
    if not origin_stop.empty else None
)

destination_platform_id = (
    destination_stop.iloc[0]["stop_id"]
    if not destination_stop.empty else None
)

# Create structured journey details
journey_details = pd.DataFrame([
    {
        "station": origin,
        "arrival": None,
        "arrival_platform": None,
        "departure": format_transport_time(
            best["origin_departure_seconds"]
        ),
        "departure_platform": get_platform_details(
            origin_platform_id
        )["platform"],
        "departure_mode": get_platform_details(
            origin_platform_id
        )["mode"]
    },
    {
        "station": best["first_transfer_station"],
        "arrival": format_transport_time(
            best["first_arrival_seconds"]
        ),
        "arrival_platform": get_platform_details(
            best["first_arrival_platform"]
        )["platform"],
        "departure": format_transport_time(
            best["second_departure_seconds"]
        ),
        "departure_platform": get_platform_details(
            best["second_boarding_platform"]
        )["platform"],
        "departure_mode": get_platform_details(
            best["second_boarding_platform"]
        )["mode"]
    },
    {
        "station": best["second_transfer_station"],
        "arrival": format_transport_time(
            best["second_arrival_seconds"]
        ),
        "arrival_platform": get_platform_details(
            best["second_arrival_platform"]
        )["platform"],
        "departure": format_transport_time(
            best["third_departure_seconds"]
        ),
        "departure_platform": get_platform_details(
            best["third_boarding_platform"]
        )["platform"],
        "departure_mode": get_platform_details(
            best["third_boarding_platform"]
        )["mode"]
    },
    {
        "station": destination,
        "arrival": format_transport_time(
            best["destination_arrival_seconds"]
        ),
        "arrival_platform": get_platform_details(
            destination_platform_id
        )["platform"],
        "departure": None,
        "departure_platform": None,
        "departure_mode": None
    }
])

print("PASSENGER JOURNEY DETAILS")
display(journey_details)


PASSENGER JOURNEY DETAILS


,station,arrival,arrival_platform,departure,departure_platform,departure_mode
0,Macquarie University Station,NaN,NaN,17:30:37,2,Metro
1,Epping Station,17:33:51,6,17:42:00,1,Train
2,Strathfield Station,18:03:00,5,18:11:24,6,Train
3,Parramatta Station,18:23:00,2,NaN,NaN,NaN


In [182]:

# STEP 89A — Reusable passenger journey formatter

def format_two_transfer_journey(
    journey,
    origin,
    destination,
    timetable=realtime_daily
):
    """
    Convert a selected two-transfer journey into a
    passenger-facing station-by-station DataFrame.
    """

    def platform_details(stop_id):
        if pd.isna(stop_id):
            return {"platform": "Unknown", "mode": "Unknown"}
        return get_platform_details(stop_id)

    def find_service_stop(trip_id, station_name, time_column, seconds):
        station_ids = set(
            station_lookup.loc[
                station_lookup["stop_name"] == station_name,
                "station_id"
            ].astype(str)
        )

        matches = timetable[
            (timetable["trip_id"] == trip_id)
            & (timetable["station_id"].astype(str).isin(station_ids))
            & (timetable[time_column] == seconds)
        ]

        return matches.iloc[0]["stop_id"] if not matches.empty else None

    origin_stop_id = find_service_stop(
        journey["first_trip_id"],
        origin,
        "departure_seconds",
        journey["origin_departure_seconds"]
    )

    destination_stop_id = find_service_stop(
        journey["third_trip_id"],
        destination,
        "arrival_seconds",
        journey["destination_arrival_seconds"]
    )

    stations = [
        {
            "station": origin,
            "arrival": None,
            "arrival_platform": None,
            "departure": format_transport_time(
                journey["origin_departure_seconds"]
            ),
            "departure_platform": platform_details(origin_stop_id)["platform"],
            "departure_mode": platform_details(origin_stop_id)["mode"]
        },
        {
            "station": journey["first_transfer_station"],
            "arrival": format_transport_time(
                journey["first_arrival_seconds"]
            ),
            "arrival_platform": platform_details(
                journey["first_arrival_platform"]
            )["platform"],
            "departure": format_transport_time(
                journey["second_departure_seconds"]
            ),
            "departure_platform": platform_details(
                journey["second_boarding_platform"]
            )["platform"],
            "departure_mode": platform_details(
                journey["second_boarding_platform"]
            )["mode"]
        },
        {
            "station": journey["second_transfer_station"],
            "arrival": format_transport_time(
                journey["second_arrival_seconds"]
            ),
            "arrival_platform": platform_details(
                journey["second_arrival_platform"]
            )["platform"],
            "departure": format_transport_time(
                journey["third_departure_seconds"]
            ),
            "departure_platform": platform_details(
                journey["third_boarding_platform"]
            )["platform"],
            "departure_mode": platform_details(
                journey["third_boarding_platform"]
            )["mode"]
        },
        {
            "station": destination,
            "arrival": format_transport_time(
                journey["destination_arrival_seconds"]
            ),
            "arrival_platform": platform_details(
                destination_stop_id
            )["platform"],
            "departure": None,
            "departure_platform": None,
            "departure_mode": None
        }
    ]

    return pd.DataFrame(stations)


# Test the formatter
formatted_journey = format_two_transfer_journey(
    clean_journeys.iloc[0],
    origin="Macquarie University Station",
    destination="Parramatta Station"
)

display(formatted_journey)


,station,arrival,arrival_platform,departure,departure_platform,departure_mode
0,Macquarie University Station,NaN,NaN,17:30:37,2,Metro
1,Epping Station,17:33:51,6,17:42:00,1,Train
2,Strathfield Station,18:03:00,5,18:11:24,6,Train
3,Parramatta Station,18:23:00,2,NaN,NaN,NaN


In [183]:

# STEP 89B — Inspect routing function signatures and required data

import inspect
import pandas as pd

functions_to_check = [
    "get_station_departures",
    "find_direct_journeys",
    "find_one_transfer_journeys",
    "recommend_journey",
    "format_two_transfer_journey"
]

for name in functions_to_check:
    function = globals().get(name)

    if callable(function):
        print(f"{name}{inspect.signature(function)}")
    else:
        print(f"{name}: NOT FOUND")

print("\nROUTING DATA")
print("Timetable rows:", len(realtime_daily))
print("Station lookup rows:", len(station_lookup))
print("Platform transfer rules:", len(platform_rules))

print("\nFIRST-LEG DATA COLUMNS")
print(first_leg_stops.columns.tolist())

print("\nSECOND-LEG CANDIDATE COLUMNS")
print(second_leg_candidates.columns.tolist())

print("\nFINAL TWO-TRANSFER COLUMNS")
print(clean_journeys.columns.tolist())


get_station_departures(daily, origin, departure_time, window_minutes=30)
find_direct_journeys(daily, origin, destination, departure_time, window_minutes=30)
find_one_transfer_journeys(daily, origin, destination, departure_time, window_minutes=30, connection_window_minutes=30)
recommend_journey(journeys, earliest_departure=None, latest_departure=None)
format_two_transfer_journey(journey, origin, destination, timetable=                             trip_id  stop_id arrival_time departure_time  \
0              0160-006-101-002:1000  2155269     04:08:00       04:08:00   
1              0160-006-101-002:1000  2155267     04:10:03       04:10:33   
2              0160-006-101-002:1000  2155265     04:12:52       04:13:22   
3              0160-006-101-002:1000  2153402     04:15:22       04:15:52   
4              0160-006-101-002:1000  2153404     04:17:59       04:18:29   
...                              ...      ...          ...            ...   
59536  L2DD.1324.165.16.M.4.92210393  21

In [184]:

# STEP 89C — Test routing with a different origin and destination

test_origin = "Chatswood Station"
test_destination = "Central Station"
test_departure = "17:30:00"

direct_test = find_direct_journeys(
    realtime_daily,
    test_origin,
    test_destination,
    test_departure,
    window_minutes=60
)

one_transfer_test = find_one_transfer_journeys(
    realtime_daily,
    test_origin,
    test_destination,
    test_departure,
    window_minutes=60,
    connection_window_minutes=45
)

print("ROUTING TEST")
print("============")
print("Origin:", test_origin)
print("Destination:", test_destination)

print("\nDirect journeys:", len(direct_test))
print("One-transfer journeys:", len(one_transfer_test))

for label, results in [
    ("DIRECT", direct_test),
    ("ONE TRANSFER", one_transfer_test)
]:
    if results.empty:
        print(f"\n{label}: No journeys found")
        continue

    print(f"\n{label} COLUMNS:")
    print(results.columns.tolist())

    print(f"\n{label} SAMPLE:")
    display(results.head(3))


ROUTING TEST
Origin: Chatswood Station
Destination: Central Station

Direct journeys: 29
One-transfer journeys: 18428

DIRECT COLUMNS:
['trip_id', 'stop_id', 'arrival_time', 'departure_time', 'stop_sequence', 'route_short_name', 'transport_mode', 'service_id', 'arrival_seconds', 'departure_seconds', 'station_id', 'scheduled_arrival_sec', 'scheduled_departure_sec', 'effective_arrival_sec', 'effective_departure_sec', 'arrival_source', 'departure_source', 'predicted_arrival_sec', 'predicted_departure_sec', 'original_arrival_seconds', 'original_departure_seconds', 'last_stop_sequence', 'destination_sequence', 'destination_arrival_seconds', 'destination_arrival_time', 'journey_duration_min', 'journey_type', 'transfers']

DIRECT SAMPLE:


,trip_id,stop_id,arrival_time,departure_time,stop_sequence,route_short_name,transport_mode,service_id,arrival_seconds,departure_seconds,...,predicted_departure_sec,original_arrival_seconds,original_departure_seconds,last_stop_sequence,destination_sequence,destination_arrival_seconds,destination_arrival_time,journey_duration_min,journey_type,transfers
0,0160-006-130-004:1000,2067142,17:33:10,17:33:55,13,M1,Metro,TA+fq+11,63190.0,63235.0,...,NaN,63190,63235,21,19,64089.0,17:48:09,14.23,Direct,0
1,0160-006-105-014:1000,2067142,17:37:10,17:37:55,13,M1,Metro,TA+fq+11,63430.0,63475.0,...,NaN,63430,63475,21,19,64329.0,17:52:09,14.23,Direct,0
2,0160-006-106-014:1000,2067142,17:41:10,17:41:55,13,M1,Metro,TA+fq+11,63670.0,63715.0,...,NaN,63670,63715,21,19,64569.0,17:56:09,14.23,Direct,0



ONE TRANSFER COLUMNS:
['boarding_id', 'trip_id', 'origin_platform', 'origin_sequence', 'origin_departure_seconds', 'origin_departure_time', 'first_line', 'first_mode', 'stop_id_x', 'station_id', 'stop_sequence', 'arrival_seconds', 'arrival_time', 'from_stop_id', 'to_stop_id', 'minimum_transfer_seconds', 'earliest_boarding_seconds', 'second_trip_id', 'second_platform', 'second_boarding_sequence', 'second_departure_seconds', 'second_departure_time', 'second_line', 'second_mode', 'destination_sequence', 'destination_arrival_seconds', 'destination_arrival_time', 'journey_duration_min', 'journey_type', 'transfers', 'stop_id_y', 'transfer_station']

ONE TRANSFER SAMPLE:


,boarding_id,trip_id,origin_platform,origin_sequence,origin_departure_seconds,origin_departure_time,first_line,first_mode,stop_id_x,station_id,...,second_line,second_mode,destination_sequence,destination_arrival_seconds,destination_arrival_time,journey_duration_min,journey_type,transfers,stop_id_y,transfer_station
0,3,0160-006-130-004:1000,2067142,13,63235.0,17:33:55,M1,Metro,2065163,206516,...,M1,Metro,19,64329.0,17:52:09,18.23,One transfer,1,2065163,Crows Nest Station
1,3,0160-006-130-004:1000,2067142,13,63235.0,17:33:55,M1,Metro,2065163,206516,...,M1,Metro,19,64329.0,17:52:09,18.23,One transfer,1,2065164,Crows Nest Station
2,3,0160-006-130-004:1000,2067142,13,63235.0,17:33:55,M1,Metro,2060115,206044,...,M1,Metro,19,64329.0,17:52:09,18.23,One transfer,1,2060115,Victoria Cross Station


In [185]:

# STEP 89D — Combine and rank direct + one-transfer journeys

import pandas as pd

direct_standard = pd.DataFrame({
    "journey_type": direct_test["journey_type"],
    "transfers": direct_test["transfers"],
    "first_trip_id": direct_test["trip_id"],
    "second_trip_id": None,
    "origin_departure_seconds": direct_test["departure_seconds"],
    "destination_arrival_seconds": direct_test["destination_arrival_seconds"],
    "journey_duration_min": direct_test["journey_duration_min"],
    "transfer_station": None
})

transfer_standard = pd.DataFrame({
    "journey_type": one_transfer_test["journey_type"],
    "transfers": one_transfer_test["transfers"],
    "first_trip_id": one_transfer_test["trip_id"],
    "second_trip_id": one_transfer_test["second_trip_id"],
    "origin_departure_seconds": one_transfer_test["origin_departure_seconds"],
    "destination_arrival_seconds": one_transfer_test["destination_arrival_seconds"],
    "journey_duration_min": one_transfer_test["journey_duration_min"],
    "transfer_station": one_transfer_test["transfer_station"]
})

combined_routes = pd.concat(
    [direct_standard, transfer_standard],
    ignore_index=True
)

# Keep only chronologically valid journeys
combined_routes = combined_routes[
    combined_routes["destination_arrival_seconds"]
    > combined_routes["origin_departure_seconds"]
].copy()

# Remove duplicate service combinations
combined_routes = combined_routes.drop_duplicates(
    subset=[
        "first_trip_id",
        "second_trip_id",
        "origin_departure_seconds",
        "destination_arrival_seconds",
        "transfer_station"
    ]
)

# Prioritise earliest arrival, then fewer transfers
ranked_routes = combined_routes.sort_values(
    by=[
        "destination_arrival_seconds",
        "transfers",
        "origin_departure_seconds"
    ],
    ascending=[True, True, False]
).reset_index(drop=True)

ranked_routes["departure"] = ranked_routes[
    "origin_departure_seconds"
].apply(format_transport_time)

ranked_routes["arrival"] = ranked_routes[
    "destination_arrival_seconds"
].apply(format_transport_time)

print("Total unique candidate journeys:", len(ranked_routes))

display(
    ranked_routes[
        [
            "journey_type",
            "departure",
            "arrival",
            "journey_duration_min",
            "transfers",
            "transfer_station"
        ]
    ].head(10)
)


Total unique candidate journeys: 5223


,journey_type,departure,arrival,journey_duration_min,transfers,transfer_station
0,Direct,17:33:55,17:48:09,14.23,0,None
1,Direct,17:37:55,17:52:09,14.23,0,None
2,One transfer,17:33:55,17:52:09,18.23,1,Crows Nest Station
3,One transfer,17:33:55,17:52:09,18.23,1,Victoria Cross Station
4,One transfer,17:33:55,17:52:09,18.23,1,Barangaroo Station
5,One transfer,17:33:55,17:52:09,18.23,1,Martin Place Station
6,One transfer,17:33:55,17:52:09,18.23,1,Gadigal Station
7,Direct,17:41:55,17:56:09,14.23,0,None
8,One transfer,17:37:55,17:56:09,18.23,1,Crows Nest Station
9,One transfer,17:37:55,17:56:09,18.23,1,Victoria Cross Station


In [186]:

# STEP 89E — Remove dominated journey options

import numpy as np

def pareto_filter_journeys(journeys):
    routes = journeys.copy()

    # Compare routes using departure, arrival and transfer count
    values = routes[
        [
            "origin_departure_seconds",
            "destination_arrival_seconds",
            "transfers"
        ]
    ].to_numpy(dtype=float)

    keep = np.ones(len(values), dtype=bool)

    for i, (dep, arr, transfers) in enumerate(values):
        dominated = (
            (values[:, 0] >= dep)
            & (values[:, 1] <= arr)
            & (values[:, 2] <= transfers)
            & (
                (values[:, 0] > dep)
                | (values[:, 1] < arr)
                | (values[:, 2] < transfers)
            )
        )

        if dominated.any():
            keep[i] = False

    return routes.loc[keep].sort_values(
        ["destination_arrival_seconds", "transfers"]
    ).reset_index(drop=True)


efficient_routes = pareto_filter_journeys(ranked_routes)

print("Before filtering:", len(ranked_routes))
print("After filtering:", len(efficient_routes))

display(
    efficient_routes[
        [
            "journey_type",
            "departure",
            "arrival",
            "journey_duration_min",
            "transfers",
            "transfer_station"
        ]
    ].head(15)
)


Before filtering: 5223
After filtering: 16


,journey_type,departure,arrival,journey_duration_min,transfers,transfer_station
0,Direct,17:33:55,17:48:09,14.23,0,None
1,Direct,17:37:55,17:52:09,14.23,0,None
2,Direct,17:41:55,17:56:09,14.23,0,None
3,Direct,17:45:55,18:00:09,14.23,0,None
4,Direct,17:49:55,18:04:09,14.23,0,None
5,Direct,17:53:55,18:08:09,14.23,0,None
6,Direct,17:57:55,18:12:09,14.23,0,None
7,Direct,18:01:55,18:16:09,14.23,0,None
8,Direct,18:05:55,18:20:09,14.23,0,None
9,Direct,18:09:55,18:24:09,14.23,0,None


In [188]:

# STEP 90A — Inspect two-transfer routing stages

stages = [
    "origin_departures",
    "first_leg_stops",
    "transfer_candidates",
    "first_arrivals",
    "first_transfer_options",
    "second_leg_candidates",
    "second_leg_stops",
    "third_transfer_options",
    "third_leg_candidates",
    "complete_journeys",
    "clean_journeys"
]

for name in stages:
    value = globals().get(name)

    if isinstance(value, pd.DataFrame):
        print(f"\n{name}: {len(value):,} rows")
        print("Columns:", value.columns.tolist())
    else:
        print(f"\n{name}: NOT FOUND")

print("\nTRANSFER RULES")
print("Columns:", all_platform_transfers.columns.tolist())
print("Rows:", len(all_platform_transfers))



origin_departures: 30 rows
Columns: ['trip_id', 'stop_id', 'arrival_time', 'departure_time', 'stop_sequence', 'route_short_name', 'transport_mode', 'service_id', 'arrival_seconds', 'departure_seconds', 'station_id', 'scheduled_arrival_sec', 'scheduled_departure_sec', 'effective_arrival_sec', 'effective_departure_sec', 'arrival_source', 'departure_source', 'predicted_arrival_sec', 'predicted_departure_sec', 'original_arrival_seconds', 'original_departure_seconds', 'last_stop_sequence']

first_leg_stops: 300 rows
Columns: ['trip_id', 'origin_sequence', 'origin_departure_seconds', 'station_id', 'stop_id', 'stop_sequence', 'arrival_seconds', 'transfer_station']

transfer_candidates: 20 rows
Columns: ['transfer_station', 'available_arrivals', 'earliest_arrival', 'earliest_arrival_time']

first_arrivals: 300 rows
Columns: ['trip_id', 'station_id', 'stop_id', 'arrival_seconds', 'origin_departure_seconds']

first_transfer_options: 1,170 rows
Columns: ['trip_id', 'station_id', 'stop_id', 'arri

In [189]:

# STEP 90B — Reusable first-leg search

def build_first_leg_stops(
    daily,
    origin,
    departure_time,
    window_minutes=60
):
    # Find services departing from the origin
    departures = get_station_departures(
        daily,
        origin,
        departure_time,
        window_minutes=window_minutes
    ).copy()

    if departures.empty:
        return pd.DataFrame()

    # Keep original boarding information
    boarding = departures[
        [
            "trip_id",
            "stop_id",
            "stop_sequence",
            "departure_seconds"
        ]
    ].rename(columns={
        "stop_id": "origin_platform",
        "stop_sequence": "origin_sequence",
        "departure_seconds": "origin_departure_seconds"
    })

    # Find subsequent stops on each service
    downstream = daily.merge(
        boarding,
        on="trip_id",
        how="inner"
    )

    downstream = downstream[
        (downstream["stop_sequence"] > downstream["origin_sequence"])
        & (
            downstream["arrival_seconds"]
            > downstream["origin_departure_seconds"]
        )
    ].copy()

    # Add readable station names
    station_names_lookup = (
        station_lookup[
            ["station_id", "stop_name"]
        ]
        .drop_duplicates("station_id")
        .rename(columns={"stop_name": "transfer_station"})
    )

    downstream = downstream.merge(
        station_names_lookup,
        on="station_id",
        how="left"
    )

    return downstream[
        [
            "trip_id",
            "origin_platform",
            "origin_sequence",
            "origin_departure_seconds",
            "station_id",
            "stop_id",
            "stop_sequence",
            "arrival_seconds",
            "transfer_station"
        ]
    ].reset_index(drop=True)


# Test against the original journey
first_leg_test = build_first_leg_stops(
    realtime_daily,
    origin="Macquarie University Station",
    departure_time="17:30:00",
    window_minutes=60
)

print("Original first-leg stops:", len(first_leg_stops))
print("Reusable function stops:", len(first_leg_test))

display(first_leg_test.head(10))


Original first-leg stops: 300
Reusable function stops: 300


,trip_id,origin_platform,origin_sequence,origin_departure_seconds,station_id,stop_id,stop_sequence,arrival_seconds,transfer_station
0,0160-006-101-015:1000,2113352,12,64237.0,212110,2121226,13,64431.0,Epping Station
1,0160-006-101-015:1000,2113352,12,64237.0,2126158,2126160,14,64778.0,Cherrybrook Station
2,0160-006-101-015:1000,2113352,12,64237.0,2154391,2154263,15,64942.0,Castle Hill Station
3,0160-006-101-015:1000,2113352,12,64237.0,2154392,2154265,16,65095.0,Hills Showground Station
4,0160-006-101-015:1000,2113352,12,64237.0,2153477,2153405,17,65250.0,Norwest Station
5,0160-006-101-015:1000,2113352,12,64237.0,2153478,2153403,18,65410.0,Bella Vista Station
6,0160-006-101-015:1000,2113352,12,64237.0,2155382,2155266,19,65565.0,Kellyville Station
7,0160-006-101-015:1000,2113352,12,64237.0,2155383,2155268,20,65740.0,Rouse Hill Station
8,0160-006-101-015:1000,2113352,12,64237.0,2155384,2155269,21,65897.0,Tallawong Station
9,0160-006-102-015:1000,2113352,12,64477.0,212110,2121226,13,64671.0,Epping Station


In [190]:

# STEP 90C — Reusable first-transfer feasibility function

def build_first_transfer_options(
    first_leg,
    transfer_rules=all_platform_transfers,
    minimum_buffer_seconds=300
):
    if first_leg.empty:
        return pd.DataFrame()

    # Prepare platform transfer rules
    rules = transfer_rules[
        [
            "station_id",
            "from_stop_id",
            "to_stop_id",
            "minimum_transfer_seconds"
        ]
    ].copy()

    # Match arrival platform to possible departure platforms
    options = first_leg.merge(
        rules,
        left_on=["station_id", "stop_id"],
        right_on=["station_id", "from_stop_id"],
        how="inner"
    )

    # Apply a minimum 5-minute transfer buffer
    options["required_transfer_seconds"] = options[
        "minimum_transfer_seconds"
    ].clip(lower=minimum_buffer_seconds)

    options["earliest_next_departure"] = (
        options["arrival_seconds"]
        + options["required_transfer_seconds"]
    )

    return options.reset_index(drop=True)


# Test against our original calculation
first_transfer_test = build_first_transfer_options(
    first_leg_test
)

print("Original transfer options:", len(first_transfer_options))
print("Reusable transfer options:", len(first_transfer_test))

# Inspect Epping transfer options
epping_options = first_transfer_test[
    first_transfer_test["transfer_station"] == "Epping Station"
]

display(
    epping_options[
        [
            "trip_id",
            "stop_id",
            "to_stop_id",
            "arrival_seconds",
            "required_transfer_seconds",
            "earliest_next_departure"
        ]
    ].head(10)
)


Original transfer options: 1170
Reusable transfer options: 1170


,trip_id,stop_id,to_stop_id,arrival_seconds,required_transfer_seconds,earliest_next_departure
0,0160-006-101-015:1000,2121226,2121221,64431.0,330.0,64761.0
1,0160-006-101-015:1000,2121226,2121222,64431.0,330.0,64761.0
2,0160-006-101-015:1000,2121226,2121223,64431.0,330.0,64761.0
3,0160-006-101-015:1000,2121226,2121225,64431.0,300.0,64731.0
4,0160-006-101-015:1000,2121226,2121226,64431.0,300.0,64731.0
21,0160-006-102-015:1000,2121226,2121221,64671.0,330.0,65001.0
22,0160-006-102-015:1000,2121226,2121222,64671.0,330.0,65001.0
23,0160-006-102-015:1000,2121226,2121223,64671.0,330.0,65001.0
24,0160-006-102-015:1000,2121226,2121225,64671.0,300.0,64971.0
25,0160-006-102-015:1000,2121226,2121226,64671.0,300.0,64971.0


In [191]:

# STEP 90D — Reusable second-leg service search

def build_second_leg_candidates(
    daily,
    transfer_options,
    connection_window_minutes=45
):
    if transfer_options.empty:
        return pd.DataFrame()

    # Candidate boarding events on the transfer platforms
    boarding_events = daily[
        [
            "trip_id",
            "station_id",
            "stop_id",
            "stop_sequence",
            "departure_seconds"
        ]
    ].rename(columns={
        "trip_id": "second_trip_id",
        "station_id": "second_station_id",
        "stop_id": "second_boarding_platform",
        "stop_sequence": "second_boarding_sequence",
        "departure_seconds": "second_departure_seconds"
    })

    # Match by the exact boarding platform
    candidates = transfer_options.merge(
        boarding_events,
        left_on=["station_id", "to_stop_id"],
        right_on=["second_station_id", "second_boarding_platform"],
        how="inner"
    )

    # Only keep feasible connections within the search window
    candidates = candidates[
        (
            candidates["second_departure_seconds"]
            >= candidates["earliest_next_departure"]
        )
        & (
            candidates["second_departure_seconds"]
            <= candidates["earliest_next_departure"]
            + connection_window_minutes * 60
        )
        & (
            candidates["trip_id"] != candidates["second_trip_id"]
        )
    ].copy()

    # Rename first-leg details for clarity
    candidates = candidates.rename(columns={
        "trip_id": "first_trip_id",
        "stop_id": "first_arrival_platform",
        "arrival_seconds": "first_arrival_seconds"
    })

    return candidates.reset_index(drop=True)


# Test against original two-transfer algorithm
second_leg_test = build_second_leg_candidates(
    realtime_daily,
    first_transfer_test,
    connection_window_minutes=45
)

print("Original second-leg candidates:", len(second_leg_candidates))
print("Reusable second-leg candidates:", len(second_leg_test))

print("\nSample Epping connections:")

display(
    second_leg_test.loc[
        second_leg_test["transfer_station"] == "Epping Station",
        [
            "first_trip_id",
            "first_arrival_platform",
            "first_arrival_seconds",
            "second_trip_id",
            "second_boarding_platform",
            "second_departure_seconds"
        ]
    ].head(10)
)


Original second-leg candidates: 7319
Reusable second-leg candidates: 9204

Sample Epping connections:


,first_trip_id,first_arrival_platform,first_arrival_seconds,second_trip_id,second_boarding_platform,second_departure_seconds
0,0160-006-101-015:1000,2121226,64431.0,123R.1324.165.16.A.8.92004317,2121221,66420.0
1,0160-006-101-015:1000,2121226,64431.0,124R.1324.165.16.A.8.92000138,2121221,65520.0
2,0160-006-101-015:1000,2121226,64431.0,156F.1324.165.16.A.8.92003296,2121221,67320.0
3,0160-006-101-015:1000,2121226,64431.0,101T.1324.165.16.T.8.92003691,2121222,65281.0
4,0160-006-101-015:1000,2121226,64431.0,122V.1324.165.16.A.8.91999806,2121222,66480.0
5,0160-006-101-015:1000,2121226,64431.0,126V.1324.165.16.A.8.91999509,2121222,65580.0
6,0160-006-101-015:1000,2121226,64431.0,128S.1324.165.16.A.8.92001023,2121222,67380.0
7,0160-006-101-015:1000,2121226,64431.0,139S.1324.165.16.A.8.92001472,2121223,66060.0
8,0160-006-101-015:1000,2121226,64431.0,140S.1324.165.16.A.8.92003085,2121223,66960.0
9,0160-006-101-015:1000,2121226,64431.0,192S.1324.165.16.B.8.92002023,2121223,65160.0


In [192]:

# STEP 90E — Compare original and reusable second-leg candidates

import pandas as pd

keys = [
    "first_trip_id",
    "first_arrival_platform",
    "first_arrival_seconds",
    "second_trip_id",
    "second_boarding_platform",
    "second_departure_seconds"
]

original_keys = second_leg_candidates[keys].drop_duplicates()
new_keys = second_leg_test[keys].drop_duplicates()

comparison = new_keys.merge(
    original_keys,
    on=keys,
    how="outer",
    indicator=True
)

print("CONNECTION COMPARISON")
print("=====================")
print(comparison["_merge"].value_counts())

extra_connections = comparison[
    comparison["_merge"] == "left_only"
].copy()

missing_connections = comparison[
    comparison["_merge"] == "right_only"
].copy()

print("\nExtra connections:", len(extra_connections))
print("Missing connections:", len(missing_connections))

# Examine connection waiting times for the extra results
if not extra_connections.empty:
    extra_details = extra_connections.merge(
        second_leg_test,
        on=keys,
        how="left"
    )

    extra_details["wait_minutes"] = (
        extra_details["second_departure_seconds"]
        - extra_details["first_arrival_seconds"]
    ) / 60

    print("\nEXTRA CONNECTION WAIT TIMES")
    print(extra_details["wait_minutes"].describe())

    print("\nSAMPLE EXTRA CONNECTIONS")
    display(
        extra_details[
            [
                "transfer_station",
                "first_arrival_seconds",
                "second_departure_seconds",
                "wait_minutes",
                "required_transfer_seconds"
            ]
        ].head(10)
    )

print("\nDUPLICATE CHECK")
print("Original duplicates:", len(second_leg_candidates) - len(original_keys))
print("New duplicates:", len(second_leg_test) - len(new_keys))


CONNECTION COMPARISON
_merge
both          7319
left_only     1885
right_only       0
Name: count, dtype: int64

Extra connections: 1885
Missing connections: 0

EXTRA CONNECTION WAIT TIMES
count    1885.000000
mean       37.935332
std        13.701532
min         5.850000
25%        27.850000
50%        45.766667
75%        48.500000
max        54.433333
Name: wait_minutes, dtype: float64

SAMPLE EXTRA CONNECTIONS


,transfer_station,first_arrival_seconds,second_departure_seconds,wait_minutes,required_transfer_seconds
0,Epping Station,64431.0,67356.0,48.750000,300.0
1,Epping Station,64431.0,67213.0,46.366667,300.0
2,Epping Station,64431.0,66480.0,34.150000,330.0
3,Epping Station,64431.0,65580.0,19.150000,330.0
4,Epping Station,64431.0,67380.0,49.150000,330.0
5,Epping Station,64431.0,67320.0,48.150000,330.0
6,Cherrybrook Station,64778.0,67585.0,46.783333,300.0
7,Cherrybrook Station,64778.0,67688.0,48.500000,300.0
8,Bella Vista Station,65410.0,68272.0,47.700000,300.0
9,Bella Vista Station,65410.0,68320.0,48.500000,300.0



DUPLICATE CHECK
Original duplicates: 0
New duplicates: 0


In [193]:

# STEP 90F — Diagnose additional second-leg connections

extra_details = extra_connections.merge(
    second_leg_test,
    on=keys,
    how="left"
)

extra_details["wait_minutes"] = (
    extra_details["second_departure_seconds"]
    - extra_details["first_arrival_seconds"]
) / 60

extra_details["after_transfer_minutes"] = (
    extra_details["second_departure_seconds"]
    - extra_details["earliest_next_departure"]
) / 60

print("EXTRA CONNECTIONS BY WAIT TIME")
display(
    pd.cut(
        extra_details["wait_minutes"],
        bins=[0, 10, 20, 30, 45, 60],
        include_lowest=True
    ).value_counts().sort_index()
)

print("\nEXTRA CONNECTIONS BY STATION")
display(
    extra_details["transfer_station"]
    .value_counts()
    .head(15)
)

print("\nEXTRA CONNECTIONS BY TRANSFER PLATFORM")
display(
    extra_details.groupby(
        ["transfer_station", "second_boarding_platform"]
    ).size().sort_values(ascending=False).head(15)
)

print("\nWAIT TIME AFTER REQUIRED TRANSFER")
print(extra_details["after_transfer_minutes"].describe())

print("\nEXTRA CONNECTIONS WITHIN 30 MINUTES OF TRANSFER READINESS")
print(
    (extra_details["after_transfer_minutes"] <= 30).sum()
)


EXTRA CONNECTIONS BY WAIT TIME


wait_minutes
(-0.001, 10.0]      87
(10.0, 20.0]       236
(20.0, 30.0]       205
(30.0, 45.0]       330
(45.0, 60.0]      1027
Name: count, dtype: int64


EXTRA CONNECTIONS BY STATION


transfer_station
Central Station           814
Sydenham Station          266
Tallawong Station         174
Epping Station             84
Martin Place Station       82
Chatswood Station          54
Barangaroo Station         43
Gadigal Station            40
Waterloo Station           37
Castle Hill Station        34
Bella Vista Station        28
Victoria Cross Station     28
Crows Nest Station         28
Macquarie Park Station     28
North Ryde Station         28
Name: count, dtype: int64


EXTRA CONNECTIONS BY TRANSFER PLATFORM


transfer_station      second_boarding_platform
Sydenham Station      204471                      175
Central Station       2000338                     160
                      2000337                     153
                      2000336                     132
                      2000341                     101
Tallawong Station     2155269                     100
                      2155270                      74
Central Station       2000340                      53
Sydenham Station      204474                       51
Epping Station        2121222                      44
Central Station       2000448                      38
                      2000447                      30
Barangaroo Station    2000461                      30
Martin Place Station  2000462                      29
Gadigal Station       2000465                      27
dtype: int64


WAIT TIME AFTER REQUIRED TRANSFER
count    1885.000000
mean       32.095385
std        13.676482
min         0.050000
25%        21.750000
50%        39.750000
75%        43.233333
max        45.000000
Name: after_transfer_minutes, dtype: float64

EXTRA CONNECTIONS WITHIN 30 MINUTES OF TRANSFER READINESS
650


In [194]:

# STEP 90G — Compare connection window definitions

def compare_connection_windows(df):
    result = df.copy()

    result["wait_from_arrival"] = (
        result["second_departure_seconds"]
        - result["first_arrival_seconds"]
    ) / 60

    result["wait_after_ready"] = (
        result["second_departure_seconds"]
        - result["earliest_next_departure"]
    ) / 60

    return result

original_check = compare_connection_windows(second_leg_candidates)
new_check = compare_connection_windows(second_leg_test)

print("ORIGINAL ALGORITHM")
print("Maximum wait from arrival:",
      original_check["wait_from_arrival"].max())
print("Maximum wait after ready:",
      original_check["wait_after_ready"].max())

print("\nNEW ALGORITHM")
print("Maximum wait from arrival:",
      new_check["wait_from_arrival"].max())
print("Maximum wait after ready:",
      new_check["wait_after_ready"].max())

print("\nCANDIDATES UNDER DIFFERENT RULES")

for minutes in [20, 30, 45]:
    print(f"\n{minutes}-minute window")

    print(
        "From arrival:",
        (new_check["wait_from_arrival"] <= minutes).sum()
    )

    print(
        "After transfer readiness:",
        (new_check["wait_after_ready"] <= minutes).sum()
    )


ORIGINAL ALGORITHM
Maximum wait from arrival: 44.983333333333334
Maximum wait after ready: 39.983333333333334

NEW ALGORITHM
Maximum wait from arrival: 54.43333333333333
Maximum wait after ready: 45.0

CANDIDATES UNDER DIFFERENT RULES

20-minute window
From arrival: 2923
After transfer readiness: 4239

30-minute window
From arrival: 5146
After transfer readiness: 6197

45-minute window
From arrival: 8177
After transfer readiness: 9204


In [195]:

# STEP 90H — Verify original 45-minute arrival-based window

arrival_window_test = second_leg_test[
    (
        second_leg_test["second_departure_seconds"]
        - second_leg_test["first_arrival_seconds"]
    ) <= 45 * 60
].copy()

keys = [
    "first_trip_id",
    "first_arrival_platform",
    "first_arrival_seconds",
    "second_trip_id",
    "second_boarding_platform",
    "second_departure_seconds"
]

original_unique = second_leg_candidates[keys].drop_duplicates()
arrival_unique = arrival_window_test[keys].drop_duplicates()

comparison = arrival_unique.merge(
    original_unique,
    on=keys,
    how="outer",
    indicator=True
)

print("Original candidates:", len(original_unique))
print("Arrival-window candidates:", len(arrival_unique))

print("\nCOMPARISON")
print(comparison["_merge"].value_counts())

print("\nExtra candidates:",
      (comparison["_merge"] == "left_only").sum())

print("Missing original candidates:",
      (comparison["_merge"] == "right_only").sum())


Original candidates: 7319
Arrival-window candidates: 8177

COMPARISON
_merge
both          7319
left_only      858
right_only       0
Name: count, dtype: int64

Extra candidates: 858
Missing original candidates: 0


In [196]:

# STEP 90I — Investigate the remaining 858 connections

extra_keys = comparison.loc[
    comparison["_merge"] == "left_only",
    keys
]

extra_858 = arrival_window_test.merge(
    extra_keys,
    on=keys,
    how="inner"
)

extra_858["wait_minutes"] = (
    extra_858["second_departure_seconds"]
    - extra_858["first_arrival_seconds"]
) / 60

print("ADDITIONAL CONNECTIONS:", len(extra_858))

print("\nWAITING TIME DISTRIBUTION")
display(
    pd.cut(
        extra_858["wait_minutes"],
        bins=[0, 10, 20, 30, 40, 45],
        include_lowest=True
    ).value_counts().sort_index()
)

print("\nTOP TRANSFER STATIONS")
display(
    extra_858["transfer_station"]
    .value_counts()
    .head(15)
)

print("\nADDITIONAL CONNECTIONS BY BOARDING PLATFORM")
display(
    extra_858.groupby(
        ["transfer_station", "second_boarding_platform"]
    ).size().sort_values(ascending=False).head(15)
)

print("\nSAMPLE CONNECTIONS")
display(
    extra_858[
        [
            "transfer_station",
            "first_trip_id",
            "second_trip_id",
            "first_arrival_seconds",
            "second_departure_seconds",
            "wait_minutes"
        ]
    ].head(10)
)


ADDITIONAL CONNECTIONS: 858

WAITING TIME DISTRIBUTION


wait_minutes
(-0.001, 10.0]     87
(10.0, 20.0]      236
(20.0, 30.0]      205
(30.0, 40.0]      228
(40.0, 45.0]      102
Name: count, dtype: int64


TOP TRANSFER STATIONS


transfer_station
Central Station      487
Sydenham Station     183
Tallawong Station    150
Epping Station        38
Name: count, dtype: int64


ADDITIONAL CONNECTIONS BY BOARDING PLATFORM


transfer_station   second_boarding_platform
Sydenham Station   204471                      141
Central Station    2000338                     123
                   2000337                     117
                   2000336                     107
Tallawong Station  2155269                      85
Central Station    2000341                      77
Tallawong Station  2155270                      65
Central Station    2000340                      38
Epping Station     2121222                      38
Sydenham Station   204474                       34
Central Station    2000333                      12
Sydenham Station   204472                        8
Central Station    2010107                       7
                   2000332                       6
dtype: int64


SAMPLE CONNECTIONS


,transfer_station,first_trip_id,second_trip_id,first_arrival_seconds,second_departure_seconds,wait_minutes
0,Epping Station,0160-006-101-015:1000,122V.1324.165.16.A.8.91999806,64431.0,66480.0,34.150000
1,Epping Station,0160-006-101-015:1000,126V.1324.165.16.A.8.91999509,64431.0,65580.0,19.150000
2,Tallawong Station,0160-006-101-015:1000,0160-006-104-015:1000,65897.0,67344.0,24.116667
3,Tallawong Station,0160-006-101-015:1000,0160-006-106-015:1000,65897.0,68544.0,44.116667
4,Tallawong Station,0160-006-101-015:1000,0160-006-118-013:1000,65897.0,67824.0,32.116667
5,Tallawong Station,0160-006-101-015:1000,0160-006-130-005:1000,65897.0,68064.0,36.116667
6,Tallawong Station,0160-006-101-015:1000,0160-006-103-015:1000,65897.0,67097.0,20.000000
7,Tallawong Station,0160-006-101-015:1000,0160-006-105-015:1000,65897.0,68297.0,40.000000
8,Tallawong Station,0160-006-101-015:1000,0160-006-114-013:1000,65897.0,66377.0,8.000000
9,Tallawong Station,0160-006-101-015:1000,0160-006-115-013:1000,65897.0,66617.0,12.000000


In [197]:

# STEP 90J — Compare original vs new connections at Epping

first_trip = "0160-006-101-015:1000"
arrival_platform = 2121226

def inspect_connections(df, label):
    matches = df[
        (df["first_trip_id"] == first_trip)
        & (
            df["first_arrival_platform"].astype(str)
            == str(arrival_platform)
        )
    ].copy()

    matches["wait_minutes"] = (
        matches["second_departure_seconds"]
        - matches["first_arrival_seconds"]
    ) / 60

    matches["departure_time"] = matches[
        "second_departure_seconds"
    ].apply(format_transport_time)

    print(f"\n{label}: {len(matches)} connections")

    return matches[
        [
            "second_trip_id",
            "second_boarding_platform",
            "departure_time",
            "wait_minutes"
        ]
    ].sort_values(
        ["second_boarding_platform", "departure_time"]
    )


original_epping = inspect_connections(
    second_leg_candidates,
    "ORIGINAL"
)

new_epping = inspect_connections(
    arrival_window_test,
    "NEW"
)

comparison_epping = new_epping.merge(
    original_epping[
        ["second_trip_id", "second_boarding_platform"]
    ].assign(in_original=True),
    on=["second_trip_id", "second_boarding_platform"],
    how="left"
)

comparison_epping["in_original"] = (
    comparison_epping["in_original"].fillna(False)
)

print("\nNEW CONNECTIONS NOT IN ORIGINAL")
display(
    comparison_epping[
        ~comparison_epping["in_original"]
    ]
)

print("\nCONNECTION COUNTS BY PLATFORM")
display(
    pd.DataFrame({
        "original": original_epping.groupby(
            "second_boarding_platform"
        ).size(),
        "new": new_epping.groupby(
            "second_boarding_platform"
        ).size()
    }).fillna(0).astype(int)
)



ORIGINAL: 26 connections

NEW: 28 connections

NEW CONNECTIONS NOT IN ORIGINAL


KeyError: "None of [Index([-2, -2, -2, -1, -1, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2,\n       -2, -2, -2, -2, -2, -2, -2, -2, -2, -2],\n      dtype='object')] are in the [columns]"

In [198]:

# STEP 90J — Compare original vs new connections at Epping
# Full corrected version

import pandas as pd

first_trip = "0160-006-101-015:1000"
arrival_platform = 2121226


def inspect_connections(df, label):
    matches = df[
        (df["first_trip_id"] == first_trip)
        & (
            df["first_arrival_platform"].astype(str)
            == str(arrival_platform)
        )
    ].copy()

    matches["wait_minutes"] = (
        matches["second_departure_seconds"]
        - matches["first_arrival_seconds"]
    ) / 60

    matches["departure_time"] = matches[
        "second_departure_seconds"
    ].apply(format_transport_time)

    print(f"\n{label}: {len(matches)} connections")

    return matches[
        [
            "second_trip_id",
            "second_boarding_platform",
            "departure_time",
            "wait_minutes"
        ]
    ].sort_values(
        ["second_boarding_platform", "departure_time"]
    ).reset_index(drop=True)


# 1. Get original Epping connections
original_epping = inspect_connections(
    second_leg_candidates,
    "ORIGINAL"
)

# 2. Get new Epping connections
new_epping = inspect_connections(
    arrival_window_test,
    "NEW"
)


# 3. Compare original and new connections
comparison_epping = new_epping.merge(
    original_epping[
        ["second_trip_id", "second_boarding_platform"]
    ].drop_duplicates().assign(in_original=True),
    on=["second_trip_id", "second_boarding_platform"],
    how="left"
)

# FIX: Explicitly convert to Boolean
comparison_epping["in_original"] = (
    comparison_epping["in_original"]
    .fillna(False)
    .astype(bool)
)


# 4. Display connections missing from original algorithm
print("\nNEW CONNECTIONS NOT IN ORIGINAL")
print("================================")

extra_epping = comparison_epping.loc[
    ~comparison_epping["in_original"]
].copy()

print("Additional Epping connections:", len(extra_epping))

display(extra_epping)


# 5. Compare connection counts by platform
print("\nCONNECTION COUNTS BY PLATFORM")
print("=============================")

platform_comparison = pd.DataFrame({
    "original": original_epping.groupby(
        "second_boarding_platform"
    ).size(),
    "new": new_epping.groupby(
        "second_boarding_platform"
    ).size()
}).fillna(0).astype(int)

platform_comparison["difference"] = (
    platform_comparison["new"]
    - platform_comparison["original"]
)

display(platform_comparison)


# 6. Summary
print("\nSUMMARY")
print("=======")
print("Original Epping connections:", len(original_epping))
print("New Epping connections:", len(new_epping))
print("Additional connections:", len(extra_epping))



ORIGINAL: 26 connections

NEW: 28 connections

NEW CONNECTIONS NOT IN ORIGINAL
Additional Epping connections: 2


,second_trip_id,second_boarding_platform,departure_time,wait_minutes,in_original
3,126V.1324.165.16.A.8.91999509,2121222,18:13:00,19.15,False
4,122V.1324.165.16.A.8.91999806,2121222,18:28:00,34.15,False



CONNECTION COUNTS BY PLATFORM


,original,new,difference
second_boarding_platform,,,
2121221,2,2,0
2121222,1,3,2
2121223,3,3,0
2121225,10,10,0
2121226,10,10,0



SUMMARY
Original Epping connections: 26
New Epping connections: 28
Additional connections: 2


In [199]:

# STEP 90K — Validate the two additional Epping services

extra_trip_ids = extra_epping["second_trip_id"].unique()

service_stops = realtime_daily[
    realtime_daily["trip_id"].isin(extra_trip_ids)
].copy()

# Add readable station and platform information
service_stops = service_stops.merge(
    transport_stops[
        ["stop_id", "stop_name", "platform_code"]
    ].drop_duplicates("stop_id"),
    on="stop_id",
    how="left"
)

for trip_id in extra_trip_ids:
    trip = service_stops[
        service_stops["trip_id"] == trip_id
    ].sort_values("stop_sequence")

    epping = trip[
        trip["stop_id"].astype(str) == "2121222"
    ]

    print("\nSERVICE:", trip_id)
    print("Total stops:", len(trip))

    if epping.empty:
        print("Epping Platform 2: NOT FOUND")
        continue

    boarding_sequence = epping.iloc[0]["stop_sequence"]

    print("Epping Platform 2 departure:",
          epping.iloc[0]["departure_time"])

    downstream = trip[
        trip["stop_sequence"] > boarding_sequence
    ]

    print("Downstream stops:", len(downstream))

    display(
        downstream[
            [
                "stop_name",
                "platform_code",
                "arrival_time"
            ]
        ].head(10)
    )



SERVICE: 126V.1324.165.16.A.8.91999509
Total stops: 11
Epping Platform 2 departure: 18:13:00
Downstream stops: 0


,stop_name,platform_code,arrival_time



SERVICE: 122V.1324.165.16.A.8.91999806
Total stops: 11
Epping Platform 2 departure: 18:28:00
Downstream stops: 0


,stop_name,platform_code,arrival_time


In [200]:

# STEP 90L — Exclude services terminating at the boarding station

def build_second_leg_candidates(
    daily,
    transfer_options,
    connection_window_minutes=45
):
    if transfer_options.empty:
        return pd.DataFrame()

    # Identify the last stop sequence of each service
    last_stops = (
        daily.groupby("trip_id")["stop_sequence"]
        .max()
        .rename("last_stop_sequence")
        .reset_index()
    )

    # Candidate boarding events
    boarding_events = daily[
        [
            "trip_id",
            "station_id",
            "stop_id",
            "stop_sequence",
            "departure_seconds"
        ]
    ].rename(columns={
        "trip_id": "second_trip_id",
        "station_id": "second_station_id",
        "stop_id": "second_boarding_platform",
        "stop_sequence": "second_boarding_sequence",
        "departure_seconds": "second_departure_seconds"
    })

    # Attach last stop of each service
    boarding_events = boarding_events.merge(
        last_stops.rename(columns={"trip_id": "second_trip_id"}),
        on="second_trip_id",
        how="left"
    )

    # Exclude services with no downstream stops
    boarding_events = boarding_events[
        boarding_events["second_boarding_sequence"]
        < boarding_events["last_stop_sequence"]
    ].copy()

    # Match exact transfer platforms
    candidates = transfer_options.merge(
        boarding_events,
        left_on=["station_id", "to_stop_id"],
        right_on=["second_station_id", "second_boarding_platform"],
        how="inner"
    )

    # Apply transfer feasibility and connection window
    candidates = candidates[
        (
            candidates["second_departure_seconds"]
            >= candidates["earliest_next_departure"]
        )
        & (
            candidates["second_departure_seconds"]
            <= candidates["first_arrival_seconds"]
            + connection_window_minutes * 60
        )
        & (
            candidates["trip_id"] != candidates["second_trip_id"]
        )
    ].copy()

    # Rename first-leg columns
    candidates = candidates.rename(columns={
        "trip_id": "first_trip_id",
        "stop_id": "first_arrival_platform",
        "arrival_seconds": "first_arrival_seconds"
    })

    return candidates.reset_index(drop=True)


# Test corrected function
second_leg_corrected = build_second_leg_candidates(
    realtime_daily,
    first_transfer_test,
    connection_window_minutes=45
)

print("Original candidates:", len(second_leg_candidates))
print("Corrected candidates:", len(second_leg_corrected))

# Compare exact service connections
keys = [
    "first_trip_id",
    "first_arrival_platform",
    "first_arrival_seconds",
    "second_trip_id",
    "second_boarding_platform",
    "second_departure_seconds"
]

comparison_corrected = (
    second_leg_corrected[keys].drop_duplicates()
    .merge(
        second_leg_candidates[keys].drop_duplicates(),
        on=keys,
        how="outer",
        indicator=True
    )
)

print("\nCOMPARISON")
print(comparison_corrected["_merge"].value_counts())

print("\nExtra connections:",
      (comparison_corrected["_merge"] == "left_only").sum())

print("Missing original connections:",
      (comparison_corrected["_merge"] == "right_only").sum())


KeyError: 'first_arrival_seconds'

In [201]:

# STEP 90L — Corrected reusable second-leg candidate search

import pandas as pd

def build_second_leg_candidates(
    daily,
    transfer_options,
    connection_window_minutes=45
):
    if transfer_options.empty:
        return pd.DataFrame()

    # 1. Identify each service's final stop
    last_stops = (
        daily.groupby("trip_id")["stop_sequence"]
        .max()
        .rename("last_stop_sequence")
        .reset_index()
    )

    # 2. Prepare second-leg boarding events
    boarding_events = daily[
        [
            "trip_id",
            "station_id",
            "stop_id",
            "stop_sequence",
            "departure_seconds"
        ]
    ].rename(columns={
        "trip_id": "second_trip_id",
        "station_id": "second_station_id",
        "stop_id": "second_boarding_platform",
        "stop_sequence": "second_boarding_sequence",
        "departure_seconds": "second_departure_seconds"
    })

    boarding_events = boarding_events.merge(
        last_stops.rename(columns={
            "trip_id": "second_trip_id"
        }),
        on="second_trip_id",
        how="left"
    )

    # 3. Exclude services that terminate at boarding station
    boarding_events = boarding_events[
        boarding_events["second_boarding_sequence"]
        < boarding_events["last_stop_sequence"]
    ].copy()

    # 4. Match transfer platforms to departing services
    candidates = transfer_options.merge(
        boarding_events,
        left_on=["station_id", "to_stop_id"],
        right_on=[
            "second_station_id",
            "second_boarding_platform"
        ],
        how="inner"
    )

    # 5. Rename first-leg columns BEFORE filtering
    candidates = candidates.rename(columns={
        "trip_id": "first_trip_id",
        "stop_id": "first_arrival_platform",
        "arrival_seconds": "first_arrival_seconds"
    })

    # 6. Apply transfer feasibility and arrival-based window
    candidates = candidates[
        (
            candidates["second_departure_seconds"]
            >= candidates["earliest_next_departure"]
        )
        & (
            candidates["second_departure_seconds"]
            <= candidates["first_arrival_seconds"]
            + connection_window_minutes * 60
        )
        & (
            candidates["first_trip_id"]
            != candidates["second_trip_id"]
        )
    ].copy()

    return candidates.reset_index(drop=True)


# 7. Run corrected function
second_leg_corrected = build_second_leg_candidates(
    realtime_daily,
    first_transfer_test,
    connection_window_minutes=45
)

print("Original candidates:", len(second_leg_candidates))
print("Corrected candidates:", len(second_leg_corrected))

# 8. Compare original and corrected service connections
keys = [
    "first_trip_id",
    "first_arrival_platform",
    "first_arrival_seconds",
    "second_trip_id",
    "second_boarding_platform",
    "second_departure_seconds"
]

comparison_corrected = (
    second_leg_corrected[keys]
    .drop_duplicates()
    .merge(
        second_leg_candidates[keys].drop_duplicates(),
        on=keys,
        how="outer",
        indicator=True
    )
)

print("\nCOMPARISON")
print(comparison_corrected["_merge"].value_counts())

print("\nExtra connections:",
      (comparison_corrected["_merge"] == "left_only").sum())

print("Missing original connections:",
      (comparison_corrected["_merge"] == "right_only").sum())


Original candidates: 7319
Corrected candidates: 7319

COMPARISON
_merge
both          7319
left_only        0
right_only       0
Name: count, dtype: int64

Extra connections: 0
Missing original connections: 0


In [202]:

# STEP 90M — Build reusable second-leg downstream stops

def build_second_leg_stops(daily, second_leg_candidates):

    if second_leg_candidates.empty:
        return pd.DataFrame()

    # All stops served by each second-leg trip
    downstream_events = daily[
        [
            "trip_id",
            "station_id",
            "stop_id",
            "stop_sequence",
            "arrival_seconds"
        ]
    ].rename(columns={
        "trip_id": "second_trip_id",
        "station_id": "second_arrival_station_id",
        "stop_id": "second_arrival_platform",
        "stop_sequence": "second_arrival_sequence",
        "arrival_seconds": "second_arrival_seconds"
    })

    # Join each boarding candidate to its trip's stops
    second_stops = second_leg_candidates.merge(
        downstream_events,
        on="second_trip_id",
        how="inner"
    )

    # Only stops AFTER the passenger boards
    second_stops = second_stops[
        second_stops["second_arrival_sequence"]
        > second_stops["second_boarding_sequence"]
    ].copy()

    # Ensure the journey moves forward in time
    second_stops = second_stops[
        second_stops["second_arrival_seconds"]
        >= second_stops["second_departure_seconds"]
    ].copy()

    return second_stops.reset_index(drop=True)


# Test the reusable function
second_leg_stops_test = build_second_leg_stops(
    realtime_daily,
    second_leg_corrected
)

print("Original downstream stops:", len(second_leg_stops))
print("Refactored downstream stops:", len(second_leg_stops_test))

# Compare stop identities and timing
keys = [
    "first_trip_id",
    "second_trip_id",
    "second_boarding_platform",
    "second_boarding_sequence",
    "second_arrival_platform",
    "second_arrival_sequence",
    "second_arrival_seconds"
]

original_keys = second_leg_stops[keys].drop_duplicates()
new_keys = second_leg_stops_test[keys].drop_duplicates()

comparison_90m = new_keys.merge(
    original_keys,
    on=keys,
    how="outer",
    indicator=True
)

print("\nCOMPARISON")
print(comparison_90m["_merge"].value_counts())

print("\nExtra downstream stops:",
      (comparison_90m["_merge"] == "left_only").sum())

print("Missing original stops:",
      (comparison_90m["_merge"] == "right_only").sum())


Original downstream stops: 73281
Refactored downstream stops: 76390


KeyError: "['second_boarding_platform'] not in index"

In [203]:

# STEP 90M — Check column names and identify the mismatch

print("ORIGINAL SECOND-LEG STOPS")
print("Rows:", len(second_leg_stops))
print("\nColumns:")
print(second_leg_stops.columns.tolist())

print("\n" + "=" * 60)

print("\nREFACTORED SECOND-LEG STOPS")

if "second_leg_stops_test" in globals():
    print("Rows:", len(second_leg_stops_test))
    print("\nColumns:")
    print(second_leg_stops_test.columns.tolist())
else:
    print("The new DataFrame was not created.")

print("\n" + "=" * 60)

print("\nCOLUMN DIFFERENCES")

if "second_leg_stops_test" in globals():
    original_cols = set(second_leg_stops.columns)
    new_cols = set(second_leg_stops_test.columns)

    print("Only in original:")
    print(sorted(original_cols - new_cols))

    print("\nOnly in refactored:")
    print(sorted(new_cols - original_cols))


ORIGINAL SECOND-LEG STOPS
Rows: 73281

Columns:
['first_trip_id', 'second_trip_id', 'first_transfer_station', 'origin_departure_seconds', 'second_boarding_sequence', 'second_departure_seconds', 'second_arrival_station_id', 'second_arrival_platform', 'second_arrival_sequence', 'second_arrival_seconds', 'second_transfer_station']


REFACTORED SECOND-LEG STOPS
Rows: 76390

Columns:
['first_trip_id', 'origin_platform', 'origin_sequence', 'origin_departure_seconds', 'station_id', 'first_arrival_platform', 'stop_sequence', 'first_arrival_seconds', 'transfer_station', 'from_stop_id', 'to_stop_id', 'minimum_transfer_seconds', 'required_transfer_seconds', 'earliest_next_departure', 'second_trip_id', 'second_station_id', 'second_boarding_platform', 'second_boarding_sequence', 'second_departure_seconds', 'last_stop_sequence', 'second_arrival_station_id', 'second_arrival_platform', 'second_arrival_sequence', 'second_arrival_seconds']


COLUMN DIFFERENCES
Only in original:
['first_transfer_station'

In [204]:

# STEP 90M — Compare original and refactored downstream stops

import pandas as pd

# 1. Standardise new column names
new_stops = second_leg_stops_test.rename(columns={
    "transfer_station": "first_transfer_station",
    "second_arrival_station_id": "second_transfer_station"
}).copy()

original_stops = second_leg_stops.copy()

# 2. Compare shared journey identity fields
keys = [
    "first_trip_id",
    "second_trip_id",
    "origin_departure_seconds",
    "second_boarding_sequence",
    "second_departure_seconds",
    "second_arrival_platform",
    "second_arrival_sequence",
    "second_arrival_seconds"
]

print("ORIGINAL ROWS:", len(original_stops))
print("REFACTORED ROWS:", len(new_stops))

# 3. Count repeated occurrences of each journey-stop combination
original_counts = (
    original_stops.groupby(keys, dropna=False)
    .size()
    .reset_index(name="original_count")
)

new_counts = (
    new_stops.groupby(keys, dropna=False)
    .size()
    .reset_index(name="new_count")
)

comparison = original_counts.merge(
    new_counts,
    on=keys,
    how="outer"
)

comparison[["original_count", "new_count"]] = (
    comparison[["original_count", "new_count"]]
    .fillna(0)
    .astype(int)
)

comparison["difference"] = (
    comparison["new_count"] - comparison["original_count"]
)

# 4. Summarise differences
print("\nCOMPARISON SUMMARY")
print("==================")
print("Matching unique stop records:",
      (comparison["difference"] == 0).sum())

print("Records with additional occurrences:",
      (comparison["difference"] > 0).sum())

print("Records with missing occurrences:",
      (comparison["difference"] < 0).sum())

print("Total additional rows:",
      comparison["difference"].clip(lower=0).sum())

print("Total missing rows:",
      (-comparison["difference"]).clip(lower=0).sum())

# 5. Show examples of additional downstream stops
print("\nEXAMPLES OF ADDITIONAL RECORDS")

display(
    comparison.loc[
        comparison["difference"] > 0,
        [
            "first_trip_id",
            "second_trip_id",
            "second_arrival_platform",
            "second_arrival_sequence",
            "original_count",
            "new_count",
            "difference"
        ]
    ].head(15)
)

# 6. Check duplicate counts
print("\nDUPLICATE CHECK")
print("Original duplicated rows by journey-stop key:",
      len(original_stops) - len(original_counts))

print("Refactored duplicated rows by journey-stop key:",
      len(new_stops) - len(new_counts))


ORIGINAL ROWS: 73281
REFACTORED ROWS: 76390

COMPARISON SUMMARY
Matching unique stop records: 73281
Records with additional occurrences: 3109
Records with missing occurrences: 0
Total additional rows: 3109
Total missing rows: 0

EXAMPLES OF ADDITIONAL RECORDS


,first_trip_id,second_trip_id,second_arrival_platform,second_arrival_sequence,original_count,new_count,difference
10,0160-006-101-015:1000,0160-006-101-016:1000,2113351,10,0,1,1
29,0160-006-101-015:1000,0160-006-101-016:1000,2113351,10,0,1,1
47,0160-006-101-015:1000,0160-006-101-016:1000,2113351,10,0,1,1
64,0160-006-101-015:1000,0160-006-101-016:1000,2113351,10,0,1,1
80,0160-006-101-015:1000,0160-006-101-016:1000,2113351,10,0,1,1
95,0160-006-101-015:1000,0160-006-101-016:1000,2113351,10,0,1,1
109,0160-006-101-015:1000,0160-006-101-016:1000,2113351,10,0,1,1
122,0160-006-101-015:1000,0160-006-102-016:1000,2113351,10,0,1,1
142,0160-006-101-015:1000,0160-006-102-016:1000,2113351,10,0,1,1
161,0160-006-101-015:1000,0160-006-102-016:1000,2113351,10,0,1,1



DUPLICATE CHECK
Original duplicated rows by journey-stop key: 0
Refactored duplicated rows by journey-stop key: 0


In [205]:

# STEP 90N — Investigate additional downstream stops

# 1. Identify records present only in the refactored result
extra_stop_keys = comparison.loc[
    comparison["difference"] > 0,
    keys
].copy()

extra_stops = new_stops.merge(
    extra_stop_keys,
    on=keys,
    how="inner"
)

# 2. Check whether each extra stop is the final stop of its trip
trip_final_sequences = (
    realtime_daily.groupby("trip_id")["stop_sequence"]
    .max()
    .rename("final_sequence")
    .reset_index()
    .rename(columns={"trip_id": "second_trip_id"})
)

extra_stops = extra_stops.merge(
    trip_final_sequences,
    on="second_trip_id",
    how="left"
)

extra_stops["is_terminal_stop"] = (
    extra_stops["second_arrival_sequence"]
    == extra_stops["final_sequence"]
)

# 3. Summary
print("TOTAL ADDITIONAL STOPS:", len(extra_stops))

print("\nTERMINAL STOP CHECK")
print(extra_stops["is_terminal_stop"].value_counts())

print("\nEXTRA STOPS BY STATION")
display(
    extra_stops.groupby(
        "second_transfer_station",
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
    .head(15)
    .rename("extra_stop_count")
    .reset_index()
)

# 4. Examine the first few additional stops
print("\nEXAMPLE ADDITIONAL STOPS")
display(
    extra_stops[
        [
            "second_trip_id",
            "second_arrival_platform",
            "second_arrival_sequence",
            "final_sequence",
            "is_terminal_stop",
            "second_arrival_seconds"
        ]
    ].head(15)
)


TOTAL ADDITIONAL STOPS: 3109

TERMINAL STOP CHECK
is_terminal_stop
False    3031
True       78
Name: count, dtype: int64

EXTRA STOPS BY STATION


,second_transfer_station,extra_stop_count
0,211310,2714
1,200060,233
2,215020,162



EXAMPLE ADDITIONAL STOPS


,second_trip_id,second_arrival_platform,second_arrival_sequence,final_sequence,is_terminal_stop,second_arrival_seconds
0,0160-006-109-014:1000,2113351,10,21,False,65244.0
1,0160-006-110-014:1000,2113351,10,21,False,65484.0
2,0160-006-111-014:1000,2113351,10,21,False,65724.0
3,0160-006-112-014:1000,2113351,10,21,False,67164.0
4,0160-006-132-004:1000,2113351,10,21,False,65004.0
5,0160-006-152-014:1000,2113351,10,21,False,66924.0
6,0160-006-153-012:1000,2113351,10,21,False,65964.0
7,0160-006-154-012:1000,2113351,10,21,False,66444.0
8,0160-006-155-012:1000,2113351,10,21,False,66684.0
9,0160-006-156-004:1000,2113351,10,21,False,66204.0


In [206]:

# STEP 90O — Investigate stations excluded by the original algorithm

import pandas as pd

extra_station_ids = (
    extra_stops["second_transfer_station"]
    .dropna()
    .unique()
)

# Station names
station_details = station_lookup[
    station_lookup["station_id"].isin(extra_station_ids)
].drop_duplicates(subset=["station_id", "stop_name"])

print("STATIONS ACCOUNTING FOR ADDITIONAL STOPS")
display(station_details.sort_values("station_id"))

# Check whether transfer rules exist at these stations
print("\nTRANSFER RULES AT THESE STATIONS")

transfer_summary = (
    all_platform_transfers[
        all_platform_transfers["station_id"].isin(extra_station_ids)
    ]
    .groupby("station_id")
    .agg(
        transfer_rules=("from_stop_id", "size"),
        arrival_platform_options=("from_stop_id", "nunique"),
        departure_platform_options=("to_stop_id", "nunique")
    )
    .reset_index()
)

display(transfer_summary)

# Check whether original second-leg stops contain these stations
print("\nORIGINAL VS REFACTORED BY STATION")

original_station_counts = (
    second_leg_stops.groupby("second_transfer_station")
    .size()
    .rename("original")
)

new_station_counts = (
    new_stops.groupby("second_transfer_station")
    .size()
    .rename("refactored")
)

station_comparison = (
    pd.concat(
        [original_station_counts, new_station_counts],
        axis=1
    )
    .fillna(0)
    .astype(int)
)

station_comparison["difference"] = (
    station_comparison["refactored"]
    - station_comparison["original"]
)

display(
    station_comparison.loc[
        station_comparison.index.isin(extra_station_ids)
    ].sort_values("difference", ascending=False)
)


STATIONS ACCOUNTING FOR ADDITIONAL STOPS


,stop_id,station_id,stop_name
18,2000467,200060,Central Station
9,2113351,211310,Macquarie University Station
42,2150414,215020,Parramatta Station



TRANSFER RULES AT THESE STATIONS


,station_id,transfer_rules,arrival_platform_options,departure_platform_options
0,200060,841,29,29
1,211310,4,2,2
2,215020,16,4,4



ORIGINAL VS REFACTORED BY STATION


,original,refactored,difference
second_transfer_station,,,
200060,0,3170,3170
211310,0,2714,2714
215020,0,162,162


In [207]:

# STEP 90P — Find original second-leg stop filtering logic

from IPython import get_ipython

history = get_ipython().history_manager.input_hist_raw

matches = []

for cell_number, source in enumerate(history):
    if (
        "second_leg_stops =" in source
        and "def build_second_leg_stops" not in source
        and "history_manager" not in source
    ):
        matches.append((cell_number, source))

print("Matching notebook cells:", len(matches))

for cell_number, source in matches[-5:]:
    print("\n" + "=" * 70)
    print(f"CELL {cell_number}")
    print("=" * 70)
    print(source[:8000])


Matching notebook cells: 1

CELL 169

# STEP 86D — Discover second-transfer stations automatically

# Find all downstream stops on the second-leg services
second_leg_stops = second_leg_candidates[
    [
        "first_trip_id",
        "second_trip_id",
        "transfer_station",
        "origin_departure_seconds",
        "second_boarding_sequence",
        "second_departure_seconds"
    ]
].rename(columns={
    "transfer_station": "first_transfer_station"
}).merge(
    realtime_daily[
        [
            "trip_id",
            "station_id",
            "stop_id",
            "stop_sequence",
            "arrival_seconds"
        ]
    ].rename(columns={
        "trip_id": "second_trip_id",
        "station_id": "second_arrival_station_id",
        "stop_id": "second_arrival_platform",
        "stop_sequence": "second_arrival_sequence",
        "arrival_seconds": "second_arrival_seconds"
    }),
    on="second_trip_id",
    how="inner"
)

# Keep only downstream stops with valid tra

In [208]:

# STEP 90Q — Show the complete original Step 86D code

from IPython import get_ipython

history = get_ipython().history_manager.input_hist_raw

source = history[169]

print("ORIGINAL STEP 86D")
print("=" * 70)

for line_number, line in enumerate(source.splitlines(), start=1):
    print(f"{line_number:03d} | {line}")


ORIGINAL STEP 86D
001 | 
002 | # STEP 86D — Discover second-transfer stations automatically
003 | 
004 | # Find all downstream stops on the second-leg services
005 | second_leg_stops = second_leg_candidates[
006 |     [
007 |         "first_trip_id",
008 |         "second_trip_id",
009 |         "transfer_station",
010 |         "origin_departure_seconds",
011 |         "second_boarding_sequence",
012 |         "second_departure_seconds"
013 |     ]
014 | ].rename(columns={
015 |     "transfer_station": "first_transfer_station"
016 | }).merge(
017 |     realtime_daily[
018 |         [
019 |             "trip_id",
020 |             "station_id",
021 |             "stop_id",
022 |             "stop_sequence",
023 |             "arrival_seconds"
024 |         ]
025 |     ].rename(columns={
026 |         "trip_id": "second_trip_id",
027 |         "station_id": "second_arrival_station_id",
028 |         "stop_id": "second_arrival_platform",
029 |         "stop_sequence": "second_arrival_seq

In [209]:

# STEP 90R — Inspect the original downstream-stop filters

from IPython import get_ipython

source = get_ipython().history_manager.input_hist_raw[169]
lines = source.splitlines()

print("ORIGINAL STEP 86D — LINES 20 TO 95")
print("=" * 60)

for i in range(19, min(95, len(lines))):
    print(f"{i + 1:03d} | {lines[i]}")


ORIGINAL STEP 86D — LINES 20 TO 95
020 |             "station_id",
021 |             "stop_id",
022 |             "stop_sequence",
023 |             "arrival_seconds"
024 |         ]
025 |     ].rename(columns={
026 |         "trip_id": "second_trip_id",
027 |         "station_id": "second_arrival_station_id",
028 |         "stop_id": "second_arrival_platform",
029 |         "stop_sequence": "second_arrival_sequence",
030 |         "arrival_seconds": "second_arrival_seconds"
031 |     }),
032 |     on="second_trip_id",
033 |     how="inner"
034 | )
035 | 
036 | # Keep only downstream stops with valid travel times
037 | second_leg_stops = second_leg_stops[
038 |     (
039 |         second_leg_stops["second_arrival_sequence"]
040 |         > second_leg_stops["second_boarding_sequence"]
041 |     )
042 |     & (
043 |         second_leg_stops["second_arrival_seconds"]
044 |         >= second_leg_stops["second_departure_seconds"]
045 |     )
046 | ].copy()
047 | 
048 | # Add station names


In [210]:

# STEP 90S — Display only the original filtering conditions

from IPython import get_ipython

source = get_ipython().history_manager.input_hist_raw[169]
lines = source.splitlines()

for start, end in [(36, 55), (56, 75)]:
    print(f"\nLINES {start}–{end}")
    print("=" * 45)
    print("\n".join(
        f"{i + 1:03d} | {lines[i]}"
        for i in range(start - 1, min(end, len(lines)))
    ))



LINES 36–55
036 | # Keep only downstream stops with valid travel times
037 | second_leg_stops = second_leg_stops[
038 |     (
039 |         second_leg_stops["second_arrival_sequence"]
040 |         > second_leg_stops["second_boarding_sequence"]
041 |     )
042 |     & (
043 |         second_leg_stops["second_arrival_seconds"]
044 |         >= second_leg_stops["second_departure_seconds"]
045 |     )
046 | ].copy()
047 | 
048 | # Add station names
049 | second_leg_stops["second_arrival_station_id"] = (
050 |     second_leg_stops["second_arrival_station_id"].astype(str)
051 | )
052 | 
053 | second_station_names = station_names.rename(columns={
054 |     "station_id": "second_arrival_station_id",
055 |     "transfer_station": "second_transfer_station"

LINES 56–75
056 | }).copy()
057 | 
058 | second_station_names["second_arrival_station_id"] = (
059 |     second_station_names["second_arrival_station_id"].astype(str)
060 | )
061 | 
062 | second_leg_stops = second_leg_stops.merge(
063 |    

In [211]:

# STEP 90T — Identify original station exclusion rules

from IPython import get_ipython

source = get_ipython().history_manager.input_hist_raw[169]
lines = source.splitlines()

print("ORIGINAL STATION FILTERS")
print("=" * 45)

for i in range(60, min(85, len(lines))):
    print(f"{i + 1}: {lines[i]}", flush=True)


ORIGINAL STATION FILTERS
61: 
62: second_leg_stops = second_leg_stops.merge(
63:     second_station_names,
64:     on="second_arrival_station_id",
65:     how="left"
66: )
67: 
68: # Exclude repeated transfer stations and the origin/destination
69: second_leg_stops = second_leg_stops[
70:     (
71:         second_leg_stops["second_transfer_station"]
72:         != second_leg_stops["first_transfer_station"]
73:     )
74:     & (
75:         ~second_leg_stops["second_transfer_station"].isin(
76:             [origin, destination]
77:         )
78:     )
79: ].copy()
80: 
81: # Summarise possible second-transfer stations
82: second_transfer_summary = (
83:     second_leg_stops.groupby("second_transfer_station")
84:     .agg(
85:         possible_arrivals=("second_trip_id", "count"),


In [212]:

# STEP 90U — Final reusable second-leg downstream search

import pandas as pd

def build_second_leg_stops(
    daily,
    second_leg_candidates,
    origin,
    destination,
    station_names
):
    if second_leg_candidates.empty:
        return pd.DataFrame()

    # 1. Prepare second-leg boarding information
    boarding = second_leg_candidates[
        [
            "first_trip_id",
            "second_trip_id",
            "transfer_station",
            "origin_departure_seconds",
            "second_boarding_sequence",
            "second_departure_seconds"
        ]
    ].rename(columns={
        "transfer_station": "first_transfer_station"
    })

    # 2. Prepare all stops on second-leg services
    downstream_events = daily[
        [
            "trip_id",
            "station_id",
            "stop_id",
            "stop_sequence",
            "arrival_seconds"
        ]
    ].rename(columns={
        "trip_id": "second_trip_id",
        "station_id": "second_arrival_station_id",
        "stop_id": "second_arrival_platform",
        "stop_sequence": "second_arrival_sequence",
        "arrival_seconds": "second_arrival_seconds"
    })

    # 3. Match boarding candidates to downstream stops
    stops = boarding.merge(
        downstream_events,
        on="second_trip_id",
        how="inner"
    )

    # 4. Keep stops after boarding with valid arrival times
    stops = stops[
        (
            stops["second_arrival_sequence"]
            > stops["second_boarding_sequence"]
        )
        & (
            stops["second_arrival_seconds"]
            >= stops["second_departure_seconds"]
        )
    ].copy()

    # 5. Attach readable station names
    stops["second_arrival_station_id"] = (
        stops["second_arrival_station_id"].astype(str)
    )

    names = station_names[
        ["station_id", "transfer_station"]
    ].rename(columns={
        "station_id": "second_arrival_station_id",
        "transfer_station": "second_transfer_station"
    }).copy()

    names["second_arrival_station_id"] = (
        names["second_arrival_station_id"].astype(str)
    )

    stops = stops.merge(
        names.drop_duplicates("second_arrival_station_id"),
        on="second_arrival_station_id",
        how="left",
        validate="many_to_one"
    )

    # 6. Exclude origin, destination and first transfer station
    stops = stops[
        (
            stops["second_transfer_station"]
            != stops["first_transfer_station"]
        )
        & (
            ~stops["second_transfer_station"].isin(
                [origin, destination]
            )
        )
    ].copy()

    return stops.reset_index(drop=True)


# 7. Test against the original algorithm
second_leg_stops_final = build_second_leg_stops(
    daily=realtime_daily,
    second_leg_candidates=second_leg_corrected,
    origin=origin,
    destination=destination,
    station_names=station_names
)

print("Original downstream stops:", len(second_leg_stops))
print("Refactored downstream stops:", len(second_leg_stops_final))

# 8. Compare all original output columns and their values
comparison_columns = second_leg_stops.columns.tolist()

comparison_final = (
    second_leg_stops_final[comparison_columns]
    .merge(
        second_leg_stops[comparison_columns],
        on=comparison_columns,
        how="outer",
        indicator=True
    )
)

print("\nCOMPARISON")
print(comparison_final["_merge"].value_counts())

print("\nExtra rows:",
      (comparison_final["_merge"] == "left_only").sum())

print("Missing rows:",
      (comparison_final["_merge"] == "right_only").sum())


Original downstream stops: 73281
Refactored downstream stops: 73281

COMPARISON
_merge
both          73281
left_only         0
right_only        0
Name: count, dtype: int64

Extra rows: 0
Missing rows: 0


In [213]:

# STEP 90V — Locate original second-transfer logic

from IPython import get_ipython

history = get_ipython().history_manager.input_hist_raw

matches = []

for cell_number, source in enumerate(history):
    if (
        "third_transfer_options =" in source
        and "def build_third_transfer_options" not in source
        and "history_manager" not in source
    ):
        matches.append((cell_number, source))

print("Matching cells:", len(matches))

for cell_number, source in matches:
    print(f"\nCELL {cell_number}")
    print("=" * 50)

    lines = source.splitlines()

    for i, line in enumerate(lines):
        if "third_transfer_options =" in line:
            start = max(0, i - 5)
            end = min(len(lines), i + 35)

            print("\n".join(
                f"{j + 1:03d} | {lines[j]}"
                for j in range(start, end)
            ))
            break


Matching cells: 1

CELL 170
019 |     "from_stop_id": "second_arrival_platform",
020 |     "to_stop_id": "third_boarding_platform",
021 |     "required_transfer_seconds": "second_required_transfer"
022 | })
023 | 
024 | third_transfer_options = second_arrivals.merge(
025 |     third_transfer_rules,
026 |     on="second_arrival_platform",
027 |     how="inner"
028 | )
029 | 
030 | # Find all departures from the third boarding platform
031 | third_departures = realtime_daily[
032 |     [
033 |         "trip_id",
034 |         "stop_id",
035 |         "stop_sequence",
036 |         "departure_seconds"
037 |     ]
038 | ].rename(columns={
039 |     "trip_id": "third_trip_id",
040 |     "stop_id": "third_departure_platform",
041 |     "stop_sequence": "third_boarding_sequence",
042 |     "departure_seconds": "third_departure_seconds"
043 | }).copy()
044 | 
045 | third_departures["third_departure_platform"] = (
046 |     third_departures["third_departure_platform"].astype(str)
047 | )
048 | 

In [214]:

# STEP 90W — Retrieve original second-transfer algorithm

from IPython import get_ipython

source = get_ipython().history_manager.input_hist_raw[170]
lines = source.splitlines()

# Print in small sections to avoid notebook output truncation
for start, end in [(1, 25), (26, 50), (51, 80)]:
    print(f"\n{'=' * 20} LINES {start}-{end} {'=' * 20}")

    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:03d} | {lines[i]}")



==================== LINES 1-25 ====================
001 | 
002 | # STEP 86E — Connect second-transfer stations to the destination
003 | 
004 | # Keep second-leg arrival events and journey history
005 | second_arrivals = second_leg_stops.copy()
006 | 
007 | second_arrivals["second_arrival_platform"] = (
008 |     second_arrivals["second_arrival_platform"].astype(str)
009 | )
010 | 
011 | # Match second-leg arrival platforms to possible third-leg platforms
012 | third_transfer_rules = platform_rules[
013 |     [
014 |         "from_stop_id",
015 |         "to_stop_id",
016 |         "required_transfer_seconds"
017 |     ]
018 | ].rename(columns={
019 |     "from_stop_id": "second_arrival_platform",
020 |     "to_stop_id": "third_boarding_platform",
021 |     "required_transfer_seconds": "second_required_transfer"
022 | })
023 | 
024 | third_transfer_options = second_arrivals.merge(
025 |     third_transfer_rules,

==================== LINES 26-50 ====================
026 |     on="seco

In [215]:

# STEP 90X — Extract second-transfer rules without truncation

from IPython import get_ipython

source = get_ipython().history_manager.input_hist_raw[170]
lines = source.splitlines()

# Extract only the code up to the third-leg departure search
start_marker = "# Keep second-leg arrival events"
end_marker = "# Find all departures from the third boarding platform"

start = next(
    i for i, line in enumerate(lines)
    if start_marker in line
)

end = next(
    i for i, line in enumerate(lines)
    if end_marker in line
)

extracted_code = "\n".join(lines[start:end])

print(extracted_code)
print("\n" + "=" * 50)
print("Original transfer options:", len(third_transfer_options))
print("Original transfer columns:")
print(third_transfer_options.columns.tolist())


# Keep second-leg arrival events and journey history
second_arrivals = second_leg_stops.copy()

second_arrivals["second_arrival_platform"] = (
    second_arrivals["second_arrival_platform"].astype(str)
)

# Match second-leg arrival platforms to possible third-leg platforms
third_transfer_rules = platform_rules[
    [
        "from_stop_id",
        "to_stop_id",
        "required_transfer_seconds"
    ]
].rename(columns={
    "from_stop_id": "second_arrival_platform",
    "to_stop_id": "third_boarding_platform",
    "required_transfer_seconds": "second_required_transfer"
})

third_transfer_options = second_arrivals.merge(
    third_transfer_rules,
    on="second_arrival_platform",
    how="inner"
)


Original transfer options: 288450
Original transfer columns:
['first_trip_id', 'second_trip_id', 'first_transfer_station', 'origin_departure_seconds', 'second_boarding_sequence', 'second_departure_seconds', 'second_arrival_station_id', 'second_arrival_platform', 'second_arrival_sequence', 

In [216]:

# STEP 90Y — Reusable second-transfer options function

import pandas as pd

def build_third_transfer_options(
    second_leg_stops,
    platform_rules
):
    if second_leg_stops.empty:
        return pd.DataFrame()

    # 1. Preserve second-leg journey history
    second_arrivals = second_leg_stops.copy()

    # 2. Standardise platform identifiers
    second_arrivals["second_arrival_platform"] = (
        second_arrivals["second_arrival_platform"].astype(str)
    )

    # 3. Prepare permitted platform transfers
    third_transfer_rules = platform_rules[
        [
            "from_stop_id",
            "to_stop_id",
            "required_transfer_seconds"
        ]
    ].copy()

    third_transfer_rules = third_transfer_rules.rename(columns={
        "from_stop_id": "second_arrival_platform",
        "to_stop_id": "third_boarding_platform",
        "required_transfer_seconds": "second_required_transfer"
    })

    third_transfer_rules["second_arrival_platform"] = (
        third_transfer_rules["second_arrival_platform"].astype(str)
    )

    # 4. Match arrival platforms to permitted boarding platforms
    transfer_options = second_arrivals.merge(
        third_transfer_rules,
        on="second_arrival_platform",
        how="inner"
    )

    return transfer_options.reset_index(drop=True)


# Run reusable function
third_transfer_test = build_third_transfer_options(
    second_leg_stops_final,
    platform_rules
)

print("Original transfer options:", len(third_transfer_options))
print("Refactored transfer options:", len(third_transfer_test))

# Compare all original columns
comparison_columns = third_transfer_options.columns.tolist()

original_counts = (
    third_transfer_options
    .groupby(comparison_columns, dropna=False)
    .size()
    .reset_index(name="original_count")
)

new_counts = (
    third_transfer_test
    .groupby(comparison_columns, dropna=False)
    .size()
    .reset_index(name="new_count")
)

comparison_90y = original_counts.merge(
    new_counts,
    on=comparison_columns,
    how="outer"
)

comparison_90y[["original_count", "new_count"]] = (
    comparison_90y[["original_count", "new_count"]]
    .fillna(0)
    .astype(int)
)

print("\nCOMPARISON")
print("Exact matching rows:",
      comparison_90y[
          comparison_90y["original_count"]
          == comparison_90y["new_count"]
      ]["original_count"].sum())

print("Extra rows:",
      (
          comparison_90y["new_count"]
          - comparison_90y["original_count"]
      ).clip(lower=0).sum())

print("Missing rows:",
      (
          comparison_90y["original_count"]
          - comparison_90y["new_count"]
      ).clip(lower=0).sum())


Original transfer options: 288450
Refactored transfer options: 288450

COMPARISON
Exact matching rows: 288450
Extra rows: 0
Missing rows: 0


In [217]:

# STEP 90Z — Inspect original third-leg connection logic

from IPython import get_ipython

source = get_ipython().history_manager.input_hist_raw[170]
lines = source.splitlines()

# Focus on the third-leg departures and connection filters
for start, end in [(30, 48), (49, 76)]:
    section = "\n".join(
        f"{i + 1:03d} | {lines[i]}"
        for i in range(start - 1, min(end, len(lines)))
    )

    print(f"\nLINES {start}–{end}")
    print(section)

print("\nORIGINAL THIRD-LEG CANDIDATES")
print("Rows:", len(third_leg_candidates))
print("Columns:", third_leg_candidates.columns.tolist())



LINES 30–48
030 | # Find all departures from the third boarding platform
031 | third_departures = realtime_daily[
032 |     [
033 |         "trip_id",
034 |         "stop_id",
035 |         "stop_sequence",
036 |         "departure_seconds"
037 |     ]
038 | ].rename(columns={
039 |     "trip_id": "third_trip_id",
040 |     "stop_id": "third_departure_platform",
041 |     "stop_sequence": "third_boarding_sequence",
042 |     "departure_seconds": "third_departure_seconds"
043 | }).copy()
044 | 
045 | third_departures["third_departure_platform"] = (
046 |     third_departures["third_departure_platform"].astype(str)
047 | )
048 | 

LINES 49–76
049 | third_leg_candidates = third_transfer_options.merge(
050 |     third_departures,
051 |     left_on="third_boarding_platform",
052 |     right_on="third_departure_platform",
053 |     how="inner"
054 | )
055 | 
056 | # Validate the second transfer
057 | third_leg_candidates = third_leg_candidates[
058 |     (
059 |         third_leg_candidates

In [218]:

# STEP 91A — Inspect third-leg connection and filtering rules

from IPython import get_ipython

source = get_ipython().history_manager.input_hist_raw[170]
lines = source.splitlines()

for start, end in [(49, 62), (63, 76)]:
    print(f"\nLINES {start}–{end}")
    print("=" * 35)

    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:03d} | {lines[i]}")



LINES 49–62
049 | third_leg_candidates = third_transfer_options.merge(
050 |     third_departures,
051 |     left_on="third_boarding_platform",
052 |     right_on="third_departure_platform",
053 |     how="inner"
054 | )
055 | 
056 | # Validate the second transfer
057 | third_leg_candidates = third_leg_candidates[
058 |     (
059 |         third_leg_candidates["third_trip_id"]
060 |         != third_leg_candidates["second_trip_id"]
061 |     )
062 |     & (

LINES 63–76
063 |         third_leg_candidates["third_trip_id"]
064 |         != third_leg_candidates["first_trip_id"]
065 |     )
066 |     & (
067 |         third_leg_candidates["third_departure_seconds"]
068 |         >= third_leg_candidates["second_arrival_seconds"]
069 |         + third_leg_candidates["second_required_transfer"]
070 |     )
071 |     & (
072 |         third_leg_candidates["third_departure_seconds"]
073 |         <= third_leg_candidates["second_arrival_seconds"] + 45 * 60
074 |     )
075 | ].copy()
076 | 


In [219]:

# STEP 91B — Display only the missing transfer-time condition

from IPython import get_ipython

lines = (
    get_ipython()
    .history_manager
    .input_hist_raw[170]
    .splitlines()
)

for line_number in range(66, 75):
    print(f"{line_number}: {lines[line_number - 1]}")


66:     & (
67:         third_leg_candidates["third_departure_seconds"]
68:         >= third_leg_candidates["second_arrival_seconds"]
69:         + third_leg_candidates["second_required_transfer"]
70:     )
71:     & (
72:         third_leg_candidates["third_departure_seconds"]
73:         <= third_leg_candidates["second_arrival_seconds"] + 45 * 60
74:     )


In [220]:

# STEP 91C — Reusable third-leg candidate search

import pandas as pd

def build_third_leg_candidates(
    daily,
    transfer_options,
    connection_window_minutes=45
):
    if transfer_options.empty:
        return pd.DataFrame()

    # 1. Prepare all possible third-leg departures
    third_departures = daily[
        [
            "trip_id",
            "stop_id",
            "stop_sequence",
            "departure_seconds"
        ]
    ].rename(columns={
        "trip_id": "third_trip_id",
        "stop_id": "third_departure_platform",
        "stop_sequence": "third_boarding_sequence",
        "departure_seconds": "third_departure_seconds"
    }).copy()

    # 2. Standardise platform IDs
    third_departures["third_departure_platform"] = (
        third_departures["third_departure_platform"].astype(str)
    )

    options = transfer_options.copy()

    options["third_boarding_platform"] = (
        options["third_boarding_platform"].astype(str)
    )

    # 3. Match transfer platforms to departing services
    candidates = options.merge(
        third_departures,
        left_on="third_boarding_platform",
        right_on="third_departure_platform",
        how="inner"
    )

    # 4. Validate third-leg connections
    candidates = candidates[
        (
            candidates["third_trip_id"]
            != candidates["second_trip_id"]
        )
        & (
            candidates["third_trip_id"]
            != candidates["first_trip_id"]
        )
        & (
            candidates["third_departure_seconds"]
            >= candidates["second_arrival_seconds"]
            + candidates["second_required_transfer"]
        )
        & (
            candidates["third_departure_seconds"]
            <= candidates["second_arrival_seconds"]
            + connection_window_minutes * 60
        )
    ].copy()

    return candidates.reset_index(drop=True)


# 5. Run reusable function
third_leg_test = build_third_leg_candidates(
    daily=realtime_daily,
    transfer_options=third_transfer_test,
    connection_window_minutes=45
)

print("Original third-leg candidates:", len(third_leg_candidates))
print("Refactored third-leg candidates:", len(third_leg_test))

# 6. Compare exact rows, including duplicate occurrences
comparison_columns = third_leg_candidates.columns.tolist()

original_counts = (
    third_leg_candidates
    .groupby(comparison_columns, dropna=False)
    .size()
    .reset_index(name="original_count")
)

new_counts = (
    third_leg_test
    .groupby(comparison_columns, dropna=False)
    .size()
    .reset_index(name="new_count")
)

comparison_91c = original_counts.merge(
    new_counts,
    on=comparison_columns,
    how="outer"
)

comparison_91c[["original_count", "new_count"]] = (
    comparison_91c[["original_count", "new_count"]]
    .fillna(0)
    .astype(int)
)

comparison_91c["difference"] = (
    comparison_91c["new_count"]
    - comparison_91c["original_count"]
)

print("\nCOMPARISON")
print("Matching rows:",
      comparison_91c.loc[
          comparison_91c["difference"] == 0,
          "original_count"
      ].sum())

print("Extra rows:",
      comparison_91c["difference"].clip(lower=0).sum())

print("Missing rows:",
      (-comparison_91c["difference"]).clip(lower=0).sum())


Original third-leg candidates: 1515898
Refactored third-leg candidates: 1515898

COMPARISON
Matching rows: 1515898
Extra rows: 0
Missing rows: 0


In [221]:

# STEP 91C — Reusable third-leg candidate search

import pandas as pd

def build_third_leg_candidates(
    daily,
    transfer_options,
    connection_window_minutes=45
):
    if transfer_options.empty:
        return pd.DataFrame()

    # 1. Prepare all possible third-leg departures
    third_departures = daily[
        [
            "trip_id",
            "stop_id",
            "stop_sequence",
            "departure_seconds"
        ]
    ].rename(columns={
        "trip_id": "third_trip_id",
        "stop_id": "third_departure_platform",
        "stop_sequence": "third_boarding_sequence",
        "departure_seconds": "third_departure_seconds"
    }).copy()

    # 2. Standardise platform IDs
    third_departures["third_departure_platform"] = (
        third_departures["third_departure_platform"].astype(str)
    )

    options = transfer_options.copy()

    options["third_boarding_platform"] = (
        options["third_boarding_platform"].astype(str)
    )

    # 3. Match transfer platforms to departing services
    candidates = options.merge(
        third_departures,
        left_on="third_boarding_platform",
        right_on="third_departure_platform",
        how="inner"
    )

    # 4. Validate third-leg connections
    candidates = candidates[
        (
            candidates["third_trip_id"]
            != candidates["second_trip_id"]
        )
        & (
            candidates["third_trip_id"]
            != candidates["first_trip_id"]
        )
        & (
            candidates["third_departure_seconds"]
            >= candidates["second_arrival_seconds"]
            + candidates["second_required_transfer"]
        )
        & (
            candidates["third_departure_seconds"]
            <= candidates["second_arrival_seconds"]
            + connection_window_minutes * 60
        )
    ].copy()

    return candidates.reset_index(drop=True)


# 5. Run reusable function
third_leg_test = build_third_leg_candidates(
    daily=realtime_daily,
    transfer_options=third_transfer_test,
    connection_window_minutes=45
)

print("Original third-leg candidates:", len(third_leg_candidates))
print("Refactored third-leg candidates:", len(third_leg_test))

# 6. Compare exact rows, including duplicate occurrences
comparison_columns = third_leg_candidates.columns.tolist()

original_counts = (
    third_leg_candidates
    .groupby(comparison_columns, dropna=False)
    .size()
    .reset_index(name="original_count")
)

new_counts = (
    third_leg_test
    .groupby(comparison_columns, dropna=False)
    .size()
    .reset_index(name="new_count")
)

comparison_91c = original_counts.merge(
    new_counts,
    on=comparison_columns,
    how="outer"
)

comparison_91c[["original_count", "new_count"]] = (
    comparison_91c[["original_count", "new_count"]]
    .fillna(0)
    .astype(int)
)

comparison_91c["difference"] = (
    comparison_91c["new_count"]
    - comparison_91c["original_count"]
)

print("\nCOMPARISON")
print("Matching rows:",
      comparison_91c.loc[
          comparison_91c["difference"] == 0,
          "original_count"
      ].sum())

print("Extra rows:",
      comparison_91c["difference"].clip(lower=0).sum())

print("Missing rows:",
      (-comparison_91c["difference"]).clip(lower=0).sum())


Original third-leg candidates: 1515898
Refactored third-leg candidates: 1515898

COMPARISON
Matching rows: 1515898
Extra rows: 0
Missing rows: 0


In [222]:

# STEP 91D — Find complete two-transfer journeys

import pandas as pd

def build_complete_two_transfer_journeys(
    daily,
    third_leg_candidates,
    destination,
    station_lookup
):
    if third_leg_candidates.empty:
        return pd.DataFrame()

    # 1. Identify destination station IDs
    destination_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"].eq(destination),
            "station_id"
        ].astype(str)
    )

    if not destination_ids:
        raise ValueError(f"Destination not found: {destination}")

    # 2. Find timetable arrivals at the destination
    destination_arrivals = daily[
        daily["station_id"].astype(str).isin(destination_ids)
    ][
        [
            "trip_id",
            "stop_id",
            "stop_sequence",
            "arrival_seconds"
        ]
    ].rename(columns={
        "trip_id": "third_trip_id",
        "stop_id": "destination_platform",
        "stop_sequence": "destination_sequence",
        "arrival_seconds": "destination_arrival_seconds"
    })

    # 3. Keep only third-leg trips serving the destination
    complete = third_leg_candidates.merge(
        destination_arrivals,
        on="third_trip_id",
        how="inner"
    )

    # 4. Destination must occur AFTER boarding
    # and the arrival time must be valid
    complete = complete[
        (
            complete["destination_sequence"]
            > complete["third_boarding_sequence"]
        )
        & (
            complete["destination_arrival_seconds"]
            >= complete["third_departure_seconds"]
        )
    ].copy()

    # 5. Calculate journey duration
    complete["journey_duration_min"] = (
        complete["destination_arrival_seconds"]
        - complete["origin_departure_seconds"]
    ) / 60

    return complete.reset_index(drop=True)


# Run the reusable function
complete_journeys_test = build_complete_two_transfer_journeys(
    daily=realtime_daily,
    third_leg_candidates=third_leg_test,
    destination=destination,
    station_lookup=station_lookup
)

print("Original complete journeys:", len(complete_journeys))
print("Refactored complete journeys:", len(complete_journeys_test))

print("\nJOURNEY DURATION SUMMARY")
if not complete_journeys_test.empty:
    display(
        complete_journeys_test[
            "journey_duration_min"
        ].describe().to_frame()
    )

    print("\nEARLIEST ARRIVALS")
    preview = complete_journeys_test.nsmallest(
        10,
        "destination_arrival_seconds"
    )[
        [
            "first_trip_id",
            "second_trip_id",
            "third_trip_id",
            "second_transfer_station",
            "journey_duration_min"
        ]
    ]

    display(preview)
else:
    print("No complete journeys found.")


Original complete journeys: 55746
Refactored complete journeys: 55746

JOURNEY DURATION SUMMARY


,journey_duration_min
count,55746.000000
mean,126.015416
std,25.828413
min,45.383333
25%,107.600000
50%,124.600000
75%,142.600000
max,246.100000



EARLIEST ARRIVALS


,first_trip_id,second_trip_id,third_trip_id,second_transfer_station,journey_duration_min
50884,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037,197F.1324.165.16.B.8.92001727,Strathfield Station,52.383333
21009,0160-006-112-013:1000,130N.1324.165.16.A.8.92003126,145N.1324.165.16.A.8.92001340,Strathfield Station,46.383333
44492,0160-006-152-015:1000,130N.1324.165.16.A.8.92003126,145N.1324.165.16.A.8.92001340,Strathfield Station,50.383333
50880,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037,145N.1324.165.16.A.8.92001340,Strathfield Station,58.383333
50963,0160-006-154-013:1000,130N.1324.165.16.A.8.92003126,145N.1324.165.16.A.8.92001340,Strathfield Station,58.383333
51990,0160-006-155-013:1000,130N.1324.165.16.A.8.92003126,145N.1324.165.16.A.8.92001340,Strathfield Station,54.383333
21012,0160-006-112-013:1000,130N.1324.165.16.A.8.92003126,165F.1324.165.124.A.8.92002689,Strathfield Station,49.383333
44495,0160-006-152-015:1000,130N.1324.165.16.A.8.92003126,165F.1324.165.124.A.8.92002689,Strathfield Station,53.383333
50883,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037,165F.1324.165.124.A.8.92002689,Strathfield Station,61.383333
50966,0160-006-154-013:1000,130N.1324.165.16.A.8.92003126,165F.1324.165.124.A.8.92002689,Strathfield Station,61.383333


In [223]:

# STEP 91E — Validate complete two-transfer journeys

import pandas as pd

# 1. Identify columns shared by both implementations
shared_columns = [
    col for col in complete_journeys.columns
    if col in complete_journeys_test.columns
]

print("Original journeys:", len(complete_journeys))
print("Refactored journeys:", len(complete_journeys_test))
print("Shared columns:", len(shared_columns))

# 2. Count occurrences of each complete journey
original_counts = (
    complete_journeys
    .groupby(shared_columns, dropna=False)
    .size()
    .reset_index(name="original_count")
)

new_counts = (
    complete_journeys_test
    .groupby(shared_columns, dropna=False)
    .size()
    .reset_index(name="new_count")
)

# 3. Compare exact records
validation = original_counts.merge(
    new_counts,
    on=shared_columns,
    how="outer"
)

validation[["original_count", "new_count"]] = (
    validation[["original_count", "new_count"]]
    .fillna(0)
    .astype(int)
)

validation["difference"] = (
    validation["new_count"]
    - validation["original_count"]
)

# 4. Validation results
print("\nVALIDATION RESULTS")
print("=" * 40)

print(
    "Matching rows:",
    validation.loc[
        validation["difference"] == 0,
        "original_count"
    ].sum()
)

print(
    "Extra rows:",
    validation["difference"].clip(lower=0).sum()
)

print(
    "Missing rows:",
    (-validation["difference"]).clip(lower=0).sum()
)

print("\nVALIDATION STATUS")

if (validation["difference"] == 0).all():
    print("PASS — All complete journeys match exactly!")
else:
    print("CHECK — Differences detected.")
    display(
        validation.loc[
            validation["difference"] != 0
        ].head(10)
    )


Original journeys: 55746
Refactored journeys: 55746
Shared columns: 20

VALIDATION RESULTS
Matching rows: 55746
Extra rows: 0
Missing rows: 0

VALIDATION STATUS
PASS — All complete journeys match exactly!


In [224]:

# STEP 91F — Combine the two-transfer routing pipeline

def find_two_transfer_journeys(
    daily,
    origin,
    destination,
    departure_time,
    window_minutes=60,
    connection_window_minutes=45,
    transfer_rules=None,
    platform_rules_df=None,
    station_names_df=None,
    station_lookup_df=None
):
    # Use existing notebook reference tables by default
    if transfer_rules is None:
        transfer_rules = all_platform_transfers

    if platform_rules_df is None:
        platform_rules_df = platform_rules

    if station_names_df is None:
        station_names_df = station_names

    if station_lookup_df is None:
        station_lookup_df = station_lookup

    # 1. Find first-leg downstream stops
    first_stops = build_first_leg_stops(
        daily,
        origin,
        departure_time,
        window_minutes=window_minutes
    )

    if first_stops.empty:
        return pd.DataFrame()

    # 2. Find possible first transfers
    first_transfers = build_first_transfer_options(
        first_stops,
        transfer_rules=transfer_rules
    )

    if first_transfers.empty:
        return pd.DataFrame()

    # 3. Find second-leg services
    second_candidates = build_second_leg_candidates(
        daily,
        first_transfers,
        connection_window_minutes=connection_window_minutes
    )

    if second_candidates.empty:
        return pd.DataFrame()

    # 4. Find downstream stops on second-leg services
    second_stops = build_second_leg_stops(
        daily,
        second_candidates,
        origin,
        destination,
        station_names_df
    )

    if second_stops.empty:
        return pd.DataFrame()

    # 5. Find possible second transfers
    second_transfers = build_third_transfer_options(
        second_stops,
        platform_rules_df
    )

    if second_transfers.empty:
        return pd.DataFrame()

    # 6. Find third-leg services
    third_candidates = build_third_leg_candidates(
        daily,
        second_transfers,
        connection_window_minutes=connection_window_minutes
    )

    if third_candidates.empty:
        return pd.DataFrame()

    # 7. Find third-leg arrivals at the destination
    complete = build_complete_two_transfer_journeys(
        daily,
        third_candidates,
        destination,
        station_lookup_df
    )

    return complete


# TEST — Original Macquarie University → Parramatta journey
two_transfer_test = find_two_transfer_journeys(
    daily=realtime_daily,
    origin="Macquarie University Station",
    destination="Parramatta Station",
    departure_time="17:30:00",
    window_minutes=60,
    connection_window_minutes=45
)

print("Original complete journeys:", len(complete_journeys))
print("Combined function journeys:", len(two_transfer_test))

# Validate the combined pipeline
if len(two_transfer_test) == len(complete_journeys):
    print("Row count matches original pipeline.")
else:
    print("Row count differs — investigate before proceeding.")

if not two_transfer_test.empty:
    print("\nFASTEST FIVE JOURNEYS")

    display(
        two_transfer_test.nsmallest(
            5,
            "journey_duration_min"
        )[
            [
                "first_transfer_station",
                "second_transfer_station",
                "journey_duration_min"
            ]
        ]
    )


Original complete journeys: 55746
Combined function journeys: 55746
Row count matches original pipeline.

FASTEST FIVE JOURNEYS


,first_transfer_station,second_transfer_station,journey_duration_min
21979,Epping Station,Strathfield Station,45.383333
21009,Epping Station,Strathfield Station,46.383333
945,Epping Station,Strathfield Station,49.383333
21012,Epping Station,Strathfield Station,49.383333
44492,Epping Station,Strathfield Station,50.383333


In [225]:

# STEP 91G — Deduplicate and Pareto-filter two-transfer journeys

import pandas as pd

def rank_two_transfer_journeys(journeys, top_n=10):
    if journeys.empty:
        return pd.DataFrame()

    routes = journeys.copy()

    # 1. Identify distinct passenger itineraries
    itinerary_keys = [
        "first_trip_id",
        "second_trip_id",
        "third_trip_id",
        "origin_departure_seconds",
        "second_boarding_sequence",
        "second_arrival_sequence",
        "third_boarding_sequence",
        "destination_arrival_seconds"
    ]

    routes = routes.drop_duplicates(
        subset=itinerary_keys
    ).copy()

    print("Unique itineraries:", len(routes))

    # 2. Sort by departure time, then arrival time
    routes = routes.sort_values(
        [
            "origin_departure_seconds",
            "destination_arrival_seconds"
        ],
        ascending=[False, True]
    ).copy()

    # 3. Pareto filtering:
    # A journey is dominated if another journey departs
    # at the same time or later and arrives earlier or equally early.

    best_arrival_so_far = float("inf")
    efficient_indices = []

    for idx, journey in routes.iterrows():
        arrival = journey["destination_arrival_seconds"]

        if arrival < best_arrival_so_far:
            efficient_indices.append(idx)
            best_arrival_so_far = arrival

    efficient = routes.loc[efficient_indices].copy()

    # 4. Rank passenger-friendly options
    efficient = efficient.sort_values(
        [
            "destination_arrival_seconds",
            "origin_departure_seconds"
        ],
        ascending=[True, False]
    )

    return efficient.head(top_n).reset_index(drop=True)


# Test ranking
recommended_two_transfers = rank_two_transfer_journeys(
    two_transfer_test,
    top_n=10
)

print("\nRECOMMENDED TWO-TRANSFER JOURNEYS")
print("================================")

if not recommended_two_transfers.empty:
    preview = recommended_two_transfers[
        [
            "first_transfer_station",
            "second_transfer_station",
            "origin_departure_seconds",
            "destination_arrival_seconds",
            "journey_duration_min"
        ]
    ].copy()

    preview["departure"] = preview[
        "origin_departure_seconds"
    ].apply(format_transport_time)

    preview["arrival"] = preview[
        "destination_arrival_seconds"
    ].apply(format_transport_time)

    display(
        preview[
            [
                "first_transfer_station",
                "second_transfer_station",
                "departure",
                "arrival",
                "journey_duration_min"
            ]
        ]
    )
else:
    print("No journeys available.")


Unique itineraries: 55746

RECOMMENDED TWO-TRANSFER JOURNEYS


,first_transfer_station,second_transfer_station,departure,arrival,journey_duration_min
0,Epping Station,Strathfield Station,17:30:37,18:23:00,52.383333
1,Epping Station,Strathfield Station,17:42:37,18:29:00,46.383333
2,Epping Station,Strathfield Station,17:46:37,18:38:00,51.383333
3,Epping Station,Strathfield Station,17:58:37,18:44:00,45.383333
4,Epping Station,Strathfield Station,18:02:37,18:56:00,53.383333
5,Epping Station,Strathfield Station,18:14:37,19:11:00,56.383333
6,Epping Station,Strathfield Station,18:26:37,19:26:00,59.383333
7,Macquarie Park Station,Central Station,18:27:54,19:41:00,73.100000


In [226]:

# STEP 91H — Inspect journey DataFrame structures

import pandas as pd

print("DIRECT JOURNEY FUNCTION")
print("=" * 50)

direct_test = find_direct_journeys(
    realtime_daily,
    origin,
    destination,
    departure_time,
    window_minutes=60
)

print("Rows:", len(direct_test))
print("Columns:")
print(direct_test.columns.tolist())

print("\nONE-TRANSFER JOURNEY FUNCTION")
print("=" * 50)

one_transfer_test = find_one_transfer_journeys(
    realtime_daily,
    origin,
    destination,
    departure_time,
    window_minutes=60,
    connection_window_minutes=45
)

print("Rows:", len(one_transfer_test))
print("Columns:")
print(one_transfer_test.columns.tolist())

print("\nTWO-TRANSFER JOURNEY FUNCTION")
print("=" * 50)

print("Rows:", len(recommended_two_transfers))
print("Columns:")
print(recommended_two_transfers.columns.tolist())


DIRECT JOURNEY FUNCTION
Rows: 0
Columns:
['trip_id', 'stop_id', 'arrival_time', 'departure_time', 'stop_sequence', 'route_short_name', 'transport_mode', 'service_id', 'arrival_seconds', 'departure_seconds', 'station_id', 'scheduled_arrival_sec', 'scheduled_departure_sec', 'effective_arrival_sec', 'effective_departure_sec', 'arrival_source', 'departure_source', 'predicted_arrival_sec', 'predicted_departure_sec', 'original_arrival_seconds', 'original_departure_seconds', 'last_stop_sequence', 'destination_sequence', 'destination_arrival_seconds', 'destination_arrival_time', 'journey_duration_min', 'journey_type', 'transfers']

ONE-TRANSFER JOURNEY FUNCTION
Rows: 3078
Columns:
['boarding_id', 'trip_id', 'origin_platform', 'origin_sequence', 'origin_departure_seconds', 'origin_departure_time', 'first_line', 'first_mode', 'stop_id_x', 'station_id', 'stop_sequence', 'arrival_seconds', 'arrival_time', 'from_stop_id', 'to_stop_id', 'minimum_transfer_seconds', 'earliest_boarding_seconds', 'secon

In [227]:

# STEP 91I — Standardise direct, one-transfer and two-transfer journeys

import pandas as pd

def standardise_journey_results(
    direct_journeys,
    one_transfer_journeys,
    two_transfer_journeys
):
    results = []

    # 1. Direct journeys
    if not direct_journeys.empty:
        direct = pd.DataFrame({
            "journey_type": "Direct",
            "transfers": 0,
            "first_trip_id": direct_journeys["trip_id"],
            "second_trip_id": None,
            "third_trip_id": None,
            "origin_departure_seconds": direct_journeys["departure_seconds"],
            "destination_arrival_seconds": direct_journeys["destination_arrival_seconds"],
            "journey_duration_min": direct_journeys["journey_duration_min"],
            "first_transfer_station": None,
            "second_transfer_station": None
        })
        results.append(direct)

    # 2. One-transfer journeys
    if not one_transfer_journeys.empty:
        one = pd.DataFrame({
            "journey_type": "One transfer",
            "transfers": 1,
            "first_trip_id": one_transfer_journeys["trip_id"],
            "second_trip_id": one_transfer_journeys["second_trip_id"],
            "third_trip_id": None,
            "origin_departure_seconds": one_transfer_journeys["origin_departure_seconds"],
            "destination_arrival_seconds": one_transfer_journeys["destination_arrival_seconds"],
            "journey_duration_min": one_transfer_journeys["journey_duration_min"],
            "first_transfer_station": one_transfer_journeys["transfer_station"],
            "second_transfer_station": None
        })
        results.append(one)

    # 3. Two-transfer journeys
    if not two_transfer_journeys.empty:
        two = pd.DataFrame({
            "journey_type": "Two transfers",
            "transfers": 2,
            "first_trip_id": two_transfer_journeys["first_trip_id"],
            "second_trip_id": two_transfer_journeys["second_trip_id"],
            "third_trip_id": two_transfer_journeys["third_trip_id"],
            "origin_departure_seconds": two_transfer_journeys["origin_departure_seconds"],
            "destination_arrival_seconds": two_transfer_journeys["destination_arrival_seconds"],
            "journey_duration_min": two_transfer_journeys["journey_duration_min"],
            "first_transfer_station": two_transfer_journeys["first_transfer_station"],
            "second_transfer_station": two_transfer_journeys["second_transfer_station"]
        })
        results.append(two)

    if not results:
        return pd.DataFrame()

    # 4. Combine all journey types
    combined = pd.concat(
        results,
        ignore_index=True
    )

    # 5. Remove duplicate journey combinations
    combined = combined.drop_duplicates(
        subset=[
            "journey_type",
            "first_trip_id",
            "second_trip_id",
            "third_trip_id",
            "origin_departure_seconds",
            "destination_arrival_seconds"
        ]
    ).copy()

    # 6. Add readable departure and arrival times
    combined["departure_time"] = combined[
        "origin_departure_seconds"
    ].apply(format_transport_time)

    combined["arrival_time"] = combined[
        "destination_arrival_seconds"
    ].apply(format_transport_time)

    # 7. Sort by arrival time, then transfers
    combined = combined.sort_values(
        [
            "destination_arrival_seconds",
            "transfers",
            "journey_duration_min"
        ]
    ).reset_index(drop=True)

    return combined


# Run unified journey function
all_journeys = standardise_journey_results(
    direct_test,
    one_transfer_test,
    recommended_two_transfers
)

print("TOTAL STANDARDISED JOURNEYS:", len(all_journeys))

print("\nJOURNEYS BY TYPE")
print(all_journeys["journey_type"].value_counts())

print("\nFIRST 15 JOURNEYS")
display(
    all_journeys[
        [
            "journey_type",
            "first_transfer_station",
            "second_transfer_station",
            "departure_time",
            "arrival_time",
            "journey_duration_min"
        ]
    ].head(15)
)


TOTAL STANDARDISED JOURNEYS: 149

JOURNEYS BY TYPE
journey_type
One transfer     141
Two transfers      8
Name: count, dtype: int64

FIRST 15 JOURNEYS


,journey_type,first_transfer_station,second_transfer_station,departure_time,arrival_time,journey_duration_min
0,Two transfers,Epping Station,Strathfield Station,17:30:37,18:23:00,52.383333
1,One transfer,Central Station,None,17:31:54,18:29:00,57.100000
2,Two transfers,Epping Station,Strathfield Station,17:42:37,18:29:00,46.383333
3,One transfer,Central Station,None,17:35:54,18:32:00,56.100000
4,One transfer,Central Station,None,17:31:54,18:32:00,60.100000
5,One transfer,Central Station,None,17:39:54,18:38:00,58.100000
6,One transfer,Central Station,None,17:35:54,18:38:00,62.100000
7,One transfer,Central Station,None,17:31:54,18:38:00,66.100000
8,Two transfers,Epping Station,Strathfield Station,17:46:37,18:38:00,51.383333
9,One transfer,Central Station,None,17:43:54,18:41:00,57.100000


In [228]:

# STEP 91J — Rank all journey types together

import pandas as pd
import numpy as np

def rank_all_journeys(journeys, top_n=10):
    if journeys.empty:
        return journeys.copy()

    routes = journeys.copy()

    # Sort to make results deterministic
    routes = routes.sort_values(
        [
            "origin_departure_seconds",
            "destination_arrival_seconds",
            "transfers",
            "journey_duration_min"
        ],
        ascending=[True, True, True, True]
    ).reset_index(drop=True)

    # A journey is dominated when another journey:
    # - departs at the same time or later
    # - arrives at the same time or earlier
    # - requires the same number or fewer transfers
    # - is strictly better in at least one dimension

    departures = routes["origin_departure_seconds"].to_numpy()
    arrivals = routes["destination_arrival_seconds"].to_numpy()
    transfers = routes["transfers"].to_numpy()

    efficient_mask = np.ones(len(routes), dtype=bool)

    for i in range(len(routes)):
        dominated_by = (
            (departures >= departures[i])
            & (arrivals <= arrivals[i])
            & (transfers <= transfers[i])
            & (
                (departures > departures[i])
                | (arrivals < arrivals[i])
                | (transfers < transfers[i])
            )
        )

        dominated_by[i] = False

        if dominated_by.any():
            efficient_mask[i] = False

    efficient = routes.loc[efficient_mask].copy()

    # Prioritise earliest arrival, then fewer transfers
    efficient = efficient.sort_values(
        [
            "destination_arrival_seconds",
            "transfers",
            "journey_duration_min"
        ]
    ).head(top_n).reset_index(drop=True)

    return efficient


# Rank the unified results
best_journeys = rank_all_journeys(
    all_journeys,
    top_n=10
)

print("Total journey combinations:", len(all_journeys))
print("Recommended journeys:", len(best_journeys))

print("\nRECOMMENDED JOURNEYS")
display(
    best_journeys[
        [
            "journey_type",
            "first_transfer_station",
            "second_transfer_station",
            "departure_time",
            "arrival_time",
            "journey_duration_min"
        ]
    ]
)


Total journey combinations: 149
Recommended journeys: 10

RECOMMENDED JOURNEYS


,journey_type,first_transfer_station,second_transfer_station,departure_time,arrival_time,journey_duration_min
0,Two transfers,Epping Station,Strathfield Station,17:30:37,18:23:00,52.383333
1,One transfer,Central Station,None,17:31:54,18:29:00,57.100000
2,Two transfers,Epping Station,Strathfield Station,17:42:37,18:29:00,46.383333
3,One transfer,Central Station,None,17:35:54,18:32:00,56.100000
4,One transfer,Central Station,None,17:39:54,18:38:00,58.100000
5,Two transfers,Epping Station,Strathfield Station,17:46:37,18:38:00,51.383333
6,One transfer,Central Station,None,17:43:54,18:41:00,57.100000
7,One transfer,Central Station,None,17:47:54,18:44:00,56.100000
8,Two transfers,Epping Station,Strathfield Station,17:58:37,18:44:00,45.383333
9,One transfer,Central Station,None,17:55:54,18:56:00,60.100000


In [229]:

# STEP 91K — Inspect the first recommended one-transfer journey

import pandas as pd

# Select the earliest-arriving one-transfer recommendation
selected = best_journeys.loc[
    best_journeys["transfers"] == 1
].iloc[0]

# Find the matching original journey
matching = one_transfer_test.loc[
    (one_transfer_test["trip_id"] == selected["first_trip_id"])
    & (one_transfer_test["second_trip_id"] == selected["second_trip_id"])
    & (
        one_transfer_test["origin_departure_seconds"]
        == selected["origin_departure_seconds"]
    )
    & (
        one_transfer_test["destination_arrival_seconds"]
        == selected["destination_arrival_seconds"]
    )
]

print("Matching original journeys:", len(matching))

if not matching.empty:
    journey = matching.iloc[0]

    fields = [
        "trip_id",
        "origin_platform",
        "origin_departure_time",
        "first_line",
        "first_mode",
        "transfer_station",
        "from_stop_id",
        "to_stop_id",
        "minimum_transfer_seconds",
        "second_trip_id",
        "second_platform",
        "second_departure_time",
        "second_line",
        "second_mode",
        "destination_arrival_time",
        "journey_duration_min"
    ]

    details = pd.DataFrame({
        "field": fields,
        "value": [
            journey.get(field, None)
            for field in fields
        ]
    })

    display(details)


Matching original journeys: 19


,field,value
0,trip_id,0160-006-106-014:1000
1,origin_platform,2113351
2,origin_departure_time,17:31:54
3,first_line,M1
4,first_mode,Metro
5,transfer_station,Central Station
6,from_stop_id,2000467
7,to_stop_id,2000338
8,minimum_transfer_seconds,366.0
9,second_trip_id,145N.1324.165.16.A.8.92001340


In [230]:

# STEP 91L — Retrieve complete stop details for a one-transfer journey

import pandas as pd

journey = matching.iloc[0]

first_trip = realtime_daily.loc[
    realtime_daily["trip_id"] == journey["trip_id"]
].sort_values("stop_sequence")

second_trip = realtime_daily.loc[
    realtime_daily["trip_id"] == journey["second_trip_id"]
].sort_values("stop_sequence")

# Identify the first-leg arrival at the transfer platform
transfer_arrival = first_trip.loc[
    first_trip["stop_id"].astype(str)
    == str(journey["from_stop_id"])
]

# Identify the second-leg destination arrival
destination_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == destination,
        "station_id"
    ].astype(str)
)

destination_arrival = second_trip.loc[
    (second_trip["station_id"].astype(str).isin(destination_ids))
    & (
        second_trip["arrival_seconds"]
        == journey["destination_arrival_seconds"]
    )
    & (
        second_trip["stop_sequence"]
        > journey["second_boarding_sequence"]
    )
]

print("FIRST LEG — ORIGIN")
print("Departure:", journey["origin_departure_time"])
print("Platform ID:", journey["origin_platform"])
print("Service:", journey["first_line"], journey["first_mode"])

print("\nTRANSFER —", journey["transfer_station"])

if not transfer_arrival.empty:
    stop = transfer_arrival.iloc[0]
    print("Arrival:", format_transport_time(stop["arrival_seconds"]))
    print("Arrival platform ID:", stop["stop_id"])
else:
    print("Transfer arrival not found")

print("Departure:", journey["second_departure_time"])
print("Departure platform ID:", journey["second_platform"])
print("Next service:", journey["second_line"], journey["second_mode"])

print("\nDESTINATION —", destination)

if not destination_arrival.empty:
    stop = destination_arrival.iloc[0]
    print("Arrival:", format_transport_time(stop["arrival_seconds"]))
    print("Arrival platform ID:", stop["stop_id"])
else:
    print("Destination arrival not found")


FIRST LEG — ORIGIN
Departure: 17:31:54
Platform ID: 2113351
Service: M1 Metro

TRANSFER — Central Station
Arrival: 17:56:09
Arrival platform ID: 2000467
Departure: 18:04:01
Departure platform ID: 2000338
Next service: T1 Train

DESTINATION — Parramatta Station
Arrival: 18:29:00
Arrival platform ID: 2150412


In [231]:

# STEP 91M — Map GTFS stop IDs to passenger-facing platform numbers

import pandas as pd

platform_ids = [
    journey["origin_platform"],
    transfer_arrival.iloc[0]["stop_id"],
    journey["second_platform"],
    destination_arrival.iloc[0]["stop_id"]
]

platform_labels = [
    "Origin departure",
    "Transfer arrival",
    "Transfer departure",
    "Destination arrival"
]

# Inspect actual platform information
platform_mapping = pd.DataFrame({
    "journey_stage": platform_labels,
    "stop_id": [str(x) for x in platform_ids]
})

stop_details = transport_stops.copy()
stop_details["stop_id"] = stop_details["stop_id"].astype(str)

platform_mapping = platform_mapping.merge(
    stop_details[
        ["stop_id", "stop_name", "platform_code"]
    ].drop_duplicates(subset=["stop_id"]),
    on="stop_id",
    how="left"
)

platform_mapping["display_platform"] = platform_mapping[
    "platform_code"
].apply(
    lambda x: f"Platform {x}"
    if pd.notna(x) and str(x).strip()
    else "Platform unavailable"
)

display(
    platform_mapping[
        [
            "journey_stage",
            "stop_name",
            "display_platform"
        ]
    ]
)


,journey_stage,stop_name,display_platform
0,Origin departure,"Macquarie University Station, Platform 1",Platform 1
1,Transfer arrival,"Central Station, Platform 27",Platform unavailable
2,Transfer departure,"Central Station, Platform 18",Platform 18
3,Destination arrival,"Parramatta Station, Platform 2",Platform 2


In [232]:

# STEP 91N — Build reusable passenger-facing platform lookup

import pandas as pd
import re

def build_platform_lookup(transport_stops):
    stops = transport_stops.copy()

    stops["stop_id"] = stops["stop_id"].astype(str)

    def extract_platform(row):
        code = row.get("platform_code")
        name = str(row.get("stop_name", ""))

        # Priority 1: Use GTFS platform_code
        if pd.notna(code) and str(code).strip():
            platform = str(code).strip()
            platform = re.sub(
                r"^Platform\s+",
                "",
                platform,
                flags=re.IGNORECASE
            )
            return f"Platform {platform}"

        # Priority 2: Extract platform number from stop_name
        match = re.search(
            r"Platform\s+([A-Za-z0-9]+)",
            name,
            flags=re.IGNORECASE
        )

        if match:
            return f"Platform {match.group(1)}"

        return "Platform unavailable"

    stops["display_platform"] = stops.apply(
        extract_platform,
        axis=1
    )

    return (
        stops[["stop_id", "display_platform"]]
        .drop_duplicates(subset=["stop_id"])
        .set_index("stop_id")["display_platform"]
        .to_dict()
    )


# Create reusable lookup
platform_lookup = build_platform_lookup(transport_stops)


def get_platform(stop_id):
    return platform_lookup.get(
        str(stop_id),
        "Platform unavailable"
    )


# Test on your actual journey
test_platforms = pd.DataFrame({
    "journey_stage": [
        "Origin departure",
        "Transfer arrival",
        "Transfer departure",
        "Destination arrival"
    ],
    "platform": [
        get_platform(journey["origin_platform"]),
        get_platform(transfer_arrival.iloc[0]["stop_id"]),
        get_platform(journey["second_platform"]),
        get_platform(destination_arrival.iloc[0]["stop_id"])
    ]
})

display(test_platforms)


,journey_stage,platform
0,Origin departure,Platform 1
1,Transfer arrival,Platform 27
2,Transfer departure,Platform 18
3,Destination arrival,Platform 2


In [233]:

# STEP 91O — Format a complete one-transfer itinerary

import pandas as pd

def format_one_transfer_itinerary(
    journey,
    daily,
    origin,
    destination
):
    first_trip = daily.loc[
        daily["trip_id"] == journey["trip_id"]
    ].sort_values("stop_sequence")

    second_trip = daily.loc[
        daily["trip_id"] == journey["second_trip_id"]
    ].sort_values("stop_sequence")

    # Find the actual transfer arrival event
    transfer_arrivals = first_trip.loc[
        (first_trip["stop_id"].astype(str) == str(journey["from_stop_id"]))
        & (first_trip["arrival_seconds"] <= journey["second_departure_seconds"])
    ]

    # Find destination arrival event
    destination_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == destination,
            "station_id"
        ].astype(str)
    )

    destination_arrivals = second_trip.loc[
        (second_trip["station_id"].astype(str).isin(destination_ids))
        & (
            second_trip["arrival_seconds"]
            == journey["destination_arrival_seconds"]
        )
        & (
            second_trip["stop_sequence"]
            > journey["second_boarding_sequence"]
        )
    ]

    if transfer_arrivals.empty or destination_arrivals.empty:
        raise ValueError("Unable to locate complete journey stop events.")

    transfer_stop = transfer_arrivals.iloc[0]
    destination_stop = destination_arrivals.iloc[0]

    # Show HH:MM only; retain seconds internally
    def display_time(seconds):
        return format_transport_time(seconds)[:5]

    itinerary = pd.DataFrame([
        {
            "stage": "Depart",
            "station": origin,
            "time": display_time(journey["origin_departure_seconds"]),
            "platform": get_platform(journey["origin_platform"]),
            "service": f'{journey["first_line"]} {journey["first_mode"]}'
        },
        {
            "stage": "Transfer arrival",
            "station": journey["transfer_station"],
            "time": display_time(transfer_stop["arrival_seconds"]),
            "platform": get_platform(transfer_stop["stop_id"]),
            "service": f'{journey["first_line"]} {journey["first_mode"]}'
        },
        {
            "stage": "Transfer departure",
            "station": journey["transfer_station"],
            "time": display_time(journey["second_departure_seconds"]),
            "platform": get_platform(journey["second_platform"]),
            "service": f'{journey["second_line"]} {journey["second_mode"]}'
        },
        {
            "stage": "Arrive",
            "station": destination,
            "time": display_time(destination_stop["arrival_seconds"]),
            "platform": get_platform(destination_stop["stop_id"]),
            "service": f'{journey["second_line"]} {journey["second_mode"]}'
        }
    ])

    return itinerary


# Test using the previously inspected journey
one_transfer_itinerary = format_one_transfer_itinerary(
    journey=journey,
    daily=realtime_daily,
    origin=origin,
    destination=destination
)

print("ONE-TRANSFER PASSENGER ITINERARY")
print("=" * 50)

display(one_transfer_itinerary)


ONE-TRANSFER PASSENGER ITINERARY


,stage,station,time,platform,service
0,Depart,Macquarie University Station,17:31,Platform 1,M1 Metro
1,Transfer arrival,Central Station,17:56,Platform 27,M1 Metro
2,Transfer departure,Central Station,18:04,Platform 18,T1 Train
3,Arrive,Parramatta Station,18:29,Platform 2,T1 Train


In [234]:

# STEP 91P — Format a complete two-transfer itinerary

import pandas as pd

def format_two_transfer_itinerary(
    journey,
    daily,
    origin,
    destination
):
    def event(trip_id, stop_id, sequence=None, time_seconds=None):
        matches = daily.loc[
            (daily["trip_id"] == trip_id)
            & (daily["stop_id"].astype(str) == str(stop_id))
        ]

        if sequence is not None:
            matches = matches.loc[
                matches["stop_sequence"] == sequence
            ]

        if time_seconds is not None:
            matches = matches.loc[
                (matches["arrival_seconds"] == time_seconds)
                | (matches["departure_seconds"] == time_seconds)
            ]

        if matches.empty:
            raise ValueError(
                f"Stop event not found: {trip_id}, {stop_id}"
            )

        return matches.iloc[0]

    def display_time(seconds):
        return format_transport_time(seconds)[:5]

    # First leg: origin departure
    first_origin = daily.loc[
        (daily["trip_id"] == journey["first_trip_id"])
        & (
            daily["departure_seconds"]
            == journey["origin_departure_seconds"]
        )
        & (
            daily["station_id"].astype(str).isin(
                station_lookup.loc[
                    station_lookup["stop_name"] == origin,
                    "station_id"
                ].astype(str)
            )
        )
    ]

    if first_origin.empty:
        raise ValueError("Origin departure event not found.")

    origin_stop = first_origin.iloc[0]

    # First transfer arrival: infer using first trip and second-leg departure
    first_transfer_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"]
            == journey["first_transfer_station"],
            "station_id"
        ].astype(str)
    )

    first_arrivals = daily.loc[
        (daily["trip_id"] == journey["first_trip_id"])
        & (daily["station_id"].astype(str).isin(first_transfer_ids))
        & (
            daily["arrival_seconds"]
            <= journey["second_departure_seconds"]
        )
    ].sort_values("arrival_seconds")

    if first_arrivals.empty:
        raise ValueError("First transfer arrival not found.")

    first_arrival = first_arrivals.iloc[-1]

    # Second leg boarding
    second_boarding = daily.loc[
        (daily["trip_id"] == journey["second_trip_id"])
        & (
            daily["stop_sequence"]
            == journey["second_boarding_sequence"]
        )
        & (
            daily["departure_seconds"]
            == journey["second_departure_seconds"]
        )
    ]

    if second_boarding.empty:
        raise ValueError("Second-leg boarding event not found.")

    second_boarding = second_boarding.iloc[0]

    # Second transfer arrival
    second_arrival = event(
        journey["second_trip_id"],
        journey["second_arrival_platform"],
        sequence=journey["second_arrival_sequence"],
        time_seconds=journey["second_arrival_seconds"]
    )

    # Third leg boarding
    third_boarding = event(
        journey["third_trip_id"],
        journey["third_departure_platform"],
        sequence=journey["third_boarding_sequence"],
        time_seconds=journey["third_departure_seconds"]
    )

    # Destination arrival
    destination_stop = event(
        journey["third_trip_id"],
        journey["destination_platform"],
        sequence=journey["destination_sequence"],
        time_seconds=journey["destination_arrival_seconds"]
    )

    stages = [
        ("Depart", origin, origin_stop, "departure_seconds"),
        ("Transfer 1 arrival", journey["first_transfer_station"],
         first_arrival, "arrival_seconds"),
        ("Transfer 1 departure", journey["first_transfer_station"],
         second_boarding, "departure_seconds"),
        ("Transfer 2 arrival", journey["second_transfer_station"],
         second_arrival, "arrival_seconds"),
        ("Transfer 2 departure", journey["second_transfer_station"],
         third_boarding, "departure_seconds"),
        ("Arrive", destination, destination_stop, "arrival_seconds")
    ]

    return pd.DataFrame([
        {
            "stage": stage,
            "station": station,
            "time": display_time(stop[time_col]),
            "platform": get_platform(stop["stop_id"]),
            "trip_id": stop["trip_id"]
        }
        for stage, station, stop, time_col in stages
    ])


# Test earliest-arriving two-transfer recommendation
two_transfer_itinerary = format_two_transfer_itinerary(
    journey=recommended_two_transfers.iloc[0],
    daily=realtime_daily,
    origin=origin,
    destination=destination
)

print("TWO-TRANSFER PASSENGER ITINERARY")
print("=" * 50)
display(two_transfer_itinerary)


TWO-TRANSFER PASSENGER ITINERARY


,stage,station,time,platform,trip_id
0,Depart,Macquarie University Station,17:30,Platform 2,0160-006-154-013:1000
1,Transfer 1 arrival,Epping Station,17:33,Platform 6,0160-006-154-013:1000
2,Transfer 1 departure,Epping Station,17:42,Platform 1,125R.1324.165.16.A.8.92001037
3,Transfer 2 arrival,Strathfield Station,18:03,Platform 5,125R.1324.165.16.A.8.92001037
4,Transfer 2 departure,Strathfield Station,18:11,Platform 6,197F.1324.165.16.B.8.92001727
5,Arrive,Parramatta Station,18:23,Platform 2,197F.1324.165.16.B.8.92001727


In [235]:

# STEP 91Q — Add service names to two-transfer itineraries

def build_trip_service_lookup(daily):
    required = ["trip_id", "route_short_name", "transport_mode"]

    missing = [col for col in required if col not in daily.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    services = (
        daily[required]
        .drop_duplicates(subset=["trip_id"])
        .set_index("trip_id")
    )

    def format_service(row):
        line = row["route_short_name"]
        mode = row["transport_mode"]

        line = str(line).strip() if pd.notna(line) else ""
        mode = str(mode).strip() if pd.notna(mode) else ""

        return " ".join(
            part for part in [line, mode] if part
        ) or "Service unavailable"

    return services.apply(format_service, axis=1).to_dict()


# Create lookup
trip_service_lookup = build_trip_service_lookup(realtime_daily)


def get_trip_service(trip_id):
    return trip_service_lookup.get(
        trip_id,
        "Service unavailable"
    )


# Add services to the existing itinerary
two_transfer_itinerary["service"] = (
    two_transfer_itinerary["trip_id"].apply(get_trip_service)
)

print("COMPLETE TWO-TRANSFER ITINERARY")
print("=" * 50)

display(
    two_transfer_itinerary[
        ["stage", "station", "time", "platform", "service"]
    ]
)


COMPLETE TWO-TRANSFER ITINERARY


,stage,station,time,platform,service
0,Depart,Macquarie University Station,17:30,Platform 2,M1 Metro
1,Transfer 1 arrival,Epping Station,17:33,Platform 6,M1 Metro
2,Transfer 1 departure,Epping Station,17:42,Platform 1,T9 Train
3,Transfer 2 arrival,Strathfield Station,18:03,Platform 5,T9 Train
4,Transfer 2 departure,Strathfield Station,18:11,Platform 6,T1 Train
5,Arrive,Parramatta Station,18:23,Platform 2,T1 Train


In [236]:

# STEP 91R — Build and test direct-journey itinerary formatter

import pandas as pd

def format_direct_itinerary(journey, daily, origin, destination):
    trip = daily.loc[
        daily["trip_id"] == journey["trip_id"]
    ].sort_values("stop_sequence")

    origin_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == origin,
            "station_id"
        ].astype(str)
    )

    destination_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == destination,
            "station_id"
        ].astype(str)
    )

    # Match the exact departure event
    departure_seconds = journey.get(
        "effective_departure_sec",
        journey.get("departure_seconds")
    )

    origin_events = trip.loc[
        trip["station_id"].astype(str).isin(origin_ids)
        & (trip["departure_seconds"] == departure_seconds)
    ]

    if origin_events.empty:
        raise ValueError("Direct journey origin event not found.")

    origin_event = origin_events.iloc[0]

    # Match the destination after boarding
    destination_events = trip.loc[
        trip["station_id"].astype(str).isin(destination_ids)
        & (trip["stop_sequence"] > origin_event["stop_sequence"])
        & (
            trip["arrival_seconds"]
            == journey["destination_arrival_seconds"]
        )
    ]

    if destination_events.empty:
        raise ValueError("Direct journey destination event not found.")

    destination_event = destination_events.iloc[0]

    def display_time(seconds):
        return format_transport_time(seconds)[:5]

    service = get_trip_service(journey["trip_id"])

    return pd.DataFrame([
        {
            "stage": "Depart",
            "station": origin,
            "time": display_time(origin_event["departure_seconds"]),
            "platform": get_platform(origin_event["stop_id"]),
            "service": service
        },
        {
            "stage": "Arrive",
            "station": destination,
            "time": display_time(destination_event["arrival_seconds"]),
            "platform": get_platform(destination_event["stop_id"]),
            "service": service
        }
    ])


# Find an actual direct journey in the current timetable
# Use the already validated M1 journey:
# Macquarie University → Epping

direct_origin = "Macquarie University Station"
direct_destination = "Epping Station"

direct_example = find_direct_journeys(
    realtime_daily,
    direct_origin,
    direct_destination,
    "17:30:00",
    window_minutes=60
)

print("Direct journeys found:", len(direct_example))

if not direct_example.empty:
    direct_itinerary = format_direct_itinerary(
        journey=direct_example.iloc[0],
        daily=realtime_daily,
        origin=direct_origin,
        destination=direct_destination
    )

    print("\nDIRECT PASSENGER ITINERARY")
    print("=" * 50)
    display(direct_itinerary)
else:
    print("No direct journeys found for this test.")


Direct journeys found: 15

DIRECT PASSENGER ITINERARY


,stage,station,time,platform,service
0,Depart,Macquarie University Station,17:30,Platform 2,M1 Metro
1,Arrive,Epping Station,17:33,Platform 6,M1 Metro


In [237]:

# STEP 91S — Universal passenger itinerary formatter

import pandas as pd

def format_journey_itinerary(
    journey,
    daily,
    origin,
    destination
):
    # Accept a pandas Series or a dictionary
    if isinstance(journey, dict):
        journey = pd.Series(journey)

    # Determine journey type
    if "transfers" in journey.index and pd.notna(journey["transfers"]):
        transfers = int(journey["transfers"])
    elif "third_trip_id" in journey.index and pd.notna(journey["third_trip_id"]):
        transfers = 2
    elif "second_trip_id" in journey.index and pd.notna(journey["second_trip_id"]):
        transfers = 1
    else:
        transfers = 0

    # Direct journey
    if transfers == 0:
        itinerary = format_direct_itinerary(
            journey,
            daily,
            origin,
            destination
        )

    # One-transfer journey
    elif transfers == 1:
        itinerary = format_one_transfer_itinerary(
            journey,
            daily,
            origin,
            destination
        )

    # Two-transfer journey
    elif transfers == 2:
        itinerary = format_two_transfer_itinerary(
            journey,
            daily,
            origin,
            destination
        )

        # Add passenger-facing service names
        itinerary["service"] = itinerary["trip_id"].apply(
            get_trip_service
        )

    else:
        raise ValueError(
            f"Unsupported number of transfers: {transfers}"
        )

    # Return a consistent schema for all journey types
    return itinerary[
        ["stage", "station", "time", "platform", "service"]
    ].reset_index(drop=True)


# TEST 1 — Direct
direct_formatted = format_journey_itinerary(
    direct_example.iloc[0],
    realtime_daily,
    direct_origin,
    direct_destination
)

# TEST 2 — One transfer
one_formatted = format_journey_itinerary(
    journey,
    realtime_daily,
    origin,
    destination
)

# TEST 3 — Two transfers
two_formatted = format_journey_itinerary(
    recommended_two_transfers.iloc[0],
    realtime_daily,
    origin,
    destination
)

print("DIRECT JOURNEY")
display(direct_formatted)

print("\nONE-TRANSFER JOURNEY")
display(one_formatted)

print("\nTWO-TRANSFER JOURNEY")
display(two_formatted)

print("\nVALIDATION")
print("Direct events:", len(direct_formatted))
print("One-transfer events:", len(one_formatted))
print("Two-transfer events:", len(two_formatted))

assert len(direct_formatted) == 2
assert len(one_formatted) == 4
assert len(two_formatted) == 6

print("PASS — All three journey types use the same itinerary format!")


DIRECT JOURNEY


,stage,station,time,platform,service
0,Depart,Macquarie University Station,17:30,Platform 2,M1 Metro
1,Arrive,Epping Station,17:33,Platform 6,M1 Metro



ONE-TRANSFER JOURNEY


,stage,station,time,platform,service
0,Depart,Macquarie University Station,17:31,Platform 1,M1 Metro
1,Transfer arrival,Central Station,17:56,Platform 27,M1 Metro
2,Transfer departure,Central Station,18:04,Platform 18,T1 Train
3,Arrive,Parramatta Station,18:29,Platform 2,T1 Train



TWO-TRANSFER JOURNEY


,stage,station,time,platform,service
0,Depart,Macquarie University Station,17:30,Platform 2,M1 Metro
1,Transfer 1 arrival,Epping Station,17:33,Platform 6,M1 Metro
2,Transfer 1 departure,Epping Station,17:42,Platform 1,T9 Train
3,Transfer 2 arrival,Strathfield Station,18:03,Platform 5,T9 Train
4,Transfer 2 departure,Strathfield Station,18:11,Platform 6,T1 Train
5,Arrive,Parramatta Station,18:23,Platform 2,T1 Train



VALIDATION
Direct events: 2
One-transfer events: 4
Two-transfer events: 6
PASS — All three journey types use the same itinerary format!


In [238]:

# STEP 91T — Connect recommended journeys to full itineraries

import pandas as pd

def get_recommended_itinerary(
    recommendation,
    daily,
    origin,
    destination,
    direct_results,
    one_transfer_results,
    two_transfer_results
):
    transfers = int(recommendation["transfers"])

    if transfers == 0:
        candidates = direct_results.loc[
            (direct_results["trip_id"] == recommendation["first_trip_id"])
            & (
                direct_results["destination_arrival_seconds"]
                == recommendation["destination_arrival_seconds"]
            )
        ].copy()

        if "departure_seconds" in candidates.columns:
            candidates = candidates.loc[
                candidates["departure_seconds"]
                == recommendation["origin_departure_seconds"]
            ]

    elif transfers == 1:
        candidates = one_transfer_results.loc[
            (one_transfer_results["trip_id"] == recommendation["first_trip_id"])
            & (
                one_transfer_results["second_trip_id"]
                == recommendation["second_trip_id"]
            )
            & (
                one_transfer_results["origin_departure_seconds"]
                == recommendation["origin_departure_seconds"]
            )
            & (
                one_transfer_results["destination_arrival_seconds"]
                == recommendation["destination_arrival_seconds"]
            )
        ].copy()

        # Match the selected transfer station
        candidates = candidates.loc[
            candidates["transfer_station"]
            == recommendation["first_transfer_station"]
        ]

    elif transfers == 2:
        candidates = two_transfer_results.loc[
            (two_transfer_results["first_trip_id"] == recommendation["first_trip_id"])
            & (
                two_transfer_results["second_trip_id"]
                == recommendation["second_trip_id"]
            )
            & (
                two_transfer_results["third_trip_id"]
                == recommendation["third_trip_id"]
            )
            & (
                two_transfer_results["origin_departure_seconds"]
                == recommendation["origin_departure_seconds"]
            )
            & (
                two_transfer_results["destination_arrival_seconds"]
                == recommendation["destination_arrival_seconds"]
            )
        ].copy()

        candidates = candidates.loc[
            (candidates["first_transfer_station"]
             == recommendation["first_transfer_station"])
            & (candidates["second_transfer_station"]
               == recommendation["second_transfer_station"])
        ]

    else:
        raise ValueError(f"Unsupported transfers: {transfers}")

    if candidates.empty:
        raise ValueError("No matching original journey found.")

    # Preserve a valid original record with full platform details.
    # If multiple distinct records remain, do not silently guess.
    if len(candidates) > 1:
        print(
            f"Note: {len(candidates)} original records match "
            "this recommendation."
        )

    original_journey = candidates.iloc[0]

    return format_journey_itinerary(
        original_journey,
        daily,
        origin,
        destination
    )


# Test the first two recommendations
for i in range(min(2, len(best_journeys))):
    recommendation = best_journeys.iloc[i]

    print(f"\nRECOMMENDATION {i + 1}")
    print(
        recommendation["departure_time"],
        "→",
        recommendation["arrival_time"],
        "|",
        recommendation["journey_type"]
    )

    itinerary = get_recommended_itinerary(
        recommendation=recommendation,
        daily=realtime_daily,
        origin=origin,
        destination=destination,
        direct_results=direct_test,
        one_transfer_results=one_transfer_test,
        two_transfer_results=two_transfer_test
    )

    display(itinerary)



RECOMMENDATION 1
17:30:37 → 18:23:00 | Two transfers


,stage,station,time,platform,service
0,Depart,Macquarie University Station,17:30,Platform 2,M1 Metro
1,Transfer 1 arrival,Epping Station,17:33,Platform 6,M1 Metro
2,Transfer 1 departure,Epping Station,17:42,Platform 1,T9 Train
3,Transfer 2 arrival,Strathfield Station,18:03,Platform 5,T9 Train
4,Transfer 2 departure,Strathfield Station,18:11,Platform 6,T1 Train
5,Arrive,Parramatta Station,18:23,Platform 2,T1 Train



RECOMMENDATION 2
17:31:54 → 18:29:00 | One transfer
Note: 19 original records match this recommendation.


,stage,station,time,platform,service
0,Depart,Macquarie University Station,17:31,Platform 1,M1 Metro
1,Transfer arrival,Central Station,17:56,Platform 27,M1 Metro
2,Transfer departure,Central Station,18:04,Platform 18,T1 Train
3,Arrive,Parramatta Station,18:29,Platform 2,T1 Train


In [239]:

# STEP 91T — Select one recommended journey

def recommend_best_journey(journeys, arrival_tolerance_minutes=5):
    if journeys.empty:
        return None

    routes = journeys.copy()

    # Find the earliest possible arrival
    earliest_arrival = routes[
        "destination_arrival_seconds"
    ].min()

    # Keep journeys arriving within 5 minutes of the earliest
    acceptable = routes.loc[
        routes["destination_arrival_seconds"]
        <= earliest_arrival + arrival_tolerance_minutes * 60
    ].copy()

    # Prefer fewer transfers, then earliest arrival,
    # then shortest journey duration
    acceptable = acceptable.sort_values(
        [
            "transfers",
            "destination_arrival_seconds",
            "journey_duration_min"
        ],
        ascending=[True, True, True]
    )

    return acceptable.iloc[0]


best_journey = recommend_best_journey(
    all_journeys,
    arrival_tolerance_minutes=5
)

print("RECOMMENDED JOURNEY")
print("=" * 45)

if best_journey is not None:
    display(
        best_journey[
            [
                "journey_type",
                "departure_time",
                "arrival_time",
                "journey_duration_min",
                "transfers",
                "first_transfer_station",
                "second_transfer_station"
            ]
        ].to_frame(name="Value")
    )
else:
    print("No journey found.")


RECOMMENDED JOURNEY


,Value
journey_type,Two transfers
departure_time,17:30:37
arrival_time,18:23:00
journey_duration_min,52.383333
transfers,2
first_transfer_station,Epping Station
second_transfer_station,Strathfield Station


In [240]:

# STEP 91U — Display the single recommended journey

def get_best_journey_itinerary(
    recommendation,
    daily,
    origin,
    destination,
    direct_results,
    one_transfer_results,
    two_transfer_results
):
    transfers = int(recommendation["transfers"])

    if transfers == 0:
        candidates = direct_results.loc[
            (direct_results["trip_id"] == recommendation["first_trip_id"])
            & (direct_results["departure_seconds"] == recommendation["origin_departure_seconds"])
            & (direct_results["destination_arrival_seconds"] == recommendation["destination_arrival_seconds"])
        ]

    elif transfers == 1:
        candidates = one_transfer_results.loc[
            (one_transfer_results["trip_id"] == recommendation["first_trip_id"])
            & (one_transfer_results["second_trip_id"] == recommendation["second_trip_id"])
            & (one_transfer_results["origin_departure_seconds"] == recommendation["origin_departure_seconds"])
            & (one_transfer_results["destination_arrival_seconds"] == recommendation["destination_arrival_seconds"])
            & (one_transfer_results["transfer_station"] == recommendation["first_transfer_station"])
        ]

    elif transfers == 2:
        candidates = two_transfer_results.loc[
            (two_transfer_results["first_trip_id"] == recommendation["first_trip_id"])
            & (two_transfer_results["second_trip_id"] == recommendation["second_trip_id"])
            & (two_transfer_results["third_trip_id"] == recommendation["third_trip_id"])
            & (two_transfer_results["origin_departure_seconds"] == recommendation["origin_departure_seconds"])
            & (two_transfer_results["destination_arrival_seconds"] == recommendation["destination_arrival_seconds"])
            & (two_transfer_results["first_transfer_station"] == recommendation["first_transfer_station"])
            & (two_transfer_results["second_transfer_station"] == recommendation["second_transfer_station"])
        ]

    else:
        raise ValueError("Unsupported journey type.")

    if candidates.empty:
        raise ValueError("Could not find the original journey record.")

    # Avoid silently selecting between distinct platform itineraries
    if len(candidates) > 1:
        raise ValueError(
            f"{len(candidates)} records match this recommendation. "
            "Additional boarding/platform identifiers are needed."
        )

    return format_journey_itinerary(
        candidates.iloc[0],
        daily,
        origin,
        destination
    )


# Generate the recommended passenger itinerary
try:
    final_itinerary = get_best_journey_itinerary(
        recommendation=best_journey,
        daily=realtime_daily,
        origin=origin,
        destination=destination,
        direct_results=direct_test,
        one_transfer_results=one_transfer_test,
        two_transfer_results=two_transfer_test
    )

    print("BEST JOURNEY — COMPLETE ITINERARY")
    display(final_itinerary)

except ValueError as error:
    print("Matching issue:", error)


BEST JOURNEY — COMPLETE ITINERARY


,stage,station,time,platform,service
0,Depart,Macquarie University Station,17:30,Platform 2,M1 Metro
1,Transfer 1 arrival,Epping Station,17:33,Platform 6,M1 Metro
2,Transfer 1 departure,Epping Station,17:42,Platform 1,T9 Train
3,Transfer 2 arrival,Strathfield Station,18:03,Platform 5,T9 Train
4,Transfer 2 departure,Strathfield Station,18:11,Platform 6,T1 Train
5,Arrive,Parramatta Station,18:23,Platform 2,T1 Train


In [241]:

# STEP 91V — Complete journey planning pipeline

def plan_journey(
    origin,
    destination,
    departure_time,
    daily=realtime_daily,
    arrival_tolerance_minutes=5
):
    # 1. Search for direct journeys
    direct = find_direct_journeys(
        daily,
        origin,
        destination,
        departure_time,
        window_minutes=60
    )

    # 2. Search for one-transfer journeys
    one_transfer = find_one_transfer_journeys(
        daily,
        origin,
        destination,
        departure_time,
        window_minutes=60,
        connection_window_minutes=45
    )

    # 3. Search for two-transfer journeys
    two_transfer = find_two_transfer_journeys(
        daily,
        origin,
        destination,
        departure_time,
        window_minutes=60,
        connection_window_minutes=45
    )

    # 4. Standardise results
    all_results = standardise_journey_results(
        direct,
        one_transfer,
        two_transfer
    )

    if all_results.empty:
        return None

    # 5. Select one recommended journey
    best = recommend_best_journey(
        all_results,
        arrival_tolerance_minutes=arrival_tolerance_minutes
    )

    # 6. Retrieve complete passenger itinerary
    itinerary = get_best_journey_itinerary(
        recommendation=best,
        daily=daily,
        origin=origin,
        destination=destination,
        direct_results=direct,
        one_transfer_results=one_transfer,
        two_transfer_results=two_transfer
    )

    # 7. Return summary and full itinerary
    return {
        "origin": origin,
        "destination": destination,
        "departure_time": best["departure_time"][:5],
        "arrival_time": best["arrival_time"][:5],
        "duration_minutes": round(best["journey_duration_min"]),
        "transfers": int(best["transfers"]),
        "journey_type": best["journey_type"],
        "itinerary": itinerary
    }


# Test the complete pipeline
result = plan_journey(
    origin="Macquarie University Station",
    destination="Parramatta Station",
    departure_time="17:30:00"
)

if result is None:
    print("No journey found.")
else:
    print("RECOMMENDED JOURNEY")
    print("=" * 45)
    print("From:", result["origin"])
    print("To:", result["destination"])
    print("Departure:", result["departure_time"])
    print("Arrival:", result["arrival_time"])
    print("Duration:", result["duration_minutes"], "minutes")
    print("Transfers:", result["transfers"])

    print("\nCOMPLETE ITINERARY")
    display(result["itinerary"])


RECOMMENDED JOURNEY
From: Macquarie University Station
To: Parramatta Station
Departure: 17:30
Arrival: 18:23
Duration: 52 minutes
Transfers: 2

COMPLETE ITINERARY


,stage,station,time,platform,service
0,Depart,Macquarie University Station,17:30,Platform 2,M1 Metro
1,Transfer 1 arrival,Epping Station,17:33,Platform 6,M1 Metro
2,Transfer 1 departure,Epping Station,17:42,Platform 1,T9 Train
3,Transfer 2 arrival,Strathfield Station,18:03,Platform 5,T9 Train
4,Transfer 2 departure,Strathfield Station,18:11,Platform 6,T1 Train
5,Arrive,Parramatta Station,18:23,Platform 2,T1 Train


In [242]:
result = plan_journey(
    origin="Central Station",
    destination="Chatswood Station",
    departure_time="09:00:00"
)

KeyboardInterrupt: 

In [243]:

# STEP 92A — Profile the two-transfer routing pipeline

import time
import pandas as pd

def benchmark_stage(name, func, *args, **kwargs):
    start = time.perf_counter()
    result = func(*args, **kwargs)
    elapsed = time.perf_counter() - start

    return result, {
        "stage": name,
        "seconds": round(elapsed, 3),
        "rows": len(result)
    }


metrics = []

# Stage 1
first, metric = benchmark_stage(
    "First-leg stops",
    build_first_leg_stops,
    realtime_daily,
    origin,
    departure_time,
    window_minutes=60
)
metrics.append(metric)

# Stage 2
transfers_1, metric = benchmark_stage(
    "First transfers",
    build_first_transfer_options,
    first,
    transfer_rules=all_platform_transfers
)
metrics.append(metric)

# Stage 3
second, metric = benchmark_stage(
    "Second-leg candidates",
    build_second_leg_candidates,
    realtime_daily,
    transfers_1,
    connection_window_minutes=45
)
metrics.append(metric)

# Stage 4
second_stops, metric = benchmark_stage(
    "Second-leg downstream stops",
    build_second_leg_stops,
    realtime_daily,
    second,
    origin,
    destination,
    station_names
)
metrics.append(metric)

# Stage 5
transfers_2, metric = benchmark_stage(
    "Second transfers",
    build_third_transfer_options,
    second_stops,
    platform_rules
)
metrics.append(metric)

# Stage 6
third, metric = benchmark_stage(
    "Third-leg candidates",
    build_third_leg_candidates,
    realtime_daily,
    transfers_2,
    connection_window_minutes=45
)
metrics.append(metric)

# Stage 7
complete, metric = benchmark_stage(
    "Complete journeys",
    build_complete_two_transfer_journeys,
    realtime_daily,
    third,
    destination,
    station_lookup
)
metrics.append(metric)

# Display benchmark results
performance = pd.DataFrame(metrics)

performance["percentage"] = (
    performance["seconds"]
    / performance["seconds"].sum()
    * 100
).round(1)

print("TOTAL ROUTING TIME:",
      round(performance["seconds"].sum(), 2),
      "seconds")

display(
    performance.sort_values(
        "seconds",
        ascending=False
    ).reset_index(drop=True)
)


TOTAL ROUTING TIME: 8.91 seconds


,stage,seconds,rows,percentage
0,Third-leg candidates,8.582,1515898,96.3
1,Complete journeys,0.138,55746,1.5
2,First-leg stops,0.074,300,0.8
3,Second-leg candidates,0.049,7319,0.5
4,Second-leg downstream stops,0.047,73281,0.5
5,Second transfers,0.016,288450,0.2
6,First transfers,0.005,1170,0.1


In [244]:

# STEP 92B — Measure potential third-leg pruning

import pandas as pd

# Earliest arrival from our validated results
best_arrival = all_journeys[
    "destination_arrival_seconds"
].min()

tolerance_seconds = 5 * 60
cutoff = best_arrival + tolerance_seconds

# Use the existing second-transfer options
original_options = len(transfers_2)

filtered_options = transfers_2.loc[
    transfers_2["second_arrival_seconds"] < cutoff
].copy()

print("THIRD-LEG SEARCH SPACE")
print("=" * 45)

print("Original transfer options:", original_options)
print("Filtered transfer options:", len(filtered_options))

reduction = (
    1 - len(filtered_options) / original_options
) * 100 if original_options else 0

print(f"Potential reduction: {reduction:.1f}%")

print("\nARRIVAL CUTOFF")
print("Best known arrival:", format_transport_time(best_arrival)[:5])
print("Cutoff:", format_transport_time(cutoff)[:5])

print("\nFILTERED TRANSFER STATIONS")
display(
    filtered_options[
        "second_transfer_station"
    ].value_counts().head(10)
)


THIRD-LEG SEARCH SPACE
Original transfer options: 288450
Filtered transfer options: 26531
Potential reduction: 90.8%

ARRIVAL CUTOFF
Best known arrival: 18:23
Cutoff: 18:28

FILTERED TRANSFER STATIONS


second_transfer_station
Central Station           7192
Epping Station            1785
Chatswood Station         1280
Martin Place Station      1124
Redfern Station           1116
Sydenham Station           996
Town Hall Station          808
North Ryde Station         730
Cherrybrook Station        730
Victoria Cross Station     716
Name: count, dtype: int64

In [245]:

# STEP 92C — Benchmark third-leg pruning

import time
import pandas as pd

start = time.perf_counter()

# Run third-leg search on the reduced transfer options
third_filtered = build_third_leg_candidates(
    realtime_daily,
    filtered_options,
    connection_window_minutes=45
)

filtered_time = time.perf_counter() - start

# Find complete journeys from the reduced candidates
complete_filtered = build_complete_two_transfer_journeys(
    realtime_daily,
    third_filtered,
    destination,
    station_lookup
)

print("THIRD-LEG PERFORMANCE")
print("=" * 45)

print("Original third-leg candidates:", len(third))
print("Filtered third-leg candidates:", len(third_filtered))

print("\nOriginal runtime:", round(
    performance.loc[
        performance["stage"] == "Third-leg candidates",
        "seconds"
    ].iloc[0], 3
), "seconds")

print("Filtered runtime:", round(filtered_time, 3), "seconds")

print("\nCOMPLETE JOURNEYS")
print("Original:", len(complete))
print("Filtered:", len(complete_filtered))

# Verify the earliest arrival is preserved
original_best_arrival = complete[
    "destination_arrival_seconds"
].min()

filtered_best_arrival = complete_filtered[
    "destination_arrival_seconds"
].min()

print("\nEARLIEST ARRIVAL VALIDATION")
print("Original:", format_transport_time(original_best_arrival)[:5])
print("Filtered:", format_transport_time(filtered_best_arrival)[:5])

assert original_best_arrival == filtered_best_arrival, (
    "Filtering changed the earliest arrival!"
)

print("\nPASS — Earliest arrival preserved.")

# STEP 92C — Benchmark third-leg pruning

import time
import pandas as pd

start = time.perf_counter()

# Run third-leg search on the reduced transfer options
third_filtered = build_third_leg_candidates(
    realtime_daily,
    filtered_options,
    connection_window_minutes=45
)

filtered_time = time.perf_counter() - start

# Find complete journeys from the reduced candidates
complete_filtered = build_complete_two_transfer_journeys(
    realtime_daily,
    third_filtered,
    destination,
    station_lookup
)

print("THIRD-LEG PERFORMANCE")
print("=" * 45)

print("Original third-leg candidates:", len(third))
print("Filtered third-leg candidates:", len(third_filtered))

print("\nOriginal runtime:", round(
    performance.loc[
        performance["stage"] == "Third-leg candidates",
        "seconds"
    ].iloc[0], 3
), "seconds")

print("Filtered runtime:", round(filtered_time, 3), "seconds")

print("\nCOMPLETE JOURNEYS")
print("Original:", len(complete))
print("Filtered:", len(complete_filtered))

# Verify the earliest arrival is preserved
original_best_arrival = complete[
    "destination_arrival_seconds"
].min()

filtered_best_arrival = complete_filtered[
    "destination_arrival_seconds"
].min()

print("\nEARLIEST ARRIVAL VALIDATION")
print("Original:", format_transport_time(original_best_arrival)[:5])
print("Filtered:", format_transport_time(filtered_best_arrival)[:5])

assert original_best_arrival == filtered_best_arrival, (
    "Filtering changed the earliest arrival!"
)

print("\nPASS — Earliest arrival preserved.")


THIRD-LEG PERFORMANCE
Original third-leg candidates: 1515898
Filtered third-leg candidates: 184277

Original runtime: 8.582 seconds
Filtered runtime: 0.333 seconds

COMPLETE JOURNEYS
Original: 55746
Filtered: 4443

EARLIEST ARRIVAL VALIDATION
Original: 18:23
Filtered: 18:23

PASS — Earliest arrival preserved.
THIRD-LEG PERFORMANCE
Original third-leg candidates: 1515898
Filtered third-leg candidates: 184277

Original runtime: 8.582 seconds
Filtered runtime: 0.235 seconds

COMPLETE JOURNEYS
Original: 55746
Filtered: 4443

EARLIEST ARRIVAL VALIDATION
Original: 18:23
Filtered: 18:23

PASS — Earliest arrival preserved.


In [246]:

# STEP 92D — Dynamic cutoff from simpler journeys

import pandas as pd

# Find the earliest arrival from direct and one-transfer journeys
direct_results = find_direct_journeys(
    realtime_daily, origin, destination, departure_time
)

one_transfer_results = find_one_transfer_journeys(
    realtime_daily, origin, destination, departure_time
)

available = [
    df["destination_arrival_seconds"].min()
    for df in [direct_results, one_transfer_results]
    if not df.empty
]

if available:
    cutoff = min(available)

    dynamic_filtered_options = transfers_2.loc[
        transfers_2["second_arrival_seconds"] < cutoff
    ].copy()

    print("Best simpler-journey arrival:",
          format_transport_time(cutoff)[:5])

    print("Original transfer options:", len(transfers_2))
    print("Filtered transfer options:", len(dynamic_filtered_options))

    print(
        "Reduction:",
        f"{(1 - len(dynamic_filtered_options) / len(transfers_2)) * 100:.1f}%"
    )
else:
    print("No direct or one-transfer journey found.")
    print("We need a different strategy for this route.")


Best simpler-journey arrival: 18:29
Original transfer options: 288450
Filtered transfer options: 29992
Reduction: 89.6%


In [247]:

# STEP 92E — Validate dynamic filtering

import time

start = time.perf_counter()

# Search only transfer options that can beat the simpler journey
third_dynamic = build_third_leg_candidates(
    realtime_daily,
    dynamic_filtered_options,
    connection_window_minutes=45
)

complete_dynamic = build_complete_two_transfer_journeys(
    realtime_daily,
    third_dynamic,
    destination,
    station_lookup
)

elapsed = time.perf_counter() - start

print("DYNAMIC ROUTING PERFORMANCE")
print("=" * 45)
print(f"Runtime: {elapsed:.3f} seconds")
print("Third-leg candidates:", len(third_dynamic))
print("Complete journeys:", len(complete_dynamic))

if not complete_dynamic.empty:
    original_arrival = complete[
        "destination_arrival_seconds"
    ].min()

    dynamic_arrival = complete_dynamic[
        "destination_arrival_seconds"
    ].min()

    print("\nEARLIEST ARRIVAL")
    print("Original:", format_transport_time(original_arrival)[:5])
    print("Dynamic:", format_transport_time(dynamic_arrival)[:5])

    assert original_arrival == dynamic_arrival, (
        "Dynamic filtering changed the earliest arrival!"
    )

    print("\nPASS — Earliest arrival preserved.")
else:
    print("\nNo faster two-transfer journey found.")
    print("The simpler journey remains the current candidate.")


DYNAMIC ROUTING PERFORMANCE
Runtime: 0.409 seconds
Third-leg candidates: 204102
Complete journeys: 5218

EARLIEST ARRIVAL
Original: 18:23
Dynamic: 18:23

PASS — Earliest arrival preserved.


In [248]:

# STEP 92F — Optimised two-transfer routing

def find_two_transfer_journeys_fast(
    daily,
    origin,
    destination,
    departure_time,
    arrival_cutoff_seconds=None,
    window_minutes=60,
    connection_window_minutes=45
):
    first = build_first_leg_stops(
        daily, origin, departure_time,
        window_minutes=window_minutes
    )

    transfers_1 = build_first_transfer_options(
        first,
        transfer_rules=all_platform_transfers
    )

    second = build_second_leg_candidates(
        daily,
        transfers_1,
        connection_window_minutes=connection_window_minutes
    )

    second_stops = build_second_leg_stops(
        daily,
        second,
        origin,
        destination,
        station_names
    )

    transfers_2 = build_third_transfer_options(
        second_stops,
        platform_rules
    )

    # Safe for finding journeys that arrive earlier
    # than the best simpler journey.
    if arrival_cutoff_seconds is not None:
        transfers_2 = transfers_2.loc[
            transfers_2["second_arrival_seconds"]
            < arrival_cutoff_seconds
        ].copy()

    if transfers_2.empty:
        return pd.DataFrame()

    third = build_third_leg_candidates(
        daily,
        transfers_2,
        connection_window_minutes=connection_window_minutes
    )

    if third.empty:
        return pd.DataFrame()

    return build_complete_two_transfer_journeys(
        daily,
        third,
        destination,
        station_lookup
    )


# Test using the best direct/one-transfer arrival
import time

start = time.perf_counter()

fast_results = find_two_transfer_journeys_fast(
    realtime_daily,
    origin,
    destination,
    departure_time,
    arrival_cutoff_seconds=cutoff
)

elapsed = time.perf_counter() - start

print(f"Optimised routing time: {elapsed:.3f} seconds")
print("Complete journeys:", len(fast_results))

if not fast_results.empty:
    print(
        "Earliest arrival:",
        format_transport_time(
            fast_results["destination_arrival_seconds"].min()
        )[:5]
    )


Optimised routing time: 0.515 seconds
Complete journeys: 5218
Earliest arrival: 18:23


In [249]:

# STEP 92G — Optimised complete journey planner

import time
import math
import pandas as pd

def plan_journey_fast(
    origin,
    destination,
    departure_time,
    daily=realtime_daily,
    arrival_tolerance_minutes=5
):
    # 1. Search direct journeys
    direct = find_direct_journeys(
        daily, origin, destination, departure_time,
        window_minutes=60
    )

    # 2. Search one-transfer journeys
    one_transfer = find_one_transfer_journeys(
        daily, origin, destination, departure_time,
        window_minutes=60,
        connection_window_minutes=45
    )

    # 3. Establish arrival cutoff from simpler journeys
    arrival_times = [
        df["destination_arrival_seconds"].min()
        for df in [direct, one_transfer]
        if not df.empty
    ]

    arrival_cutoff = min(arrival_times) if arrival_times else None

    # 4. Search two-transfer journeys with pruning
    two_transfer = find_two_transfer_journeys_fast(
        daily,
        origin,
        destination,
        departure_time,
        arrival_cutoff_seconds=arrival_cutoff,
        window_minutes=60,
        connection_window_minutes=45
    )

    # 5. Combine all journey types
    all_results = standardise_journey_results(
        direct,
        one_transfer,
        two_transfer
    )

    if all_results.empty:
        return None

    # 6. Recommend the best journey
    best = recommend_best_journey(
        all_results,
        arrival_tolerance_minutes=arrival_tolerance_minutes
    )

    # 7. Build the passenger itinerary
    itinerary = get_best_journey_itinerary(
        best,
        daily,
        origin,
        destination,
        direct,
        one_transfer,
        two_transfer
    )

    return {
        "origin": origin,
        "destination": destination,
        "departure_time": best["departure_time"][:5],
        "arrival_time": best["arrival_time"][:5],
        "duration_minutes": math.ceil(best["journey_duration_min"]),
        "transfers": int(best["transfers"]),
        "journey_type": best["journey_type"],
        "itinerary": itinerary
    }


# Benchmark the complete workflow
start = time.perf_counter()

result_fast = plan_journey_fast(
    "Macquarie University Station",
    "Parramatta Station",
    "17:30:00"
)

elapsed = time.perf_counter() - start

print(f"TOTAL SEARCH TIME: {elapsed:.3f} seconds")

if result_fast is None:
    print("No journey found.")
else:
    print("\nRECOMMENDED JOURNEY")
    print("Departure:", result_fast["departure_time"])
    print("Arrival:", result_fast["arrival_time"])
    print("Duration:", result_fast["duration_minutes"], "minutes")
    print("Transfers:", result_fast["transfers"])

    display(result_fast["itinerary"])


TOTAL SEARCH TIME: 0.444 seconds

RECOMMENDED JOURNEY
Departure: 17:30
Arrival: 18:23
Duration: 53 minutes
Transfers: 2


,stage,station,time,platform,service
0,Depart,Macquarie University Station,17:30,Platform 2,M1 Metro
1,Transfer 1 arrival,Epping Station,17:33,Platform 6,M1 Metro
2,Transfer 1 departure,Epping Station,17:42,Platform 1,T9 Train
3,Transfer 2 arrival,Strathfield Station,18:03,Platform 5,T9 Train
4,Transfer 2 departure,Strathfield Station,18:11,Platform 6,T1 Train
5,Arrive,Parramatta Station,18:23,Platform 2,T1 Train


In [250]:

# STEP 92H — Test the optimised planner across different routes

import time
import pandas as pd

test_routes = [
    ("Macquarie University Station", "Epping Station", "17:30:00"),
    ("Macquarie University Station", "Parramatta Station", "17:30:00"),
    ("Chatswood Station", "Central Station", "17:30:00"),
    ("Central Station", "Parramatta Station", "17:30:00"),
]

results = []

for start_station, end_station, depart_at in test_routes:
    start = time.perf_counter()

    try:
        journey = plan_journey_fast(
            start_station,
            end_station,
            depart_at
        )

        elapsed = time.perf_counter() - start

        if journey is None:
            results.append({
                "origin": start_station,
                "destination": end_station,
                "status": "No journey",
                "transfers": None,
                "arrival": None,
                "runtime_seconds": round(elapsed, 3)
            })
        else:
            results.append({
                "origin": start_station,
                "destination": end_station,
                "status": "Success",
                "transfers": journey["transfers"],
                "arrival": journey["arrival_time"],
                "runtime_seconds": round(elapsed, 3)
            })

    except Exception as error:
        results.append({
            "origin": start_station,
            "destination": end_station,
            "status": f"ERROR: {str(error)[:100]}",
            "transfers": None,
            "arrival": None,
            "runtime_seconds": round(
                time.perf_counter() - start, 3
            )
        })

display(pd.DataFrame(results))


,origin,destination,status,transfers,arrival,runtime_seconds
0,Macquarie University Station,Epping Station,Success,0,17:33,0.138
1,Macquarie University Station,Parramatta Station,Success,2,18:23,0.434
2,Chatswood Station,Central Station,Success,0,17:48,0.151
3,Central Station,Parramatta Station,Success,0,17:59,0.446


In [251]:

# STEP 92I — Compare original and optimised recommendations

import pandas as pd

original_direct = find_direct_journeys(
    realtime_daily, origin, destination, departure_time
)

original_one = find_one_transfer_journeys(
    realtime_daily, origin, destination, departure_time
)

# Reuse previously calculated complete two-transfer journeys
original_two = complete

original_all = standardise_journey_results(
    original_direct,
    original_one,
    original_two
)

original_best = recommend_best_journey(
    original_all,
    arrival_tolerance_minutes=5
)

fast_best = result_fast

comparison = pd.DataFrame([
    {
        "version": "Original",
        "departure": original_best["departure_time"][:5],
        "arrival": original_best["arrival_time"][:5],
        "transfers": int(original_best["transfers"]),
        "duration_min": round(original_best["journey_duration_min"], 1)
    },
    {
        "version": "Optimised",
        "departure": fast_best["departure_time"],
        "arrival": fast_best["arrival_time"],
        "transfers": fast_best["transfers"],
        "duration_min": round(
            (original_best["journey_duration_min"])
            if (
                fast_best["departure_time"] == original_best["departure_time"][:5]
                and fast_best["arrival_time"] == original_best["arrival_time"][:5]
            )
            else fast_best["duration_minutes"],
            1
        )
    }
])

display(comparison)

same_recommendation = (
    comparison.loc[0, "departure"] == comparison.loc[1, "departure"]
    and comparison.loc[0, "arrival"] == comparison.loc[1, "arrival"]
    and comparison.loc[0, "transfers"] == comparison.loc[1, "transfers"]
)

print(
    "\nVALIDATION:",
    "PASS — Same recommendation"
    if same_recommendation
    else "DIFFERENCE — Needs investigation"
)


,version,departure,arrival,transfers,duration_min
0,Original,17:30,18:23,2,52.4
1,Optimised,17:30,18:23,2,52.4



VALIDATION: PASS — Same recommendation


In [252]:

# STEP 92J — Test one-transfer recommendations

import time
import pandas as pd

test_pairs = [
    ("Macquarie University Station", "Town Hall Station"),
    ("Macquarie University Station", "Redfern Station"),
    ("Epping Station", "Parramatta Station"),
    ("Chatswood Station", "Parramatta Station"),
    ("North Ryde Station", "Strathfield Station")
]

results = []
one_transfer_example = None

for start_station, end_station in test_pairs:
    start = time.perf_counter()

    try:
        journey = plan_journey_fast(
            start_station,
            end_station,
            "17:30:00"
        )

        elapsed = time.perf_counter() - start

        if journey is None:
            status = "No journey"
            transfers = None
        else:
            status = "Success"
            transfers = journey["transfers"]

            if transfers == 1 and one_transfer_example is None:
                one_transfer_example = journey

        results.append({
            "origin": start_station,
            "destination": end_station,
            "status": status,
            "transfers": transfers,
            "runtime_seconds": round(elapsed, 3)
        })

    except Exception as error:
        results.append({
            "origin": start_station,
            "destination": end_station,
            "status": str(error)[:100],
            "transfers": None,
            "runtime_seconds": round(
                time.perf_counter() - start, 3
            )
        })

display(pd.DataFrame(results))

if one_transfer_example is not None:
    print("\nONE-TRANSFER ITINERARY")
    display(one_transfer_example["itinerary"])
else:
    print("\nNo successful one-transfer recommendation found.")


,origin,destination,status,transfers,runtime_seconds
0,Macquarie University Station,Town Hall Station,19 records match this recommendation. Addition...,None,0.233
1,Macquarie University Station,Redfern Station,19 records match this recommendation. Addition...,None,0.141
2,Epping Station,Parramatta Station,8 records match this recommendation. Additiona...,None,0.216
3,Chatswood Station,Parramatta Station,19 records match this recommendation. Addition...,None,0.863
4,North Ryde Station,Strathfield Station,5 records match this recommendation. Additiona...,None,0.129



No successful one-transfer recommendation found.


In [253]:

# STEP 92K — Investigate ambiguous one-transfer journeys

import pandas as pd

test_origin = "Macquarie University Station"
test_destination = "Town Hall Station"
test_departure = "17:30:00"

direct_test = find_direct_journeys(
    realtime_daily, test_origin, test_destination, test_departure
)

one_test = find_one_transfer_journeys(
    realtime_daily, test_origin, test_destination, test_departure
)

two_test = find_two_transfer_journeys_fast(
    realtime_daily,
    test_origin,
    test_destination,
    test_departure,
    arrival_cutoff_seconds=min(
        [
            df["destination_arrival_seconds"].min()
            for df in [direct_test, one_test]
            if not df.empty
        ],
        default=None
    )
)

combined_test = standardise_journey_results(
    direct_test, one_test, two_test
)

best_test = recommend_best_journey(
    combined_test,
    arrival_tolerance_minutes=5
)

print("RECOMMENDED JOURNEY")
display(best_test.to_frame().T)

print("\nONE-TRANSFER DATAFRAME COLUMNS")
print(one_test.columns.tolist())

# Identify matching trip IDs and journey times
matches = one_test.loc[
    (one_test["first_trip_id"] == best_test["first_trip_id"])
    & (one_test["second_trip_id"] == best_test["second_trip_id"])
    & (
        one_test["origin_departure_seconds"]
        == best_test["origin_departure_seconds"]
    )
    & (
        one_test["destination_arrival_seconds"]
        == best_test["destination_arrival_seconds"]
    )
].copy()

print("\nMATCHING RECORDS:", len(matches))

# Show only columns useful for diagnosing differences
diagnostic_columns = [
    col for col in matches.columns
    if any(
        term in col.lower()
        for term in [
            "stop_id",
            "station",
            "platform",
            "sequence",
            "trip_id",
            "seconds"
        ]
    )
]

display(matches[diagnostic_columns].head(25))

print("\nDISTINCT MATCHING RECORDS:",
      len(matches.drop_duplicates()))

print("\nCOLUMNS THAT DIFFER BETWEEN MATCHES:")
varying_columns = [
    col for col in matches.columns
    if matches[col].nunique(dropna=False) > 1
]
print(varying_columns)


RECOMMENDED JOURNEY


,journey_type,transfers,first_trip_id,second_trip_id,third_trip_id,origin_departure_seconds,destination_arrival_seconds,journey_duration_min,first_transfer_station,second_transfer_station,departure_time,arrival_time
0,One transfer,1,0160-006-106-014:1000,86-D.1324.165.48.A.8.92000788,None,63114.0,65082.0,32.8,Central Station,None,17:31:54,18:04:42



ONE-TRANSFER DATAFRAME COLUMNS
['boarding_id', 'trip_id', 'origin_platform', 'origin_sequence', 'origin_departure_seconds', 'origin_departure_time', 'first_line', 'first_mode', 'stop_id_x', 'station_id', 'stop_sequence', 'arrival_seconds', 'arrival_time', 'from_stop_id', 'to_stop_id', 'minimum_transfer_seconds', 'earliest_boarding_seconds', 'second_trip_id', 'second_platform', 'second_boarding_sequence', 'second_departure_seconds', 'second_departure_time', 'second_line', 'second_mode', 'destination_sequence', 'destination_arrival_seconds', 'destination_arrival_time', 'journey_duration_min', 'journey_type', 'transfers', 'stop_id_y', 'transfer_station']


KeyError: 'first_trip_id'

In [254]:

# STEP 92K — Inspect one-transfer journey structure

import pandas as pd

print("ONE-TRANSFER DATAFRAME")
print("Total records:", len(one_test))

print("\nALL COLUMN NAMES:")
for column in one_test.columns:
    print("-", column)

print("\nRECOMMENDED JOURNEY:")
display(best_test.to_frame().T)

print("\nSAMPLE ONE-TRANSFER RECORD:")
display(one_test.head(1).T)

print("\nPOTENTIAL TRIP / TRANSFER IDENTIFIERS:")
identifier_columns = [
    col for col in one_test.columns
    if any(
        keyword in col.lower()
        for keyword in ["trip", "transfer", "stop_id", "platform"]
    )
]

print(identifier_columns)


ONE-TRANSFER DATAFRAME
Total records: 4167

ALL COLUMN NAMES:
- boarding_id
- trip_id
- origin_platform
- origin_sequence
- origin_departure_seconds
- origin_departure_time
- first_line
- first_mode
- stop_id_x
- station_id
- stop_sequence
- arrival_seconds
- arrival_time
- from_stop_id
- to_stop_id
- minimum_transfer_seconds
- earliest_boarding_seconds
- second_trip_id
- second_platform
- second_boarding_sequence
- second_departure_seconds
- second_departure_time
- second_line
- second_mode
- destination_sequence
- destination_arrival_seconds
- destination_arrival_time
- journey_duration_min
- journey_type
- transfers
- stop_id_y
- transfer_station

RECOMMENDED JOURNEY:


,journey_type,transfers,first_trip_id,second_trip_id,third_trip_id,origin_departure_seconds,destination_arrival_seconds,journey_duration_min,first_transfer_station,second_transfer_station,departure_time,arrival_time
0,One transfer,1,0160-006-106-014:1000,86-D.1324.165.48.A.8.92000788,None,63114.0,65082.0,32.8,Central Station,None,17:31:54,18:04:42



SAMPLE ONE-TRANSFER RECORD:


,0
boarding_id,1
trip_id,0160-006-106-014:1000
origin_platform,2113351
origin_sequence,10
origin_departure_seconds,63114.0
origin_departure_time,17:31:54
first_line,M1
first_mode,Metro
stop_id_x,2000467
station_id,200060



POTENTIAL TRIP / TRANSFER IDENTIFIERS:
['trip_id', 'origin_platform', 'stop_id_x', 'from_stop_id', 'to_stop_id', 'minimum_transfer_seconds', 'second_trip_id', 'second_platform', 'transfers', 'stop_id_y', 'transfer_station']


In [255]:

# STEP 92L — Investigate ambiguous one-transfer records

import pandas as pd

matches = one_test.loc[
    (one_test["trip_id"] == best_test["first_trip_id"])
    & (one_test["second_trip_id"] == best_test["second_trip_id"])
    & (
        one_test["origin_departure_seconds"]
        == best_test["origin_departure_seconds"]
    )
    & (
        one_test["destination_arrival_seconds"]
        == best_test["destination_arrival_seconds"]
    )
    & (
        one_test["transfer_station"]
        == best_test["first_transfer_station"]
    )
].copy()

print("MATCHING RECORDS:", len(matches))

# Display actual passenger-relevant transfer details
columns = [
    "trip_id",
    "origin_platform",
    "arrival_seconds",
    "from_stop_id",
    "to_stop_id",
    "minimum_transfer_seconds",
    "second_trip_id",
    "second_platform",
    "second_departure_seconds",
    "transfer_station"
]

display(matches[columns].drop_duplicates())

print("\nDISTINCT FULL RECORDS:", len(matches.drop_duplicates()))

print("\nCOLUMNS THAT DIFFER:")
varying = [
    col for col in matches.columns
    if matches[col].nunique(dropna=False) > 1
]

for col in varying:
    print(f"{col}: {matches[col].nunique(dropna=False)} unique values")


MATCHING RECORDS: 19


,trip_id,origin_platform,arrival_seconds,from_stop_id,to_stop_id,minimum_transfer_seconds,second_trip_id,second_platform,second_departure_seconds,transfer_station
0,0160-006-106-014:1000,2113351,64569.0,2000467,2000337,348.0,86-D.1324.165.48.A.8.92000788,2000337,64921.0,Central Station



DISTINCT FULL RECORDS: 19

COLUMNS THAT DIFFER:
stop_id_y: 19 unique values


In [256]:

# STEP 92M — Fix duplicate one-transfer itinerary matching

# Preserve the original formatter for other journey types
original_get_best_journey_itinerary = get_best_journey_itinerary

def get_best_journey_itinerary(
    recommendation,
    daily,
    origin,
    destination,
    direct_results,
    one_transfer_results,
    two_transfer_results
):
    transfers = int(recommendation["transfers"])

    # Keep existing behaviour for direct and two-transfer journeys
    if transfers != 1:
        return original_get_best_journey_itinerary(
            recommendation,
            daily,
            origin,
            destination,
            direct_results,
            one_transfer_results,
            two_transfer_results
        )

    # Match the original one-transfer journey
    matches = one_transfer_results.loc[
        (one_transfer_results["trip_id"] == recommendation["first_trip_id"])
        & (one_transfer_results["second_trip_id"] == recommendation["second_trip_id"])
        & (
            one_transfer_results["origin_departure_seconds"]
            == recommendation["origin_departure_seconds"]
        )
        & (
            one_transfer_results["destination_arrival_seconds"]
            == recommendation["destination_arrival_seconds"]
        )
        & (
            one_transfer_results["transfer_station"]
            == recommendation["first_transfer_station"]
        )
    ].copy()

    if matches.empty:
        raise ValueError("No matching one-transfer journey found.")

    # Ignore only the duplicate stop_id_y field
    identity_columns = [
        col for col in matches.columns
        if col != "stop_id_y"
    ]

    unique_matches = matches.drop_duplicates(
        subset=identity_columns
    )

    if len(unique_matches) != 1:
        raise ValueError(
            f"{len(unique_matches)} distinct one-transfer journeys remain. "
            "Cannot safely select a platform."
        )

    # All matching records represent the same passenger journey
    return format_journey_itinerary(
        unique_matches.iloc[0],
        daily,
        origin,
        destination
    )


# Test the previously failing route
result_test = plan_journey_fast(
    "Macquarie University Station",
    "Town Hall Station",
    "17:30:00"
)

print("RECOMMENDED JOURNEY")
print("Departure:", result_test["departure_time"])
print("Arrival:", result_test["arrival_time"])
print("Transfers:", result_test["transfers"])

display(result_test["itinerary"])


RECOMMENDED JOURNEY
Departure: 17:31
Arrival: 18:04
Transfers: 1


,stage,station,time,platform,service
0,Depart,Macquarie University Station,17:31,Platform 1,M1 Metro
1,Transfer arrival,Central Station,17:56,Platform 27,M1 Metro
2,Transfer departure,Central Station,18:02,Platform 17,T8 Train
3,Arrive,Town Hall Station,18:04,Platform 6,T8 Train


In [257]:

# STEP 92N — Regression test for one-transfer journeys

import time
import pandas as pd

test_pairs = [
    ("Macquarie University Station", "Town Hall Station"),
    ("Macquarie University Station", "Redfern Station"),
    ("Epping Station", "Parramatta Station"),
    ("Chatswood Station", "Parramatta Station"),
    ("North Ryde Station", "Strathfield Station")
]

results = []

for start_station, end_station in test_pairs:
    start = time.perf_counter()

    try:
        journey = plan_journey_fast(
            start_station,
            end_station,
            "17:30:00"
        )

        elapsed = time.perf_counter() - start

        if journey is None:
            results.append({
                "origin": start_station,
                "destination": end_station,
                "status": "No journey",
                "transfers": None,
                "arrival": None,
                "runtime_seconds": round(elapsed, 3)
            })
            continue

        expected_rows = 2 * (journey["transfers"] + 1)
        actual_rows = len(journey["itinerary"])

        assert actual_rows == expected_rows, (
            f"Expected {expected_rows} itinerary rows, "
            f"got {actual_rows}"
        )

        results.append({
            "origin": start_station,
            "destination": end_station,
            "status": "PASS",
            "transfers": journey["transfers"],
            "arrival": journey["arrival_time"],
            "runtime_seconds": round(elapsed, 3)
        })

    except Exception as error:
        results.append({
            "origin": start_station,
            "destination": end_station,
            "status": f"ERROR: {str(error)[:90]}",
            "transfers": None,
            "arrival": None,
            "runtime_seconds": round(
                time.perf_counter() - start, 3
            )
        })

display(pd.DataFrame(results))


,origin,destination,status,transfers,arrival,runtime_seconds
0,Macquarie University Station,Town Hall Station,PASS,1,18:04,0.176
1,Macquarie University Station,Redfern Station,PASS,1,18:06,0.126
2,Epping Station,Parramatta Station,PASS,1,18:14,0.204
3,Chatswood Station,Parramatta Station,PASS,1,18:23,1.045
4,North Ryde Station,Strathfield Station,PASS,1,18:07,0.123


In [258]:

# STEP 93A — Validate station choices for Streamlit

import pandas as pd

station_options = (
    station_lookup["stop_name"]
    .dropna()
    .astype(str)
    .str.strip()
)

station_options = sorted(
    name for name in station_options.unique()
    if name
)

print("STATION COVERAGE")
print("=" * 45)
print("Total unique stations:", len(station_options))
print("First 20 stations:")
for name in station_options[:20]:
    print(" -", name)

print("\nDATA QUALITY CHECKS")
print("Duplicate names:", station_lookup["stop_name"].duplicated().sum())
print("Missing names:", station_lookup["stop_name"].isna().sum())

required_stations = [
    "Macquarie University Station",
    "Parramatta Station",
    "Central Station",
    "Town Hall Station"
]

print("\nREQUIRED STATIONS")
for name in required_stations:
    print(
        name,
        "— FOUND" if name in station_options else "— MISSING"
    )


STATION COVERAGE
Total unique stations: 191
First 20 stations:
 - Allawah Station
 - Arncliffe Station
 - Artarmon Station
 - Ashfield Station
 - Asquith Station
 - Auburn Station
 - Banksia Station
 - Bankstown Station
 - Barangaroo Station
 - Bardwell Park Station
 - Beecroft Station
 - Bella Vista Station
 - Berala Station
 - Berowra Station
 - Beverly Hills Station
 - Bexley North Station
 - Birrong Station
 - Blacktown Station
 - Bondi Junction Station
 - Bridge Street Light Rail

DATA QUALITY CHECKS
Duplicate names: 327
Missing names: 0

REQUIRED STATIONS
Macquarie University Station — FOUND
Parramatta Station — FOUND
Central Station — FOUND
Town Hall Station — FOUND


In [259]:

# STEP 93B — Check which stations are supported by routing data

import pandas as pd

lookup = (
    station_lookup[["station_id", "stop_name"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

lookup["station_id"] = lookup["station_id"].astype(str)

daily_stations = realtime_daily.copy()
daily_stations["station_id"] = daily_stations["station_id"].astype(str)

coverage = (
    daily_stations.groupby("station_id")
    .agg(
        timetable_events=("trip_id", "size"),
        unique_services=("trip_id", "nunique"),
        departures=("departure_seconds", "count"),
        arrivals=("arrival_seconds", "count")
    )
    .reset_index()
)

station_coverage = lookup.merge(
    coverage,
    on="station_id",
    how="left"
)

station_coverage["supported"] = (
    station_coverage["timetable_events"].fillna(0) > 0
)

supported_stations = sorted(
    station_coverage.loc[
        station_coverage["supported"], "stop_name"
    ].unique()
)

unsupported_stations = sorted(
    station_coverage.loc[
        ~station_coverage["supported"], "stop_name"
    ].unique()
)

print("STATION COVERAGE")
print("=" * 45)
print("Stations in lookup:", lookup["stop_name"].nunique())
print("Stations with timetable events:", len(supported_stations))
print("Stations without timetable events:", len(unsupported_stations))

print("\nUNSUPPORTED STATIONS")
for station in unsupported_stations[:30]:
    print("-", station)

print("\nSAMPLE SUPPORTED STATIONS")
display(
    station_coverage.loc[
        station_coverage["supported"],
        [
            "stop_name",
            "timetable_events",
            "unique_services"
        ]
    ]
    .sort_values("timetable_events", ascending=False)
    .head(10)
)


STATION COVERAGE
Stations in lookup: 191
Stations with timetable events: 191
Stations without timetable events: 0

UNSUPPORTED STATIONS

SAMPLE SUPPORTED STATIONS


,stop_name,timetable_events,unique_services
18,Central Station,3673,3188
40,Town Hall Station,1788,1788
41,Wynyard Station,1466,1466
39,Redfern Station,1418,1418
42,Circular Quay,1029,1029
26,Lidcombe Station,978,978
29,Strathfield Station,861,861
20,Sydenham Station,828,828
12,Chatswood Station,798,798
16,Martin Place Station,756,756


In [260]:

# STEP 93C — Inspect routing engine dependencies

import inspect

routing_functions = [
    "plan_journey_fast",
    "find_direct_journeys",
    "find_one_transfer_journeys",
    "find_two_transfer_journeys_fast",
    "build_first_leg_stops",
    "build_first_transfer_options",
    "build_second_leg_candidates",
    "build_second_leg_stops",
    "build_third_transfer_options",
    "build_third_leg_candidates",
    "build_complete_two_transfer_journeys",
    "standardise_journey_results",
    "recommend_best_journey",
    "get_best_journey_itinerary",
    "format_journey_itinerary",
    "format_direct_itinerary",
    "format_one_transfer_itinerary",
    "format_two_transfer_itinerary",
    "get_platform",
    "get_trip_service",
    "format_transport_time"
]

inspection = []

for name in routing_functions:
    obj = globals().get(name)

    if obj is None:
        status = "MISSING"
        source = None
    elif not callable(obj):
        status = "NOT CALLABLE"
        source = None
    else:
        status = "AVAILABLE"
        try:
            source = inspect.getsourcefile(obj) or "Notebook"
        except (TypeError, OSError):
            source = "Notebook / interactive"

    inspection.append({
        "function": name,
        "status": status,
        "source": source
    })

dependency_report = pd.DataFrame(inspection)

display(dependency_report)

print("\nAVAILABLE:",
      (dependency_report["status"] == "AVAILABLE").sum())

print("MISSING:",
      (dependency_report["status"] == "MISSING").sum())


,function,status,source
0,plan_journey_fast,AVAILABLE,/var/folders/w6/tgkpq1sx16s4f4k9pbscyzx80000gn...
1,find_direct_journeys,AVAILABLE,/var/folders/w6/tgkpq1sx16s4f4k9pbscyzx80000gn...
2,find_one_transfer_journeys,AVAILABLE,/var/folders/w6/tgkpq1sx16s4f4k9pbscyzx80000gn...
3,find_two_transfer_journeys_fast,AVAILABLE,/var/folders/w6/tgkpq1sx16s4f4k9pbscyzx80000gn...
4,build_first_leg_stops,AVAILABLE,/var/folders/w6/tgkpq1sx16s4f4k9pbscyzx80000gn...
5,build_first_transfer_options,AVAILABLE,/var/folders/w6/tgkpq1sx16s4f4k9pbscyzx80000gn...
6,build_second_leg_candidates,AVAILABLE,/var/folders/w6/tgkpq1sx16s4f4k9pbscyzx80000gn...
7,build_second_leg_stops,AVAILABLE,/var/folders/w6/tgkpq1sx16s4f4k9pbscyzx80000gn...
8,build_third_transfer_options,AVAILABLE,/var/folders/w6/tgkpq1sx16s4f4k9pbscyzx80000gn...
9,build_third_leg_candidates,AVAILABLE,/var/folders/w6/tgkpq1sx16s4f4k9pbscyzx80000gn...



AVAILABLE: 21
MISSING: 0


In [261]:

# Test: Kogarah → Victoria Cross at 8:00 AM

import time

origin = "Kogarah Station"
destination = "Victoria Cross Station"
departure_time = "08:00:00"

start = time.perf_counter()

result = plan_journey_fast(
    origin,
    destination,
    departure_time
)

elapsed = time.perf_counter() - start

print("KOGARAH → VICTORIA CROSS")
print("=" * 45)
print(f"Search time: {elapsed:.3f} seconds")

if result is None:
    print("No journey found.")
else:
    print("Departure:", result["departure_time"])
    print("Arrival:", result["arrival_time"])
    print("Duration:", result["duration_minutes"], "minutes")
    print("Transfers:", result["transfers"])
    print("\nFULL ITINERARY")
    display(result["itinerary"])


KOGARAH → VICTORIA CROSS
Search time: 0.230 seconds
Departure: 08:07
Arrival: 08:46
Duration: 40 minutes
Transfers: 1

FULL ITINERARY


,stage,station,time,platform,service
0,Depart,Kogarah Station,08:07,Platform 1,T4 Train
1,Transfer arrival,Sydenham Station,08:20,Platform 5,T4 Train
2,Transfer departure,Sydenham Station,08:32,Platform 1,M1 Metro
3,Arrive,Victoria Cross Station,08:46,Platform 2,M1 Metro


In [262]:

import pandas as pd

routes = find_one_transfer_journeys(
    realtime_daily,
    "Kogarah Station",
    "Victoria Cross Station",
    "08:00:00",
    window_minutes=60,
    connection_window_minutes=45
).copy()

# Only departures from 8 AM onwards
routes = routes.loc[
    routes["origin_departure_seconds"] >= 8 * 3600
].copy()

# Remove duplicate representations
routes = routes.drop_duplicates(
    subset=[
        "trip_id", "second_trip_id",
        "origin_departure_seconds",
        "arrival_seconds",
        "second_departure_seconds",
        "destination_arrival_seconds",
        "transfer_station",
        "from_stop_id", "to_stop_id"
    ]
)

routes["transfer_wait_min"] = (
    routes["second_departure_seconds"]
    - routes["arrival_seconds"]
) / 60

for source, target in [
    ("origin_departure_seconds", "departure"),
    ("arrival_seconds", "transfer_arrival"),
    ("second_departure_seconds", "transfer_departure"),
    ("destination_arrival_seconds", "arrival")
]:
    routes[target] = routes[source].apply(
        lambda seconds: format_transport_time(seconds)[:5]
    )

# Earliest arrival is the main priority
fastest_routes = routes.sort_values(
    [
        "destination_arrival_seconds",
        "transfer_wait_min",
        "origin_departure_seconds"
    ]
)

print("FASTEST ARRIVALS — SEARCH FROM 08:00")
display(
    fastest_routes[
        [
            "departure",
            "transfer_station",
            "transfer_arrival",
            "transfer_departure",
            "transfer_wait_min",
            "arrival",
            "journey_duration_min",
            "first_line",
            "second_line"
        ]
    ].head(10).reset_index(drop=True)
)


FASTEST ARRIVALS — SEARCH FROM 08:00


,departure,transfer_station,transfer_arrival,transfer_departure,transfer_wait_min,arrival,journey_duration_min,first_line,second_line
0,08:07,Martin Place Station,08:33,08:42,8.650000,08:46,39.82,T4,M1
1,08:07,Central Station,08:28,08:38,9.833333,08:46,39.82,T4,M1
2,08:07,Sydenham Station,08:20,08:32,11.700000,08:46,39.82,T4,M1
3,08:07,Martin Place Station,08:33,08:46,12.650000,08:50,43.82,T4,M1
4,08:07,Central Station,08:28,08:42,13.833333,08:50,43.82,T4,M1
5,08:07,Sydenham Station,08:20,08:36,15.700000,08:50,43.82,T4,M1
6,08:17,Sydenham Station,08:30,08:40,9.700000,08:54,37.65,T4,M1
7,08:07,Martin Place Station,08:33,08:49,16.483333,08:54,47.65,T4,M1
8,08:07,Central Station,08:28,08:46,17.833333,08:54,47.65,T4,M1
9,08:07,Sydenham Station,08:20,08:40,19.700000,08:54,47.65,T4,M1


In [263]:

# STEP 94D — Compare transfer feasibility for earliest arrivals

import pandas as pd

earliest_arrival = routes["destination_arrival_seconds"].min()

candidates = routes.loc[
    routes["destination_arrival_seconds"] == earliest_arrival
].copy()

candidates["transfer_wait_min"] = (
    candidates["second_departure_seconds"]
    - candidates["arrival_seconds"]
) / 60

candidates["walking_min"] = (
    candidates["minimum_transfer_seconds"] / 60
)

candidates["spare_min"] = (
    candidates["second_departure_seconds"]
    - candidates["arrival_seconds"]
    - candidates["minimum_transfer_seconds"]
) / 60

candidates = candidates.sort_values(
    ["transfer_wait_min", "spare_min"],
    ascending=[True, False]
)

display(
    candidates[
        [
            "departure",
            "transfer_station",
            "transfer_arrival",
            "transfer_departure",
            "transfer_wait_min",
            "walking_min",
            "spare_min",
            "arrival"
        ]
    ].head(10).reset_index(drop=True)
)


,departure,transfer_station,transfer_arrival,transfer_departure,transfer_wait_min,walking_min,spare_min,arrival
0,08:07,Martin Place Station,08:33,08:42,8.650000,6.6,2.050000,08:46
1,08:07,Central Station,08:28,08:38,9.833333,7.9,1.933333,08:46
2,08:07,Sydenham Station,08:20,08:32,11.700000,8.9,2.800000,08:46


In [264]:

# STEP 94G — Safest connection first, shorter transfer second

import pandas as pd

origin = "Kogarah Station"
destination = "Victoria Cross Station"

journeys = find_one_transfer_journeys(
    realtime_daily, origin, destination, "08:00:00",
    window_minutes=60,
    connection_window_minutes=45
).copy()

earliest = journeys["destination_arrival_seconds"].min()

journeys = journeys.loc[
    journeys["destination_arrival_seconds"] == earliest
].copy()

journeys["transfer_wait_min"] = (
    journeys["second_departure_seconds"] - journeys["arrival_seconds"]
) / 60

journeys["walking_min"] = journeys["minimum_transfer_seconds"] / 60

journeys["spare_min"] = (
    journeys["transfer_wait_min"] - journeys["walking_min"]
)

# Exclude connections with less than 2 minutes spare
journeys = journeys.loc[journeys["spare_min"] >= 2].copy()

journeys = journeys.drop_duplicates(
    subset=[
        "trip_id", "second_trip_id",
        "transfer_station", "from_stop_id", "to_stop_id",
        "origin_departure_seconds",
        "destination_arrival_seconds"
    ]
)

# Primary recommendation: greatest spare time
ranked = journeys.sort_values(
    ["spare_min", "transfer_wait_min"],
    ascending=[False, True]
)

recommended = ranked.iloc[0] if not ranked.empty else None

# Alternative: shortest transfer at a different station
alternatives = ranked.loc[
    ranked["transfer_station"] != recommended["transfer_station"]
] if recommended is not None else ranked

alternative = (
    alternatives.sort_values("transfer_wait_min").iloc[0]
    if not alternatives.empty else None
)

for label, journey in [
    ("RECOMMENDED — MORE TRANSFER BUFFER", recommended),
    ("ALTERNATIVE — SHORTER TRANSFER", alternative)
]:
    if journey is None:
        continue

    print(f"\n{label}")
    print("Via:", journey["transfer_station"])
    print("Transfer window:", round(journey["transfer_wait_min"], 1), "min")
    print("Estimated walking:", round(journey["walking_min"], 1), "min")
    print("Spare time:", round(journey["spare_min"], 1), "min")

    display(
        format_journey_itinerary(
            journey, realtime_daily, origin, destination
        )
    )



RECOMMENDED — MORE TRANSFER BUFFER
Via: Sydenham Station
Transfer window: 11.7 min
Estimated walking: 8.9 min
Spare time: 2.8 min


,stage,station,time,platform,service
0,Depart,Kogarah Station,08:07,Platform 1,T4 Train
1,Transfer arrival,Sydenham Station,08:20,Platform 5,T4 Train
2,Transfer departure,Sydenham Station,08:32,Platform 1,M1 Metro
3,Arrive,Victoria Cross Station,08:46,Platform 2,M1 Metro



ALTERNATIVE — SHORTER TRANSFER
Via: Martin Place Station
Transfer window: 8.6 min
Estimated walking: 6.6 min
Spare time: 2.1 min


,stage,station,time,platform,service
0,Depart,Kogarah Station,08:07,Platform 1,T4 Train
1,Transfer arrival,Martin Place Station,08:33,Platform 2,T4 Train
2,Transfer departure,Martin Place Station,08:42,Platform 4,M1 Metro
3,Arrive,Victoria Cross Station,08:46,Platform 2,M1 Metro


In [265]:

# STEP 94H — Passenger-friendly itinerary display

def display_passenger_journey(label, journey, origin, destination):
    if journey is None:
        return

    itinerary = format_journey_itinerary(
        journey, realtime_daily, origin, destination
    )

    departure = itinerary.iloc[0]["time"]
    arrival = itinerary.iloc[-1]["time"]

    duration = round(journey["journey_duration_min"])

    print(f"\n{label}")
    print(f"{departure} → {arrival}")
    print(f"Duration: {duration} min | Transfers: 1")
    print(f"Via: {journey['transfer_station']}")

    # Display only passenger-facing information
    display(
        itinerary[
            ["stage", "station", "time", "platform", "service"]
        ].reset_index(drop=True)
    )


display_passenger_journey(
    "RECOMMENDED — VIA SYDENHAM",
    recommended,
    origin,
    destination
)

display_passenger_journey(
    "ALTERNATIVE — VIA MARTIN PLACE",
    alternative,
    origin,
    destination
)



RECOMMENDED — VIA SYDENHAM
08:07 → 08:46
Duration: 40 min | Transfers: 1
Via: Sydenham Station


,stage,station,time,platform,service
0,Depart,Kogarah Station,08:07,Platform 1,T4 Train
1,Transfer arrival,Sydenham Station,08:20,Platform 5,T4 Train
2,Transfer departure,Sydenham Station,08:32,Platform 1,M1 Metro
3,Arrive,Victoria Cross Station,08:46,Platform 2,M1 Metro



ALTERNATIVE — VIA MARTIN PLACE
08:07 → 08:46
Duration: 40 min | Transfers: 1
Via: Martin Place Station


,stage,station,time,platform,service
0,Depart,Kogarah Station,08:07,Platform 1,T4 Train
1,Transfer arrival,Martin Place Station,08:33,Platform 2,T4 Train
2,Transfer departure,Martin Place Station,08:42,Platform 4,M1 Metro
3,Arrive,Victoria Cross Station,08:46,Platform 2,M1 Metro


In [266]:

# STEP 95 — Reusable recommended + alternative selector

def select_one_transfer_options(journeys, min_spare_minutes=2):
    if journeys is None or journeys.empty:
        return None, None

    df = journeys.copy()

    df["transfer_wait_min"] = (
        df["second_departure_seconds"] - df["arrival_seconds"]
    ) / 60

    df["walking_min"] = df["minimum_transfer_seconds"] / 60

    df["spare_min"] = df["transfer_wait_min"] - df["walking_min"]

    # Keep connections meeting the minimum transfer buffer
    df = df.loc[df["spare_min"] >= min_spare_minutes].copy()

    if df.empty:
        return None, None

    # Earliest arrival is the first priority
    earliest = df["destination_arrival_seconds"].min()
    df = df.loc[df["destination_arrival_seconds"] == earliest].copy()

    df = df.drop_duplicates(
        subset=[
            "trip_id", "second_trip_id",
            "transfer_station", "from_stop_id", "to_stop_id",
            "origin_departure_seconds",
            "destination_arrival_seconds"
        ]
    )

    # Recommended: greatest transfer buffer
    ranked = df.sort_values(
        ["spare_min", "transfer_wait_min"],
        ascending=[False, True]
    )

    recommended = ranked.iloc[0]

    # Alternative: different interchange, shortest transfer
    other_stations = ranked.loc[
        ranked["transfer_station"] != recommended["transfer_station"]
    ]

    alternative = (
        other_stations.sort_values("transfer_wait_min").iloc[0]
        if not other_stations.empty else None
    )

    return recommended, alternative


# Test using Kogarah → Victoria Cross
test_journeys = find_one_transfer_journeys(
    realtime_daily,
    "Kogarah Station",
    "Victoria Cross Station",
    "08:00:00"
)

rec, alt = select_one_transfer_options(test_journeys)

display_passenger_journey(
    "RECOMMENDED",
    rec,
    "Kogarah Station",
    "Victoria Cross Station"
)

display_passenger_journey(
    "ALTERNATIVE",
    alt,
    "Kogarah Station",
    "Victoria Cross Station"
)



RECOMMENDED
08:07 → 08:46
Duration: 40 min | Transfers: 1
Via: Sydenham Station


,stage,station,time,platform,service
0,Depart,Kogarah Station,08:07,Platform 1,T4 Train
1,Transfer arrival,Sydenham Station,08:20,Platform 5,T4 Train
2,Transfer departure,Sydenham Station,08:32,Platform 1,M1 Metro
3,Arrive,Victoria Cross Station,08:46,Platform 2,M1 Metro



ALTERNATIVE
08:07 → 08:46
Duration: 40 min | Transfers: 1
Via: Martin Place Station


,stage,station,time,platform,service
0,Depart,Kogarah Station,08:07,Platform 1,T4 Train
1,Transfer arrival,Martin Place Station,08:33,Platform 2,T4 Train
2,Transfer departure,Martin Place Station,08:42,Platform 4,M1 Metro
3,Arrive,Victoria Cross Station,08:46,Platform 2,M1 Metro


In [267]:

import time

origin = "Kogarah Station"
destination = "Richmond Station"
departure_time = "09:00:00"

start = time.perf_counter()

result = plan_journey_fast(
    origin,
    destination,
    departure_time,
    daily=realtime_daily
)

elapsed = time.perf_counter() - start

print(f"Search: {origin} → {destination}")
print(f"Departure search time: {departure_time}")
print(f"Processing time: {elapsed:.3f} seconds")

display(result)


Search: Kogarah Station → Richmond Station
Departure search time: 09:00:00
Processing time: 1.530 seconds


{'origin': 'Kogarah Station',
 'destination': 'Richmond Station',
 'departure_time': '09:17',
 'arrival_time': '11:05',
 'duration_minutes': 108,
 'transfers': 1,
 'journey_type': 'One transfer',
 'itinerary':                 stage           station   time     platform   service
 0              Depart   Kogarah Station  09:17   Platform 1  T4 Train
 1    Transfer arrival   Redfern Station  09:36  Platform 11  T4 Train
 2  Transfer departure   Redfern Station  09:47   Platform 4  T1 Train
 3              Arrive  Richmond Station  11:05   Platform 2  T1 Train}

In [268]:

# STEP 97 — Test Arrive By filtering

import pandas as pd

def filter_arrive_by(all_results, arrival_time):
    if all_results is None or all_results.empty:
        return pd.DataFrame()

    hours, minutes, seconds = map(int, arrival_time.split(":"))
    deadline_seconds = hours * 3600 + minutes * 60 + seconds

    eligible = all_results.loc[
        all_results["destination_arrival_seconds"] <= deadline_seconds
    ].copy()

    if eligible.empty:
        return eligible

    # Arrive By: prefer the latest possible departure
    return eligible.sort_values(
        ["origin_departure_seconds", "destination_arrival_seconds"],
        ascending=[False, True]
    )


# Example: Arrive at Richmond by 09:00
# Search an earlier departure window to find eligible journeys

origin = "Kogarah Station"
destination = "Richmond Station"

direct = find_direct_journeys(
    realtime_daily, origin, destination, "06:00:00",
    window_minutes=180
)

one = find_one_transfer_journeys(
    realtime_daily, origin, destination, "06:00:00",
    window_minutes=180
)

two = find_two_transfer_journeys_fast(
    realtime_daily, origin, destination, "06:00:00",
    window_minutes=180
)

all_results = standardise_journey_results(direct, one, two)

arrive_by_results = filter_arrive_by(all_results, "09:00:00")

print("Journeys arriving by 09:00:")
display(
    arrive_by_results[
        [
            "journey_type",
            "departure_time",
            "arrival_time",
            "journey_duration_min",
            "transfers"
        ]
    ].head(10)
)


Journeys arriving by 09:00:


,journey_type,departure_time,arrival_time,journey_duration_min,transfers
7,One transfer,06:47:00,08:35:00,108.0,1
8,One transfer,06:37:00,08:35:00,118.0,1
10,Two transfers,06:37:00,08:35:00,118.0,2
11,Two transfers,06:37:00,08:35:00,118.0,2
12,Two transfers,06:37:00,08:35:00,118.0,2
13,Two transfers,06:37:00,08:35:00,118.0,2
14,Two transfers,06:37:00,08:35:00,118.0,2
15,Two transfers,06:37:00,08:35:00,118.0,2
16,Two transfers,06:37:00,08:35:00,118.0,2
17,Two transfers,06:37:00,08:35:00,118.0,2


In [269]:

# STEP 99 — Validate Arrive By for Kogarah → Richmond

import time
import pandas as pd

origin = "Kogarah Station"
destination = "Richmond Station"
arrival_deadline = "09:00:00"

start = time.perf_counter()

# Search departures between 04:00 and 09:00
direct = find_direct_journeys(
    realtime_daily, origin, destination,
    "04:00:00", window_minutes=300
)

one = find_one_transfer_journeys(
    realtime_daily, origin, destination,
    "04:00:00", window_minutes=300
)

two = find_two_transfer_journeys_fast(
    realtime_daily, origin, destination,
    "04:00:00", window_minutes=300
)

results = standardise_journey_results(direct, one, two)

deadline_seconds = 9 * 3600

valid = results.loc[
    (results["destination_arrival_seconds"] <= deadline_seconds)
    & (results["origin_departure_seconds"] <= deadline_seconds)
].copy()

valid = valid.drop_duplicates(
    subset=[
        "first_trip_id",
        "second_trip_id",
        "third_trip_id",
        "origin_departure_seconds",
        "destination_arrival_seconds"
    ]
)

valid = valid.sort_values(
    ["origin_departure_seconds", "transfers", "destination_arrival_seconds"],
    ascending=[False, True, True]
)

print(f"Search completed in {time.perf_counter() - start:.2f} seconds")
print(f"Valid journeys arriving by {arrival_deadline}: {len(valid)}")

display(
    valid[
        [
            "journey_type",
            "departure_time",
            "arrival_time",
            "journey_duration_min",
            "transfers"
        ]
    ].head(10).reset_index(drop=True)
)


Search completed in 65.59 seconds
Valid journeys arriving by 09:00:00: 1362


,journey_type,departure_time,arrival_time,journey_duration_min,transfers
0,One transfer,06:47:00,08:35:00,108.0,1
1,One transfer,06:37:00,08:35:00,118.0,1
2,Two transfers,06:37:00,08:35:00,118.0,2
3,Two transfers,06:37:00,08:35:00,118.0,2
4,Two transfers,06:37:00,08:35:00,118.0,2
5,Two transfers,06:37:00,08:35:00,118.0,2
6,Two transfers,06:37:00,08:35:00,118.0,2
7,Two transfers,06:37:00,08:35:00,118.0,2
8,Two transfers,06:37:00,08:35:00,118.0,2
9,Two transfers,06:37:00,08:35:00,118.0,2


In [270]:

# STEP 100 — Faster Arrive By search using hourly windows

import time
import pandas as pd

def plan_arrive_by_test(origin, destination, arrival_time, daily):
    h, m, s = map(int, arrival_time.split(":"))
    deadline = h * 3600 + m * 60 + s

    start = time.perf_counter()

    # Search hourly departure windows, working backwards.
    # Extend to 5 hours before the deadline.
    for hours_back in range(1, 6):
        window_end = deadline - (hours_back - 1) * 3600
        window_start = max(0, window_end - 3600)

        search_time = (
            f"{window_start // 3600:02d}:"
            f"{(window_start % 3600) // 60:02d}:00"
        )

        direct = find_direct_journeys(
            daily, origin, destination,
            search_time, window_minutes=60
        )

        one = find_one_transfer_journeys(
            daily, origin, destination,
            search_time, window_minutes=60
        )

        two = find_two_transfer_journeys_fast(
            daily, origin, destination,
            search_time, window_minutes=60,
            arrival_cutoff_seconds=deadline + 1
        )

        results = standardise_journey_results(direct, one, two)

        if results.empty:
            continue

        valid = results.loc[
            (results["destination_arrival_seconds"] <= deadline)
            & (results["origin_departure_seconds"] >= window_start)
            & (results["origin_departure_seconds"] < window_end)
        ].copy()

        if not valid.empty:
            valid = valid.sort_values(
                ["origin_departure_seconds", "transfers",
                 "destination_arrival_seconds"],
                ascending=[False, True, True]
            )

            print(f"Found valid journeys in window starting {search_time}")
            print(f"Processing time: {time.perf_counter() - start:.2f}s")
            return valid

    print("No journey found within the five-hour lookback.")
    print(f"Processing time: {time.perf_counter() - start:.2f}s")
    return pd.DataFrame()


arrive_by_fast = plan_arrive_by_test(
    "Kogarah Station",
    "Richmond Station",
    "09:00:00",
    realtime_daily
)

if not arrive_by_fast.empty:
    display(
        arrive_by_fast[
            ["journey_type", "departure_time", "arrival_time",
             "journey_duration_min", "transfers"]
        ].head(10).reset_index(drop=True)
    )


Found valid journeys in window starting 06:00:00
Processing time: 2.88s


,journey_type,departure_time,arrival_time,journey_duration_min,transfers
0,One transfer,06:47:00,08:35:00,108.0,1
1,One transfer,06:37:00,08:35:00,118.0,1
2,Two transfers,06:37:00,08:35:00,118.0,2
3,Two transfers,06:37:00,08:35:00,118.0,2
4,Two transfers,06:37:00,08:35:00,118.0,2
5,Two transfers,06:37:00,08:35:00,118.0,2
6,Two transfers,06:37:00,08:35:00,118.0,2
7,Two transfers,06:37:00,08:35:00,118.0,2
8,Two transfers,06:37:00,08:35:00,118.0,2
9,Two transfers,06:37:00,08:35:00,118.0,2


In [271]:

# STEP 101 — Recommended + Alternative for Arrive By

import pandas as pd

origin = "Kogarah Station"
destination = "Richmond Station"

df = arrive_by_fast.copy()

# Remove equivalent journey records
df = df.drop_duplicates(
    subset=[
        "first_trip_id",
        "second_trip_id",
        "third_trip_id",
        "origin_departure_seconds",
        "destination_arrival_seconds"
    ]
)

# Latest departure first, fewer transfers second
df = df.sort_values(
    [
        "origin_departure_seconds",
        "transfers",
        "destination_arrival_seconds"
    ],
    ascending=[False, True, True]
)

recommended = df.iloc[0]

# Prefer a different journey with the same latest departure;
# otherwise use the next-latest departure
other = df.iloc[1:].copy()

different_route = other.loc[
    (other["first_trip_id"] != recommended["first_trip_id"]) |
    (other["second_trip_id"] != recommended["second_trip_id"]) |
    (other["third_trip_id"] != recommended["third_trip_id"])
]

alternative = (
    different_route.iloc[0]
    if not different_route.empty else None
)

for label, journey in [
    ("RECOMMENDED", recommended),
    ("ALTERNATIVE", alternative)
]:
    if journey is None:
        continue

    print(f"\n{label}")
    print(
        f"{journey['departure_time'][:5]} → "
        f"{journey['arrival_time'][:5]}"
    )
    print(f"Transfers: {journey['transfers']}")

    itinerary = get_best_journey_itinerary(
        journey,
        direct,
        one,
        two,
        origin,
        destination
    )

    display(itinerary)



RECOMMENDED
06:47 → 08:35
Transfers: 1


TypeError: get_best_journey_itinerary() missing 1 required positional argument: 'two_transfer_results'

In [272]:

# STEP 101 — ARRIVE BY: RECOMMENDED + ALTERNATIVE
# Kogarah Station → Richmond Station | Arrive by 09:00

import pandas as pd
import inspect

origin = "Kogarah Station"
destination = "Richmond Station"
arrival_deadline = "09:00:00"

deadline_seconds = sum(
    int(v) * multiplier
    for v, multiplier in zip(
        arrival_deadline.split(":"),
        [3600, 60, 1]
    )
)

# 1. Use the results from our faster Arrive By search
df = arrive_by_fast.copy()

df = df.loc[
    df["destination_arrival_seconds"] <= deadline_seconds
].copy()

if df.empty:
    print("No journeys found that meet the arrival deadline.")

else:
    # 2. Remove duplicate journey records
    df = df.drop_duplicates(
        subset=[
            "first_trip_id",
            "second_trip_id",
            "third_trip_id",
            "origin_departure_seconds",
            "destination_arrival_seconds"
        ]
    )

    # 3. Rank journeys:
    # Latest departure first, fewer transfers second,
    # earlier arrival third
    df = df.sort_values(
        [
            "origin_departure_seconds",
            "transfers",
            "destination_arrival_seconds"
        ],
        ascending=[False, True, True]
    ).reset_index(drop=True)

    recommended = df.iloc[0]

    # 4. Find a meaningful alternative
    # Prefer a different interchange or trip combination
    other = df.iloc[1:].copy()

    if not other.empty:
        different = other.loc[
            (
                other["first_transfer_station"].fillna("")
                != str(recommended.get("first_transfer_station") or "")
            )
            |
            (
                other["second_transfer_station"].fillna("")
                != str(recommended.get("second_transfer_station") or "")
            )
        ]

        alternative = (
            different.iloc[0]
            if not different.empty
            else other.iloc[0]
        )
    else:
        alternative = None

    # 5. Rebuild raw results for itinerary formatting.
    # These cover the hourly window containing the selected journeys.
    selected = [recommended]
    if alternative is not None:
        selected.append(alternative)

    earliest_departure = min(
        int(j["origin_departure_seconds"]) for j in selected
    )

    search_start = max(
        0, (earliest_departure // 3600) * 3600
    )

    search_time = (
        f"{search_start // 3600:02d}:"
        f"{(search_start % 3600) // 60:02d}:00"
    )

    window_minutes = max(
        60,
        int((deadline_seconds - search_start) / 60) + 1
    )

    raw_direct = find_direct_journeys(
        realtime_daily,
        origin,
        destination,
        search_time,
        window_minutes=window_minutes
    )

    raw_one = find_one_transfer_journeys(
        realtime_daily,
        origin,
        destination,
        search_time,
        window_minutes=window_minutes
    )

    raw_two = find_two_transfer_journeys_fast(
        realtime_daily,
        origin,
        destination,
        search_time,
        window_minutes=window_minutes,
        arrival_cutoff_seconds=deadline_seconds + 1
    )

    # 6. Check the actual function signature
    # and pass the required arguments correctly
    signature = inspect.signature(get_best_journey_itinerary)

    print("Itinerary function signature:", signature)

    def build_itinerary(journey):
        params = list(signature.parameters)

        # Common signature:
        # (recommendation, daily, origin, destination,
        #  direct_results, one_transfer_results,
        #  two_transfer_results)
        if len(params) == 7:
            return get_best_journey_itinerary(
                journey,
                realtime_daily,
                origin,
                destination,
                raw_direct,
                raw_one,
                raw_two
            )

        raise ValueError(
            "Unexpected itinerary function signature: "
            + str(signature)
        )

    # 7. Display passenger-friendly itineraries
    for label, journey in [
        ("RECOMMENDED", recommended),
        ("ALTERNATIVE", alternative)
    ]:
        if journey is None:
            continue

        print("\n" + "=" * 55)
        print(label)
        print("=" * 55)

        print(
            f"Depart: {journey['departure_time'][:5]} | "
            f"Arrive: {journey['arrival_time'][:5]}"
        )
        print(
            f"Duration: {round(journey['journey_duration_min'])} min | "
            f"Transfers: {int(journey['transfers'])}"
        )

        itinerary = build_itinerary(journey)

        if itinerary is not None and not itinerary.empty:
            display(
                itinerary[
                    ["stage", "station", "time", "platform", "service"]
                ].reset_index(drop=True)
            )
        else:
            print("Itinerary details unavailable.")


Itinerary function signature: (recommendation, daily, origin, destination, direct_results, one_transfer_results, two_transfer_results)

RECOMMENDED
Depart: 06:47 | Arrive: 08:35
Duration: 108 min | Transfers: 1


,stage,station,time,platform,service
0,Depart,Kogarah Station,06:47,Platform 1,T4 Train
1,Transfer arrival,Redfern Station,07:06,Platform 11,T4 Train
2,Transfer departure,Redfern Station,07:17,Platform 4,T1 Train
3,Arrive,Richmond Station,08:35,Platform 2,T1 Train



ALTERNATIVE
Depart: 06:37 | Arrive: 08:35
Duration: 118 min | Transfers: 2


,stage,station,time,platform,service
0,Depart,Kogarah Station,06:37,Platform 1,T4 Train
1,Transfer 1 arrival,Rockdale Station,06:38,Platform 2,T4 Train
2,Transfer 1 departure,Rockdale Station,06:49,Platform 2,T4 Train
3,Transfer 2 arrival,Redfern Station,07:06,Platform 11,T4 Train
4,Transfer 2 departure,Redfern Station,07:17,Platform 4,T1 Train
5,Arrive,Richmond Station,08:35,Platform 2,T1 Train


In [273]:

# STEP 102 — Add journey date input

from datetime import datetime

def parse_journey_date(date_text):
    """Convert DD/MM/YYYY into a Python date."""
    return datetime.strptime(date_text, "%d/%m/%Y").date()


def prepare_journey_search(
    origin,
    destination,
    journey_date,
    journey_time,
    time_mode="depart_at"
):
    date = parse_journey_date(journey_date)

    if time_mode not in ("depart_at", "arrive_by"):
        raise ValueError("Choose depart_at or arrive_by")

    datetime.strptime(journey_time, "%H:%M")

    return {
        "origin": origin,
        "destination": destination,
        "journey_date": date,
        "day_of_week": date.strftime("%A"),
        "journey_time": journey_time + ":00",
        "time_mode": time_mode
    }


# Test
search = prepare_journey_search(
    origin="Kogarah Station",
    destination="Richmond Station",
    journey_date="29/10/2026",
    journey_time="09:00",
    time_mode="arrive_by"
)

display(search)


{'origin': 'Kogarah Station',
 'destination': 'Richmond Station',
 'journey_date': datetime.date(2026, 10, 29),
 'day_of_week': 'Thursday',
 'journey_time': '09:00:00',
 'time_mode': 'arrive_by'}

In [274]:

# STEP 104 — Inspect available GTFS calendar information

import pandas as pd

print("=== CALENDAR DATA ===")

for name in ["calendar", "calendar_dates", "trips", "trip_service_lookup"]:
    if name in globals():
        obj = globals()[name]

        print(f"\n{name}")
        print("Type:", type(obj).__name__)

        if isinstance(obj, pd.DataFrame):
            print("Rows:", len(obj))
            print("Columns:", list(obj.columns))
            display(obj.head(3))
        else:
            print("Preview:", str(obj)[:500])
    else:
        print(f"\n{name}: Not found in notebook variables")

print("\n=== ROUTING DATA ===")
print("Columns:", list(realtime_daily.columns))

if "service_id" in realtime_daily.columns:
    print("Unique service IDs:", realtime_daily["service_id"].nunique())
else:
    print("service_id is not directly available in realtime_daily")


=== CALENDAR DATA ===

calendar
Type: DataFrame
Rows: 2336
Columns: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,AA61+1,1,1,1,1,1,0,0,20261027,20270125
1,AA61#1+1,1,1,1,1,1,0,0,20261027,20270125
2,AB22+1,0,0,0,0,0,1,1,20261027,20270125



calendar_dates
Type: DataFrame
Rows: 37650
Columns: ['service_id', 'date', 'exception_type']


,service_id,date,exception_type
0,AA61+1,20261225,2
1,AA61+1,20261228,2
2,AA61+1,20270101,2



trips
Type: DataFrame
Rows: 211419
Columns: ['route_id', 'service_id', 'trip_id', 'shape_id', 'trip_headsign', 'direction_id', 'block_id', 'wheelchair_accessible', 'route_direction', 'trip_note', 'bikes_allowed']


,route_id,service_id,trip_id,shape_id,trip_headsign,direction_id,block_id,wheelchair_accessible,route_direction,trip_note,bikes_allowed
0,1-SC0-1-sj2-1,AA61#1+1,1.AA61.1-SC0-1-sj2-1.1.R,1-SC0-1-sj2-1.1.R,Kiama,1,NaN,1,Bomaderry to Kiama,NaN,NaN
1,1-SC0-1-sj2-1,AB22#1+1,1.AB22.1-SC0-1-sj2-1.1.R,1-SC0-1-sj2-1.1.R,Kiama,1,NaN,1,Bomaderry to Kiama,NaN,NaN
2,1-SC0-1-sj2-1,AB22#1+1,10.AB22.1-SC0-1-sj2-1.2.H,1-SC0-1-sj2-1.2.H,Bomaderry,0,NaN,1,Kiama to Bomaderry,NaN,NaN



trip_service_lookup
Type: dict
Preview: {'0160-006-101-002:1000': 'M1 Metro', '0160-006-101-003:1000': 'M1 Metro', '0160-006-101-004:1000': 'M1 Metro', '0160-006-101-005:1000': 'M1 Metro', '0160-006-101-006:1000': 'M1 Metro', '0160-006-101-007:1000': 'M1 Metro', '0160-006-101-008:1000': 'M1 Metro', '0160-006-101-009:1000': 'M1 Metro', '0160-006-101-010:1000': 'M1 Metro', '0160-006-101-011:1000': 'M1 Metro', '0160-006-101-012:1000': 'M1 Metro', '0160-006-101-013:1000': 'M1 Metro', '0160-006-101-014:1000': 'M1 Metro', '0160-006-101-015:

=== ROUTING DATA ===
Columns: ['trip_id', 'stop_id', 'arrival_time', 'departure_time', 'stop_sequence', 'route_short_name', 'transport_mode', 'service_id', 'arrival_seconds', 'departure_seconds', 'station_id', 'scheduled_arrival_sec', 'scheduled_departure_sec', 'effective_arrival_sec', 'effective_departure_sec', 'arrival_source', 'departure_source', 'predicted_arrival_sec', 'predicted_departure_sec', 'original_arrival_seconds', 'original_departure_secon

In [275]:

# STEP 105 — Determine active GTFS services for a selected date

import pandas as pd
from datetime import datetime

def get_active_services(journey_date, calendar, calendar_dates):
    """
    Return active GTFS service IDs for a date in DD/MM/YYYY format.
    Includes regular schedules and calendar exceptions.
    """

    date = datetime.strptime(journey_date, "%d/%m/%Y").date()
    date_key = int(date.strftime("%Y%m%d"))
    weekday = date.strftime("%A").lower()

    # Regular calendar services
    regular = calendar.loc[
        (pd.to_numeric(calendar["start_date"]) <= date_key) &
        (pd.to_numeric(calendar["end_date"]) >= date_key) &
        (pd.to_numeric(calendar[weekday]) == 1),
        "service_id"
    ]

    active = set(regular.astype(str))

    # Date-specific exceptions
    exceptions = calendar_dates.loc[
        pd.to_numeric(calendar_dates["date"]) == date_key
    ]

    for row in exceptions.itertuples(index=False):
        if int(row.exception_type) == 1:
            active.add(str(row.service_id))
        elif int(row.exception_type) == 2:
            active.discard(str(row.service_id))

    return active


# Test different dates
test_dates = [
    "29/10/2026",  # Thursday
    "31/10/2026",  # Saturday
    "01/11/2026",  # Sunday
    "25/12/2026"   # Christmas Day
]

for date in test_dates:
    services = get_active_services(date, calendar, calendar_dates)

    available = set(realtime_daily["service_id"].astype(str))
    overlap = services & available

    print(f"\nDate: {date}")
    print(f"Active GTFS services: {len(services)}")
    print(f"Services present in realtime_daily: {len(overlap)} / {len(available)}")
    print(f"Matching timetable rows: {realtime_daily['service_id'].astype(str).isin(services).sum():,}")



Date: 29/10/2026
Active GTFS services: 692
Services present in realtime_daily: 3 / 5
Matching timetable rows: 17,111

Date: 31/10/2026
Active GTFS services: 449
Services present in realtime_daily: 0 / 5
Matching timetable rows: 0

Date: 01/11/2026
Active GTFS services: 364
Services present in realtime_daily: 0 / 5
Matching timetable rows: 0

Date: 25/12/2026
Active GTFS services: 321
Services present in realtime_daily: 0 / 5
Matching timetable rows: 0


In [276]:

# STEP 106 — Inspect planned GTFS timetable sources

import pandas as pd

for name in [
    "stop_times",
    "planned_trips",
    "scheduled_daily",
    "trips",
    "routes"
]:
    if name in globals():
        obj = globals()[name]

        print(f"\n{name}")
        print("Type:", type(obj).__name__)

        if isinstance(obj, pd.DataFrame):
            print("Rows:", f"{len(obj):,}")
            print("Columns:", list(obj.columns))
            display(obj.head(2))
        else:
            print("Preview:", str(obj)[:200])
    else:
        print(f"\n{name}: Not loaded")

print("\nCurrent real-time timetable:")
print("Rows:", f"{len(realtime_daily):,}")
print("Unique services:", realtime_daily["service_id"].nunique())



stop_times
Type: DataFrame
Rows: 5,705,967
Columns: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type', 'shape_dist_traveled', 'timepoint', 'stop_note']


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,stop_note
0,1.AA61.1-SC0-1-sj2-1.1.R,22:50:00,22:50:00,254111,1,NaN,0,0,0.00,1,NaN
1,1.AA61.1-SC0-1-sj2-1.1.R,23:04:00,23:04:00,253572,2,NaN,0,0,16208.96,1,NaN



planned_trips: Not loaded

scheduled_daily: Not loaded

trips
Type: DataFrame
Rows: 211,419
Columns: ['route_id', 'service_id', 'trip_id', 'shape_id', 'trip_headsign', 'direction_id', 'block_id', 'wheelchair_accessible', 'route_direction', 'trip_note', 'bikes_allowed']


,route_id,service_id,trip_id,shape_id,trip_headsign,direction_id,block_id,wheelchair_accessible,route_direction,trip_note,bikes_allowed
0,1-SC0-1-sj2-1,AA61#1+1,1.AA61.1-SC0-1-sj2-1.1.R,1-SC0-1-sj2-1.1.R,Kiama,1,NaN,1,Bomaderry to Kiama,NaN,NaN
1,1-SC0-1-sj2-1,AB22#1+1,1.AB22.1-SC0-1-sj2-1.1.R,1-SC0-1-sj2-1.1.R,Kiama,1,NaN,1,Bomaderry to Kiama,NaN,NaN



routes
Type: DataFrame
Rows: 165
Columns: ['boarding_id', 'trip_id', 'origin_platform', 'origin_sequence', 'origin_departure_seconds', 'origin_departure_time', 'first_line', 'first_mode', 'stop_id_x', 'station_id', 'stop_sequence', 'arrival_seconds', 'arrival_time', 'from_stop_id', 'to_stop_id', 'minimum_transfer_seconds', 'earliest_boarding_seconds', 'second_trip_id', 'second_platform', 'second_boarding_sequence', 'second_departure_seconds', 'second_departure_time', 'second_line', 'second_mode', 'destination_sequence', 'destination_arrival_seconds', 'destination_arrival_time', 'journey_duration_min', 'journey_type', 'transfers', 'stop_id_y', 'transfer_station', 'transfer_wait_min', 'departure', 'transfer_arrival', 'transfer_departure', 'arrival']


,boarding_id,trip_id,origin_platform,origin_sequence,origin_departure_seconds,origin_departure_time,first_line,first_mode,stop_id_x,station_id,...,journey_duration_min,journey_type,transfers,stop_id_y,transfer_station,transfer_wait_min,departure,transfer_arrival,transfer_departure,arrival
0,0,610A.1324.165.48.T.8.92001719,2217191,4,29220.0,08:07:00,T4,Train,204475,204420,...,39.82,One transfer,1,204471,Sydenham Station,11.700000,08:07,08:20,08:32,08:46
6,0,610A.1324.165.48.T.8.92001719,2217191,4,29220.0,08:07:00,T4,Train,2000344,200060,...,39.82,One transfer,1,2000467,Central Station,9.833333,08:07,08:28,08:38,08:46



Current real-time timetable:
Rows: 59,541
Unique services: 5


In [277]:

# STEP 107 — Inspect mappings for scheduled journey construction

import pandas as pd

for name in [
    "transport_stops",
    "station_lookup",
    "station_names",
    "platform_lookup",
    "gtfs_routes",
    "routes_gtfs"
]:
    if name not in globals():
        print(f"\n{name}: NOT FOUND")
        continue

    obj = globals()[name]
    print(f"\n{name}: {type(obj).__name__}")

    if isinstance(obj, pd.DataFrame):
        print("Rows:", f"{len(obj):,}")
        print("Columns:", list(obj.columns))
        display(obj.head(2))

    elif isinstance(obj, dict):
        print("Entries:", f"{len(obj):,}")
        print("Sample:", list(obj.items())[:3])

    else:
        print("Preview:", str(obj)[:300])

print("\n=== TRIP ID MATCHING ===")

planned_ids = set(trips["trip_id"].astype(str))
current_ids = set(realtime_daily["trip_id"].astype(str))

print("Current trip IDs:", len(current_ids))
print("Matching planned trip IDs:", len(current_ids & planned_ids))
print("Unmatched current trip IDs:", len(current_ids - planned_ids))



transport_stops: DataFrame
Rows: 568
Columns: ['stop_id', 'stop_code', 'stop_name', 'stop_lat', 'stop_lon', 'location_type', 'parent_station', 'wheelchair_boarding', 'level_id', 'platform_code', 'station_id', 'transport_mode']


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code,station_id,transport_mode
0,2113341,2113341,"Macquarie Park Station, Platform 1",-33.78498208,151.12833711,NaN,211340,1,Level -3,1,211340,Metro
1,2113342,2113342,"Macquarie Park Station, Platform 2",-33.78506593,151.12825481,NaN,211340,1,Level -3,2,211340,Metro



station_lookup: DataFrame
Rows: 518
Columns: ['stop_id', 'station_id', 'stop_name']


,stop_id,station_id,stop_name
0,2155269,2155384,Tallawong Station
1,2155267,2155383,Rouse Hill Station



station_names: DataFrame
Rows: 191
Columns: ['station_id', 'transfer_station']


,station_id,transfer_station
0,2155384,Tallawong Station
1,2155383,Rouse Hill Station



platform_lookup: dict
Entries: 568
Sample: [('2113341', 'Platform 1'), ('2113342', 'Platform 2'), ('2113351', 'Platform 1')]

gtfs_routes: NOT FOUND

routes_gtfs: NOT FOUND

=== TRIP ID MATCHING ===
Current trip IDs: 3705
Matching planned trip IDs: 3705
Unmatched current trip IDs: 0


In [278]:

# STEP 108 — Build scheduled train/metro timetable for a selected date

import pandas as pd
import time

def gtfs_time_to_seconds(series):
    """Convert GTFS HH:MM:SS, including hours >= 24, into seconds."""
    parts = series.astype(str).str.split(":", expand=True)
    return (
        pd.to_numeric(parts[0], errors="coerce") * 3600
        + pd.to_numeric(parts[1], errors="coerce") * 60
        + pd.to_numeric(parts[2], errors="coerce")
    )


def get_scheduled_timetable_for_date(journey_date):
    start = time.perf_counter()

    # 1. Active services for the requested date
    active_services = get_active_services(
        journey_date, calendar, calendar_dates
    )

    # 2. Trips operating on that date
    active_trips = trips.loc[
        trips["service_id"].astype(str).isin(active_services),
        ["trip_id", "service_id", "route_id"]
    ].copy()

    # 3. Select stop times for active trips
    daily = stop_times.loc[
        stop_times["trip_id"].isin(active_trips["trip_id"]),
        ["trip_id", "stop_id", "arrival_time",
         "departure_time", "stop_sequence"]
    ].copy()

    daily = daily.merge(
        active_trips,
        on="trip_id",
        how="inner",
        validate="many_to_one"
    )

    # 4. Keep known train and metro platforms
    stops_map = transport_stops[
        ["stop_id", "station_id", "transport_mode"]
    ].drop_duplicates("stop_id").copy()

    daily["stop_id"] = daily["stop_id"].astype(str)
    stops_map["stop_id"] = stops_map["stop_id"].astype(str)

    daily = daily.merge(
        stops_map,
        on="stop_id",
        how="inner",
        validate="many_to_one"
    )

    # 5. Map route labels using existing trip metadata where possible
    known_lines = realtime_daily[
        ["trip_id", "route_short_name"]
    ].drop_duplicates("trip_id")

    daily = daily.merge(
        known_lines,
        on="trip_id",
        how="left",
        validate="many_to_one"
    )

    # 6. Convert scheduled times into seconds
    daily["arrival_seconds"] = gtfs_time_to_seconds(
        daily["arrival_time"]
    )
    daily["departure_seconds"] = gtfs_time_to_seconds(
        daily["departure_time"]
    )

    daily["scheduled_arrival_sec"] = daily["arrival_seconds"]
    daily["scheduled_departure_sec"] = daily["departure_seconds"]

    daily["effective_arrival_sec"] = daily["arrival_seconds"]
    daily["effective_departure_sec"] = daily["departure_seconds"]

    daily["arrival_source"] = "scheduled"
    daily["departure_source"] = "scheduled"

    daily["predicted_arrival_sec"] = pd.NA
    daily["predicted_departure_sec"] = pd.NA

    daily["original_arrival_seconds"] = daily["arrival_seconds"]
    daily["original_departure_seconds"] = daily["departure_seconds"]

    daily = daily.sort_values(
        ["trip_id", "stop_sequence"]
    ).reset_index(drop=True)

    print(f"Date: {journey_date}")
    print(f"Active services: {len(active_services):,}")
    print(f"Active trips: {len(active_trips):,}")
    print(f"Train/metro stop events: {len(daily):,}")
    print(f"Missing line labels: {daily['route_short_name'].isna().sum():,}")
    print(f"Processing time: {time.perf_counter() - start:.2f}s")

    return daily


# Test tomorrow: Friday, 30 October 2026
scheduled_tomorrow = get_scheduled_timetable_for_date("30/10/2026")

display(scheduled_tomorrow.head(5))


Date: 30/10/2026
Active services: 720
Active trips: 60,344
Train/metro stop events: 67,620
Missing line labels: 67,620
Processing time: 2.37s


,trip_id,stop_id,arrival_time,departure_time,stop_sequence,service_id,route_id,station_id,transport_mode,route_short_name,...,scheduled_arrival_sec,scheduled_departure_sec,effective_arrival_sec,effective_departure_sec,arrival_source,departure_source,predicted_arrival_sec,predicted_departure_sec,original_arrival_seconds,original_departure_seconds
0,0178-001-101-002:1000,2155269,04:05:00,04:05:00,1,TA+R59+11,3-M1-sj2-1,2155384,Metro,NaN,...,14700,14700,14700,14700,scheduled,scheduled,<NA>,<NA>,14700,14700
1,0178-001-101-002:1000,2113351,04:31:48,04:32:18,10,TA+R59+11,3-M1-sj2-1,211310,Metro,NaN,...,16308,16338,16308,16338,scheduled,scheduled,<NA>,<NA>,16308,16338
2,0178-001-101-002:1000,2113341,04:33:47,04:34:17,11,TA+R59+11,3-M1-sj2-1,211340,Metro,NaN,...,16427,16457,16427,16457,scheduled,scheduled,<NA>,<NA>,16427,16457
3,0178-001-101-002:1000,2113361,04:35:51,04:36:21,12,TA+R59+11,3-M1-sj2-1,211320,Metro,NaN,...,16551,16581,16551,16581,scheduled,scheduled,<NA>,<NA>,16551,16581
4,0178-001-101-002:1000,2067142,04:41:21,04:42:06,13,TA+R59+11,3-M1-sj2-1,206710,Metro,NaN,...,16881,16926,16881,16926,scheduled,scheduled,<NA>,<NA>,16881,16926


In [279]:

# STEP 109 — Recover scheduled train and metro line labels

import pandas as pd

# Inspect route IDs before assigning labels
route_summary = (
    scheduled_tomorrow
    .groupby(["route_id", "transport_mode"], dropna=False)
    .agg(
        trips=("trip_id", "nunique"),
        stop_events=("stop_id", "size")
    )
    .reset_index()
    .sort_values("stop_events", ascending=False)
)

print("=== ROUTE IDS IN TOMORROW'S TIMETABLE ===")
display(route_summary)

# Extract route identifier from patterns such as 3-M1-sj2-1
scheduled_tomorrow["route_short_name"] = (
    scheduled_tomorrow["route_id"]
    .astype(str)
    .str.extract(r"^\d+-([A-Za-z]+\d+)-", expand=False)
)

# Use transport mode as a fallback, but keep unknown lines
# identifiable rather than inventing a route number.
scheduled_tomorrow["service_label"] = (
    scheduled_tomorrow["route_short_name"].fillna(
        scheduled_tomorrow["transport_mode"]
    )
)

print("\n=== LINE LABEL COVERAGE ===")
print("Total rows:", f"{len(scheduled_tomorrow):,}")
print(
    "Missing line numbers:",
    f"{scheduled_tomorrow['route_short_name'].isna().sum():,}"
)

display(
    scheduled_tomorrow[
        [
            "trip_id",
            "route_id",
            "route_short_name",
            "transport_mode",
            "service_label"
        ]
    ].drop_duplicates("route_id").head(20)
)


=== ROUTE IDS IN TOMORROW'S TIMETABLE ===


,route_id,transport_mode,trips,stop_events
13,3-M1-sj2-1,Metro,445,13795
5,2-T2-sj2-1,Train,397,8043
11,2-T8-sj2-1,Train,468,7530
7,2-T4-sj2-1,Train,371,7176
12,2-T9-sj2-1,Train,364,5478
3,2-T1-N-sj2-1,Train,293,5346
4,2-T1-W-sj2-1,Train,296,4898
34,78-L3-sj2-1,Light Rail,279,4166
33,78-L2-sj2-1,Light Rail,278,3888
6,2-T3-sj2-1,Train,98,2256



=== LINE LABEL COVERAGE ===
Total rows: 67,620
Missing line numbers: 1,731


,trip_id,route_id,route_short_name,transport_mode,service_label
0,0178-001-101-002:1000,3-M1-sj2-1,M1,Metro,M1
13795,1--B.1324.166.116.B.8.92000948,2-T2-sj2-1,T2,Train,T2
13815,1--C.1324.166.116.B.8.92001634,2-T8-sj2-1,T8,Train,T8
14469,10-T.1324.166.64.B.8.91999827,2-T3-sj2-1,T3,Train,T3
14500,101A.1324.166.64.T.8.92005032,2-T9-sj2-1,T9,Train,T9
14987,103B.1324.166.64.T.8.92357014,2-T5-sj2-1,T5,Train,T5
14998,103C.1324.166.64.T.8.92001044,2-T1-W-sj2-1,T1,Train,T1
15019,103D.1324.166.64.T.8.92001046,2-T1-N-sj2-1,T1,Train,T1
34691,281B.1324.166.64.H.8.92003336,2-CCN-sj2-1,NaN,Train,Train
36264,314B.1324.166.64.H.8.92001548,2-SCO-sj2-1,NaN,Train,Train


In [280]:

# STEP 110 — Complete scheduled route labels and validate routing data

import pandas as pd

# Extract the full route code between the GTFS prefix and -sj2-
# Examples:
# 2-T1-W-sj2-1 -> T1
# 2-CCN-sj2-1  -> CCN
# 76-223-sj2-1 -> 223

route_code = scheduled_tomorrow["route_id"].astype(str).str.extract(
    r"^\d+-(.+?)-sj2-\d+$",
    expand=False
)

# Remove directional suffixes for T1 variants
route_code = route_code.str.replace(
    r"^(T\d+)-[A-Z]$",
    r"\1",
    regex=True
)

scheduled_tomorrow["route_short_name"] = route_code

# Keep a fallback for unusual route IDs
scheduled_tomorrow["route_short_name"] = (
    scheduled_tomorrow["route_short_name"]
    .fillna(scheduled_tomorrow["route_id"].astype(str))
)

# Preserve the mode used by the routing engine
scheduled_tomorrow["transport_mode"] = (
    scheduled_tomorrow["transport_mode"].astype(str)
)

# Check station coverage
known_stations = set(station_names["station_id"].astype(str))

scheduled_tomorrow["station_id"] = (
    scheduled_tomorrow["station_id"].astype(str)
)

print("=== SCHEDULED TIMETABLE VALIDATION ===")
print("Rows:", f"{len(scheduled_tomorrow):,}")
print("Trips:", f"{scheduled_tomorrow['trip_id'].nunique():,}")
print(
    "Missing line labels:",
    scheduled_tomorrow["route_short_name"].isna().sum()
)
print(
    "Unknown station IDs:",
    (~scheduled_tomorrow["station_id"].isin(known_stations)).sum()
)

print("\n=== ROUTE COVERAGE ===")
display(
    scheduled_tomorrow.groupby(
        ["route_short_name", "transport_mode"]
    ).agg(
        trips=("trip_id", "nunique"),
        events=("stop_id", "size")
    ).reset_index().sort_values(
        "events", ascending=False
    ).head(25)
)

print("\n=== REQUIRED ROUTING COLUMNS ===")

required_columns = [
    "trip_id",
    "stop_id",
    "arrival_time",
    "departure_time",
    "stop_sequence",
    "route_short_name",
    "transport_mode",
    "service_id",
    "arrival_seconds",
    "departure_seconds",
    "station_id"
]

for col in required_columns:
    print(f"{col}: {'OK' if col in scheduled_tomorrow.columns else 'MISSING'}")


=== SCHEDULED TIMETABLE VALIDATION ===
Rows: 67,620
Trips: 4,138
Missing line labels: 0
Unknown station IDs: 4005

=== ROUTE COVERAGE ===


,route_short_name,transport_mode,trips,events
22,M1,Metro,445,13795
25,T1,Train,589,10244
26,T2,Train,397,8043
32,T8,Train,468,7530
28,T4,Train,371,7176
33,T9,Train,364,5478
20,L3,Light Rail,279,4166
19,L2,Light Rail,278,3888
27,T3,Train,98,2256
29,T5,Train,101,1885



=== REQUIRED ROUTING COLUMNS ===
trip_id: OK
stop_id: OK
arrival_time: OK
departure_time: OK
stop_sequence: OK
route_short_name: OK
transport_mode: OK
service_id: OK
arrival_seconds: OK
departure_seconds: OK
station_id: OK


In [281]:

# STEP 111 — Investigate station coverage

import pandas as pd

known_stations = set(station_names["station_id"].astype(str))

unknown = scheduled_tomorrow.loc[
    ~scheduled_tomorrow["station_id"].astype(str).isin(known_stations)
].copy()

print("=== UNKNOWN STATION EVENTS ===")
print("Total:", f"{len(unknown):,}")
print("Unique station IDs:", unknown["station_id"].nunique())

print("\n=== BREAKDOWN BY TRANSPORT MODE ===")
display(
    unknown.groupby("transport_mode")
    .agg(
        events=("stop_id", "size"),
        stations=("station_id", "nunique")
    )
    .reset_index()
)

print("\n=== BREAKDOWN BY ROUTE ===")
display(
    unknown.groupby(["route_short_name", "transport_mode"])
    .agg(
        events=("stop_id", "size"),
        stations=("station_id", "nunique")
    )
    .reset_index()
    .sort_values("events", ascending=False)
    .head(15)
)

print("\n=== CHECK KOGARAH AND RICHMOND ===")

for station in ["Kogarah Station", "Richmond Station"]:
    matches = station_names.loc[
        station_names["transfer_station"] == station
    ]

    if matches.empty:
        print(f"{station}: NOT FOUND in station_names")
        continue

    ids = set(matches["station_id"].astype(str))

    events = scheduled_tomorrow.loc[
        scheduled_tomorrow["station_id"].astype(str).isin(ids)
    ]

    print(
        f"{station}: {len(events):,} scheduled events, "
        f"{events['trip_id'].nunique():,} trips"
    )


=== UNKNOWN STATION EVENTS ===
Total: 4,005
Unique station IDs: 9

=== BREAKDOWN BY TRANSPORT MODE ===


,transport_mode,events,stations
0,Metro,4005,9



=== BREAKDOWN BY ROUTE ===


,route_short_name,transport_mode,events,stations
0,M1,Metro,4005,9



=== CHECK KOGARAH AND RICHMOND ===
Kogarah Station: 235 scheduled events, 235 trips
Richmond Station: 84 scheduled events, 84 trips


In [282]:

# STEP 112 — Test scheduled Arrive By journey

import time
import pandas as pd

origin = "Kogarah Station"
destination = "Richmond Station"
journey_date = "30/10/2026"
arrive_by_time = "09:00"

start = time.perf_counter()

# Use the planned timetable, not today's real-time data
scheduled_arrive_by = plan_arrive_by_test(
    origin,
    destination,
    arrive_by_time,
    daily=scheduled_tomorrow
)

print(f"\n=== PLANNED JOURNEY: {journey_date} ===")
print(f"From: {origin}")
print(f"To: {destination}")
print(f"Arrive by: {arrive_by_time}")
print(f"Processing time: {time.perf_counter() - start:.2f}s")

if scheduled_arrive_by is None or scheduled_arrive_by.empty:
    print("\nNo scheduled journeys found for this search.")
else:
    print(f"\nJourneys found: {len(scheduled_arrive_by):,}")

    display(
        scheduled_arrive_by[
            [
                "journey_type",
                "transfers",
                "departure_time",
                "arrival_time",
                "journey_duration_min",
                "first_transfer_station",
                "second_transfer_station"
            ]
        ].head(10)
    )


ValueError: not enough values to unpack (expected 3, got 2)

In [283]:

import traceback
import inspect

print("=== FUNCTION SIGNATURE ===")
print(inspect.signature(plan_arrive_by_test))

print("\n=== FULL ERROR TRACEBACK ===")

try:
    scheduled_arrive_by = plan_arrive_by_test(
        "Kogarah Station",
        "Richmond Station",
        "09:00",
        daily=scheduled_tomorrow
    )

    print("Function completed successfully.")
    print("Returned type:", type(scheduled_arrive_by).__name__)

except Exception:
    traceback.print_exc()


=== FUNCTION SIGNATURE ===
(origin, destination, arrival_time, daily)

=== FULL ERROR TRACEBACK ===


Traceback (most recent call last):
  File "/var/folders/w6/tgkpq1sx16s4f4k9pbscyzx80000gn/T/ipykernel_99174/1836313717.py", line 10, in <module>
    scheduled_arrive_by = plan_arrive_by_test(
                          ^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/w6/tgkpq1sx16s4f4k9pbscyzx80000gn/T/ipykernel_99174/3595825426.py", line 7, in plan_arrive_by_test
    h, m, s = map(int, arrival_time.split(":"))
    ^^^^^^^
ValueError: not enough values to unpack (expected 3, got 2)


In [284]:

import time

start = time.perf_counter()

scheduled_arrive_by = plan_arrive_by_test(
    origin="Kogarah Station",
    destination="Richmond Station",
    arrival_time="09:00:00",
    daily=scheduled_tomorrow
)

print("=== SCHEDULED JOURNEY ===")
print("Date: 30/10/2026")
print("Kogarah Station → Richmond Station")
print("Arrive by: 09:00")
print(f"Processing time: {time.perf_counter() - start:.2f}s")

if scheduled_arrive_by is None or scheduled_arrive_by.empty:
    print("No journeys found.")
else:
    print(f"Journeys found: {len(scheduled_arrive_by):,}")
    display(scheduled_arrive_by.head(10))


Found valid journeys in window starting 07:00:00
Processing time: 3.77s
=== SCHEDULED JOURNEY ===
Date: 30/10/2026
Kogarah Station → Richmond Station
Arrive by: 09:00
Processing time: 3.77s
Journeys found: 3


,journey_type,transfers,first_trip_id,second_trip_id,third_trip_id,origin_departure_seconds,destination_arrival_seconds,journey_duration_min,first_transfer_station,second_transfer_station,departure_time,arrival_time
0,One transfer,1,602B.1324.166.64.T.8.92002528,172F.1324.166.64.A.8.92000952,None,26952,30900,65.8,Central Station,None,07:29:12,08:35:00
1,One transfer,1,614C.1324.166.64.T.8.91999408,172F.1324.166.64.A.8.92000952,None,26250,30900,77.5,Central Station,None,07:17:30,08:35:00
2,One transfer,1,601B.1324.166.64.T.8.92003576,172F.1324.166.64.A.8.92000952,None,25524,30900,89.6,Central Station,None,07:05:24,08:35:00


In [285]:

# STEP 115 — Display the recommended scheduled journey itinerary

import pandas as pd

best = scheduled_arrive_by.iloc[0]

first_trip = scheduled_tomorrow.loc[
    scheduled_tomorrow["trip_id"] == best["first_trip_id"]
].sort_values("stop_sequence").copy()

second_trip = scheduled_tomorrow.loc[
    scheduled_tomorrow["trip_id"] == best["second_trip_id"]
].sort_values("stop_sequence").copy()

transfer_name = best["first_transfer_station"]

# Look up station IDs
name_to_id = dict(
    zip(
        station_names["transfer_station"],
        station_names["station_id"].astype(str)
    )
)

origin_id = name_to_id["Kogarah Station"]
transfer_id = name_to_id[transfer_name]
destination_id = name_to_id["Richmond Station"]

def get_event(trip, station_id, event_type):
    rows = trip.loc[
        trip["station_id"].astype(str) == station_id
    ]

    if rows.empty:
        raise ValueError(
            f"Station {station_id} not found on selected trip"
        )

    row = rows.iloc[0]

    return {
        "station": station_id,
        "time": row[
            "departure_time" if event_type == "Depart" else "arrival_time"
        ][:5],
        "platform": platform_lookup.get(
            str(row["stop_id"]), "Unknown"
        ),
        "service": (
            f"{row['route_short_name']} "
            f"{row['transport_mode']}"
        )
    }

events = [
    ("Depart", get_event(first_trip, origin_id, "Depart")),
    ("Arrive", get_event(first_trip, transfer_id, "Arrive")),
    ("Depart", get_event(second_trip, transfer_id, "Depart")),
    ("Arrive", get_event(second_trip, destination_id, "Arrive"))
]

station_id_to_name = dict(
    zip(
        station_names["station_id"].astype(str),
        station_names["transfer_station"]
    )
)

itinerary = pd.DataFrame([
    {
        "stage": stage,
        "station": station_id_to_name.get(
            event["station"], event["station"]
        ),
        "time": event["time"],
        "platform": event["platform"],
        "service": event["service"]
    }
    for stage, event in events
])

print("=== RECOMMENDED SCHEDULED JOURNEY ===")
print("Date: 30/10/2026")
print("Kogarah Station → Richmond Station")
print("Arrive by: 09:00")
print(
    f"Departure: {best['departure_time'][:5]} | "
    f"Arrival: {best['arrival_time'][:5]}"
)
print(f"Transfer: {transfer_name}")

display(itinerary)


=== RECOMMENDED SCHEDULED JOURNEY ===
Date: 30/10/2026
Kogarah Station → Richmond Station
Arrive by: 09:00
Departure: 07:29 | Arrival: 08:35
Transfer: Central Station


,stage,station,time,platform,service
0,Depart,Kogarah Station,07:29,Platform 4,T4 Train
1,Arrive,Central Station,07:09,Platform 25,T4 Train
2,Depart,Central Station,07:15,Platform 18,T1 Train
3,Arrive,Richmond Station,08:35,Platform 2,T1 Train


In [286]:

# STEP 116 — Diagnose scheduled journey timing

best = scheduled_arrive_by.iloc[0]

first_trip = scheduled_tomorrow.loc[
    scheduled_tomorrow["trip_id"] == best["first_trip_id"]
].sort_values("stop_sequence")

second_trip = scheduled_tomorrow.loc[
    scheduled_tomorrow["trip_id"] == best["second_trip_id"]
].sort_values("stop_sequence")

print("=== FIRST TRIP: KOGARAH TO CENTRAL ===")
print("Trip ID:", best["first_trip_id"])

display(
    first_trip[
        first_trip["station_id"].astype(str).isin([
            origin_id, transfer_id
        ])
    ][[
        "stop_sequence",
        "station_id",
        "stop_id",
        "arrival_time",
        "departure_time",
        "route_short_name"
    ]]
)

print("\n=== SECOND TRIP: CENTRAL TO RICHMOND ===")
print("Trip ID:", best["second_trip_id"])

display(
    second_trip[
        second_trip["station_id"].astype(str).isin([
            transfer_id, destination_id
        ])
    ][[
        "stop_sequence",
        "station_id",
        "stop_id",
        "arrival_time",
        "departure_time",
        "route_short_name"
    ]]
)

print("\n=== ROUTING RESULT ===")
print("Origin departure:", best["departure_time"])
print("Destination arrival:", best["arrival_time"])
print("Transfer station:", best["first_transfer_station"])


=== FIRST TRIP: KOGARAH TO CENTRAL ===
Trip ID: 602B.1324.166.64.T.8.92002528


,stop_sequence,station_id,stop_id,arrival_time,departure_time,route_short_name
50853,16,221710,2217194,07:28:42,07:29:12,T4
50868,6,200060,2000345,07:09:00,07:10:00,T4



=== SECOND TRIP: CENTRAL TO RICHMOND ===
Trip ID: 172F.1324.166.64.A.8.92000952


,stop_sequence,station_id,stop_id,arrival_time,departure_time,route_short_name
27538,1,200060,2000338,07:15:01,07:15:01,T1
27557,32,275320,2753672,08:35:00,08:35:00,T1



=== ROUTING RESULT ===
Origin departure: 07:29:12
Destination arrival: 08:35:00
Transfer station: Central Station


In [287]:

# STEP 117 — FIXED SCHEDULED ARRIVE-BY JOURNEY PLANNER
# Validates direction, chronology and transfer feasibility.
# Supports direct and one-transfer journeys.
# Does not yet search two-transfer journeys.

import pandas as pd
import numpy as np
import time

def normalise_time(value):
    parts = str(value).split(":")
    if len(parts) == 2:
        return str(value) + ":00"
    if len(parts) == 3:
        return str(value)
    raise ValueError("Time must be HH:MM or HH:MM:SS")


def seconds_to_clock(seconds):
    seconds = int(seconds)
    return f"{(seconds // 3600) % 24:02d}:{(seconds % 3600) // 60:02d}"


def fixed_arrive_by(
    origin,
    destination,
    arrival_time,
    daily,
    max_lookback_hours=5,
    minimum_transfer_minutes=5
):
    start = time.perf_counter()

    arrival_time = normalise_time(arrival_time)
    h, m, s = map(int, arrival_time.split(":"))
    deadline = h * 3600 + m * 60 + s

    earliest_departure = max(
        0, deadline - max_lookback_hours * 3600
    )

    station_ids = dict(
        zip(
            station_names["transfer_station"],
            station_names["station_id"].astype(str)
        )
    )

    if origin not in station_ids or destination not in station_ids:
        raise ValueError("Origin or destination not found")

    origin_id = station_ids[origin]
    destination_id = station_ids[destination]

    data = daily.copy()
    data["station_id"] = data["station_id"].astype(str)
    data["stop_id"] = data["stop_id"].astype(str)

    data["arrival_seconds"] = pd.to_numeric(
        data["arrival_seconds"], errors="coerce"
    )
    data["departure_seconds"] = pd.to_numeric(
        data["departure_seconds"], errors="coerce"
    )
    data["stop_sequence"] = pd.to_numeric(
        data["stop_sequence"], errors="coerce"
    )

    data = data.dropna(
        subset=[
            "arrival_seconds",
            "departure_seconds",
            "stop_sequence"
        ]
    )

    # Index all station events to avoid repeatedly scanning timetable.
    station_events = {
        station: group
        for station, group in data.groupby("station_id")
    }

    origin_events = station_events.get(
        origin_id, data.iloc[0:0]
    )

    destination_events = station_events.get(
        destination_id, data.iloc[0:0]
    )

    origin_events = origin_events.loc[
        origin_events["departure_seconds"].between(
            earliest_departure, deadline
        )
    ]

    destination_events = destination_events.loc[
        destination_events["arrival_seconds"] <= deadline
    ]

    results = []

    # --------------------------------------------------
    # DIRECT JOURNEYS
    # --------------------------------------------------

    direct = origin_events.merge(
        destination_events,
        on="trip_id",
        suffixes=("_origin", "_destination")
    )

    direct = direct.loc[
        (direct["stop_sequence_origin"] <
         direct["stop_sequence_destination"])
        &
        (direct["departure_seconds_origin"] <
         direct["arrival_seconds_destination"])
    ]

    for row in direct.itertuples(index=False):
        results.append({
            "journey_type": "Direct",
            "transfers": 0,
            "first_trip_id": row.trip_id,
            "second_trip_id": None,
            "origin_departure_seconds":
                row.departure_seconds_origin,
            "destination_arrival_seconds":
                row.arrival_seconds_destination,
            "first_transfer_station": None,
            "first_transfer_id": None,
            "first_transfer_arrival_seconds": None,
            "first_transfer_departure_seconds": None,
            "first_transfer_arrival_stop_id": None,
            "first_transfer_departure_stop_id": None,
            "origin_stop_id": row.stop_id_origin,
            "destination_stop_id": row.stop_id_destination,
            "first_line": row.route_short_name_origin,
            "second_line": None
        })

    # --------------------------------------------------
    # ONE-TRANSFER JOURNEYS
    # --------------------------------------------------

    # Candidate first legs: origin -> interchange
    origin_trip_ids = origin_events["trip_id"].unique()

    first_leg_stops = data.loc[
        data["trip_id"].isin(origin_trip_ids)
    ].copy()

    first_legs = origin_events.merge(
        first_leg_stops,
        on="trip_id",
        suffixes=("_origin", "_transfer")
    )

    first_legs = first_legs.loc[
        (first_legs["stop_sequence_origin"] <
         first_legs["stop_sequence_transfer"])
        &
        (first_legs["arrival_seconds_transfer"] >
         first_legs["departure_seconds_origin"])
        &
        (first_legs["arrival_seconds_transfer"] < deadline)
        &
        (first_legs["station_id_transfer"] != origin_id)
        &
        (first_legs["station_id_transfer"] != destination_id)
    ]

    # Candidate second legs: interchange -> destination
    destination_trip_ids = destination_events[
        "trip_id"
    ].unique()

    second_leg_stops = data.loc[
        data["trip_id"].isin(destination_trip_ids)
    ].copy()

    second_legs = second_leg_stops.merge(
        destination_events,
        on="trip_id",
        suffixes=("_transfer", "_destination")
    )

    second_legs = second_legs.loc[
        (second_legs["stop_sequence_transfer"] <
         second_legs["stop_sequence_destination"])
        &
        (second_legs["departure_seconds_transfer"] <
         second_legs["arrival_seconds_destination"])
        &
        (second_legs["station_id_transfer"] != origin_id)
        &
        (second_legs["station_id_transfer"] != destination_id)
    ]

    # Join legs by transfer station
    connections = first_legs.merge(
        second_legs,
        left_on="station_id_transfer",
        right_on="station_id_transfer",
        suffixes=("_first", "_second")
    )

    # Check transfer is chronological and has enough time.
    min_transfer_seconds = minimum_transfer_minutes * 60

    connections = connections.loc[
        (connections["trip_id_first"] !=
         connections["trip_id_second"])
        &
        (
            connections["departure_seconds_transfer_second"]
            >=
            connections["arrival_seconds_transfer_first"]
            + min_transfer_seconds
        )
        &
        (
            connections["arrival_seconds_destination"]
            <= deadline
        )
    ]

    id_to_name = dict(
        zip(
            station_names["station_id"].astype(str),
            station_names["transfer_station"]
        )
    )

    for row in connections.itertuples(index=False):
        results.append({
            "journey_type": "One transfer",
            "transfers": 1,
            "first_trip_id": row.trip_id_first,
            "second_trip_id": row.trip_id_second,
            "origin_departure_seconds":
                row.departure_seconds_origin,
            "destination_arrival_seconds":
                row.arrival_seconds_destination,
            "first_transfer_station": id_to_name.get(
                str(row.station_id_transfer),
                str(row.station_id_transfer)
            ),
            "first_transfer_id":
                str(row.station_id_transfer),
            "first_transfer_arrival_seconds":
                row.arrival_seconds_transfer_first,
            "first_transfer_departure_seconds":
                row.departure_seconds_transfer_second,
            "first_transfer_arrival_stop_id":
                row.stop_id_transfer_first,
            "first_transfer_departure_stop_id":
                row.stop_id_transfer_second,
            "origin_stop_id": row.stop_id_origin,
            "destination_stop_id": row.stop_id_destination,
            "first_line": row.route_short_name_origin,
            "second_line": row.route_short_name_destination
        })

    if not results:
        print("No valid journeys found.")
        return pd.DataFrame()

    result = pd.DataFrame(results)

    result["journey_duration_min"] = (
        result["destination_arrival_seconds"]
        - result["origin_departure_seconds"]
    ) / 60

    result["departure_time"] = result[
        "origin_departure_seconds"
    ].apply(seconds_to_clock)

    result["arrival_time"] = result[
        "destination_arrival_seconds"
    ].apply(seconds_to_clock)

    result = result.sort_values(
        [
            "origin_departure_seconds",
            "transfers",
            "destination_arrival_seconds"
        ],
        ascending=[False, True, True]
    )

    # Remove identical journey options, but retain alternatives.
    result = result.drop_duplicates(
        subset=[
            "first_trip_id",
            "second_trip_id",
            "origin_departure_seconds",
            "destination_arrival_seconds",
            "first_transfer_id"
        ]
    ).reset_index(drop=True)

    print(f"Valid journeys: {len(result):,}")
    print(f"Search time: {time.perf_counter() - start:.2f}s")

    return result


def show_fixed_itinerary(journey, daily, origin, destination):
    """Display exact events used by the validated journey."""

    station_id_to_name = dict(
        zip(
            station_names["station_id"].astype(str),
            station_names["transfer_station"]
        )
    )

    def event(stage, station, seconds, stop_id, line, mode):
        return {
            "stage": stage,
            "station": station,
            "time": seconds_to_clock(seconds),
            "platform": platform_lookup.get(
                str(stop_id), "Unknown"
            ),
            "service": f"{line} {mode}"
        }

    first_trip = daily.loc[
        daily["trip_id"] == journey["first_trip_id"]
    ]

    first_mode = first_trip["transport_mode"].iloc[0]

    itinerary_rows = [
        event(
            "Depart",
            origin,
            journey["origin_departure_seconds"],
            journey["origin_stop_id"],
            journey["first_line"],
            first_mode
        )
    ]

    if journey["transfers"] == 1:
        second_trip = daily.loc[
            daily["trip_id"] == journey["second_trip_id"]
        ]

        second_mode = second_trip[
            "transport_mode"
        ].iloc[0]

        itinerary_rows.extend([
            event(
                "Arrive",
                journey["first_transfer_station"],
                journey["first_transfer_arrival_seconds"],
                journey["first_transfer_arrival_stop_id"],
                journey["first_line"],
                first_mode
            ),
            event(
                "Depart",
                journey["first_transfer_station"],
                journey["first_transfer_departure_seconds"],
                journey["first_transfer_departure_stop_id"],
                journey["second_line"],
                second_mode
            ),
            event(
                "Arrive",
                destination,
                journey["destination_arrival_seconds"],
                journey["destination_stop_id"],
                journey["second_line"],
                second_mode
            )
        ])
    else:
        itinerary_rows.append(
            event(
                "Arrive",
                destination,
                journey["destination_arrival_seconds"],
                journey["destination_stop_id"],
                journey["first_line"],
                first_mode
            )
        )

    return pd.DataFrame(itinerary_rows)


# --------------------------------------------------
# TEST: KOGARAH -> RICHMOND, 30/10/2026
# --------------------------------------------------

fixed_scheduled_results = fixed_arrive_by(
    origin="Kogarah Station",
    destination="Richmond Station",
    arrival_time="09:00",
    daily=scheduled_tomorrow,
    max_lookback_hours=5,
    minimum_transfer_minutes=5
)

if not fixed_scheduled_results.empty:

    print("\n=== RECOMMENDED JOURNEY ===")

    best = fixed_scheduled_results.iloc[0]

    print(
        f"{best['departure_time']} → {best['arrival_time']} | "
        f"{best['journey_duration_min']:.0f} min | "
        f"{best['transfers']} transfer(s)"
    )

    display(
        show_fixed_itinerary(
            best,
            scheduled_tomorrow,
            "Kogarah Station",
            "Richmond Station"
        )
    )

    print("\n=== ALTERNATIVE JOURNEYS ===")

    display(
        fixed_scheduled_results[
            [
                "journey_type",
                "departure_time",
                "arrival_time",
                "journey_duration_min",
                "first_transfer_station"
            ]
        ].iloc[1:11]
    )


Valid journeys: 96
Search time: 0.09s

=== RECOMMENDED JOURNEY ===
06:47 → 08:35 | 108 min | 1 transfer(s)


,stage,station,time,platform,service
0,Depart,Kogarah Station,06:47,Platform 1,T4 Train
1,Arrive,Redfern Station,07:06,Platform 11,T4 Train
2,Depart,Redfern Station,07:17,Platform 4,T1 Train
3,Arrive,Richmond Station,08:35,Platform 2,T1 Train



=== ALTERNATIVE JOURNEYS ===


,journey_type,departure_time,arrival_time,journey_duration_min,first_transfer_station
1,One transfer,06:47,08:35,108.0,Central Station
2,One transfer,06:37,08:35,118.0,Redfern Station
3,One transfer,06:37,08:35,118.0,Central Station
4,One transfer,06:27,08:35,128.0,Redfern Station
5,One transfer,06:27,08:35,128.0,Central Station
6,One transfer,06:11,08:05,114.0,Redfern Station
7,One transfer,06:11,08:05,114.0,Central Station
8,One transfer,06:11,08:35,144.0,Redfern Station
9,One transfer,06:11,08:35,144.0,Central Station
10,One transfer,05:57,08:05,128.0,Redfern Station


In [288]:

# STEP 118 — Date-aware Arrive By journey planner

from datetime import datetime
import pandas as pd
import time

def plan_journey_by_date(
    origin,
    destination,
    journey_date,
    arrival_time,
    today_realtime_date=None
):
    selected_date = datetime.strptime(
        journey_date, "%d/%m/%Y"
    ).date()

    today = datetime.now().date()

    if selected_date < today:
        raise ValueError(
            "Please select today or a future journey date."
        )

    # Validate GTFS calendar coverage
    calendar_start = pd.to_datetime(
        calendar["start_date"].astype(str),
        format="%Y%m%d"
    ).min().date()

    calendar_end = pd.to_datetime(
        calendar["end_date"].astype(str),
        format="%Y%m%d"
    ).max().date()

    if not calendar_start <= selected_date <= calendar_end:
        raise ValueError(
            f"No planned timetable coverage for {journey_date}. "
            f"GTFS covers {calendar_start:%d/%m/%Y} "
            f"to {calendar_end:%d/%m/%Y}."
        )

    # Only use real-time data if its actual service date is known
    if (
        selected_date == today
        and today_realtime_date == journey_date
    ):
        daily = realtime_daily
        source = "Real-time timetable"
    else:
        daily = get_scheduled_timetable_for_date(
            journey_date
        )

        # Apply complete route labels
        route_codes = daily["route_id"].astype(str).str.extract(
            r"^\d+-(.+?)-sj2-\d+$",
            expand=False
        )

        daily["route_short_name"] = (
            route_codes.str.replace(
                r"^(T\d+)-[A-Z]$",
                r"\1",
                regex=True
            ).fillna(daily["route_id"].astype(str))
        )

        source = "Scheduled GTFS timetable"

    print(f"\n=== JOURNEY PLANNER ===")
    print(f"Date: {journey_date}")
    print(f"Source: {source}")
    print(f"From: {origin}")
    print(f"To: {destination}")
    print(f"Arrive by: {arrival_time}")

    results = fixed_arrive_by(
        origin=origin,
        destination=destination,
        arrival_time=arrival_time,
        daily=daily
    )

    if results.empty:
        print("No journeys found.")
        return results, daily

    best = results.iloc[0]

    print("\n=== RECOMMENDED JOURNEY ===")
    print(
        f"{best['departure_time']} → {best['arrival_time']} | "
        f"{best['journey_duration_min']:.0f} min | "
        f"{best['transfers']} transfer(s)"
    )

    display(
        show_fixed_itinerary(
            best, daily, origin, destination
        )
    )

    print("\n=== ALTERNATIVES ===")
    display(
        results[
            [
                "journey_type",
                "departure_time",
                "arrival_time",
                "journey_duration_min",
                "first_transfer_station"
            ]
        ].iloc[1:6]
    )

    return results, daily


# TEST — TOMORROW
date_results, date_timetable = plan_journey_by_date(
    origin="Kogarah Station",
    destination="Richmond Station",
    journey_date="30/10/2026",
    arrival_time="09:00"
)


Date: 30/10/2026
Active services: 720
Active trips: 60,344
Train/metro stop events: 67,620
Missing line labels: 67,620
Processing time: 0.89s

=== JOURNEY PLANNER ===
Date: 30/10/2026
Source: Scheduled GTFS timetable
From: Kogarah Station
To: Richmond Station
Arrive by: 09:00
Valid journeys: 96
Search time: 0.03s

=== RECOMMENDED JOURNEY ===
06:47 → 08:35 | 108 min | 1 transfer(s)


,stage,station,time,platform,service
0,Depart,Kogarah Station,06:47,Platform 1,T4 Train
1,Arrive,Redfern Station,07:06,Platform 11,T4 Train
2,Depart,Redfern Station,07:17,Platform 4,T1 Train
3,Arrive,Richmond Station,08:35,Platform 2,T1 Train



=== ALTERNATIVES ===


,journey_type,departure_time,arrival_time,journey_duration_min,first_transfer_station
1,One transfer,06:47,08:35,108.0,Central Station
2,One transfer,06:37,08:35,118.0,Redfern Station
3,One transfer,06:37,08:35,118.0,Central Station
4,One transfer,06:27,08:35,128.0,Redfern Station
5,One transfer,06:27,08:35,128.0,Central Station
